In [6]:
### Loading various functions, dictionaries, packages.
cell1_start_time = time(); start_load_pkgs = time()
    using Pkg;  
#    Pkg.add("Plots"); using Plots;  
    Pkg.add("JSON3"); using JSON3;
    Pkg.add("Dates"); using Dates;
    Pkg.add("JLD2"); using JLD2
    Pkg.add("HypothesisTests"); using HypothesisTests
    Pkg.add("DataFrames"); using DataFrames
    Pkg.add("CSV"); using CSV
    Pkg.add("XLSX"); using XLSX
    Pkg.add("FASTX"); using FASTX
    Pkg.add("Combinatorics"); using Combinatorics
    Pkg.add("StatsBase"); using StatsBase
    using Statistics
    using Printf
date_now = Dates.format(now(), "yyyy_mm_dd__HMMp"); println(date_now)
println(pwd()); cd("/Users/ryhisner"); println(pwd())
#    Pkg.add("LaTeXStrings");  using LaTeXStrings
#    Pkg.add("PyPlot"); using PyPlot
#    Pkg.add("PlotlyJS"); using PlotlyJS
#    Pkg.add("PGFPlotsX"); using PGFPlotsX
#    Pkg.add("UnicodePlots"); using UnicodePlots
#    Pkg.add("InspectDR"); using InspectDR
#    Pkg.add("GLMakie"); using GLMakie
#    Pkg.add("CairoMakie"); using CairoMakie
#    Pkg.add("WGLMakie"); using WGLMakie
#    Pkg.add("GMT"); using GMT
#####################################################################################################################################
### Turns time in seconds to hours, minutes, & seconds
function seconds_to_hrs_min_sec(t)
    hours = 0
    minutes = 0
    seconds = 0
    hours = t÷3600
    minutes = (t%3600)÷60
    if t > 0.0001
        seconds = (t%3600)%60
    end
    hours_int = Int(hours)
    minutes_int = Int(minutes)
    minutes_str = split(string(minutes_int), ".")[1]
    hours_fin = split(string(hours_int), ".")[1]
    minutes_fin = ""
    minutes_fin = lpad(minutes_str, 2, "0")
    seconds_rd = round(digits=2, seconds)
    seconds_1 = string(split(string(seconds_rd), ".")[1])
    seconds_2 = string(split(string(seconds_rd), ".")[2])
    seconds_left = lpad(seconds_1, 2, "0")
    seconds_right = rpad(seconds_2, 2, "0")
    seconds_fin = "$(seconds_left).$(seconds_right)"
    final_time = "$hours_int:$minutes_fin:$seconds_fin"
    final_time2 = "$hours_int hr, $minutes_int min, $seconds_fin sec"
    return final_time, final_time2
end
#################################################################################
function add_leading_zero(int_str::String)
    int_str2 = int_str
    if length(int_str) == 1 && int_str ≠ "0"
        int_str2 = "0"*int_str
    end
    return int_str2
end     
######################################################################################################################################
### Adds zero to truncated digit so all numbers have same # of digits & line up nicely E.g. if number = 9.1 & digits_rd = 3, it returns 9.100
### Adds zero to truncated digit so all numbers have same # of digits & line up nicely E.g. if number = 9.1 & digits_rd = 3, it returns 9.100
function add_zeros_to_rounded(number::Float64, digits_rd::Int)
    fmt = Printf.Format("%.$(digits_rd)f")
    num_final = Printf.format(fmt, number)
    return num_final
end
#####################################################################################################################################
load_pkgs_runtime = time() - start_load_pkgs
load_pkgs_hms1, load_pkgs_hms2 = seconds_to_hrs_min_sec(load_pkgs_runtime)
println("Time to Load Packages = ", load_pkgs_hms1); println("Time to Load Packages = ", load_pkgs_hms2)
#####################################################################################################################################
hydrophobic_index_dict = Dict{String,Int}("L"=>97, "I"=>99, "F"=>100, "W"=>97, "V"=>76, "M"=>74, "C"=>49, "Y"=>63, "A"=>41, "T"=>13, "E"=>-31, "H"=>8, "G"=>0, "S"=>-5, "Q"=>-10, "D"=>-55, "R"=>-14, "K"=>-23, "N"=>-28, "P"=>5, "*"=>0)
#####################################################################################################################################
wuhan_ref_seq = "..."
ref_seq = wuhan_ref_seq
#####################################################################################################################################
######################################################################################################################################
clade_set_complete = Set(["recombinant", "19A", "19B", "20A", "20B", "20C", "20D", "20E", "20F", "20G", "20H", "20I", "20J", "21A", "21B", "21C", "21D", "21E", "21F", "21G", "21H", "21I", "21J", "21K", "21L", "21M", "22A", "22B", "22C", "22D", "22E", "22F", "23A", "23B", "23C", "23D", "23E", "23F", "23G", "23H", "23I", "24A", "24B", "24C", "24D", "24E", "24F", "24G", "24H", "24I", "25A", "25B", "25C", "25D", "25E", "25F", "25G", "25H", "25I"])
clade_arr_complete = ["recombinant", "19A", "19B", "20A", "20B", "20C", "20D", "20E", "20F", "20G", "20H", "20I", "20J", "21A", "21B", "21C", "21D", "21E", "21F", "21G", "21H", "21I", "21J", "21K", "21L", "21M", "22A", "22B", "22C", "22D", "22E", "22F", "23A", "23B", "23C", "23D", "23E", "23F", "23G", "23H", "23I", "24A", "24B", "24C", "24D", "24E", "24F", "24G", "24H", "24I", "25A", "25B", "25C", "25D", "25E", "25F", "25G", "25H", "25I"]
clade_to_pango = Dict("recombinant"=>"recombinant", "19A"=>"B", "19B"=>"A", "20A"=>"B.1", "20B"=>"B.1.1", "20C"=>"B.1", "20D"=>"B.1.1.1", "20E"=>"B.1.177", "20F"=>"D.2", "20G"=>"B.1.2", "20H"=>"B.1.351", "20I"=>"B.1.1.7", "20J"=>"P.1", "21A"=>"B.1.617.2", "21B"=>"B.1.617.1", "21C"=>"B.1.427", "21D"=>"B.1.525", "21E"=>"P.3", "21F"=>"B.1.526", "21G"=>"C.37", "21H"=>"B.1.621", "21I"=>"B.1.617.2", "21J"=>"B.1.617.2", "21K"=>"BA.1", "21L"=>"BA.2", "21M"=>"BA.3", "22A"=>"BA.4", "22B"=>"BA.5", "22C"=>"BA.2.12.1", "22D"=>"BA.2.75", "22E"=>"BQ.1", "22F"=>"XBB", "23A"=>"XBB.1.5", "23B"=>"XBB.1.16", "23C"=>"CH.1.1", "23D"=>"XBB.1.9", "23E"=>"XBB.2.3", "23F"=>"EG.5.1", "23G"=>"XBB.1.5.70", "23H"=>"HK.3", "23I"=>"BA.2.86", "24A"=>"JN.1", "24B"=>"JN.1.11.1", "24C"=>"KP.3", "24D"=>"XDV.1", "24E"=>"KP.3.1.1", "24F"=>"XEC", "24G"=>"KP.2.3", "24H"=>"LF.7", "24I"=>"MV.1", "25A"=>"LP.8.1", "25B"=>"NB.1.8.1", "25C"=>"XFG", "25D"=>"MC.10.2.1", "25E"=>"PY.1", "25F"=>"NW.1.2", "25G"=>"XFC", "25H"=>"XFJ", "25I"=>"BA.3.2")
######################################################################################################################################
EPI_ISL(n) = split(n, "|")[2]
country(n) = split(n, "/")[2]
sequence_date(n) = split(n, "|")[3]
seq_lab(n) = split(n, "/")[3]
US_state(n) = split(split(n, "/")[3], "-")[1]
######################################################################################################################################
######################################################################################################################################
AAsub_gene(n) = aa_gene_comprehensive_dict[n]
AAsub_gene_num(n) = [aa_gene_comprehensive_dict[n], aa_pos_comprehensive_dict[n]]
mut_num_pos_only(n) = aa_pos_comprehensive_dict[n]
AAsub_gene_num_pos_only(n) = [aa_gene_comprehensive_dict[n], aa_pos_comprehensive_dict[n]]
mut_gene_Dict = Dict{String,Int}("ORF1a"=>1, "ORF1b"=>2, "S"=>3, "E"=>4, "M"=>5, "N"=>6, "ORF3a"=>7, "ORF6"=>8, "ORF7a"=>9, "ORF7b"=>10, "ORF8"=>11, "ORF9b"=>12)
#################################
AA_gene_sortKey_2(n) = (mut_gene_Dict[aa_gene_comprehensive_dict[n]], aa_pos_comprehensive_dict[n])
AA_gene_sortKey(n) = (mut_gene_Dict[aa_gene_comprehensive_dict[n[1]]], aa_pos_comprehensive_dict[n[1]])
AA_ct_sortKey1(n) = (1000÷mut_gene_Dict[aa_gene_comprehensive_dict[n[1]]], aa_pos_comprehensive_dict[n[1]])
AA_ct_sortKey2(n) = (n[2], AA_ct_sortKey1(n))
AA_gene_pos_sortKey(n) = (mut_gene_Dict[aa_gene_comprehensive_dict[n[1]]], aa_pos_comprehensive_dict[n[1]])
AA_gene_pos_sortKey_2(n) = (mut_gene_Dict[aa_gene_comprehensive_dict[n]], aa_pos_comprehensive_dict[n])
AA_ct_pos_sortKey1(n) = (1000÷mut_gene_Dict[aa_gene_comprehensive_dict[n[1]]], aa_pos_comprehensive_dict[n[1]])
AA_ct_pos_sortKey2(n) = (n[2], AA_ct_pos_sortKey1(n))
######################################################################################################################################
function AA_order_key(mut)
    gene = aa_gene_comprehensive_dict[mut]
    AApos = aa_pos_comprehensive_dict[mut]
    gene_pos = gene_print_order[gene]
    return (gene_pos, AApos)
end
#####################################################################################################################################
function pango_variant_sort_key(pango::String)
    dotparts = split(pango, ".")
    k1 = string(dotparts[1])
    k2 = 0
    k3 = 0
    k4 = 0
    if length(dotparts) ≥ 2
        k2 = parse(Int,String(dotparts[2]))
    end
    if length(dotparts) ≥ 3
        k3 = parse(Int,String(dotparts[3]))
    end
    if length(dotparts) ≥ 4
        k4 = parse(Int,String(dotparts[4]))
    end
    return (k1, k2, k3, k4)
end
######################################################################################################################################
gene_print_order = Dict{String,Int}("S"=>1, "N"=>2, "E"=>3, "M"=>4, "ORF3a"=>5, "ORF6"=>6, "ORF7a"=>7, "ORF7b"=>8, "ORF8"=>9, "ORF9b"=>10, "ORF1a"=>12, "ORF1b"=>13)
######################################################################################################################################
function pango_minus_X_fx(pango::String, minus::Int)
    unaliased = pango_to_pango_unaliased_v2[pango]
    dot_ct = count(".", unaliased)
    println(dot_ct)
    if dot_ct ≥ minus
        dotsplits = split(unaliased, ".")
        minus_X_unaliased = join(dotsplits[1:dot_ct+1-minus], ".")
        minus_X_pango = pango_unaliased_to_pango[minus_X_unaliased]
        println("$(pango), $(unaliased), minus-$(minus) = $(minus_X_unaliased)")
        return minus_X_pango
    else
        return pango
    end
end
####################################################################################
function pango_unaliased_minus_X_fx(unaliased::String, minus::Int)
    dot_ct = count(".", unaliased)
    if dot_ct ≥ minus 
        dotsplits = split(unaliased, ".")
        println(dotsplits)
        minus_X_unaliased = join(dotsplits[1:dot_ct+1-minus], ".")
        minus_X_pango = pango_unaliased_to_pango[minus_X_unaliased]
        println("$(unaliased), minus-$(minus) = $(minus_X_unaliased)")
        return minus_X
    else
        return minus_X_pango
    end
end
##########################################################################################################################################################################
function read_fasta(filepath::String)
    reader = FASTX.FASTA.Reader(open(filepath, "r"))
    fasta_in = [record for record in reader]
    close(reader)
    return[String(FASTX.FASTX.description(rec)) for rec in fasta_in],
    [uppercase(String(FASTX.FASTA.sequence(rec))) for rec in fasta_in]
end
##########################################################################################################################################################################
wuhan_ref_seq = "..."
refAA_ORF1a = "MESLVPGFNEKTHVQLSLPVLQVRDVLVRGFGDSVEEVLSEARQHLKDGTCGLVEVEKGVLPQLEQPYVFIKRSDARTAPHGHVMVELVAELEGIQYGRSGETLGVLVPHVGEIPVAYRKVLLRKNGNKGAGGHSYGADLKSFDLGDELGTDPYEDFQENWNTKHSSGVTRELMRELNGGAYTRYVDNNFCGPDGYPLECIKDLLARAGKASCTLSEQLDFIDTKRGVYCCREHEHEIAWYTERSEKSYELQTPFEIKLAKKFDTFNGECPNFVFPLNSIIKTIQPRVEKKKLDGFMGRIRSVYPVASPNECNQMCLSTLMKCDHCGETSWQTGDFVKATCEFCGTENLTKEGATTCGYLPQNAVVKIYCPACHNSEVGPEHSLAEYHNESGLKTILRKGGRTIAFGGCVFSYVGCHNKCAYWVPRASANIGCNHTGVVGEGSEGLNDNLLEILQKEKVNINIVGDFKLNEEIAIILASFSASTSAFVETVKGLDYKAFKQIVESCGNFKVTKGKAKKGAWNIGEQKSILSPLYAFASEAARVVRSIFSRTLETAQNSVRVLQKAAITILDGISQYSLRLIDAMMFTSDLATNNLVVMAYITGGVVQLTSQWLTNIFGTVYEKLKPVLDWLEEKFKEGVEFLRDGWEIVKFISTCACEIVGGQIVTCAKEIKESVQTFFKLVNKFLALCADSIIIGGAKLKALNLGETFVTHSKGLYRKCVKSREETGLLMPLKAPKEIIFLEGETLPTEVLTEEVVLKTGDLQPLEQPTSEAVEAPLVGTPVCINGLMLLEIKDTEKYCALAPNMMVTNNTFTLKGGAPTKVTFGDDTVIEVQGYKSVNITFELDERIDKVLNEKCSAYTVELGTEVNEFACVVADAVIKTLQPVSELLTPLGIDLDEWSMATYYLFDESGEFKLASHMYCSFYPPDEDEEEGDCEEEEFEPSTQYEYGTEDDYQGKPLEFGATSAALQPEEEQEEDWLDDDSQQTVGQQDGSEDNQTTTIQTIVEVQPQLEMELTPVVQTIEVNSFSGYLKLTDNVYIKNADIVEEAKKVKPTVVVNAANVYLKHGGGVAGALNKATNNAMQVESDDYIATNGPLKVGGSCVLSGHNLAKHCLHVVGPNVNKGEDIQLLKSAYENFNQHEVLLAPLLSAGIFGADPIHSLRVCVDTVRTNVYLAVFDKNLYDKLVSSFLEMKSEKQVEQKIAEIPKEEVKPFITESKPSVEQRKQDDKKIKACVEEVTTTLEETKFLTENLLLYIDINGNLHPDSATLVSDIDITFLKKDAPYIVGDVVQEGVLTAVVIPTKKAGGTTEMLAKALRKVPTDNYITTYPGQGLNGYTVEEAKTVLKKCKSAFYILPSIISNEKQEILGTVSWNLREMLAHAEETRKLMPVCVETKAIVSTIQRKYKGIKIQEGVVDYGARFYFYTSKTTVASLINTLNDLNETLVTMPLGYVTHGLNLEEAARYMRSLKVPATVSVSSPDAVTAYNGYLTSSSKTPEEHFIETISLAGSYKDWSYSGQSTQLGIEFLKRGDKSVYYTSNPTTFHLDGEVITFDNLKTLLSLREVRTIKVFTTVDNINLHTQVVDMSMTYGQQFGPTYLDGADVTKIKPHNSHEGKTFYVLPNDDTLRVEAFEYYHTTDPSFLGRYMSALNHTKKWKYPQVNGLTSIKWADNNCYLATALLTLQQIELKFNPPALQDAYYRARAGEAANFCALILAYCNKTVGELGDVRETMSYLFQHANLDSCKRVLNVVCKTCGQQQTTLKGVEAVMYMGTLSYEQFKKGVQIPCTCGKQATKYLVQQESPFVMMSAPPAQYELKHGTFTCASEYTGNYQCGHYKHITSKETLYCIDGALLTKSSEYKGPITDVFYKENSYTTTIKPVTYKLDGVVCTEIDPKLDNYYKKDNSYFTEQPIDLVPNQPYPNASFDNFKFVCDNIKFADDLNQLTGYKKPASRELKVTFFPDLNGDVVAIDYKHYTPSFKKGAKLLHKPIVWHVNNATNKATYKPNTWCIRCLWSTKPVETSNSFDVLKSEDAQGMDNLACEDLKPVSEEVVENPTIQKDVLECNVKTTEVVGDIILKPANNSLKITEEVGHTDLMAAYVDNSSLTIKKPNELSRVLGLKTLATHGLAAVNSVPWDTIANYAKPFLNKVVSTTTNIVTRCLNRVCTNYMPYFFTLLLQLCTFTRSTNSRIKASMPTTIAKNTVKSVGKFCLEASFNYLKSPNFSKLINIIIWFLLLSVCLGSLIYSTAALGVLMSNLGMPSYCTGYREGYLNSTNVTIATYCTGSIPCSVCLSGLDSLDTYPSLETIQITISSFKWDLTAFGLVAEWFLAYILFTRFFYVLGLAAIMQLFFSYFAVHFISNSWLMWLIINLVQMAPISAMVRMYIFFASFYYVWKSYVHVVDGCNSSTCMMCYKRNRATRVECTTIVNGVRRSFYVYANGGKGFCKLHNWNCVNCDTFCAGSTFISDEVARDLSLQFKRPINPTDQSSYIVDSVTVKNGSIHLYFDKAGQKTYERHSLSHFVNLDNLRANNTKGSLPINVIVFDGKSKCEESSAKSASVYYSQLMCQPILLLDQALVSDVGDSAEVAVKMFDAYVNTFSSTFNVPMEKLKTLVATAEAELAKNVSLDNVLSTFISAARQGFVDSDVETKDVVECLKLSHQSDIEVTGDSCNNYMLTYNKVENMTPRDLGACIDCSARHINAQVAKSHNIALIWNVKDFMSLSEQLRKQIRSAAKKNNLPFKLTCATTRQVVNVVTTKIALKGGKIVNNWLKQLIKVTLVFLFVAAIFYLITPVHVMSKHTDFSSEIIGYKAIDGGVTRDIASTDTCFANKHADFDTWFSQRGGSYTNDKACPLIAAVITREVGFVVPGLPGTILRTTNGDFLHFLPRVFSAVGNICYTPSKLIEYTDFATSACVLAAECTIFKDASGKPVPYCYDTNVLEGSVAYESLRPDTRYVLMDGSIIQFPNTYLEGSVRVVTTFDSEYCRHGTCERSEAGVCVSTSGRWVLNNDYYRSLPGVFCGVDAVNLLTNMFTPLIQPIGALDISASIVAGGIVAIVVTCLAYYFMRFRRAFGEYSHVVAFNTLLFLMSFTVLCLTPVYSFLPGVYSVIYLYLTFYLTNDVSFLAHIQWMVMFTPLVPFWITIAYIICISTKHFYWFFSNYLKRRVVFNGVSFSTFEEAALCTFLLNKEMYLKLRSDVLLPLTQYNRYLALYNKYKYFSGAMDTTSYREAACCHLAKALNDFSNSGSDVLYQPPQTSITSAVLQSGFRKMAFPSGKVEGCMVQVTCGTTTLNGLWLDDVVYCPRHVICTSEDMLNPNYEDLLIRKSNHNFLVQAGNVQLRVIGHSMQNCVLKLKVDTANPKTPKYKFVRIQPGQTFSVLACYNGSPSGVYQCAMRPNFTIKGSFLNGSCGSVGFNIDYDCVSFCYMHHMELPTGVHAGTDLEGNFYGPFVDRQTAQAAGTDTTITVNVLAWLYAAVINGDRWFLNRFTTTLNDFNLVAMKYNYEPLTQDHVDILGPLSAQTGIAVLDMCASLKELLQNGMNGRTILGSALLEDEFTPFDVVRQCSGVTFQSAVKRTIKGTHHWLLLTILTSLLVLVQSTQWSLFFFLYENAFLPFAMGIIAMSAFAMMFVKHKHAFLCLFLLPSLATVAYFNMVYMPASWVMRIMTWLDMVDTSLSGFKLKDCVMYASAVVLLILMTARTVYDDGARRVWTLMNVLTLVYKVYYGNALDQAISMWALIISVTSNYSGVVTTVMFLARGIVFMCVEYCPIFFITGNTLQCIMLVYCFLGYFCTCYFGLFCLLNRYFRLTLGVYDYLVSTQEFRYMNSQGLLPPKNSIDAFKLNIKLLGVGGKPCIKVATVQSKMSDVKCTSVVLLSVLQQLRVESSSKLWAQCVQLHNDILLAKDTTEAFEKMVSLLSVLLSMQGAVDINKLCEEMLDNRATLQAIASEFSSLPSYAAFATAQEAYEQAVANGDSEVVLKKLKKSLNVAKSEFDRDAAMQRKLEKMADQAMTQMYKQARSEDKRAKVTSAMQTMLFTMLRKLDNDALNNIINNARDGCVPLNIIPLTTAAKLMVVIPDYNTYKNTCDGTTFTYASALWEIQQVVDADSKIVQLSEISMDNSPNLAWPLIVTALRANSAVKLQNNELSPVALRQMSCAAGTTQTACTDDNALAYYNTTKGGRFVLALLSDLQDLKWARFPKSDGTGTIYTELEPPCRFVTDTPKGPKVKYLYFIKGLNNLNRGMVLGSLAATVRLQAGNATEVPANSTVLSFCAFAVDAAKAYKDYLASGGQPITNCVKMLCTHTGTGQAITVTPEANMDQESFGGASCCLYCRCHIDHPNPKGFCDLKGKYVQIPTTCANDPVGFTLKNTVCTVCGMWKGYGCSCDQLREPMLQSADAQSFLNGFAV"
refAA_ORF1b = "NRVCGVSAARLTPCGTGTSTDVVYRAFDIYNDKVAGFAKFLKTNCCRFQEKDEDDNLIDSYFVVKRHTFSNYQHEETIYNLLKDCPAVAKHDFFKFRIDGDMVPHISRQRLTKYTMADLVYALRHFDEGNCDTLKEILVTYNCCDDDYFNKKDWYDFVENPDILRVYANLGERVRQALLKTVQFCDAMRNAGIVGVLTLDNQDLNGNWYDFGDFIQTTPGSGVPVVDSYYSLLMPILTLTRALTAESHVDTDLTKPYIKWDLLKYDFTEERLKLFDRYFKYWDQTYHPNCVNCLDDRCILHCANFNVLFSTVFPPTSFGPLVRKIFVDGVPFVVSTGYHFRELGVVHNQDVNLHSSRLSFKELLVYAADPAMHAASGNLLLDKRTTCFSVAALTNNVAFQTVKPGNFNKDFYDFAVSKGFFKEGSSVELKHFFFAQDGNAAISDYDYYRYNLPTMCDIRQLLFVVEVVDKYFDCYDGGCINANQVIVNNLDKSAGFPFNKWGKARLYYDSMSYEDQDALFAYTKRNVIPTITQMNLKYAISAKNRARTVAGVSICSTMTNRQFHQKLLKSIAATRGATVVIGTSKFYGGWHNMLKTVYSDVENPHLMGWDYPKCDRAMPNMLRIMASLVLARKHTTCCSLSHRFYRLANECAQVLSEMVMCGGSLYVKPGGTSSGDATTAYANSVFNICQAVTANVNALLSTDGNKIADKYVRNLQHRLYECLYRNRDVDTDFVNEFYAYLRKHFSMMILSDDAVVCFNSTYASQGLVASIKNFKSVLYYQNNVFMSEAKCWTETDLTKGPHEFCSQHTMLVKQGDDYVYLPYPDPSRILGAGCFVDDIVKTDGTLMIERFVSLAIDAYPLTKHPNQEYADVFHLYLQYIRKLHDELTGHMLDMYSVMLTNDNTSRYWEPEFYEAMYTPHTVLQAVGACVLCNSQTSLRCGACIRRPFLCCKCCYDHVISTSHKLVLSVNPYVCNAPGCDVTDVTQLYLGGMSYYCKSHKPPISFPLCANGQVFGLYKNTCVGSDNVTDFNAIATCDWTNAGDYILANTCTERLKLFAAETLKATEETFKLSYGIATVREVLSDRELHLSWEVGKPRPPLNRNYVFTGYRVTKNSKVQIGEYTFEKGDYGDAVVYRGTTTYKLNVGDYFVLTSHTVMPLSAPTLVPQEHYVRITGLYPTLNISDEFSSNVANYQKVGMQKYSTLQGPPGTGKSHFAIGLALYYPSARIVYTACSHAAVDALCEKALKYLPIDKCSRIIPARARVECFDKFKVNSTLEQYVFCTVNALPETTADIVVFDEISMATNYDLSVVNARLRAKHYVYIGDPAQLPAPRTLLTKGTLEPEYFNSVCRLMKTIGPDMFLGTCRRCPAEIVDTVSALVYDNKLKAHKDKSAQCFKMFYKGVITHDVSSAINRPQIGVVREFLTRNPAWRKAVFISPYNSQNAVASKILGLPTQTVDSSQGSEYDYVIFTQTTETAHSCNVNRFNVAITRAKVGILCIMSDRDLYDKLQFTSLEIPRRNVATLQAENVTGLFKDCSKVITGLHPTQAPTHLSVDTKFKTEGLCVDIPGIPKDMTYRRLISMMGFKMNYQVNGYPNMFITREEAIRHVRAWIGFDVEGCHATREAVGTNLPLQLGFSTGVNLVAVPTGYVDTPNNTDFSRVSAKPPPGDQFKHLIPLMYKGLPWNVVRIKIVQMLSDTLKNLSDRVVFVLWAHGFELTSMKYFVKIGPERTCCLCDRRATCFSTASDTYACWHHSIGFDYVYNPFMIDVQQWGFTGNLQSNHDLYCQVHGNAHVASCDAIMTRCLAVHECFVKRVDWTIEYPIIGDELKINAACRKVQHMVVKAALLADKFPVLHDIGNPKAIKCVPQADVEWKFYDAQPCSDKAYKIEELFYSYATHSDKFTDGVCLFWNCNVDRYPANSIVCRFDTRVLSNLNLPGCDGGSLYVNKHAFHTPAFDKSAFVNLKQLPFFYYSDSPCESHGKQVVSDIDYVPLKSATCITRCNLGGAVCRHHANEYRLYLDAYNMMISAGFSLWVYKQFDTYNLWNTFTRLQSLENVAFNVVNKGHFDGQQGEVPVSIINNTVYTKVDGVDVELFENKTTLPVNVAFELWAKRNIKPVPEVKILNNLGVDIAANTVIWDYKRDAPAHISTIGVCSMTDIAKKPTETICAPLTVFFDGRVDGQVDLFRNARNGVLITEGSVKGLQPSVGPKQASLNGVTLIGEAVKTQFNYYKKVDGVVQQLPETYFTQSRNLQEFKPRSQMEIDFLELAMDEFIERYKLEGYAFEHIVYGDFSHSQLGGLHLLIGLAKRFKESPFELEDFIPMDSTVKNYFITDAQTGSSKCVCSVIDLLLDDFVEIIKSQDLSVVSKVVKVTIDYTEISFMLWCKDGHVETFYPKLQSSQAWQPGVAMPNLYKMQRMLLEKCDLQNYGDSATLPKGIMMNVAKYTQLCQYLNTLTLAVPYNMRVIHFGAGSDKGVAPGTAVLRQWLPTGTLLVDSDLNDFVSDADSTLIGDCATVHTANKWDLIISDMYDPKTKNVTKENDSKEGFFTYICGFIQQKLALGGSVAIKITEHSWNADLYKLMGHFAWWTAFVTNVNASSSEAFLIGCNYLGKPREQIDGYVMHANYIFWRNTNPIQLSSYSLFDMSKFPLKLRGTAVMSLKEGQINDMILSLLSKGRLIIRENNRVVISSDVLVNN*"
refAA_S = "MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSSVLHSTQDLFLPFFSNVTWFHAIHVSGTNGTKRFDNPVLPFNDGVYFASTEKSNIIRGWIFGTTLDSKTQSLLIVNNATNVVIKVCEFQFCNDPFLGVYYHKNNKSWMESEFRVYSSANNCTFEYVSQPFLMDLEGKQGNFKNLREFVFKNIDGYFKIYSKHTPINLVRDLPQGFSALEPLVDLPIGINITRFQTLLALHRSYLTPGDSSSGWTAGAAAYYVGYLQPRTFLLKYNENGTITDAVDCALDPLSETKCTLKSFTVEKGIYQTSNFRVQPTESIVRFPNITNLCPFGEVFNATRFASVYAWNRKRISNCVADYSVLYNSASFSTFKCYGVSPTKLNDLCFTNVYADSFVIRGDEVRQIAPGQTGKIADYNYKLPDDFTGCVIAWNSNNLDSKVGGNYNYLYRLFRKSNLKPFERDISTEIYQAGSTPCNGVEGFNCYFPLQSYGFQPTNGVGYQPYRVVVLSFELLHAPATVCGPKKSTNLVKNKCVNFNFNGLTGTGVLTESNKKFLPFQQFGRDIADTTDAVRDPQTLEILDITPCSFGGVSVITPGTNTSNQVAVLYQDVNCTEVPVAIHADQLTPTWRVYSTGSNVFQTRAGCLIGAEHVNNSYECDIPIGAGICASYQTQTNSPRRARSVASQSIIAYTMSLGAENSVAYSNNSIAIPTNFTISVTTEILPVSMTKTSVDCTMYICGDSTECSNLLLQYGSFCTQLNRALTGIAVEQDKNTQEVFAQVKQIYKTPPIKDFGGFNFSQILPDPSKPSKRSFIEDLLFNKVTLADAGFIKQYGDCLGDIAARDLICAQKFNGLTVLPPLLTDEMIAQYTSALLAGTITSGWTFGAGAALQIPFAMQMAYRFNGIGVTQNVLYENQKLIANQFNSAIGKIQDSLSSTASALGKLQDVVNQNAQALNTLVKQLSSNFGAISSVLNDILSRLDKVEAEVQIDRLITGRLQSLQTYVTQQLIRAAEIRASANLAATKMSECVLGQSKRVDFCGKGYHLMSFPQSAPHGVVFLHVTYVPAQEKNFTTAPAICHDGKAHFPREGVFVSNGTHWFVTQRNFYEPQIITTDNTFVSGNCDVVIGIVNNTVYDPLQPELDSFKEELDKYFKNHTSPDVDLGDISGINASVVNIQKEIDRLNEVAKNLNESLIDLQELGKYEQYIKWPWYIWLGFIAGLIAIVMVTIMLCCMTSCCSCLKGCCSCGSCCKFDEDDSEPVLKGVKLHYT*"
refAA_ORF3a = "MDLFMRIFTIGTVTLKQGEIKDATPSDFVRATATIPIQASLPFGWLIVGVALLAVFQSASKIITLKKRWQLALSKGVHFVCNLLLLFVTVYSHLLLVAAGLEAPFLYLYALVYFLQSINFVRIIMRLWLCWKCRSKNPLLYDANYFLCWHTNCYDYCIPYNSVTSSIVITSGDGTTSPISEHDYQIGGYTEKWESGVKDCVVLHSYFTSDYYQLYSTQLSTDTGVEHVTFFIYNKIVDEPEEHVQIHTIDGSSGVVNPVMEPIYDEPTTTTSVPL*"
refAA_E = "MYSFVSEETGTLIVNSVLLFLAFVVFLLVTLAILTALRLCAYCCNIVNVSLVKPSFYVYSRVKNLNSSRVPDLLV*"
refAA_M = "MADSNGTITVEELKKLLEQWNLVIGFLFLTWICLLQFAYANRNRFLYIIKLIFLWLLWPVTLACFVLAAVYRINWITGGIAIAMACLVGLMWLSYFIASFRLFARTRSMWSFNPETNILLNVPLHGTILTRPLLESELVIGAVILRGHLRIAGHHLGRCDIKDLPKEITVATSRTLSYYKLGASQRVAGDSGFAAYSRYRIGNYKLNTDHSSSSDNIALLVQ*"
refAA_ORF6 = "MFHLVDFQVTIAEILLIIMRTFKVSIWNLDYIINLIIKNLSKSLTENKYSQLDEEQPMEID*"
refAA_ORF7a = "MKIILFLALITLATCELYHYQECVRGTTVLLKEPCSSGTYEGNSPFHPLADNKFALTCFSTQFAFACPDGVKHVYQLRARSVSPKLFIRQEEVQELYSPIFLIVAAIVFITLCFTLKRKTE*"
refAA_ORF7b = "MIELSLIDFYLCFLAFLLFLVLIMLIIFWFSLELQDHNETCHA*"
refAA_ORF8 = "MKFLVFLGIITTVAAFHQECSLQSCTQHQPYVVDDPCPIHFYSKWYIRVGARKSAPLIELCVDEAGSKSPIQYIDIGNYTVSCLPFTINCQEPKLGSLVVRCSFYEDFLEYHDVRVVLDFI*"
refAA_N = "MSDNGPQNQRNAPRITFGGPSDSTGSNQNGERSGARSKQRRPQGLPNNTASWFTALTQHGKEDLKFPRGQGVPINTNSSPDDQIGYYRRATRRIRGGDGKMKDLSPRWYFYYLGTGPEAGLPYGANKDGIIWVATEGALNTPKDHIGTRNPANNAAIVLQLPQGTTLPKGFYAEGSRGGSQASSRSSSRSRNSSRNSTPGSSRGTSPARMAGNGGDAALALLLLDRLNQLESKMSGKGQQQQGQTVTKKSAAEASKKPRQKRTATKAYNVTQAFGRRGPEQTQGNFGDQELIRQGTDYKHWPQIAQFAPSASAFFGMSRIGMEVTPSGTWLTYTGAIKLDDKDPNFKDQVILLNKHIDAYKTFPPTEPKKDKKKKADETQALPQRQKKQQTVTLLPAADLDDFSKQLQQSMSSADSTQA*"
refAA_ORF9b = "MDPKISEMHPALRLVDPQIQLAVTRMENAVGRDQNNVGPKVYPIILRLGSPLSLNMARKTLNSLEDKAFQLTPIAVQMTKLATTEELPDEFVVVTVK*"
######################################################################################################################################
gene_array = ["ORF1a", "ORF1b", "S", "ORF3a", "E", "M", "ORF6", "ORF7a", "ORF7b", "ORF8", "N", "ORF9b"]
gene_order_dict = Dict{String,Int}("ORF1a"=>1, "ORF1b"=>2, "S"=>3, "ORF3a"=>4, "E"=>4, "M"=>6, "ORF6"=>7, "ORF7a"=>8, "ORF7b"=>9, "ORF8"=>10, "N"=>11, "ORF9b"=>12)
gene_order_dict = Dict{String,Int}("ORF1a"=>1, "ORF1b"=>2, "S"=>3, "ORF3a"=>4, "E"=>4, "M"=>6, "ORF6"=>7, "ORF7a"=>8, "ORF7b"=>9, "ORF8"=>10, "N"=>11, "ORF9b"=>12)
gene_AA_sortKey(m) = (gene_order_dict[aa_gene_comprehensive_dict[m]], aa_pos_comprehensive_dict[m])
gene_AApos_sortKey(m) = (gene_order_dict[aa_gene_comprehensive_dict[m]], aa_pos_comprehensive_dict[m])
######################################################################################################################################
gene_AA_dict = Dict{String,String}()
gene_AA_dict["ORF1a"] = "MESLVPGFNEKTHVQLSLPVLQVRDVLVRGFGDSVEEVLSEARQHLKDGTCGLVEVEKGVLPQLEQPYVFIKRSDARTAPHGHVMVELVAELEGIQYGRSGETLGVLVPHVGEIPVAYRKVLLRKNGNKGAGGHSYGADLKSFDLGDELGTDPYEDFQENWNTKHSSGVTRELMRELNGGAYTRYVDNNFCGPDGYPLECIKDLLARAGKASCTLSEQLDFIDTKRGVYCCREHEHEIAWYTERSEKSYELQTPFEIKLAKKFDTFNGECPNFVFPLNSIIKTIQPRVEKKKLDGFMGRIRSVYPVASPNECNQMCLSTLMKCDHCGETSWQTGDFVKATCEFCGTENLTKEGATTCGYLPQNAVVKIYCPACHNSEVGPEHSLAEYHNESGLKTILRKGGRTIAFGGCVFSYVGCHNKCAYWVPRASANIGCNHTGVVGEGSEGLNDNLLEILQKEKVNINIVGDFKLNEEIAIILASFSASTSAFVETVKGLDYKAFKQIVESCGNFKVTKGKAKKGAWNIGEQKSILSPLYAFASEAARVVRSIFSRTLETAQNSVRVLQKAAITILDGISQYSLRLIDAMMFTSDLATNNLVVMAYITGGVVQLTSQWLTNIFGTVYEKLKPVLDWLEEKFKEGVEFLRDGWEIVKFISTCACEIVGGQIVTCAKEIKESVQTFFKLVNKFLALCADSIIIGGAKLKALNLGETFVTHSKGLYRKCVKSREETGLLMPLKAPKEIIFLEGETLPTEVLTEEVVLKTGDLQPLEQPTSEAVEAPLVGTPVCINGLMLLEIKDTEKYCALAPNMMVTNNTFTLKGGAPTKVTFGDDTVIEVQGYKSVNITFELDERIDKVLNEKCSAYTVELGTEVNEFACVVADAVIKTLQPVSELLTPLGIDLDEWSMATYYLFDESGEFKLASHMYCSFYPPDEDEEEGDCEEEEFEPSTQYEYGTEDDYQGKPLEFGATSAALQPEEEQEEDWLDDDSQQTVGQQDGSEDNQTTTIQTIVEVQPQLEMELTPVVQTIEVNSFSGYLKLTDNVYIKNADIVEEAKKVKPTVVVNAANVYLKHGGGVAGALNKATNNAMQVESDDYIATNGPLKVGGSCVLSGHNLAKHCLHVVGPNVNKGEDIQLLKSAYENFNQHEVLLAPLLSAGIFGADPIHSLRVCVDTVRTNVYLAVFDKNLYDKLVSSFLEMKSEKQVEQKIAEIPKEEVKPFITESKPSVEQRKQDDKKIKACVEEVTTTLEETKFLTENLLLYIDINGNLHPDSATLVSDIDITFLKKDAPYIVGDVVQEGVLTAVVIPTKKAGGTTEMLAKALRKVPTDNYITTYPGQGLNGYTVEEAKTVLKKCKSAFYILPSIISNEKQEILGTVSWNLREMLAHAEETRKLMPVCVETKAIVSTIQRKYKGIKIQEGVVDYGARFYFYTSKTTVASLINTLNDLNETLVTMPLGYVTHGLNLEEAARYMRSLKVPATVSVSSPDAVTAYNGYLTSSSKTPEEHFIETISLAGSYKDWSYSGQSTQLGIEFLKRGDKSVYYTSNPTTFHLDGEVITFDNLKTLLSLREVRTIKVFTTVDNINLHTQVVDMSMTYGQQFGPTYLDGADVTKIKPHNSHEGKTFYVLPNDDTLRVEAFEYYHTTDPSFLGRYMSALNHTKKWKYPQVNGLTSIKWADNNCYLATALLTLQQIELKFNPPALQDAYYRARAGEAANFCALILAYCNKTVGELGDVRETMSYLFQHANLDSCKRVLNVVCKTCGQQQTTLKGVEAVMYMGTLSYEQFKKGVQIPCTCGKQATKYLVQQESPFVMMSAPPAQYELKHGTFTCASEYTGNYQCGHYKHITSKETLYCIDGALLTKSSEYKGPITDVFYKENSYTTTIKPVTYKLDGVVCTEIDPKLDNYYKKDNSYFTEQPIDLVPNQPYPNASFDNFKFVCDNIKFADDLNQLTGYKKPASRELKVTFFPDLNGDVVAIDYKHYTPSFKKGAKLLHKPIVWHVNNATNKATYKPNTWCIRCLWSTKPVETSNSFDVLKSEDAQGMDNLACEDLKPVSEEVVENPTIQKDVLECNVKTTEVVGDIILKPANNSLKITEEVGHTDLMAAYVDNSSLTIKKPNELSRVLGLKTLATHGLAAVNSVPWDTIANYAKPFLNKVVSTTTNIVTRCLNRVCTNYMPYFFTLLLQLCTFTRSTNSRIKASMPTTIAKNTVKSVGKFCLEASFNYLKSPNFSKLINIIIWFLLLSVCLGSLIYSTAALGVLMSNLGMPSYCTGYREGYLNSTNVTIATYCTGSIPCSVCLSGLDSLDTYPSLETIQITISSFKWDLTAFGLVAEWFLAYILFTRFFYVLGLAAIMQLFFSYFAVHFISNSWLMWLIINLVQMAPISAMVRMYIFFASFYYVWKSYVHVVDGCNSSTCMMCYKRNRATRVECTTIVNGVRRSFYVYANGGKGFCKLHNWNCVNCDTFCAGSTFISDEVARDLSLQFKRPINPTDQSSYIVDSVTVKNGSIHLYFDKAGQKTYERHSLSHFVNLDNLRANNTKGSLPINVIVFDGKSKCEESSAKSASVYYSQLMCQPILLLDQALVSDVGDSAEVAVKMFDAYVNTFSSTFNVPMEKLKTLVATAEAELAKNVSLDNVLSTFISAARQGFVDSDVETKDVVECLKLSHQSDIEVTGDSCNNYMLTYNKVENMTPRDLGACIDCSARHINAQVAKSHNIALIWNVKDFMSLSEQLRKQIRSAAKKNNLPFKLTCATTRQVVNVVTTKIALKGGKIVNNWLKQLIKVTLVFLFVAAIFYLITPVHVMSKHTDFSSEIIGYKAIDGGVTRDIASTDTCFANKHADFDTWFSQRGGSYTNDKACPLIAAVITREVGFVVPGLPGTILRTTNGDFLHFLPRVFSAVGNICYTPSKLIEYTDFATSACVLAAECTIFKDASGKPVPYCYDTNVLEGSVAYESLRPDTRYVLMDGSIIQFPNTYLEGSVRVVTTFDSEYCRHGTCERSEAGVCVSTSGRWVLNNDYYRSLPGVFCGVDAVNLLTNMFTPLIQPIGALDISASIVAGGIVAIVVTCLAYYFMRFRRAFGEYSHVVAFNTLLFLMSFTVLCLTPVYSFLPGVYSVIYLYLTFYLTNDVSFLAHIQWMVMFTPLVPFWITIAYIICISTKHFYWFFSNYLKRRVVFNGVSFSTFEEAALCTFLLNKEMYLKLRSDVLLPLTQYNRYLALYNKYKYFSGAMDTTSYREAACCHLAKALNDFSNSGSDVLYQPPQTSITSAVLQSGFRKMAFPSGKVEGCMVQVTCGTTTLNGLWLDDVVYCPRHVICTSEDMLNPNYEDLLIRKSNHNFLVQAGNVQLRVIGHSMQNCVLKLKVDTANPKTPKYKFVRIQPGQTFSVLACYNGSPSGVYQCAMRPNFTIKGSFLNGSCGSVGFNIDYDCVSFCYMHHMELPTGVHAGTDLEGNFYGPFVDRQTAQAAGTDTTITVNVLAWLYAAVINGDRWFLNRFTTTLNDFNLVAMKYNYEPLTQDHVDILGPLSAQTGIAVLDMCASLKELLQNGMNGRTILGSALLEDEFTPFDVVRQCSGVTFQSAVKRTIKGTHHWLLLTILTSLLVLVQSTQWSLFFFLYENAFLPFAMGIIAMSAFAMMFVKHKHAFLCLFLLPSLATVAYFNMVYMPASWVMRIMTWLDMVDTSLSGFKLKDCVMYASAVVLLILMTARTVYDDGARRVWTLMNVLTLVYKVYYGNALDQAISMWALIISVTSNYSGVVTTVMFLARGIVFMCVEYCPIFFITGNTLQCIMLVYCFLGYFCTCYFGLFCLLNRYFRLTLGVYDYLVSTQEFRYMNSQGLLPPKNSIDAFKLNIKLLGVGGKPCIKVATVQSKMSDVKCTSVVLLSVLQQLRVESSSKLWAQCVQLHNDILLAKDTTEAFEKMVSLLSVLLSMQGAVDINKLCEEMLDNRATLQAIASEFSSLPSYAAFATAQEAYEQAVANGDSEVVLKKLKKSLNVAKSEFDRDAAMQRKLEKMADQAMTQMYKQARSEDKRAKVTSAMQTMLFTMLRKLDNDALNNIINNARDGCVPLNIIPLTTAAKLMVVIPDYNTYKNTCDGTTFTYASALWEIQQVVDADSKIVQLSEISMDNSPNLAWPLIVTALRANSAVKLQNNELSPVALRQMSCAAGTTQTACTDDNALAYYNTTKGGRFVLALLSDLQDLKWARFPKSDGTGTIYTELEPPCRFVTDTPKGPKVKYLYFIKGLNNLNRGMVLGSLAATVRLQAGNATEVPANSTVLSFCAFAVDAAKAYKDYLASGGQPITNCVKMLCTHTGTGQAITVTPEANMDQESFGGASCCLYCRCHIDHPNPKGFCDLKGKYVQIPTTCANDPVGFTLKNTVCTVCGMWKGYGCSCDQLREPMLQSADAQSFLNGFAV*"
gene_AA_dict["ORF1b"] = "RVCGVSAARLTPCGTGTSTDVVYRAFDIYNDKVAGFAKFLKTNCCRFQEKDEDDNLIDSYFVVKRHTFSNYQHEETIYNLLKDCPAVAKHDFFKFRIDGDMVPHISRQRLTKYTMADLVYALRHFDEGNCDTLKEILVTYNCCDDDYFNKKDWYDFVENPDILRVYANLGERVRQALLKTVQFCDAMRNAGIVGVLTLDNQDLNGNWYDFGDFIQTTPGSGVPVVDSYYSLLMPILTLTRALTAESHVDTDLTKPYIKWDLLKYDFTEERLKLFDRYFKYWDQTYHPNCVNCLDDRCILHCANFNVLFSTVFPPTSFGPLVRKIFVDGVPFVVSTGYHFRELGVVHNQDVNLHSSRLSFKELLVYAADPAMHAASGNLLLDKRTTCFSVAALTNNVAFQTVKPGNFNKDFYDFAVSKGFFKEGSSVELKHFFFAQDGNAAISDYDYYRYNLPTMCDIRQLLFVVEVVDKYFDCYDGGCINANQVIVNNLDKSAGFPFNKWGKARLYYDSMSYEDQDALFAYTKRNVIPTITQMNLKYAISAKNRARTVAGVSICSTMTNRQFHQKLLKSIAATRGATVVIGTSKFYGGWHNMLKTVYSDVENPHLMGWDYPKCDRAMPNMLRIMASLVLARKHTTCCSLSHRFYRLANECAQVLSEMVMCGGSLYVKPGGTSSGDATTAYANSVFNICQAVTANVNALLSTDGNKIADKYVRNLQHRLYECLYRNRDVDTDFVNEFYAYLRKHFSMMILSDDAVVCFNSTYASQGLVASIKNFKSVLYYQNNVFMSEAKCWTETDLTKGPHEFCSQHTMLVKQGDDYVYLPYPDPSRILGAGCFVDDIVKTDGTLMIERFVSLAIDAYPLTKHPNQEYADVFHLYLQYIRKLHDELTGHMLDMYSVMLTNDNTSRYWEPEFYEAMYTPHTVLQAVGACVLCNSQTSLRCGACIRRPFLCCKCCYDHVISTSHKLVLSVNPYVCNAPGCDVTDVTQLYLGGMSYYCKSHKPPISFPLCANGQVFGLYKNTCVGSDNVTDFNAIATCDWTNAGDYILANTCTERLKLFAAETLKATEETFKLSYGIATVREVLSDRELHLSWEVGKPRPPLNRNYVFTGYRVTKNSKVQIGEYTFEKGDYGDAVVYRGTTTYKLNVGDYFVLTSHTVMPLSAPTLVPQEHYVRITGLYPTLNISDEFSSNVANYQKVGMQKYSTLQGPPGTGKSHFAIGLALYYPSARIVYTACSHAAVDALCEKALKYLPIDKCSRIIPARARVECFDKFKVNSTLEQYVFCTVNALPETTADIVVFDEISMATNYDLSVVNARLRAKHYVYIGDPAQLPAPRTLLTKGTLEPEYFNSVCRLMKTIGPDMFLGTCRRCPAEIVDTVSALVYDNKLKAHKDKSAQCFKMFYKGVITHDVSSAINRPQIGVVREFLTRNPAWRKAVFISPYNSQNAVASKILGLPTQTVDSSQGSEYDYVIFTQTTETAHSCNVNRFNVAITRAKVGILCIMSDRDLYDKLQFTSLEIPRRNVATLQAENVTGLFKDCSKVITGLHPTQAPTHLSVDTKFKTEGLCVDIPGIPKDMTYRRLISMMGFKMNYQVNGYPNMFITREEAIRHVRAWIGFDVEGCHATREAVGTNLPLQLGFSTGVNLVAVPTGYVDTPNNTDFSRVSAKPPPGDQFKHLIPLMYKGLPWNVVRIKIVQMLSDTLKNLSDRVVFVLWAHGFELTSMKYFVKIGPERTCCLCDRRATCFSTASDTYACWHHSIGFDYVYNPFMIDVQQWGFTGNLQSNHDLYCQVHGNAHVASCDAIMTRCLAVHECFVKRVDWTIEYPIIGDELKINAACRKVQHMVVKAALLADKFPVLHDIGNPKAIKCVPQADVEWKFYDAQPCSDKAYKIEELFYSYATHSDKFTDGVCLFWNCNVDRYPANSIVCRFDTRVLSNLNLPGCDGGSLYVNKHAFHTPAFDKSAFVNLKQLPFFYYSDSPCESHGKQVVSDIDYVPLKSATCITRCNLGGAVCRHHANEYRLYLDAYNMMISAGFSLWVYKQFDTYNLWNTFTRLQSLENVAFNVVNKGHFDGQQGEVPVSIINNTVYTKVDGVDVELFENKTTLPVNVAFELWAKRNIKPVPEVKILNNLGVDIAANTVIWDYKRDAPAHISTIGVCSMTDIAKKPTETICAPLTVFFDGRVDGQVDLFRNARNGVLITEGSVKGLQPSVGPKQASLNGVTLIGEAVKTQFNYYKKVDGVVQQLPETYFTQSRNLQEFKPRSQMEIDFLELAMDEFIERYKLEGYAFEHIVYGDFSHSQLGGLHLLIGLAKRFKESPFELEDFIPMDSTVKNYFITDAQTGSSKCVCSVIDLLLDDFVEIIKSQDLSVVSKVVKVTIDYTEISFMLWCKDGHVETFYPKLQSSQAWQPGVAMPNLYKMQRMLLEKCDLQNYGDSATLPKGIMMNVAKYTQLCQYLNTLTLAVPYNMRVIHFGAGSDKGVAPGTAVLRQWLPTGTLLVDSDLNDFVSDADSTLIGDCATVHTANKWDLIISDMYDPKTKNVTKENDSKEGFFTYICGFIQQKLALGGSVAIKITEHSWNADLYKLMGHFAWWTAFVTNVNASSSEAFLIGCNYLGKPREQIDGYVMHANYIFWRNTNPIQLSSYSLFDMSKFPLKLRGTAVMSLKEGQINDMILSLLSKGRLIIRENNRVVISSDVLVNN*"
gene_AA_dict["S"] = "MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSSVLHSTQDLFLPFFSNVTWFHAIHVSGTNGTKRFDNPVLPFNDGVYFASTEKSNIIRGWIFGTTLDSKTQSLLIVNNATNVVIKVCEFQFCNDPFLGVYYHKNNKSWMESEFRVYSSANNCTFEYVSQPFLMDLEGKQGNFKNLREFVFKNIDGYFKIYSKHTPINLVRDLPQGFSALEPLVDLPIGINITRFQTLLALHRSYLTPGDSSSGWTAGAAAYYVGYLQPRTFLLKYNENGTITDAVDCALDPLSETKCTLKSFTVEKGIYQTSNFRVQPTESIVRFPNITNLCPFGEVFNATRFASVYAWNRKRISNCVADYSVLYNSASFSTFKCYGVSPTKLNDLCFTNVYADSFVIRGDEVRQIAPGQTGKIADYNYKLPDDFTGCVIAWNSNNLDSKVGGNYNYLYRLFRKSNLKPFERDISTEIYQAGSTPCNGVEGFNCYFPLQSYGFQPTNGVGYQPYRVVVLSFELLHAPATVCGPKKSTNLVKNKCVNFNFNGLTGTGVLTESNKKFLPFQQFGRDIADTTDAVRDPQTLEILDITPCSFGGVSVITPGTNTSNQVAVLYQDVNCTEVPVAIHADQLTPTWRVYSTGSNVFQTRAGCLIGAEHVNNSYECDIPIGAGICASYQTQTNSPRRARSVASQSIIAYTMSLGAENSVAYSNNSIAIPTNFTISVTTEILPVSMTKTSVDCTMYICGDSTECSNLLLQYGSFCTQLNRALTGIAVEQDKNTQEVFAQVKQIYKTPPIKDFGGFNFSQILPDPSKPSKRSFIEDLLFNKVTLADAGFIKQYGDCLGDIAARDLICAQKFNGLTVLPPLLTDEMIAQYTSALLAGTITSGWTFGAGAALQIPFAMQMAYRFNGIGVTQNVLYENQKLIANQFNSAIGKIQDSLSSTASALGKLQDVVNQNAQALNTLVKQLSSNFGAISSVLNDILSRLDKVEAEVQIDRLITGRLQSLQTYVTQQLIRAAEIRASANLAATKMSECVLGQSKRVDFCGKGYHLMSFPQSAPHGVVFLHVTYVPAQEKNFTTAPAICHDGKAHFPREGVFVSNGTHWFVTQRNFYEPQIITTDNTFVSGNCDVVIGIVNNTVYDPLQPELDSFKEELDKYFKNHTSPDVDLGDISGINASVVNIQKEIDRLNEVAKNLNESLIDLQELGKYEQYIKWPWYIWLGFIAGLIAIVMVTIMLCCMTSCCSCLKGCCSCGSCCKFDEDDSEPVLKGVKLHYT*"
gene_AA_dict["E"] = "MYSFVSEETGTLIVNSVLLFLAFVVFLLVTLAILTALRLCAYCCNIVNVSLVKPSFYVYSRVKNLNSSRVPDLLV*"
gene_AA_dict["M"] = "MADSNGTITVEELKKLLEQWNLVIGFLFLTWICLLQFAYANRNRFLYIIKLIFLWLLWPVTLACFVLAAVYRINWITGGIAIAMACLVGLMWLSYFIASFRLFARTRSMWSFNPETNILLNVPLHGTILTRPLLESELVIGAVILRGHLRIAGHHLGRCDIKDLPKEITVATSRTLSYYKLGASQRVAGDSGFAAYSRYRIGNYKLNTDHSSSSDNIALLVQ*"
gene_AA_dict["N"] = "MSDNGPQNQRNAPRITFGGPSDSTGSNQNGERSGARSKQRRPQGLPNNTASWFTALTQHGKEDLKFPRGQGVPINTNSSPDDQIGYYRRATRRIRGGDGKMKDLSPRWYFYYLGTGPEAGLPYGANKDGIIWVATEGALNTPKDHIGTRNPANNAAIVLQLPQGTTLPKGFYAEGSRGGSQASSRSSSRSRNSSRNSTPGSSRGTSPARMAGNGGDAALALLLLDRLNQLESKMSGKGQQQQGQTVTKKSAAEASKKPRQKRTATKAYNVTQAFGRRGPEQTQGNFGDQELIRQGTDYKHWPQIAQFAPSASAFFGMSRIGMEVTPSGTWLTYTGAIKLDDKDPNFKDQVILLNKHIDAYKTFPPTEPKKDKKKKADETQALPQRQKKQQTVTLLPAADLDDFSKQLQQSMSSADSTQA*"
gene_AA_dict["ORF3a"] = "MDLFMRIFTIGTVTLKQGEIKDATPSDFVRATATIPIQASLPFGWLIVGVALLAVFQSASKIITLKKRWQLALSKGVHFVCNLLLLFVTVYSHLLLVAAGLEAPFLYLYALVYFLQSINFVRIIMRLWLCWKCRSKNPLLYDANYFLCWHTNCYDYCIPYNSVTSSIVITSGDGTTSPISEHDYQIGGYTEKWESGVKDCVVLHSYFTSDYYQLYSTQLSTDTGVEHVTFFIYNKIVDEPEEHVQIHTIDGSSGVVNPVMEPIYDEPTTTTSVPL*"
gene_AA_dict["ORF6"] = "MFHLVDFQVTIAEILLIIMRTFKVSIWNLDYIINLIIKNLSKSLTENKYSQLDEEQPMEID*"
gene_AA_dict["ORF7a"] = "MKIILFLALITLATCELYHYQECVRGTTVLLKEPCSSGTYEGNSPFHPLADNKFALTCFSTQFAFACPDGVKHVYQLRARSVSPKLFIRQEEVQELYSPIFLIVAAIVFITLCFTLKRKTE*"
gene_AA_dict["ORF7b"] = "MIELSLIDFYLCFLAFLLFLVLIMLIIFWFSLELQDHNETCHA*"
gene_AA_dict["ORF8"] = "MKFLVFLGIITTVAAFHQECSLQSCTQHQPYVVDDPCPIHFYSKWYIRVGARKSAPLIELCVDEAGSKSPIQYIDIGNYTVSCLPFTINCQEPKLGSLVVRCSFYEDFLEYHDVRVVLDFI*"
gene_AA_dict["ORF9b"] = "MDPKISEMHPALRLVDPQIQLAVTRMENAVGRDQNNVGPKVYPIILRLGSPLSLNMARKTLNSLEDKAFQLTPIAVQMTKLATTEELPDEFVVVTVK*"
######################################################################################################################################
aa_site_to_index = Dict{String, Int}()
aa_index_to_site = Dict{Int, String}()
aa_index = 0
for gene in gene_array
    for i in 1:length(gene_AA_dict[gene])
        aa_index += 1
        gene_n_pos = "$(gene):$(i)"
        aa_site_to_index[gene_n_pos] = aa_index
    end
end
for (genepos, aaindex) in aa_site_to_index
    aa_index_to_site[aaindex] = genepos
end
######################################################################################################################################
NSP_AA_size = Dict{String,Int}("NSP1"=>180, "NSP2"=>638, "NSP3"=>1945, "NSP4"=>500, "NSP5"=>306, "NSP6"=>290, "NSP7"=>83, "NSP8"=>198, "NSP9"=>113, "NSP10"=>139, "NSP11"=>0, "NSP12"=>932, "NSP13"=>601, "NSP14"=>527, "NSP15"=>346, "NSP16"=>299)                                                                # "NSP12"=>BitSet(1:923), 
NSP_ranges = Dict{String,String}("NSP1"=>"ORF1a:1-180", "NSP2"=>"ORF1a:181-818", "NSP3"=>"ORF1a:819-2763", "NSP4"=>"ORF1a:2764-3263", "NSP5"=>"ORF1a:3264-3569", "NSP6"=>"ORF1a:3570-3859", "NSP7"=>"ORF1a:3860-3942", "NSP8"=>"ORF1a:3943-4140", "NSP9"=>"ORF1a:4141-4253", "NSP10"=>"ORF1a:4254-4392", "NSP11"=>"", "NSP12"=>"1a:4393-1b:923", "NSP13"=>"ORF1b:924-1524", "NSP14"=>"ORF1b:1525-2051", "NSP15"=>"ORF1b:2052-2397", "NSP16"=>"ORF1b:2398-2696", "S"=>"S:1-1274", "ORF3a"=>"ORF3a:1-276", "E"=>"E:1-76", "M"=>"M:1-223", "ORF6"=>"ORF6:1-62", "ORF7a"=>"ORF7a:1-122", "ORF7b"=>"ORF7b:1-44", "ORF8"=>"ORF8:1-122", "N"=>"N:1-420", "ORF9b"=>"ORF9b:1-98")
NSP_ranges_num_only = Dict{String, BitSet}("NSP1"=>BitSet(1:180), "NSP2"=>BitSet(181:818), "NSP3"=>BitSet(819:2763), "NSP4"=>BitSet(2764:3263), "NSP5"=>BitSet(3264:3569), "NSP6"=>BitSet(3570:3859), "NSP7"=>BitSet(3860:3942), "NSP8"=>BitSet(3943:4140), "NSP9"=>BitSet(4141:4253), "NSP10"=>BitSet(4254:4392), "NSP11"=>BitSet(), "NSP12"=>BitSet([4393:4401; 1:923]), "NSP13"=>BitSet(924:1524), "NSP14"=>BitSet(1525:2051), "NSP15"=>BitSet(2052:2397), "NSP16"=>BitSet(2398:2696), "S"=>BitSet(1:1274), "ORF3a"=>BitSet(1:276), "E"=>BitSet(1:76), "M"=>BitSet(1:223), "ORF6"=>BitSet(1:62), "ORF7a"=>BitSet(1:122), "ORF7b"=>BitSet(1:44), "ORF8"=>BitSet(1:122), "N"=>BitSet(1:420), "ORF9b"=>BitSet(1:98))
NSP_ranges1a = Dict{Int, BitSet}(1=>BitSet(1:180), 2=>BitSet(181:818), 3=>BitSet(819:2763), 4=>BitSet(2764:3263), 5=>BitSet(3264:3569), 6=>BitSet(3570:3859), 7=>BitSet(3860:3942), 8=>BitSet(3943:4140), 9=>BitSet(4141:4253), 10=>BitSet(4254:4392), 12=>BitSet(4393:4401))
NSP_ranges1b = Dict{Int, BitSet}(12=>BitSet(1:923), 13=>BitSet(924:1524), 14=>BitSet(1525:2051), 15=>BitSet(2052:2397), 16=>BitSet(2398:2696))
NSP1a_add = Dict{Int,Int}(1=>0, 2=>180, 3=>818, 4=>2763, 5=>3263, 6=>3569, 7=>3859, 8=>3942, 9=>4140, 10=>4253, 11=>0, 12=>4392)
NSP1b_add = Dict{Int,Int}(12=>-9, 13=>923, 14=>1524, 15=>2051, 16=>2397)
NSP1ab_add = Dict{Int,Int}(1=>0, 2=>180, 3=>818, 4=>2763, 5=>3263, 6=>3569, 7=>3859, 8=>3942, 9=>4140, 10=>4353, 11=>0, 12=>-9, 13=>923, 14=>1524, 15=>2051, 16=>2397)
######################################################################################################################################
ORF_size_dict = Dict{String,Int}()
for (orf, aaseq) in gene_AA_dict
    orf_len = length(aaseq)
    ORF_size_dict[orf] = orf_len
end
###########################################################################################################################################################################
###########################################################################################################################################################################
AA_residues = Set(["A", "C", "D", "E", "F", "G", "H", "I", "K", "L", "M", "N", "P", "Q", "R", "S", "T", "V", "W", "Y", "-", "*", "X"])
aa_pos_comprehensive_dict = Dict{String,Int}()
aa_gene_comprehensive_dict = Dict{String,String}()
aa_gene_and_pos_comprehensive_dict = Dict{String,String}()
refAA_comprehensive_dict = Dict{String,String}()
qryAA_comprehensive_dict = Dict{String,String}()
global nonspike_muts = Set{String}()
for (orf, orf_len) in ORF_size_dict
    for aa1 in AA_residues
        for aa2 in AA_residues
            for i in 1:orf_len
                orf_mut = "$(orf):$(aa1)$(i)$(aa2)"
                orf_pos_mut = "$(orf):$(i)"
                aa_pos_comprehensive_dict[orf_mut] = i
                aa_pos_comprehensive_dict[orf_pos_mut] = i
                aa_gene_comprehensive_dict[orf_mut] = orf
                aa_gene_comprehensive_dict[orf_pos_mut] = orf
                aa_gene_and_pos_comprehensive_dict[orf_mut] = orf_pos_mut
                aa_gene_and_pos_comprehensive_dict[orf_pos_mut] = orf_pos_mut
                if orf ≠ "S"
                    push!(nonspike_muts, orf_mut)
                    push!(nonspike_muts, orf_pos_mut)
                end
                refAA_comprehensive_dict[orf_mut] = aa1
                qryAA_comprehensive_dict[orf_mut] = aa2
            end
        end
    end
end
aa_gene_comprehensive_dict["NTD_disulfide"] = "S"
aa_pos_comprehensive_dict["NTD_disulfide"] = 1
aa_gene_and_pos_comprehensive_dict["NTD_disulfide"] = "S:NTD_disulfide"
refAA_comprehensive_dict["NTD_disulfide"] = "NTDdisulfide"
qryAA_comprehensive_dict["NTD_disulfide"] = "NTD_disulfide"
aa_gene_comprehensive_dict["NTD:disulfide"] = "S"
aa_pos_comprehensive_dict["NTD:disulfide"] = 1
aa_gene_and_pos_comprehensive_dict["NTD:disulfide"] = "S:NTD_disulfide"
refAA_comprehensive_dict["NTD:disulfide"] = "NTDdisulfide"
qryAA_comprehensive_dict["NTD:disulfide"] = "NTD_disulfide"
######################################################## Below: 100% comprehensive ref_nuc & qry_nuc dicts #################################################################
ref_nuc_comprehensive_dict = Dict{String,String}()
qry_nuc_comprehensive_dict = Dict{String,String}()
nuc_mut_int_comprehensive_dict = Dict{String,Int}()
nuc_mut_int_string_comprehensive_dict = Dict{String,String}()
nuc_residues1 = ["T", "C", "A", "G"]
nuc_residues2 = ["T", "C", "A", "G", "Y", "R", "K", "W", "M", "S", "-", "N"]
for nres1 in nuc_residues1
    for nres2 in nuc_residues2
        for i in 1:30000
            mut = "$(nres1)$(i)$(nres2)"
            nucmpos = i
            ref_nuc_comprehensive_dict[mut] = nres1
            qry_nuc_comprehensive_dict[mut] = nres2
            nuc_mut_int_comprehensive_dict[mut] = i
            nuc_mut_int_string_comprehensive_dict[mut] = string(i)
        end
    end
end
##########################################################################################################################################################################
##########################################################################################################################################################################
NSP_muts_pos_dict = Dict{String,Int}()
NSP_muts_gene_dict = Dict{String,String}()
NSP_ref_AA_dict = Dict{String,String}()
NSP_qry_AA_dict = Dict{String,String}()
NSP_set = Set(["NSP1", "NSP2", "NSP3", "NSP4", "NSP5", "NSP6", "NSP7", "NSP8", "NSP9", "NSP10", "NSP12", "NSP13", "NSP14", "NSP15", "NSP16"])
for nsp in NSP_set
    nsp_len = NSP_AA_size[nsp]
    for aa1 in AA_residues
        for aa2 in AA_residues
            for i in 1:nsp_len
                nspmut = "$(nsp):$(aa1)$(i)$(aa2)"
                nsppos = "$(nsp):$(i)"
                NSP_muts_gene_dict[nspmut] = nsp
                NSP_muts_pos_dict[nspmut] = i
                NSP_muts_gene_dict[nsppos] = nsp
                NSP_muts_pos_dict[nsppos] = i
                NSP_ref_AA_dict[nspmut] = aa1
                NSP_qry_AA_dict[nspmut] = aa2  
            end
        end
    end
end
###########################################################################################################################################################################
###########################################################################################################################################################################
function AAmut_to_AApos(m)
    gn = split(m, ":")[1]
    mt = split(m, ":")[2]
    pos = mt[2:end-1]
    AApos = gn*":"*pos
    return AApos
end
########################################################################################################################################################################
########################################################################################################################################################################
#####################################################################################################################################
function mixed2nuc(mix_mut::String)
    nuc_mut = mix_mut
    qrynuc = qry_nuc_comprehensive_dict[mix_mut]
    refnuc = ref_nuc_comprehensive_dict[mix_mut]
    pos_str = nuc_mut_int_string_comprehensive_dict[mix_mut]
    ref_n_pos = refnuc*pos_str
    normal_qry_nucs = Set(["T", "C", "A", "G", "N"])
    if length(mix_mut) ≥ 4
        if !(qrynuc in  normal_qry_nucs)
            if refnuc == "T"
                if qrynuc == "Y"
                    nuc_mut = ref_n_pos*"C"
                elseif qrynuc == "W"
                    nuc_mut = ref_n_pos*"A"
                elseif qrynuc == "K"
                    nuc_mut = ref_n_pos*"G"
                elseif qrynuc == "M"
                    nuc_mut = "$(ref_n_pos)C, $(ref_n_pos)A"
                elseif qrynuc == "S"
                    nuc_mut = "$(ref_n_pos)C, $(ref_n_pos)G"
                elseif qrynuc == "R"
                    nuc_mut = "$(ref_n_pos)A, $(ref_n_pos)G"
                end
            elseif refnuc == "C"
                if qrynuc == "Y"
                    nuc_mut = ref_n_pos*"T"
                elseif qrynuc == "M"
                    nuc_mut = ref_n_pos*"A"
                elseif qrynuc == "S"
                    nuc_mut = ref_n_pos*"G"
                elseif qrynuc == "R"
                    nuc_mut = "$(ref_n_pos)A, $(ref_n_pos)G"
                elseif qrynuc == "W"
                    nuc_mut = "$(ref_n_pos)T, $(ref_n_pos)A"
                elseif qrynuc == "K"
                    nuc_mut = "$(ref_n_pos)T, $(ref_n_pos)G"
                end
            elseif refnuc == "A"
                if qrynuc == "R"
                    nuc_mut = ref_n_pos*"G"
                elseif qrynuc == "W"
                    nuc_mut = ref_n_pos*"T"
                elseif qrynuc == "M"
                    nuc_mut = ref_n_pos*"C"
                elseif qrynuc == "Y"
                    nuc_mut = "$(ref_n_pos)T, $(ref_n_pos)C"
                elseif qrynuc == "K"
                    nuc_mut = "$(ref_n_pos)T, $(ref_n_pos)G"
                elseif qrynuc == "S"
                    nuc_mut = "$(ref_n_pos)C, $(ref_n_pos)G"
                end
            elseif refnuc == "G"
                if qrynuc == "R"
                    nuc_mut = ref_n_pos*"A"
                elseif qrynuc == "K"
                    nuc_mut = ref_n_pos*"T"
                elseif qrynuc == "S"
                    nuc_mut = ref_n_pos*"C"
                elseif qrynuc == "Y"
                    nuc_mut = "$(ref_n_pos)T, $(ref_n_pos)C"
                elseif qrynuc == "W"
                    nuc_mut = "$(ref_n_pos)T, $(ref_n_pos)A"
                elseif qrynuc == "M"
                    nuc_mut = "$(ref_n_pos)C, $(ref_n_pos)A"
                end
            end
        end
    end
    return nuc_mut
end
######################################################################################################################################
####################### Making gene AA & nuc references for all designated variants #################################################
#####################################################################################################################################
gene_AA_pango_dict = Dict{String, Dict{String,String}}()
nuc_genome_pango_dict = Dict{String,String}()
pango_consensus_set = Set{String}()
headers1a, seqs1a = read_fasta("___pango_consensus_sequences/pango_consensus_AA_ORF1a_2025_06_25_NNL.fasta")
for i in 1:length(headers1a)
    pango = headers1a[i]
    push!(pango_consensus_set, pango)
end
for pango in pango_consensus_set
    gene_AA_pango_dict[pango] = Dict{String,String}()
end
################################################################################################
for gene in gene_array
    aa_file = "___pango_consensus_sequences/pango_consensus_AA_$(gene)_2025_06_25_NNL.fasta"
    headers, seqs = read_fasta(aa_file)
    for i in 1:length(headers)
        pango = headers[i]
        aa_seq = seqs[i]
        gene_AA_pango_dict[pango][gene] = aa_seq
    end
end
nuc_file = "___pango_consensus_sequences/pango_consensus_sequences_genome_nuc_2025_06_25_NNL.fasta"
nuc_headers, nuc_seqs = read_fasta(nuc_file)
for i in 1:length(nuc_headers)
    pango = nuc_headers[i]
    nuc_seq = nuc_seqs[i]
    if length(nuc_seq) ≥ 28000
        nuc_genome_pango_dict[pango] = nuc_seq
    end
end
seqs_in_nuc_genome_pango_dict = length(nuc_genome_pango_dict)
println("seqs_in_nuc_genome_pango_dict = $(seqs_in_nuc_genome_pango_dict)")
######################################################################################################################################
######################################################################################################################################
gene_hydrophobic_dict = Dict{String,Float64}()
for (gene, aaseq) in gene_AA_dict
    gene_hydrophobe_sum = 0
    for aa in aaseq
        hydrophobe_score = hydrophobic_index_dict[string(aa)]
        gene_hydrophobe_sum += hydrophobe_score
    end
    gene_hydrophobe_score = gene_hydrophobe_sum/length(aaseq)
    gene_hydrophobic_dict[gene] = gene_hydrophobe_score
end 
######################################################################################################################################
AA_res_set = Set(["A", "C", "D", "E", "F", "G", "H", "I", "K", "L", "M", "N", "P", "Q", "R", "S", "T", "V", "W", "Y", "-", "*"])
AA_res_set_noDel = Set(["A", "C", "D", "E", "F", "G", "H", "I", "K", "L", "M", "N", "P", "Q", "R", "S", "T", "V", "W", "Y", "*"])
AA_res_pairs = Set{Tuple{String,String}}()
for res1 in AA_res_set_noDel
    for res2 in AA_res_set
        push!(AA_res_pairs, (res1, res2) )
    end
end
###########################################################################################################################################################################
AA_triplets = Dict{String,String}("TTT"=>"F", "TTC"=>"F", "TTA"=>"L", "TTG"=>"L", "TCT"=>"S", "TCC"=>"S", "TCA"=>"S", "TCG"=>"S", "TAT"=>"Y", "TAC"=>"Y", "TAA"=>"*", "TAG"=>"*", "TGT"=>"C", "TGC"=>"C", "TGA"=>"*", "TGG"=>"W", "CTT"=>"L", "CTC"=>"L", "CTA"=>"L", "CTG"=>"L", "CCT"=>"P", "CCC"=>"P", "CCA"=>"P", "CCG"=>"P", "CAT"=>"H", "CAC"=>"H", "CAA"=>"Q", "CAG"=>"Q", "CGT"=>"R", "CGC"=>"R", "CGA"=>"R", "CGG"=>"R", "ATT"=>"I", "ATC"=>"I", "ATA"=>"I", "ATG"=>"M", "ACT"=>"T", "ACC"=>"T", "ACA"=>"T", "ACG"=>"T", "AAT"=>"N", "AAC"=>"N", "AAA"=>"K", "AAG"=>"K", "AGT"=>"S", "AGC"=>"S", "AGA"=>"R", "AGG"=>"R", "GTT"=>"V", "GTC"=>"V", "GTA"=>"V", "GTG"=>"V", "GCT"=>"A", "GCC"=>"A", "GCA"=>"A", "GCG"=>"A", "GAT"=>"D", "GAC"=>"D", "GAA"=>"E", "GAG"=>"E", "GGT"=>"G", "GGC"=>"G", "GGA"=>"G", "GGG"=>"G", "TT-"=>"X", "TC-"=>"X", "TA-"=>"X", "TG-"=>"X", "T-T"=>"X", "T-C"=>"X", "T-A"=>"X", "T-G"=>"X", "T--"=>"X", "CT-"=>"X", "CC-"=>"X", "CA-"=>"X", "CG-"=>"X", "C-T"=>"X", "C-C"=>"X", "C-A"=>"X", "C-G"=>"X", "C--"=>"X", "AT-"=>"X", "AC-"=>"X", "AA-"=>"X", "AG-"=>"X", "A-T"=>"X", "A-C"=>"X", "A-A"=>"X", "A-G"=>"X", "A--"=>"X", "GT-"=>"X", "GC-"=>"X", "GA-"=>"X", "GG-"=>"X", "G-T"=>"X", "G-C"=>"X", "G-A"=>"X", "G-G"=>"X", "G--"=>"X", "-TT"=>"X", "-TC"=>"X", "-TA"=>"X", "-TG"=>"X", "-T-"=>"X", "-CT"=>"X", "-CC"=>"X", "-CA"=>"X", "-CG"=>"X", "-C-"=>"X", "-AT"=>"X", "-AC"=>"X", "-AA"=>"X", "-AG"=>"X", "-A-"=>"X", "-GT"=>"X", "-GC"=>"X", "-GA"=>"X", "-GG"=>"X", "-G-"=>"X", "--T"=>"X", "--C"=>"X", "--A"=>"X", "--G"=>"X", "---"=>"X", "NTT"=>"X", "TNT"=>"X", "TTN"=>"X", "NTC"=>"X", "TNC"=>"X", "TCN"=>"X", "NTA"=>"X", "TNA"=>"X", "TAN"=>"X", "NTG"=>"X", "TNG"=>"X", "TGN"=>"X", "NT-"=>"X", "TN-"=>"X", "T-N"=>"X", "NTN"=>"X", "TNN"=>"X", "NCT"=>"X", "CNT"=>"X", "CTN"=>"X", "NCC"=>"X", "CNC"=>"X", "CCN"=>"X", "NCA"=>"X", "CNA"=>"X", "CAN"=>"X", "NCG"=>"X", "CNG"=>"X", "CGN"=>"X", "NC-"=>"X", "CN-"=>"X", "C-N"=>"X", "NCN"=>"X", "CNN"=>"X", "NAT"=>"X", "ANT"=>"X", "ATN"=>"X", "NAC"=>"X", "ANC"=>"X", "ACN"=>"X", "NAA"=>"X", "ANA"=>"X", "AAN"=>"X", "NAG"=>"X", "ANG"=>"X", "AGN"=>"X", "NA-"=>"X", "AN-"=>"X", "A-N"=>"X", "NAN"=>"X", "ANN"=>"X", "NGT"=>"X", "GNT"=>"X", "GTN"=>"X", "NGC"=>"X", "GNC"=>"X", "GCN"=>"X", "NGA"=>"X", "GNA"=>"X", "GAN"=>"X", "NGG"=>"X", "GNG"=>"X", "GGN"=>"X", "NG-"=>"X", "GN-"=>"X", "G-N"=>"X", "NGN"=>"X", "GNN"=>"X", "N-T"=>"X", "-NT"=>"X", "-TN"=>"X", "N-C"=>"X", "-NC"=>"X", "-CN"=>"X", "N-A"=>"X", "-NA"=>"X", "-AN"=>"X", "N-G"=>"X", "-NG"=>"X", "-GN"=>"X", "N--"=>"X", "-N-"=>"X", "--N"=>"X", "N-N"=>"X", "-NN"=>"X", "NNT"=>"X", "NNC"=>"X", "NNA"=>"X", "NNG"=>"X", "NN-"=>"X", "NNN"=>"X")                 
AA_triplet_dels = Dict{String,String}("TTT"=>"F", "TTC"=>"F", "TTA"=>"L", "TTG"=>"L", "TCT"=>"S", "TCC"=>"S", "TCA"=>"S", "TCG"=>"S", "TAT"=>"Y", "TAC"=>"Y", "TAA"=>"*", "TAG"=>"*", "TGT"=>"C", "TGC"=>"C", "TGA"=>"*", "TGG"=>"W", "CTT"=>"L", "CTC"=>"L", "CTA"=>"L", "CTG"=>"L", "CCT"=>"P", "CCC"=>"P", "CCA"=>"P", "CCG"=>"P", "CAT"=>"H", "CAC"=>"H", "CAA"=>"Q", "CAG"=>"Q", "CGT"=>"R", "CGC"=>"R", "CGA"=>"R", "CGG"=>"R", "ATT"=>"I", "ATC"=>"I", "ATA"=>"I", "ATG"=>"M", "ACT"=>"T", "ACC"=>"T", "ACA"=>"T", "ACG"=>"T", "AAT"=>"N", "AAC"=>"N", "AAA"=>"K", "AAG"=>"K", "AGT"=>"S", "AGC"=>"S", "AGA"=>"R", "AGG"=>"R", "GTT"=>"V", "GTC"=>"V", "GTA"=>"V", "GTG"=>"V", "GCT"=>"A", "GCC"=>"A", "GCA"=>"A", "GCG"=>"A", "GAT"=>"D", "GAC"=>"D", "GAA"=>"E", "GAG"=>"E", "GGT"=>"G", "GGC"=>"G", "GGA"=>"G", "GGG"=>"G", "TT-"=>"X", "TC-"=>"X", "TA-"=>"X", "TG-"=>"X", "T-T"=>"X", "T-C"=>"X", "T-A"=>"X", "T-G"=>"X", "T--"=>"-", "CT-"=>"X", "CC-"=>"X", "CA-"=>"X", "CG-"=>"X", "C-T"=>"X", "C-C"=>"X", "C-A"=>"X", "C-G"=>"X", "C--"=>"-", "AT-"=>"X", "AC-"=>"X", "AA-"=>"X", "AG-"=>"X", "A-T"=>"X", "A-C"=>"X", "A-A"=>"X", "A-G"=>"X", "A--"=>"-", "GT-"=>"X", "GC-"=>"X", "GA-"=>"X", "GG-"=>"X", "G-T"=>"X", "G-C"=>"X", "G-A"=>"X", "G-G"=>"X", "G--"=>"-", "-TT"=>"X", "-TC"=>"X", "-TA"=>"X", "-TG"=>"X", "-T-"=>"X", "-CT"=>"X", "-CC"=>"X", "-CA"=>"X", "-CG"=>"X", "-C-"=>"X", "-AT"=>"X", "-AC"=>"X", "-AA"=>"X", "-AG"=>"X", "-A-"=>"X", "-GT"=>"X", "-GC"=>"X", "-GA"=>"X", "-GG"=>"X", "-G-"=>"X", "--T"=>"-", "--C"=>"-", "--A"=>"-", "--G"=>"-", "---"=>"-", "NTT"=>"X", "TNT"=>"X", "TTN"=>"X", "NTC"=>"X", "TNC"=>"X", "TCN"=>"X", "NTA"=>"X", "TNA"=>"X", "TAN"=>"X", "NTG"=>"X", "TNG"=>"X", "TGN"=>"X", "NT-"=>"X", "TN-"=>"X", "T-N"=>"X", "NTN"=>"X", "TNN"=>"X", "NCT"=>"X", "CNT"=>"X", "CTN"=>"X", "NCC"=>"X", "CNC"=>"X", "CCN"=>"X", "NCA"=>"X", "CNA"=>"X", "CAN"=>"X", "NCG"=>"X", "CNG"=>"X", "CGN"=>"X", "NC-"=>"X", "CN-"=>"X", "C-N"=>"X", "NCN"=>"X", "CNN"=>"X", "NAT"=>"X", "ANT"=>"X", "ATN"=>"X", "NAC"=>"X", "ANC"=>"X", "ACN"=>"X", "NAA"=>"X", "ANA"=>"X", "AAN"=>"X", "NAG"=>"X", "ANG"=>"X", "AGN"=>"X", "NA-"=>"X", "AN-"=>"X", "A-N"=>"X", "NAN"=>"X", "ANN"=>"X", "NGT"=>"X", "GNT"=>"X", "GTN"=>"X", "NGC"=>"X", "GNC"=>"X", "GCN"=>"X", "NGA"=>"X", "GNA"=>"X", "GAN"=>"X", "NGG"=>"X", "GNG"=>"X", "GGN"=>"X", "NG-"=>"X", "GN-"=>"X", "G-N"=>"X", "NGN"=>"X", "GNN"=>"X", "N-T"=>"X", "-NT"=>"X", "-TN"=>"X", "N-C"=>"X", "-NC"=>"X", "-CN"=>"X", "N-A"=>"X", "-NA"=>"X", "-AN"=>"X", "N-G"=>"X", "-NG"=>"X", "-GN"=>"X", "N--"=>"X", "-N-"=>"X", "--N"=>"X", "N-N"=>"X", "-NN"=>"X", "NNT"=>"X", "NNC"=>"X", "NNA"=>"X", "NNG"=>"X", "NN-"=>"X", "NNN"=>"X")                 
############################################################################################################################################################################
############################################################################################################################################################################
nonleap_month_day_dict = Dict{Int,Int}(0=>0, 1=>31, 2=>28, 3=>31, 4=>30, 5=>31, 6=>30, 7=>31, 8=>31, 9=>30, 10=>31, 11=>30, 12=>31)
leap_month_day_dict = Dict{Int,Int}(0=>0, 1=>31, 2=>29, 3=>31, 4=>30, 5=>31, 6=>30, 7=>31, 8=>31, 9=>30, 10=>31, 11=>30, 12=>31)
###################
index_to_tuple = Dict{Int, Tuple{Int,Int,Int}}()
tuple_to_index = Dict{Tuple{Int,Int,Int}, Int}()        
###########################################################
index = 0
for year in 2020:2027
    for month in 1:12
        if year%4 == 0
            month_days = leap_month_day_dict[month]
            for day in 1:month_days
                index += 1
                index_to_tuple[index] = (year, month, day)
            end
        else
            month_days = nonleap_month_day_dict[month]
            for day in 1:month_days
                index += 1
                index_to_tuple[index] = (year, month, day)
            end
        end
    end
end
for (index, date) in index_to_tuple
    tuple_to_index[date] = index
end
for y in 2020:2027
    tuple_to_index[(y, 0, 0)] = y*1000000
    index_to_tuple[y*1000000] = (y, 0, 0)
    for m in 1:12
        tuple_to_index[(y, m, 0)] = y*1000000 + m*1000
        index_to_tuple[y*1000000 + m*1000] = (y, m, 0)
    end
end
tuple_to_index[(0, 0, 0)] = 1000000000
index_to_tuple[1000000000] = (0, 0, 0)
############################################################################################################################################################################
############################################################################################################################################################################
function mut_ct_by_date_range_all_UNIVERSAL(date1::Int, date2::Int, all_dict::Dict{Int, Dict{String,Int}})
    for i in 1:3000
        if !haskey(all_dict, i)
            all_dict[i] = Dict{String,Int}()
        end
    end
    date1_to_date2_ct = Dict{String,Int}()
    for i in date1:date2
        for (mut, count) in all_dict[i]
            date1_to_date2_ct[mut] = get!(date1_to_date2_ct, mut, 0) + count
        end
    end
    return date1_to_date2_ct
end
############################################################################################################################################################################
############################################################################################################################################################################
function nuc_mut_pos_only_ct_by_date_range_all_UNIVERSAL(date1::Int, date2::Int, all_dict::Dict{Int, Dict{Int,Int}})
    for i in 1:3000
        if !haskey(all_dict, i)
            all_dict[i] = Dict{Int,Int}()
        end
    end
    date1_to_date2_ct = Dict()
    for i in date1:date2
        for (mut, count) in all_dict[i]
            date1_to_date2_ct[mut] = get!(date1_to_date2_ct, mut, 0) + count
        end
    end
    return date1_to_date2_ct
end
############################################################################################################################################################################
############################################################################################################################################################################
function ORF1abMut_to_NSP(mut::String)
    NSPmut = ""
#    NSP_muts = Dict("NSP$(i)" => Dict{String,Int}() for i in 1:16 if i ≠ 11)
    gene = aa_gene_comprehensive_dict[mut]
    pos = aa_pos_comprehensive_dict[mut]
    refAA = ref_AA_dict[mut]
    qryAA = qry_AA_dict[mut]
    if gene == "ORF1a"
        for (NSP, range) in NSP_ranges1a
            if pos in range
                NSPpos = pos - NSP1a_add[NSP]
                NSPmut = "NSP$(NSP):$(refAA)$(NSPpos)$(qryAA)"
            end
        end
    end
    if gene == "ORF1b"
        for (NSP, range) in NSP_ranges1b
            if pos in range
                NSPpos = pos - NSP1b_add[NSP]
                NSPmut = "NSP$(NSP):$(refAA)$(NSPpos)$(qryAA)"
            end
        end
    end
    return NSPmut
end
#####################################################################################################################################
function NSPmut_to_ORF1ab(NSPmut::String)
    ORFmut = ""
    ORFpos = ""
    NSP_num = parse(Int, split(NSPmut, ":")[1][4:end])
    NSP_pos = NSP_muts_pos_dict[NSPmut]
    refAA = NSP_ref_AA_dict[NSPmut]
    qryAA = NSP_qry_AA_dict[NSPmut]
    if NSPnum in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
        ORF1_pos = NSP1a_add[NSP_num] + NSP_pos
        ORFmut = "ORF1a:$(refAA)$(ORF1_pos)$(qryAA)"
    end
    if NSPnum in [13, 14, 15, 16]
        ORF1_pos = NSP1b_add[NSP_num] + NSP_pos
        ORFmut = "ORF1b:$(refAA)$(ORF1_pos)$(qryAA)"
    end
    if NSP_num == 12
        if NSP_pos in [1:8...]
            ORF1_pos = NSP1a_add[NSP_num] + NSP_pos
            ORFmut = "ORF1a:$(refAA)$(ORF1_pos)$(qryAA)"
        end
        if NSP_pos in [10:932...]
            ORF1_pos = NSP1b_add[NSP_num] + NSP_pos
            ORFmut = "ORF1b:$(refAA)$(ORF1_pos)$(qryAA)"
        end
    end
    ORF1pos = parse(Int, ORFpos)
    return ORFmut
end
#####################################################################################################################################
function NSP_range_to_ORF1ab_range(NSP_range::String)
    NSP = split(NSP_range, ":")[1]
    if length(NSP) < 3
        return ""
    end
    if NSP[1:3] ≠ "NSP"
        return ""
    end
    NSP_int = parse(Int, NSP[4:end])
    range = split(NSP_range, ":")[2]
    NSPpos1_int = parse(Int, split(range, "-")[1])
    NSPpos2_int = parse(Int, split(range, "-")[2])
    ORF_int1 = 0
    ORF_int2 = 0
    ORF1_AorB_pos1 = ""
    ORF1_AorB_pos2 = ""
    if NSP_int in [1:10...]
        ORF_int1 = NSPpos1_int + NSP1a_add[NSP_int]
        ORF_int2 = NSPpos2_int + NSP1a_add[NSP_int]
        ORF1_AorB_pos1 = "ORF1a"
        ORF1_AorB_pos2 = ""
    end
    if NSP_int in [13:16...]
        ORF_int1 = NSPpos1_int + NSP1b_add[NSP_int]
        ORF_int2 = NSPpos2_int + NSP1b_add[NSP_int]
        ORF1_AorB_pos1 = "ORF1b"
        ORF1_AorB_pos2 = ""
    end
    if NSP_int == 12
        if NSPpos1_int in [1:9...]
            ORF_int1 = NSPpos1_int + NSP1a_add[NSP_int]
            ORF1_AorB_pos1 = "1a"
        else
            ORF_int1 = NSPpos1_int + NSP1b_add[NSP_int]
            ORF1_AorB_pos1 = "ORF1b"
        end
        if NSPpos2_int in [1:9...]
            ORF_int2 = NSPpos2_int + NSP1a_add[NSP_int]
            ORF1_AorB_pos2 = "1a"
        end
        if !(NSPpos2_int in [1:9...])
            ORF_int2 = NSPpos2_int + NSP1b_add[NSP_int]
            if ORF1_AorB_pos1 == "1a"
                ORF1_AorB_pos2 = "1b:"
            else
            ORF1_AorB_pos2 = ""
            end
        end
    end
    ORF_int1_str = string(ORF_int1)
    ORF_int2_str = string(ORF_int2)
    ORF_range = ORF1_AorB_pos1*":"*ORF_int1_str*"-"*ORF1_AorB_pos2*ORF_int2_str
    return ORF_range
end
#####################################################################################################################################
function ORF1ab_range_to_NSP_range(ab_range::String)
    ab = split(ab_range, ":")[1]
    range = split(ab_range, ":")[2]
    pos1 = split(range, "-")[1]
    pos2 = split(range, "-")[2]
    pos1_int = parse(Int, pos1)
    pos2_int = parse(Int, pos2)
    NSPint1 = 0
    NSPint2 = 0
    NSPpos1 = ""
    NSPpos2 = ""
    NSPrange = ""
    NSPrange_pt1 = ""
    top_ct = 0
    if ab == "ORF1a"
        ct = 0
        for (NSP, rng) in NSP_ranges1a
            if pos1_int in rng && pos2_int in rng
                NSPint1 = pos1_int - NSP1a_add[NSP]
                NSPint2 = pos2_int - NSP1a_add[NSP]
                NSPpos1 = string(NSPint1)
                NSPpos2 = string(NSPint2)
                NSPstr = string(NSP)
                NSPrange = "NSP"*NSPstr*":"*NSPpos1*"-"*NSPpos2
            end
            if pos1_int in rng && !(pos2_int in rng)
                ct += 1
                NSPint1 = pos1_int - NSP1a_add[NSP]
                NSPpos1 = string(NSPint1)
                NSPstr = string(NSP)
                NSPrange_pt1 = "NSP"*NSPstr*":"*NSPpos1*"-"
            end
            if ct > 0
                top_ct += 1
                if pos2_int in rng
                    NSPint2 = pos2_int - NSP1a_add[NSP]
                    NSPpos2 = string(NSPint2)
                    NSPstr = string(NSP)
                    NSPrange_pt2 = "NSP"*NSPstr*":"*NSPpos2
                    NSPrange = NSPrange_pt1*NSPrange_pt2
                end
            end
        end
    end
    if ab == "ORF1b"
        ct = 0
        for (NSP, rng) in NSP_ranges1b
            if pos1_int in rng && pos2_int in rng
                NSPint1 = pos1_int - NSP1b_add[NSP]
                NSPint2 = pos2_int - NSP1b_add[NSP]
                NSPpos1 = string(NSPint1)
                NSPpos2 = string(NSPint2)
                NSPstr = string(NSP)
                NSPrange = "NSP"*NSPstr*":"*NSPpos1*"-"*NSPpos2
            end
            if pos1_int in rng && !(pos2_int in rng)
                ct += 1
                NSPint1 = pos1_int - NSP1b_add[NSP]
                NSPpos1 = string(NSPint1)
                NSPstr = string(NSP)
                NSPrange_pt1 = "NSP"*NSPstr*":"*NSPpos1*"-"
            end
            if ct > 0
                top_ct += 1
                if pos2_int in rng
                    NSPint2 = pos2_int - NSP1b_add[NSP]
                    NSPpos2 = string(NSPint2)
                    NSPstr = string(NSP)
                    NSPrange_pt2 = "NSP"*NSPstr*":"*NSPpos2
                    NSPrange = NSPrange_pt1*NSPrange_pt2
                end
            end
        end
    end 
    return NSPrange
end
######################################################################################################################################
function multiepi_to_epis(multi)  
    epi_num_only_pre(n) = split(n, "_")[3]
    epi_num_only_first(n) = parse(Int, split(epi_num_only_pre(n), "-")[1])
    epi_num_only_last(n) = parse(Int, split(epi_num_only_pre(n), "-")[2])
    first = epi_num_only_first(multi)
    last = epi_num_only_last(multi)
    epi_arr = Vector{String}()
    for i in first:last
        i_str = string(i)
        epi = "EPI_ISL_"*i_str
        push!(epi_arr, epi)
    end
    return epi_arr
end
######################################################################################################################################
function stringlist_to_strings(txt::String)
    epi_num_only_pre(n) = split(n, "_")[3]
    function epi_sortkey(epi)
        epinum = epi_num_only_pre(epi)
        epi_key = (length(epinum), epinum)
        return epi_key
    end
    arr_of_strings1 = Vector{String}()
    arr_of_strings2 = Vector{String}()
    no_newlines = replace(txt, "\n" =>" ")
    for seq in split(no_newlines, ", ")
        if '-' in seq
            multis = multiepi_to_epis(seq)
            for mseq in multis
                push!(arr_of_strings2, mseq)
            end
        else 
            push!(arr_of_strings2, seq)
        end
    end
    sort_arr_of_strings2 = sort(collect(arr_of_strings2), by = x -> epi_sortkey(x))    
    return sort_arr_of_strings2
end
######################################################################################################################################
function stringlist_to_set(txt::String)
    epi_num_only_pre(n) = split(n, "_")[3]
    function epi_sortkey(epi)
        epinum = epi_num_only_pre(epi)
        epi_key = (length(epinum), epinum)
        return epi_key
    end
    set_of_strings = Set{String}()
    no_newlines = replace(txt, "\n" =>" ")
    for seq in split(no_newlines, ", ")
        if '-' in seq
            multis = multiepi_to_epis(seq)
            for mseq in multis
                push!(set_of_strings, mseq)
            end
        else 
            push!(set_of_strings, seq)
        end
    end
    return set_of_strings
end  
######################################################################################################################################
function stringlist_to_set(txt::String)
    set_of_strings = Set{String}()
    no_newlines = replace(txt, "\n" =>" ")
    for seq in split(no_newlines, ", ")
        if '-' in seq
            multis = multiepi_to_epis(seq)
            for mseq in multis
                push!(set_of_strings, mseq)
            end
        else 
            push!(set_of_strings, seq)
        end
    end   
    return set_of_strings
end
#######################################################################################################################################
function list_to_strings(list::String)
    string_vec = string.(split(list, ", "))
    return string_vec
end
##################################################
function list_to_set(list::String)
    string_vec = string.(split(list, ", "))
    string_set = Set{String}()
    for str in string_vec
        push!(string_set, str)
    end
    return string_set
end
#####################################################################################################################################
function cross_check_set(old_seq_set::Set{String}, new_seq_set::Set{String})
    new_set2 = Set{String}()
    old_set2 = Set{String}()
    old_seq_set_len = length(old_seq_set)
    new_seq_set_len = length(new_seq_set)
    difference = new_seq_set_len - old_seq_set_len
    println("Number of Sequences in Old List = $(old_seq_set_len)")
    println("Number of Sequences in New List = $(new_seq_set_len)")
    println("Difference between Old & New Sets = $(difference)")
    println()
    for seq in new_seq_set
        if !(seq in old_seq_set)
            push!(new_set2, seq)
        end
        if seq in old_seq_set
            push!(old_set2, seq)
        end
    end
    new_len = length(new_set2)
    old_len = length(old_set2)
    if difference == new_len
        println("The numbers check out: difference between old & new sets = Number of new seqs")
    else
        println("The numbers don't check out: difference between old & new sets not equal to Number of new seqs")
    end
    println()
    println("Number of New Sequences = $(new_len)")
    println("Number of Old Sequences = $(old_len)")
    new_set2_sort = sort(collect(new_set2), by = x -> (length(x), x))
    return new_set2, new_set2_sort
end
############################################################################################################################################################################
############################################################################################################################################################################
cell1_runtime = round(digits=1, time() - cell1_start_time)
c1runtime1, c1runtime2 = seconds_to_hrs_min_sec(cell1_runtime)
println("Cell1 Runtime v0 = $(cell1_runtime) seconds")
println("Cell1 Runtime v1 = $(c1runtime1)")
println("Cell1 Runtime v2 = $(c1runtime2)"); println()
date_now = Dates.format(now(), "yyyy_mm_dd__HMMp"); println(date_now); print("\n"^1)
######################################################################################################################################

   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.to

2026_05_03__1951PM
/Users/ryhisner
/Users/ryhisner
Time to Load Packages = 0:00:19.24
Time to Load Packages = 0 hr, 0 min, 19.24 sec
seqs_in_nuc_genome_pango_dict = 3586
Cell1 Runtime v0 = 53.0 seconds
Cell1 Runtime v1 = 0:00:53.00
Cell1 Runtime v2 = 0 hr, 0 min, 53.00 sec

2026_05_03__1951PM



In [35]:
### Fx: load_all_seq_dicts, 2026_03_14 (Loads HQCS dataset)  | many large ones not needed & therefore commented out  
load_all_start = time();   print("\n"^1); date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime); print("\n"^1)
function load_all_seq_dicts(date::String, fx_name::String, ndjson_name::String, max_AA_mut::Int, revs_thresh::Int, qc_max::Int)
    start_all = time()
    filename = "$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)"
############################################################################################################################################################################
############################################################################################################################################################################
########################################################### HUGE files often not necessary to load #########################################################################
############################################################################################################################################################################
############################################################################################################################################################################
    global AA_muts_seq_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_seq.jld2", "AA_muts_seq")
#    global AA_dels_seq_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_dels_seq.jld2", "AA_dels_seq")    
#    global nuc_muts_seq_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_nuc_muts_seq.jld2", "nuc_muts_seq")
#    global nuc_muts_seq_no_dels_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_nuc_muts_seq_no_dels.jld2", "nuc_muts_seq_no_dels")
#    global seq_pango_all = load("2026_02_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000_seq_pango.jld2", "seq_pango")
#    global seq_date_index_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_date_index.jld2", "seq_date_index")
#    global date_nuc_mut_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_date_nuc_mut_ct.jld2", "date_nuc_mut_ct")
#    global date_nuc_mut_ct_no_dels_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_date_nuc_mut_ct_no_dels.jld2", "date_nuc_mut_ct_no_dels")
#    global date_AA_mut_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_date_AA_mut_ct.jld2", "date_AA_mut_ct")
#    global date_AA_mut_ct_no_dels_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_date_AA_mut_ct_no_dels.jld2", "date_AA_mut_ct_no_dels")
#    global date_AA_mut_ct_pos_only_no_dels_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_date_AA_mut_ct_pos_only_no_dels.jld2", "date_AA_mut_ct_pos_only_no_dels")
#    global seq_lab_set = load("2026_01_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000/2026_01_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000_seq_lab_set.jld2", "seq_lab_set")
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################ 
############################################################################################################################################################################
    global seq_ct_by_year_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_ct_by_year.jld2", "seq_ct_by_year")
    global seq_ct_by_year_month_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_ct_by_year_month.jld2", "seq_ct_by_year_month")
    global seq_ct_by_year_month_day_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_ct_by_year_month_day.jld2", "seq_ct_by_year_month_day")
    global pango_date_index_ct = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_pango_date_index_ct.jld2", "pango_date_index_ct")
############################################################################################################################################################################
    global avg_private_AA_per_circ_seq = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_avg_private_AA_per_circ_seq.jld2", "avg_private_AA_per_circ_seq")
    global all_seq_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_all_seq_ct.jld2", "all_seq_ct")
    global qualifying_seq_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_qualifying_seq_ct.jld2", "qualifying_seq_ct")
    global nuc_muts_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_nuc_muts_ct.jld2", "nuc_muts_ct")
    global nuc_muts_ct_no_dels_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_nuc_muts_ct_no_dels.jld2", "nuc_muts_ct_no_dels")
    global AA_muts_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct.jld2", "AA_muts_ct")
    global AA_dels_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_dels_ct.jld2", "AA_dels_ct")
    global AA_muts_ct_no_dels_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_no_dels.jld2", "AA_muts_ct_no_dels")
    global AA_muts_ct_pos_only_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_pos_only.jld2", "AA_muts_ct_pos_only")
    global AA_muts_ct_pos_only_no_dels_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_pos_only_no_dels.jld2", "AA_muts_ct_pos_only_no_dels")
    global gene_mut_density_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_gene_mut_density.jld2", "gene_mut_density")
    global domain_mut_density_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_domain_mut_density.jld2", "domain_mut_density")
    global nuc_muts_ct_sort_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_nuc_muts_ct_sort.jld2", "nuc_muts_ct_sort")
    global nuc_muts_ct_sort_by_seq_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_nuc_muts_ct_sort_by_seq_ct.jld2", "nuc_muts_ct_sort_by_seq_ct")
    global nuc_muts_ct_no_dels_sort_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_nuc_muts_ct_no_dels_sort.jld2", "nuc_muts_ct_no_dels_sort")
    global nuc_muts_ct_no_dels_sort_by_seq_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_nuc_muts_ct_no_dels_sort_by_seq_ct.jld2", "nuc_muts_ct_no_dels_sort_by_seq_ct")
    global AA_muts_ct_sort_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_sort.jld2", "AA_muts_ct_sort")
    global AA_muts_ct_sort_by_seq_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_sort_by_seq_ct.jld2", "AA_muts_ct_sort_by_seq_ct")
    global AA_muts_ct_pos_only_sort_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_pos_only_sort.jld2", "AA_muts_ct_pos_only_sort")
    global AA_muts_ct_pos_only_sort_by_seq_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_pos_only_sort_by_seq_ct.jld2", "AA_muts_ct_pos_only_sort_by_seq_ct")
    global AA_muts_ct_no_dels_sort_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_no_dels_sort.jld2", "AA_muts_ct_no_dels_sort")
    global AA_muts_ct_no_dels_sort_by_seq_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_no_dels_sort_by_seq_ct.jld2", "AA_muts_ct_no_dels_sort_by_seq_ct")
    global AA_muts_ct_pos_only_no_dels_sort_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_pos_only_no_dels_sort.jld2", "AA_muts_ct_pos_only_no_dels_sort")
    global AA_muts_ct_pos_only_no_dels_sort_by_seq_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_pos_only_no_dels_sort_by_seq_ct.jld2", "AA_muts_ct_pos_only_no_dels_sort_by_seq_ct")
    global gene_mut_density_sort_by_gene_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_gene_mut_density_sort_by_gene.jld2", "gene_mut_density_sort_by_gene")
    global gene_mut_density_sort_by_density_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_gene_mut_density_sort_by_density.jld2", "gene_mut_density_sort_by_density")
    global domain_mut_density_sort_by_gene_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_domain_mut_density_sort_by_gene.jld2", "domain_mut_density_sort_by_gene")
    global domain_mut_density_sort_by_density_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_domain_mut_density_sort_by_density.jld2", "domain_mut_density_sort_by_density")
    global nuc_dels_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_nuc_dels_ct.jld2", "nuc_dels_ct")
############################################################################################################################################################################
    global AA_no_dels_sub_count_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_no_dels_sub_count.jld2", "AA_no_dels_sub_count")
    global total_AA_subs_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_total_AA_subs.jld2", "total_AA_subs")
    global pango_date_index_ct = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_pango_date_index_ct.jld2", "pango_date_index_ct")
#    global clade_date_index_ct = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_clade_date_index_ct.jld2", "clade_date_index_ct")
#    global seq_pango_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_pango_all.jld2", "seq_pango_all")
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
    global country_set = load("2026_01_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000/2026_01_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000_country_set.jld2", "country_set")
    global clade_set = load("2026_01_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000/2026_01_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000_clade_set.jld2", "clade_set")
    global pango_set = load("2026_01_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000/2026_01_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000_pango_set.jld2", "pango_set")
############################################################################################################################################################################
############################################################################################################################################################################
#    global ORF9b_CTD_muts_seq_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_ORF9b_CTD_muts_seq.jld2", "ORF9b_CTD_muts_seq")
#    global multi_ORF9b_CTD_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_multi_ORF9b_CTD.jld2", "multi_ORF9b_CTD")
#    global multi_ORF9b_CTD_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_multi_ORF9b_CTD_ct.jld2", "multi_ORF9b_CTD_ct")
#    global qualifying_ORF9b_double_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_qualifying_ORF9b_double_ct.jld2", "qualifying_ORF9b_double_ct")
    
#    global ORF9b_CTD_muts_seq_all_relaxed_qc_10_1_30 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut8_maxRevs1_qcMax30_ORF9b_CTD_muts_seq_relaxed_qc_10_1_30.jld2", "ORF9b_CTD_muts_seq_relaxed_qc_10_1_30")
#    global multi_ORF9b_CTD_all_relaxed_qc_10_1_30 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut8_maxRevs1_qcMax30_multi_ORF9b_CTD_relaxed_qc_10_1_30.jld2", "multi_ORF9b_CTD_relaxed_qc_10_1_30")
#    global multi_ORF9b_CTD_ct_all_relaxed_qc_10_1_30 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut8_maxRevs1_qcMax30_multi_ORF9b_CTD_ct_relaxed_qc_10_1_30.jld2", "multi_ORF9b_CTD_ct_relaxed_qc_10_1_30")
#    global qualifying_ORF9b_double_ct_all_relaxed_qc_10_1_30 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut8_maxRevs1_qcMax30_qualifying_ORF9b_double_ct_relaxed_qc_10_1_30.jld2", "qualifying_ORF9b_double_ct_relaxed_qc_10_1_30")

#    global ORF9b_CTD_muts_seq_all_relaxed_qc_15_1_50 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut15_maxRevs1_qcMax50_ORF9b_CTD_muts_seq_relaxed_qc_15_1_50.jld2", "ORF9b_CTD_muts_seq_relaxed_qc_50_1_15")
#    global multi_ORF9b_CTD_all_relaxed_qc_15_1_50 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut15_maxRevs1_qcMax50_multi_ORF9b_CTD_relaxed_qc_15_1_50.jld2", "multi_ORF9b_CTD_relaxed_qc_50_1_15")
#    global multi_ORF9b_CTD_ct_all_relaxed_qc_15_1_50 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut15_maxRevs1_qcMax50_multi_ORF9b_CTD_ct_relaxed_qc_15_1_50.jld2", "multi_ORF9b_CTD_ct_relaxed_qc_50_1_15")
#    global qualifying_ORF9b_double_ct_all_relaxed_qc_15_1_50 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut15_maxRevs1_qcMax50_qualifying_ORF9b_double_ct_relaxed_qc_15_1_50.jld2", "qualifying_ORF9b_double_ct_relaxed_qc_50_1_15")

#    global ORF9b_CTD_muts_seq_all_relaxed_qc_25_3_200 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut25_maxRevs3_qcMax200_ORF9b_CTD_muts_seq_relaxed_qc_25_3_200.jld2", "ORF9b_CTD_muts_seq_relaxed_qc_200_3_25")
#    global multi_ORF9b_CTD_all_relaxed_qc_25_3_200 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut25_maxRevs3_qcMax200_multi_ORF9b_CTD_relaxed_qc_25_3_200.jld2", "multi_ORF9b_CTD_relaxed_qc_200_3_25")
#    global multi_ORF9b_CTD_ct_all_relaxed_qc_25_3_200 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut25_maxRevs3_qcMax200_multi_ORF9b_CTD_ct_relaxed_qc_25_3_200.jld2", "multi_ORF9b_CTD_ct_relaxed_qc_200_3_25")
#    global qualifying_ORF9b_double_ct_all_relaxed_qc_25_3_200 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut25_maxRevs3_qcMax200_qualifying_ORF9b_double_ct_relaxed_qc_25_3_200.jld2", "qualifying_ORF9b_double_ct_relaxed_qc_200_3_25")
#    global avg_private_AA_per_circ_seq2 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_avg_private_AA_per_circ_seq2.jld2", "avg_private_AA_per_circ_seq2")
    ##################### Below: Dicts for all seqs using higher (less restrictive) QC filters ##########################################
#    global AA_muts_seq_all_8000qc_filter = load("2026_01_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000/2026_01_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000_AA_muts_seq.jld2", "AA_muts_seq")
#    global AA_muts_seq_all_999qc_filter = load("dictionaries/2025-08-24_all_private_muts_EPI_ISL_400000_20080000_maxAAmut90_maxRevs4_qcMax999_AA_muts_seq.jld2", "AA_muts_seq")
############################################################################################################################################################################
    println("Dictionaries loaded!");   println()
    finish_all = time() - start_all;    finish_all_rd = round(digits=1, finish_all)
    load_hms1, load_hms2 = seconds_to_hrs_min_sec(finish_all)
    println("Total Time to Load ALL Dictionaries = $(finish_all_rd) seconds")  # println("Total Time to Load ALL Dictionaries = $(load_hms1)")
    println("Total Time to Load ALL Dictionaries = $(load_hms2)")
end
############################################################################################################################################################################
#date = "2026_01_06"
#nuc_muts_seq_all__90_5_8000 = load("2026_01_06_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000_nuc_muts_seq.jld2", "nuc_muts_seq")
############################################################################################################################################################################
date = "2026_04_14"
seq_labeled_subs_all = load("2026_02_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000_seq_labeled_subs.jld2", "seq_labeled_subs")
############################################################################################################################################################################
date = "2026_02_08"
clade_pct_date_index = load("dictionaries/$(date)__clade_pct_date_index.jld2", "clade_pct_date_index")
clade_pct_date_tuple = load("dictionaries/$(date)__clade_pct_date_tuple.jld2", "clade_pct_date_tuple")
clade_pct_date_string = load("dictionaries/$(date)__clade_pct_date_string.jld2", "clade_pct_date_string")
pango_pct_date_index = load("dictionaries/$(date)__pango_pct_date_index.jld2", "pango_pct_date_index")
pango_pct_date_tuple = load("dictionaries/$(date)__pango_pct_date_tuple.jld2", "pango_pct_date_tuple")
pango_pct_date_string = load("dictionaries/$(date)__pango_pct_date_string.jld2", "pango_pct_date_string")
pango_unaliased_pct_date_index = load("dictionaries/$(date)__pango_unaliased_pct_date_index.jld2", "pango_unaliased_pct_date_index")
pango_unaliased_pct_date_tuple = load("dictionaries/$(date)__pango_unaliased_pct_date_tuple.jld2", "pango_unaliased_pct_date_tuple")
pango_unaliased_pct_date_string = load("dictionaries/$(date)__pango_unaliased_pct_date_string.jld2", "pango_unaliased_pct_date_string")
pango_to_pango_unaliased_v2 = load("dictionaries/$(date)__pango_to_pango_unaliased_v2.jld2", "pango_to_pango_unaliased_v2")
pango_unaliased_to_pango_prefix = load("dictionaries/$(date)__pango_unaliased_to_pango_prefix.jld2", "pango_unaliased_to_pango_prefix")
pango_unaliased_to_pango = load("dictionaries/$(date)__pango_unaliased_to_pango.jld2", "pango_unaliased_to_pango")
pango_unaliased_predecessor_meta_dict = load("dictionaries/$(date)__pango_unaliased_predecessor_meta_dict.jld2", "pango_unaliased_predecessor_meta_dict")
pango_predecessor_meta_dict = load("dictionaries/$(date)__pango_predecessor_meta_dict.jld2", "pango_predecessor_meta_dict")
pango_unaliased_set = load("dictionaries/$(date)__pango_unaliased_set.jld2", "pango_unaliased_set")
pango_unaliased_date_index_ct = load("dictionaries/$(date)__pango_unaliased_date_index_ct.jld2", "pango_unaliased_date_index_ct")
clade_date_index_cumul = load("dictionaries/$(date)__clade_date_index_cumul.jld2", "clade_date_index_cumul")
pango_date_index_cumul = load("dictionaries/$(date)__pango_date_index_cumul.jld2", "pango_date_index_cumul")
pango_unaliased_date_index_cumul = load("dictionaries/$(date)__pango_unaliased_date_index_cumul.jld2", "pango_unaliased_date_index_cumul")
clade_total = load("dictionaries/$(date)__clade_total.jld2", "clade_total")
pango_total = load("dictionaries/$(date)__pango_total.jld2", "pango_total")
pango_unaliased_total = load("dictionaries/$(date)__pango_unaliased_total.jld2", "pango_unaliased_total")
AAmut_date_index_cumul = load("dictionaries/$(date)__AAmut_date_index_cumul.jld2", "AAmut_date_index_cumul")
nucmut_date_index_cumul = load("dictionaries/$(date)__nucmut_date_index_cumul.jld2", "nucmut_date_index_cumul")
pango_nuc_sub_WT = load("dictionaries/$(date)__pango_nuc_sub_WT.jld2", "pango_nuc_sub_WT")
pango_nuc_del_WT = load("dictionaries/$(date)__pango_nuc_del_WT.jld2", "pango_nuc_del_WT")
pango_nuc_sub_private = load("dictionaries/$(date)__pango_nuc_sub_private.jld2", "pango_nuc_sub_private")
pango_nuc_del_private = load("dictionaries/$(date)__pango_nuc_del_private.jld2", "pango_nuc_del_private")
pango_nuc_sub_revs = load("dictionaries/$(date)__pango_nuc_sub_revs.jld2", "pango_nuc_sub_revs")
pango_nuc_del_revs = load("dictionaries/$(date)__pango_nuc_del_revs.jld2", "pango_nuc_del_revs")
pango_AAsub_WT = load("dictionaries/$(date)__pango_AAsub_WT.jld2", "pango_AAsub_WT")
pango_AAsub_WT_pos_only = load("dictionaries/$(date)__pango_AAsub_WT_pos_only.jld2", "pango_AAsub_WT_pos_only")
pango_AAdel_WT = load("dictionaries/$(date)__pango_AAdel_WT.jld2", "pango_AAdel_WT")
pango_AAsub_private = load("dictionaries/$(date)__pango_AAsub_private.jld2", "pango_AAsub_private")
pango_AAdel_private = load("dictionaries/$(date)__pango_AAdel_private.jld2", "pango_AAdel_private")
pango_AAsub_revs = load("dictionaries/$(date)__pango_AAsub_revs.jld2", "pango_AAsub_revs")
pango_AAdel_revs = load("dictionaries/$(date)__pango_AAdel_revs.jld2", "pango_AAdel_revs")
pango_frameshifts_WT = load("dictionaries/$(date)__pango_frameshifts_WT.jld2", "pango_frameshifts_WT")
pango_designation_date = load("dictionaries/$(date)__pango_designation_date.jld2", "pango_designation_date")
clade_pango_set = load("dictionaries/$(date)__clade_pango_set.jld2", "clade_pango_set")
###########################################################################
delete!(pango_AAsub_WT["B.55"], "")
delete!(pango_AAsub_WT["B"], "")
###########################################################################
pango_AAsub_WT["B.1.1.529"] = union(pango_AAsub_WT["BA.1"], pango_AAsub_WT["BA.2"])
pango_AAsub_WT_pos_only["B.1.1.529"] = union(pango_AAsub_WT_pos_only["BA.1"], pango_AAsub_WT_pos_only["BA.2"])
pango_AAdel_WT["B.1.1.529"] = union(pango_AAdel_WT["BA.1"], pango_AAdel_WT["BA.2"])
pango_nuc_sub_WT["B.1.1.529"] = union(pango_nuc_sub_WT["BA.1"], pango_nuc_sub_WT["BA.2"])
pango_nuc_del_WT["B.1.1.529"] = union(pango_nuc_del_WT["BA.1"], pango_nuc_del_WT["BA.2"])
###########################################################################
pango_AAsub_WT["LF.3.1"] = pango_AAsub_WT["LF.3"]
pango_AAsub_WT_pos_only["LF.3.1"] = pango_AAsub_WT_pos_only["LF.3"]
pango_AAdel_WT["LF.3.1"] = pango_AAdel_WT["LF.3"]
pango_nuc_sub_WT["LF.3.1"] = pango_nuc_sub_WT["LF.3"]
pango_nuc_del_WT["LF.3.1"] = pango_nuc_del_WT["LF.3"]
###########################################################################
println("Done!")
load_all_fx_runtime = time() - load_all_start; load_all_fx_runtime_rd = round(digits=3, load_all_fx_runtime)
load_all_fx_hms1, load_all_fx_hms2 = seconds_to_hrs_min_sec(load_all_fx_runtime)
println("Total Time to Run load_all Fx = $(load_all_fx_runtime_rd) seconds")  # println("Total Time to Run load_all Fx = $(load_all_fx_hms1)") 
println("Total Time to Run load_all Fx = $(load_all_fx_hms2)"); print("\n"^1)
######################################################################################################################################
######################################################################################################################################


2026_05_04__807AM
8:07.23_AM

Done!
Total Time to Run load_all Fx = 47.034 seconds
Total Time to Run load_all Fx = 0 hr, 0 min, 47.03 sec



In [60]:
### Execute Load All Dicts Fx, EPI_ISL_400000_20300000 | 2026_02_08 | QC--5-1-5 | Runtime = 38 sec #########################
print("\n"^1); date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime); print("\n"^1)
start_load_all_dicts = time()
HQCS_date = "2026_01_06"
HQCS_fx_name = "all_private_muts"
HQCS_ndjson_name = "EPI_ISL_400001_20300000"
HQCS_max_AA_mut = 5
HQCS_revs_thresh = 1
HQCS_qc_max = 5
HQCS_qc_string = "$(HQCS_max_AA_mut)_$(HQCS_revs_thresh)_$(HQCS_qc_max)"
println("HQCS_qc_string = $(HQCS_qc_string)")
load_all_seq_dicts(HQCS_date, HQCS_fx_name, HQCS_ndjson_name, HQCS_max_AA_mut, HQCS_revs_thresh, HQCS_qc_max)

gene_protein_order = Dict{String, Int}("NSP1"=>1, "NSP2"=>2, "NSP3"=>3, "NSP4"=>4, "NSP5"=>5, "NSP6"=>6, "NSP7"=>7, "NSP8"=>8, "NSP9"=>9, "NSP10"=>10, "NSP12"=>12, "NSP13"=>13, "NSP14"=>14, "NSP15"=>15, "NSP16"=>16, "ORF3a"=>17, "ORF6"=>18, "ORF7a"=>19, "ORF7b"=>20, "ORF8"=>21, "ORF9b"=>22, "S"=>23, "E"=>24, "M"=>25, "N"=>26)
domain_order = Dict{String, Int}("NSP3_Ubl1"=>1, "NSP3_HVR"=>2, "NSP3_Mac1"=>3, "NSP3_Mac2"=>4, "NSP3_Mac3"=>5, "NSP3_DPUP"=>6, "NSP3_Ubl2"=>7, "NSP3_PLpro"=>8, "NSP3_NAB"=>9, "NSP3_BSM"=>10, "NSP3_TM1"=>11, "NSP3_Ecto3"=>12, "NSP3_TM234HLX"=>13, "NSP3_Y1"=>14, "NSP3_CoVY"=>15, "NSP4_TM1"=>16, "NSP4_Ecto4"=>17, "NSP4_TM2_TM6"=>18, "NSP4_CTD"=>19, "NSP6_AmphHlx"=>20, "NSP6_MAE"=>21, "NSP6_cyto_CTD"=>22, "NSP12_NiRAN"=>23, "NSP12_intrfce"=>24, "NSP12_fingers"=>25, "NSP12_palm"=>26, "NSP12_palmLnk"=>27, "NSP12_thumb"=>28, "NSP13_ZBD"=>29, "NSP13_stalk"=>30, "NSP13_1B"=>31, "NSP13_RecA1"=>32, "NSP13_RecA2"=>33, "NSP14_nsp10"=>34, "NSP14_EXON"=>35, "NSP14_hinge1"=>36, "NSP14_hinge2"=>37, "NSP14_N7MTase"=>38, "NSP15_NTD"=>39, "NSP15_MD"=>40, "NSP15_endoU"=>41, "S_S1"=>42, "S_S2"=>43, "S_NTD"=>44, "S_N2R"=>45, "S_RBD"=>46, "S_RBM"=>47, "S_SD1"=>48, "S_SD2"=>49, "S_630_loop"=>50, "S_FCS_region"=>51, "S_Beta1"=>52, "S_3H"=>53, "S_IL770"=>54, "S_FPPR"=>55, "S_FP"=>56, "S_HR1"=>57, "S_CH"=>58, "S_CD"=>59, "S_Beta2"=>60, "S_2turnHelix"=>61, "S_HR2"=>62, "S_TM"=>63, "S_CT"=>64, "ORF3a_SignalP"=>65, "ORF3a_NTD"=>66, "ORF3a_TM1"=>67, "ORF3a_TM12Lnk"=>68, "ORF3a_TM2"=>69, "ORF3a_TM3"=>70, "ORF3a_cytosl1"=>71, "ORF3a_Loop"=>72, "ORF3a_3DB"=>73, "ORF3a_CTD"=>74, "E_TM"=>75, "E_cytosol"=>76, "E_CTD"=>77, "N_N1"=>78, "N_N2"=>79, "N_N3"=>80, "N_N4"=>81, "N_N5"=>82, "N_SR"=>83, "N_L_helix"=>84, "N_CBP"=>85, "N_9b_overlap"=>86)    

finish_load_all_dicts = time() - start_load_all_dicts
finish_load_all_dicts_rd = round(digits=1, finish_load_all_dicts)

println("all_seq_ct_all = $(all_seq_ct_all)"); println()
println("Total Time to Load ALL Dictionaries = $(finish_load_all_dicts_rd)")
####################################################################################################################################
#################################### Create  date_index_seq_ct_all  Dictionary #####################################################
############################# Only needed for special calculations, so commented out here ##########################################
####################################################################################################################################
#date_index_seq_ct_all = Dict{Int, Int}()
#for d_index in values(seq_date_index_all)
#    date_index_seq_ct_all[d_index] = get(date_index_seq_ct_all, d_index, 0) + 1
#end
function date_index_range_seq_ct(date1::Int, date2::Int)
    date1_date2_seq_ct = 0
    for d_index in date1:date2
        date1_date2_seq_ct += date_index_seq_ct_all[d_index]
    end
    return date1_date2_seq_ct
end   
###################################################################################################################################
############################### Create  nuc_muts_ct_pos_only_no_dels_all  Dictionary ##############################################
###################################################################################################################################
dict_make_start = time()
nuc_muts_ct_pos_only_no_dels_all = Dict{Int, Int}()
for i in 1:30000
    nuc_muts_ct_pos_only_no_dels_all[i] = 0
end
for (nuc_mut, count) in nuc_muts_ct_no_dels_all
    pos = nuc_mut_int_comprehensive_dict[nuc_mut]
    nuc_muts_ct_pos_only_no_dels_all[pos] += count
end
##################################################################################################################################################
########################################################### Create pango_index_date_set ##########################################################
##################################################################################################################################################
pango_index_date_set = Set{String}()
for pango in keys(pango_date_index_ct)
    push!(pango_index_date_set, pango)
end
println(length(pango_index_date_set))
############################################################################################################################################################################
############################################################################################################################################################################
###################################### Note: Section below takes forever & usually not needed. Comment out normally.########################################################
############################################################################################################################################################################
############################################################################################################################################################################
#seq_AA_total_all = Dict{String, Int}()
#for i in 400000:20300000
#    epi = "EPI_ISL_$(i)"
#    seq_AA_total_all[epi] = 0
#end
#sizehint!(seq_AA_total_all, 19900001)
#last_time_check = time()
#aa_ct = 0
#forbidden_epis = Set(["O-1530551633/2023", "O-4336284453/2023"])
#for (mut, seq_set) in AA_muts_seq_all
#    for seq in seq_set
#        if !(seq in forbidden_epis)
#            seq_AA_total_all[seq] += 1
#        end
#    end
#    aa_ct += 1
#    if aa_ct%2000 == 0
#        last2000time = round(digits=1, time() - last_time_check)
#        println("$(aa_ct) Finished! | Time: $(last2000time)")
#    end
#end
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
println("length(keys(pango_to_pango_unaliased_v2) = $(length(keys(pango_to_pango_unaliased_v2)))")
dict_make_runtime = round(digits=2, time() - dict_make_start)
println("Total Time to Make nuc_muts_ct_pos_only_no_dels_all Dict = $(dict_make_runtime) seconds"); print("\n"^1)
print("\n"^1); println("HQCS_qc_string = $(HQCS_qc_string)"); print("\n"^2) 
#################################################################################################################################
#################################################################################################################################


2026_05_04__947PM
9:47.59_PM

HQCS_qc_string = 5_1_5
Dictionaries loaded!

Total Time to Load ALL Dictionaries = 102.5 seconds
Total Time to Load ALL Dictionaries = 0 hr, 1 min, 42.51 sec
all_seq_ct_all = 17283528

Total Time to Load ALL Dictionaries = 102.6
4297
length(keys(pango_to_pango_unaliased_v2) = 5657
Total Time to Make nuc_muts_ct_pos_only_no_dels_all Dict = 0.3 seconds



In [14]:
### Execute Load All Dicts Fx, EPI_ISL_400000_20300000 | 2026_02_08 | QC--10-1-30 | Runtime = 2 min 45 sec #########################
print("\n"^1); date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime); print("\n"^1)
start_load_all_dicts = time()
HQCS_date = "2026_01_06"
HQCS_fx_name = "all_private_muts"
HQCS_ndjson_name = "EPI_ISL_400001_20300000"
HQCS_max_AA_mut = 10
HQCS_revs_thresh = 1
HQCS_qc_max = 30
HQCS_qc_string = "$(HQCS_max_AA_mut)_$(HQCS_revs_thresh)_$(HQCS_qc_max)"
println("HQCS_qc_string = $(HQCS_qc_string)")
load_all_seq_dicts(HQCS_date, HQCS_fx_name, HQCS_ndjson_name, HQCS_max_AA_mut, HQCS_revs_thresh, HQCS_qc_max)

gene_protein_order = Dict{String, Int}("NSP1"=>1, "NSP2"=>2, "NSP3"=>3, "NSP4"=>4, "NSP5"=>5, "NSP6"=>6, "NSP7"=>7, "NSP8"=>8, "NSP9"=>9, "NSP10"=>10, "NSP12"=>12, "NSP13"=>13, "NSP14"=>14, "NSP15"=>15, "NSP16"=>16, "ORF3a"=>17, "ORF6"=>18, "ORF7a"=>19, "ORF7b"=>20, "ORF8"=>21, "ORF9b"=>22, "S"=>23, "E"=>24, "M"=>25, "N"=>26)
domain_order = Dict{String, Int}("NSP3_Ubl1"=>1, "NSP3_HVR"=>2, "NSP3_Mac1"=>3, "NSP3_Mac2"=>4, "NSP3_Mac3"=>5, "NSP3_DPUP"=>6, "NSP3_Ubl2"=>7, "NSP3_PLpro"=>8, "NSP3_NAB"=>9, "NSP3_BSM"=>10, "NSP3_TM1"=>11, "NSP3_Ecto3"=>12, "NSP3_TM234HLX"=>13, "NSP3_Y1"=>14, "NSP3_CoVY"=>15, "NSP4_TM1"=>16, "NSP4_Ecto4"=>17, "NSP4_TM2_TM6"=>18, "NSP4_CTD"=>19, "NSP6_AmphHlx"=>20, "NSP6_MAE"=>21, "NSP6_cyto_CTD"=>22, "NSP12_NiRAN"=>23, "NSP12_intrfce"=>24, "NSP12_fingers"=>25, "NSP12_palm"=>26, "NSP12_palmLnk"=>27, "NSP12_thumb"=>28, "NSP13_ZBD"=>29, "NSP13_stalk"=>30, "NSP13_1B"=>31, "NSP13_RecA1"=>32, "NSP13_RecA2"=>33, "NSP14_nsp10"=>34, "NSP14_EXON"=>35, "NSP14_hinge1"=>36, "NSP14_hinge2"=>37, "NSP14_N7MTase"=>38, "NSP15_NTD"=>39, "NSP15_MD"=>40, "NSP15_endoU"=>41, "S_S1"=>42, "S_S2"=>43, "S_NTD"=>44, "S_N2R"=>45, "S_RBD"=>46, "S_RBM"=>47, "S_SD1"=>48, "S_SD2"=>49, "S_630_loop"=>50, "S_FCS_region"=>51, "S_Beta1"=>52, "S_3H"=>53, "S_IL770"=>54, "S_FPPR"=>55, "S_FP"=>56, "S_HR1"=>57, "S_CH"=>58, "S_CD"=>59, "S_Beta2"=>60, "S_2turnHelix"=>61, "S_HR2"=>62, "S_TM"=>63, "S_CT"=>64, "ORF3a_SignalP"=>65, "ORF3a_NTD"=>66, "ORF3a_TM1"=>67, "ORF3a_TM12Lnk"=>68, "ORF3a_TM2"=>69, "ORF3a_TM3"=>70, "ORF3a_cytosl1"=>71, "ORF3a_Loop"=>72, "ORF3a_3DB"=>73, "ORF3a_CTD"=>74, "E_TM"=>75, "E_cytosol"=>76, "E_CTD"=>77, "N_N1"=>78, "N_N2"=>79, "N_N3"=>80, "N_N4"=>81, "N_N5"=>82, "N_SR"=>83, "N_L_helix"=>84, "N_CBP"=>85, "N_9b_overlap"=>86)    

println("all_seq_ct_all = $(all_seq_ct_all)"); println()
finish_load_all_dicts = time() - start_load_all_dicts
finish_load_all_dicts_rd = round(digits=1, finish_load_all_dicts)
println("Total Time to Load ALL Dictionaries = $(finish_load_all_dicts_rd)")
####################################################################################################################################
#################################### Create  date_index_seq_ct_all  Dictionary #####################################################
############################# Only needed for special calculations, so commented out here ##########################################
####################################################################################################################################
#date_index_seq_ct_all = Dict{Int, Int}()
#for d_index in values(seq_date_index_all)
#    date_index_seq_ct_all[d_index] = get(date_index_seq_ct_all, d_index, 0) + 1
#end
function date_index_range_seq_ct(date1::Int, date2::Int)
    date1_date2_seq_ct = 0
    for d_index in date1:date2
        date1_date2_seq_ct += date_index_seq_ct_all[d_index]
    end
    return date1_date2_seq_ct
end   
###################################################################################################################################
############################### Create  nuc_muts_ct_pos_only_no_dels_all  Dictionary ##############################################
###################################################################################################################################
dict_make_start = time()
nuc_muts_ct_pos_only_no_dels_all = Dict{Int, Int}()
for i in 1:30000
    nuc_muts_ct_pos_only_no_dels_all[i] = 0
end
for (nuc_mut, count) in nuc_muts_ct_no_dels_all
    pos = nuc_mut_int_comprehensive_dict[nuc_mut]
    nuc_muts_ct_pos_only_no_dels_all[pos] += count
end
############################################################################################################################################################################
############################################################################################################################################################################
###################################### Note: Section below takes forever & usually not needed. Comment out normally.########################################################
############################################################################################################################################################################
############################################################################################################################################################################
#seq_AA_total_all = Dict{String, Int}()
#for i in 400000:20300000
#    epi = "EPI_ISL_$(i)"
#    seq_AA_total_all[epi] = 0
#end
#sizehint!(seq_AA_total_all, 19900001)
#last_time_check = time()
#aa_ct = 0
#forbidden_epis = Set(["O-1530551633/2023", "O-4336284453/2023"])
#for (mut, seq_set) in AA_muts_seq_all
#    for seq in seq_set
#        if !(seq in forbidden_epis)
#            seq_AA_total_all[seq] += 1
#        end
#    end
#    aa_ct += 1
#    if aa_ct%2000 == 0
#        last2000time = round(digits=1, time() - last_time_check)
#        println("$(aa_ct) Finished! | Time: $(last2000time)")
#    end
#end
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
################################################### Create pango_index_date_set ###########################################################
pango_index_date_set = Set{String}()
for pango in keys(pango_date_index_ct)
    push!(pango_index_date_set, pango)
end
println("length of pango_index_date_set = $(length(pango_index_date_set)))")
#################################################################################################################################
pango_key_len = length(pango_to_pango_unaliased_v2)
println("length(keys(pango_to_pango_unaliased_v2) = $(pango_key_len)")
dict_make_runtime = round(digits=2, time() - dict_make_start)
println("Total Time to Make nuc_muts_ct_pos_only_no_dels_all Dict = $(dict_make_runtime) seconds"); print("\n"^1)
#################################################################################################################################
#################################################################################################################################


2026_05_03__847PM
8:47.18_PM

HQCS_qc_string = 10_1_30
Dictionaries loaded!

Total Time to Load ALL Dictionaries = 87.6 seconds
Total Time to Load ALL Dictionaries = 0 hr, 1 min, 27.59 sec
all_seq_ct_all = 17283528

Total Time to Load ALL Dictionaries = 87.6
length of pango_index_date_set = 4303)
length(keys(pango_to_pango_unaliased_v2) = 5657
Total Time to Make nuc_muts_ct_pos_only_no_dels_all Dict = 0.15 seconds



In [133]:
### Fx: chronic_load_dicts2_DQ, 2026_04_08 version (Loads EPCI dataset) | Loading EPCI Datasets #################
print("\n"^1); date_now = Dates.format(now(), "yyyy_mm_dd__HMMp"); println(date_now); nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime); print("\n"^1)
start_chr_load_fx = time()
function chronic_load_dicts2_DQ(ndjson_name::String, folder_name::String, date::String, rep_thresh::Int, revs_thresh::Int, print_ct_thresh::Int, DQ_mut_thresh::Int, DatePctDQThresh::Int, abs_min_mut_thresh::Int)
    start_date = Dates.format(now(), "yyyy_mm_dd__HMMp"); println(start_date)
######################################################################################################################################
#    global cumulative_AA_mut_ct_by_date_index
#        cumulative_AA_mut_ct_by_date_index = load("$(folder_name)/$(folder_name)__cumulative_AA_mut_ct_by_date_index.jld2", "cumulative_AA_mut_ct_by_date_index")
#    global cumulative_nuc_mut_ct_by_date_index
#        cumulative_nuc_mut_ct_by_date_index = load("$(folder_name)/$(folder_name)__cumulative_nuc_mut_ct_by_date_index.jld2", "cumulative_nuc_mut_ct_by_date_index")
######################################################################################################################################
    global seq_AA_insertions_WT
        seq_AA_insertions_WT = load("$(folder_name)/$(folder_name)__seq_AA_insertions_WT.jld2", "seq_AA_insertions_WT")
    global seq_nuc_insertions_WT
        seq_nuc_insertions_WT = load("$(folder_name)/$(folder_name)__seq_nuc_insertions_WT.jld2", "seq_nuc_insertions_WT")    
######################################################################################################################################
    global total_chr_AA_subs
        total_chr_AA_subs = load("$(folder_name)/$(folder_name)__total_chr_AA_subs.jld2", "total_chr_AA_subs")
######################################################################################################################################
    global total_nuc_revs_seq
        total_nuc_revs_seq = load("$(folder_name)/$(folder_name)__total_nuc_revs_seq.jld2", "total_nuc_revs_seq")
    global seq_nuc_total_revs
        seq_nuc_total_revs = load("$(folder_name)/$(folder_name)__seq_nuc_total_revs.jld2", "seq_nuc_total_revs")
    global total_AA_revs_seq
        total_AA_revs_seq = load("$(folder_name)/$(folder_name)__total_AA_revs_seq.jld2", "total_AA_revs_seq")
    global seq_AA_total_revs
        seq_AA_total_revs = load("$(folder_name)/$(folder_name)__seq_AA_total_revs.jld2", "seq_AA_total_revs")
    global seq_AA_revs
        seq_AA_revs = load("$(folder_name)/$(folder_name)__seq_AA_revs.jld2", "seq_AA_revs")
######################################################################################################################################
    global AA_muts_ct_chr_all_ratio
        AA_muts_ct_chr_all_ratio = load("$(folder_name)/$(folder_name)__AA_muts_ct_chr_all_ratio.jld2", "AA_muts_ct_chr_all_ratio")
    global AA_muts_ct_no_dels_chr_all_ratio
        AA_muts_ct_no_dels_chr_all_ratio = load("$(folder_name)/$(folder_name)__AA_muts_ct_no_dels_chr_all_ratio.jld2", "AA_muts_ct_no_dels_chr_all_ratio")
    global AA_muts_ct_pos_only_no_dels_chr_all_ratio
        AA_muts_ct_pos_only_no_dels_chr_all_ratio = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_no_dels_chr_all_ratio.jld2", "AA_muts_ct_pos_only_no_dels_chr_all_ratio")
    global AA_muts_ct_no_dels_no_revs_chr_all_ratio
        AA_muts_ct_no_dels_no_revs_chr_all_ratio = load("$(folder_name)/$(folder_name)__AA_muts_ct_no_dels_no_revs_chr_all_ratio.jld2", "AA_muts_ct_no_dels_no_revs_chr_all_ratio")
    global AA_muts_ct_pos_only_no_dels_no_revs_chr_all_ratio
        AA_muts_ct_pos_only_no_dels_no_revs_chr_all_ratio = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_no_dels_no_revs_chr_all_ratio.jld2", "AA_muts_ct_pos_only_no_dels_no_revs_chr_all_ratio")
###################################################################################################################################### 
    global nuc_muts_ct_no_dels_chr_all_ratio
        nuc_muts_ct_no_dels_chr_all_ratio = load("$(folder_name)/$(folder_name)__nuc_muts_ct_no_dels_chr_all_ratio.jld2", "nuc_muts_ct_no_dels_chr_all_ratio")
    global nuc_muts_ct_pos_only_no_dels_chr_all_ratio
        nuc_muts_ct_pos_only_no_dels_chr_all_ratio = load("$(folder_name)/$(folder_name)__nuc_muts_ct_pos_only_no_dels_chr_all_ratio.jld2", "nuc_muts_ct_pos_only_no_dels_chr_all_ratio")
    global nuc_muts_ct_no_dels_no_revs_chr_all_ratio
        nuc_muts_ct_no_dels_no_revs_chr_all_ratio = load("$(folder_name)/$(folder_name)__nuc_muts_ct_no_dels_no_revs_chr_all_ratio.jld2", "nuc_muts_ct_no_dels_no_revs_chr_all_ratio")
    global nuc_muts_ct_pos_only_no_dels_no_revs_chr_all_ratio
        nuc_muts_ct_pos_only_no_dels_no_revs_chr_all_ratio = load("$(folder_name)/$(folder_name)__nuc_muts_ct_pos_only_no_dels_no_revs_chr_all_ratio.jld2", "nuc_muts_ct_pos_only_no_dels_no_revs_chr_all_ratio")
###################################################################################################################################### 
####################################################################################################################################
    global AA_muts_ct_pos_only_adj_score
        AA_muts_ct_pos_only_adj_score = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_score.jld2", "AA_muts_ct_pos_only_adj_score")
    global AA_muts_ct_adj
        AA_muts_ct_adj = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj.jld2", "AA_muts_ct_adj")
    global AA_muts_ct_pos_only_adj_score_no_dels
        AA_muts_ct_pos_only_adj_score_no_dels = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_score_no_dels.jld2", "AA_muts_ct_pos_only_adj_score_no_dels")
    global nuc_muts_ct_adj
        nuc_muts_ct_adj = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj.jld2", "nuc_muts_ct_adj")
    global AA_muts_ct_adj_score
        AA_muts_ct_adj_score = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_score.jld2", "AA_muts_ct_adj_score")
    global AA_muts_ct_adj_score_no_dels
        AA_muts_ct_adj_score_no_dels = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_score_no_dels.jld2", "AA_muts_ct_adj_score_no_dels")
    global AA_muts_ct_pos_only_adj
        AA_muts_ct_pos_only_adj = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj.jld2", "AA_muts_ct_pos_only_adj")
    global AA_muts_ct_pos_only_adj_no_dels
        AA_muts_ct_pos_only_adj_no_dels = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_no_dels.jld2", "AA_muts_ct_pos_only_adj_no_dels")
    global nuc_muts_ct_adj_score_no_dels
        nuc_muts_ct_adj_score_no_dels = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_score_no_dels.jld2", "nuc_muts_ct_adj_score_no_dels")
    global nuc_muts_ct_adj_score
        nuc_muts_ct_adj_score = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_score.jld2", "nuc_muts_ct_adj_score")
    global nuc_muts_ct_adj_no_dels
        nuc_muts_ct_adj_no_dels = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_no_dels.jld2", "nuc_muts_ct_adj_no_dels")
    global AA_muts_ct_adj_no_dels
        AA_muts_ct_adj_no_dels = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_no_dels.jld2", "AA_muts_ct_adj_no_dels")
####################################################################################################################################
####################################################################################################################################
    global seq_ct_by_year
        seq_ct_by_year = load("$(folder_name)/$(folder_name)__seq_ct_by_year.jld2", "seq_ct_by_year")
    global seq_ct_by_year_month
        seq_ct_by_year_month = load("$(folder_name)/$(folder_name)__seq_ct_by_year_month.jld2", "seq_ct_by_year_month")
    global seq_ct_by_year_month_day
        seq_ct_by_year_month_day = load("$(folder_name)/$(folder_name)__seq_ct_by_year_month_day.jld2", "seq_ct_by_year_month_day")
    global seq_date_tuple
        seq_date_tuple = load("$(folder_name)/$(folder_name)__seq_date_tuple.jld2", "seq_date_tuple")
    global date_nuc_mut_ct_no_dels
        date_nuc_mut_ct_no_dels = load("$(folder_name)/$(folder_name)__date_nuc_mut_ct_no_dels.jld2", "date_nuc_mut_ct_no_dels")
    global date_AA_mut_ct_no_dels
        date_AA_mut_ct_no_dels = load("$(folder_name)/$(folder_name)__date_AA_mut_ct_no_dels.jld2", "date_AA_mut_ct_no_dels")
    global date_AA_mut_ct
        date_AA_mut_ct = load("$(folder_name)/$(folder_name)__date_AA_mut_ct.jld2", "date_AA_mut_ct")
    global date_AA_mut_ct_pos_only_no_dels
        date_AA_mut_ct_pos_only_no_dels = load("$(folder_name)/$(folder_name)__date_AA_mut_ct_pos_only_no_dels.jld2", "date_AA_mut_ct_pos_only_no_dels")
####################################################################################################################################
    global seq_collection_date
        seq_collection_date = load("$(folder_name)/$(folder_name)__seq_collection_date.jld2", "seq_collection_date")
    global seq_date_index
        seq_date_index = load("$(folder_name)/$(folder_name)__seq_date_index.jld2", "seq_date_index")
    global date_nuc_mut_ct
        date_nuc_mut_ct = load("$(folder_name)/$(folder_name)__date_nuc_mut_ct.jld2", "date_nuc_mut_ct")
####################################################################################################################################
    global seq_AA_del_ranges
        seq_AA_del_ranges = load("$(folder_name)/$(folder_name)__seq_AA_del_ranges.jld2", "seq_AA_del_ranges")
    global seq_AA_muts_pos_only_no_dels
        seq_AA_muts_pos_only_no_dels = load("$(folder_name)/$(folder_name)__seq_AA_muts_pos_only_no_dels.jld2", "seq_AA_muts_pos_only_no_dels")
    global seq_AA_muts_WT_pos_only
        seq_AA_muts_WT_pos_only = load("$(folder_name)/$(folder_name)__seq_AA_muts_WT_pos_only.jld2", "seq_AA_muts_WT_pos_only")
    global seq_AA_del_ranges_WT
        seq_AA_del_ranges_WT = load("$(folder_name)/$(folder_name)__seq_AA_del_ranges_WT.jld2", "seq_AA_del_ranges_WT")
    global seq_AA_muts_WT
        seq_AA_muts_WT = load("$(folder_name)/$(folder_name)__seq_AA_muts_WT.jld2", "seq_AA_muts_WT")
    global seq_AA_muts
        seq_AA_muts = load("$(folder_name)/$(folder_name)__seq_AA_muts.jld2", "seq_AA_muts")
    global seq_AA_muts_no_dels
        seq_AA_muts_no_dels = load("$(folder_name)/$(folder_name)__seq_AA_muts_no_dels.jld2", "seq_AA_muts_no_dels")
    global seq_AA_muts_pos_only
        seq_AA_muts_pos_only = load("$(folder_name)/$(folder_name)__seq_AA_muts_pos_only.jld2", "seq_AA_muts_pos_only")
    global seq_mixed_AA_muts
        seq_mixed_AA_muts = load("$(folder_name)/$(folder_name)__seq_mixed_AA_muts.jld2", "seq_mixed_AA_muts")
    global seq_unknown_AA
        seq_unknown_AA = load("$(folder_name)/$(folder_name)__seq_unknown_AA.jld2", "seq_unknown_AA")
    global seq_unknown_AA_ranges
        seq_unknown_AA_ranges = load("$(folder_name)/$(folder_name)__seq_unknown_AA_ranges.jld2", "seq_unknown_AA_ranges")
####################################################################################################################################
    global seq_nuc_muts
        seq_nuc_muts = load("$(folder_name)/$(folder_name)__seq_nuc_muts.jld2", "seq_nuc_muts")
    global seq_nuc_muts_WT
        seq_nuc_muts_WT = load("$(folder_name)/$(folder_name)__seq_nuc_muts_WT.jld2", "seq_nuc_muts_WT")
    global seq_nuc_del_ranges_WT
        seq_nuc_del_ranges_WT = load("$(folder_name)/$(folder_name)__seq_nuc_del_ranges_WT.jld2", "seq_nuc_del_ranges_WT")
    global seq_nuc_dropout
        seq_nuc_dropout = load("$(folder_name)/$(folder_name)__seq_nuc_dropout.jld2", "seq_nuc_dropout")
    global seq_nuc_del_ranges
        seq_nuc_del_ranges = load("$(folder_name)/$(folder_name)__seq_nuc_del_ranges.jld2", "seq_nuc_del_ranges")
    global seq_mixed_nucs
        seq_mixed_nucs = load("$(folder_name)/$(folder_name)__seq_mixed_nucs.jld2", "seq_mixed_nucs")
###################################################################################################################################
    global seq_clade
        seq_clade = load("$(folder_name)/$(folder_name)__seq_clade.jld2", "seq_clade")
    global seq_pango
        seq_pango = load("$(folder_name)/$(folder_name)__seq_pango.jld2", "seq_pango")
    global seq_pango_unaliased
        seq_pango_unaliased = load("$(folder_name)/$(folder_name)__seq_pango_unaliased.jld2", "seq_pango_unaliased")    
    global seq_clade_display
        seq_clade_display = load("$(folder_name)/$(folder_name)__seq_clade_display.jld2", "seq_clade_display")
    global seq_clade_ct
        seq_clade_ct = load("$(folder_name)/$(folder_name)__seq_clade_ct.jld2", "seq_clade_ct")
    global seq_pango_ct
        seq_pango_ct = load("$(folder_name)/$(folder_name)__seq_pango_ct.jld2", "seq_pango_ct")
    global seq_pango_unaliased_ct
        seq_pango_unaliased_ct = load("$(folder_name)/$(folder_name)__seq_pango_unaliased_ct.jld2", "seq_pango_unaliased_ct")
####################################################################################################################################
    global seq_US_state
        seq_US_state = load("$(folder_name)/$(folder_name)__seq_US_state.jld2", "seq_US_state")
    global seq_country
        seq_country = load("$(folder_name)/$(folder_name)__seq_country.jld2", "seq_country")
    global seq_lab_dict
        seq_lab_dict = load("$(folder_name)/$(folder_name)__seq_lab_dict.jld2", "seq_lab_dict")
####################################################################################################################################
    global gene_mut_density
        gene_mut_density = load("$(folder_name)/$(folder_name)__gene_mut_density.jld2", "gene_mut_density")
    global domain_mut_density
        domain_mut_density = load("$(folder_name)/$(folder_name)__domain_mut_density.jld2", "domain_mut_density")
####################################################################################################################################
    global nuc_muts_ct
        nuc_muts_ct = load("$(folder_name)/$(folder_name)__nuc_muts_ct.jld2", "nuc_muts_ct")
    global nuc_dels_ct
        nuc_dels_ct = load("$(folder_name)/$(folder_name)__nuc_dels_ct.jld2", "nuc_dels_ct")
    global nuc_muts_ct_no_dels
        nuc_muts_ct_no_dels = load("$(folder_name)/$(folder_name)__nuc_muts_ct_no_dels.jld2", "nuc_muts_ct_no_dels")
######################
    global nuc_dels_seq
        nuc_dels_seq = load("$(folder_name)/$(folder_name)__nuc_dels_seq.jld2", "nuc_dels_seq")
    global nuc_muts_seq
        nuc_muts_seq = load("$(folder_name)/$(folder_name)__nuc_muts_seq.jld2", "nuc_muts_seq")
    global nuc_dels_seq_WT
        nuc_dels_seq_WT = load("$(folder_name)/$(folder_name)__nuc_dels_seq_WT.jld2", "nuc_dels_seq_WT")
    global nuc_muts_seq_WT
        nuc_muts_seq_WT = load("$(folder_name)/$(folder_name)__nuc_muts_seq_WT.jld2", "nuc_muts_seq_WT")
##################################################################
    global AA_muts_ct
        AA_muts_ct = load("$(folder_name)/$(folder_name)__AA_muts_ct.jld2", "AA_muts_ct")
    global AA_muts_ct_no_dels_no_revs
        AA_muts_ct_no_dels_no_revs = load("$(folder_name)/$(folder_name)__AA_muts_ct_no_dels_no_revs.jld2", "AA_muts_ct_no_dels_no_revs")
    global AA_muts_ct_pos_only
        AA_muts_ct_pos_only = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only.jld2", "AA_muts_ct_pos_only")
    global AA_dels_ct
        AA_dels_ct = load("$(folder_name)/$(folder_name)__AA_dels_ct.jld2", "AA_dels_ct")
    global AA_muts_ct_pos_only_no_dels
        AA_muts_ct_pos_only_no_dels = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_no_dels.jld2", "AA_muts_ct_pos_only_no_dels")
    global AA_muts_ct_no_dels
        AA_muts_ct_no_dels = load("$(folder_name)/$(folder_name)__AA_muts_ct_no_dels.jld2", "AA_muts_ct_no_dels")
############################## 
    global AA_muts_seq
        AA_muts_seq = load("$(folder_name)/$(folder_name)__AA_muts_seq.jld2", "AA_muts_seq")
    global AA_muts_seq_pos_only
        AA_muts_seq_pos_only = load("$(folder_name)/$(folder_name)__AA_muts_seq_pos_only.jld2", "AA_muts_seq_pos_only")
    global AA_muts_seq_pos_only_no_dels
        AA_muts_seq_pos_only_no_dels = load("$(folder_name)/$(folder_name)__AA_muts_seq_pos_only_no_dels.jld2", "AA_muts_seq_pos_only_no_dels")
    global AA_dels_seq
        AA_dels_seq = load("$(folder_name)/$(folder_name)__AA_dels_seq.jld2", "AA_dels_seq")
##############################
    global AA_muts_seq_WT
        AA_muts_seq_WT = load("$(folder_name)/$(folder_name)__AA_muts_seq_WT.jld2", "AA_muts_seq_WT")
    global AA_muts_seq_WT_pos_only
        AA_muts_seq_WT_pos_only = load("$(folder_name)/$(folder_name)__AA_muts_seq_WT_pos_only.jld2", "AA_muts_seq_WT_pos_only")
    global AA_dels_seq_WT
        AA_dels_seq_WT = load("$(folder_name)/$(folder_name)__AA_dels_seq_WT.jld2", "AA_dels_seq_WT")
#####################################################################################################################################
#####################################################################################################################################
    global NSP_muts
        NSP_muts = load("$(folder_name)/$(folder_name)__NSP_muts.jld2", "NSP_muts")
    global NSP_muts_no_dels
        NSP_muts_no_dels = load("$(folder_name)/$(folder_name)__NSP_muts_no_dels.jld2", "NSP_muts_no_dels")
#####################################################################################################################################
    global non_rep_seqs_AA
        non_rep_seqs_AA = load("$(folder_name)/$(folder_name)__non_rep_seqs_AA.jld2", "non_rep_seqs_AA")
    global non_rep_seqs_AA_pos_only
        non_rep_seqs_AA_pos_only = load("$(folder_name)/$(folder_name)__non_rep_seqs_AA_pos_only.jld2", "non_rep_seqs_AA_pos_only")
    global non_rep_seqs_AA_pos_only_no_dels
        non_rep_seqs_AA_pos_only_no_dels = load("$(folder_name)/$(folder_name)__non_rep_seqs_AA_pos_only_no_dels.jld2", "non_rep_seqs_AA_pos_only_no_dels")
#####################################################################################################################################
    global rep_seq_grps_AA_muts_WT
        rep_seq_grps_AA_muts_WT = load("$(folder_name)/$(folder_name)__rep_seq_grps_AA_muts_WT.jld2", "rep_seq_grps_AA_muts_WT")
    global rep_seq_grps_AA_muts_WT_pos_only
        rep_seq_grps_AA_muts_WT_pos_only = load("$(folder_name)/$(folder_name)__rep_seq_grps_AA_muts_WT_pos_only.jld2", "rep_seq_grps_AA_muts_WT_pos_only")
    global rep_seq_grps_AA_dels_WT
        rep_seq_grps_AA_dels_WT = load("$(folder_name)/$(folder_name)__rep_seq_grps_AA_dels_WT.jld2", "rep_seq_grps_AA_dels_WT")
    global rep_seq_grps_nuc_muts_WT
        rep_seq_grps_nuc_muts_WT = load("$(folder_name)/$(folder_name)__rep_seq_grps_nuc_muts_WT.jld2", "rep_seq_grps_nuc_muts_WT")
    global rep_seq_grps_nuc_dels_WT
        rep_seq_grps_nuc_dels_WT = load("$(folder_name)/$(folder_name)__rep_seq_grps_nuc_dels_WT.jld2", "rep_seq_grps_nuc_dels_WT")
    global nuc_muts_rep_seq_grps_WT
        nuc_muts_rep_seq_grps_WT = load("$(folder_name)/$(folder_name)__nuc_muts_rep_seq_grps_WT.jld2", "nuc_muts_rep_seq_grps_WT")
    global nuc_dels_rep_seq_grps_WT
        nuc_dels_rep_seq_grps_WT = load("$(folder_name)/$(folder_name)__nuc_dels_rep_seq_grps_WT.jld2", "nuc_dels_rep_seq_grps_WT")
    global AA_dels_rep_seq_grps_WT
        AA_dels_rep_seq_grps_WT = load("$(folder_name)/$(folder_name)__AA_dels_rep_seq_grps_WT.jld2", "AA_dels_rep_seq_grps_WT")
    global AA_muts_rep_seq_grps_WT
        AA_muts_rep_seq_grps_WT = load("$(folder_name)/$(folder_name)__AA_muts_rep_seq_grps_WT.jld2", "AA_muts_rep_seq_grps_WT")
    global AA_muts_rep_seq_grps_WT_pos_only
        AA_muts_rep_seq_grps_WT_pos_only = load("$(folder_name)/$(folder_name)__AA_muts_rep_seq_grps_WT_pos_only.jld2", "AA_muts_rep_seq_grps_WT_pos_only")
#####################################################################################################################################
#####################################################################################################################################
    global rep_seq_grps_maxmut_pango
        rep_seq_grps_maxmut_pango = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_pango.jld2", "rep_seq_grps_maxmut_pango")
    global rep_seq_grps_maxmut_clade
        rep_seq_grps_maxmut_clade = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_clade.jld2", "rep_seq_grps_maxmut_clade")
    global rep_seq_grps_maxmut_pango_unaliased
        rep_seq_grps_maxmut_pango_unaliased = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_pango_unaliased.jld2", "rep_seq_grps_maxmut_pango_unaliased")
#####################################################################################################################################
    global rep_seq_grps_maxmut_dels
        rep_seq_grps_maxmut_dels = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_dels.jld2", "rep_seq_grps_maxmut_dels")
    global rep_seq_grps_maxmut_AA_pos_only_no_dels
        rep_seq_grps_maxmut_AA_pos_only_no_dels = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_AA_pos_only_no_dels.jld2", "rep_seq_grps_maxmut_AA_pos_only_no_dels")
    global rep_seq_grps_maxmut_nuc
        rep_seq_grps_maxmut_nuc = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_nuc.jld2", "rep_seq_grps_maxmut_nuc")
    global rep_seq_grps_maxmut_AA_pos_only
        rep_seq_grps_maxmut_AA_pos_only = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_AA_pos_only.jld2", "rep_seq_grps_maxmut_AA_pos_only")
    global rep_seq_grps_maxmut_nuc_dropout
        rep_seq_grps_maxmut_nuc_dropout = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_nuc_dropout.jld2", "rep_seq_grps_maxmut_nuc_dropout")
    global rep_seq_grps_maxmut_nuc_no_dels
        rep_seq_grps_maxmut_nuc_no_dels = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_nuc_no_dels.jld2", "rep_seq_grps_maxmut_nuc_no_dels")
    global rep_seq_grps_maxmut_mixed_nucs
        rep_seq_grps_maxmut_mixed_nucs = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_mixed_nucs.jld2", "rep_seq_grps_maxmut_mixed_nucs")
    global rep_seq_grps_maxmut_AA_dels
        rep_seq_grps_maxmut_AA_dels = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_AA_dels.jld2", "rep_seq_grps_maxmut_AA_dels")
    global rep_seq_grps_maxmut_del_ranges_ct
        rep_seq_grps_maxmut_del_ranges_ct = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_del_ranges_ct.jld2", "rep_seq_grps_maxmut_del_ranges_ct")
    global rep_seq_grps_maxmut_AA
        rep_seq_grps_maxmut_AA = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_AA.jld2", "rep_seq_grps_maxmut_AA")
    global rep_seq_grps_maxmut_mixed_AA_muts
        rep_seq_grps_maxmut_mixed_AA_muts = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_mixed_AA_muts.jld2", "rep_seq_grps_maxmut_mixed_AA_muts")
    global rep_seq_grps_maxmut_unknown_AA
        rep_seq_grps_maxmut_unknown_AA = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_unknown_AA.jld2", "rep_seq_grps_maxmut_unknown_AA")
    global rep_seq_grps_maxmut_unknown_AA_ranges
        rep_seq_grps_maxmut_unknown_AA_ranges = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_unknown_AA_ranges.jld2", "rep_seq_grps_maxmut_unknown_AA_ranges")
    global rep_seq_grps_maxmut_seqs
        rep_seq_grps_maxmut_seqs = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_seqs.jld2", "rep_seq_grps_maxmut_seqs")
    global rep_seq_grps_maxmut_AA_no_dels
        rep_seq_grps_maxmut_AA_no_dels = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_AA_no_dels.jld2", "rep_seq_grps_maxmut_AA_no_dels")
    global rep_seq_grps_maxmut_nuc_muts_WT
        rep_seq_grps_maxmut_nuc_muts_WT = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_nuc_muts_WT.jld2", "rep_seq_grps_maxmut_nuc_muts_WT")
    global rep_seq_grps_maxmut_nuc_dels_WT
        rep_seq_grps_maxmut_nuc_dels_WT = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_nuc_dels_WT.jld2", "rep_seq_grps_maxmut_nuc_dels_WT")
    global rep_seq_grps_maxmut_AA_muts_WT
        rep_seq_grps_maxmut_AA_muts_WT = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_AA_muts_WT.jld2", "rep_seq_grps_maxmut_AA_muts_WT")
    global rep_seq_grps_maxmut_AA_dels_WT
        rep_seq_grps_maxmut_AA_dels_WT = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_AA_dels_WT.jld2", "rep_seq_grps_maxmut_AA_dels_WT")
    global rep_seq_grps_maxmut_AA_muts_WT_pos_only
        rep_seq_grps_maxmut_AA_muts_WT_pos_only = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_AA_muts_WT_pos_only.jld2", "rep_seq_grps_maxmut_AA_muts_WT_pos_only")
################################################################################################################################
#####################################################################################################################################
    global rep_seq_grps_seqs
        rep_seq_grps_seqs = load("$(folder_name)/$(folder_name)__rep_seq_grps_seqs.jld2", "rep_seq_grps_seqs")
    global rep_seq_grps_pango
        rep_seq_grps_pango = load("$(folder_name)/$(folder_name)__rep_seq_grps_pango.jld2", "rep_seq_grps_pango")
    global rep_seq_grps_clade
        rep_seq_grps_clade = load("$(folder_name)/$(folder_name)__rep_seq_grps_clade.jld2", "rep_seq_grps_clade")
    global rep_seq_grps_pango_unaliased
        rep_seq_grps_pango_unaliased = load("$(folder_name)/$(folder_name)__rep_seq_grps_pango_unaliased.jld2", "rep_seq_grps_pango_unaliased")
    global rep_seq_grps_muts
        rep_seq_grps_muts = load("$(folder_name)/$(folder_name)__rep_seq_grps_muts.jld2", "rep_seq_grps_muts")
    global rep_seq_grps_mixed_nucs
        rep_seq_grps_mixed_nucs = load("$(folder_name)/$(folder_name)__rep_seq_grps_mixed_nucs.jld2", "rep_seq_grps_mixed_nucs")
    global rep_seq_grps_muts_no_dels
        rep_seq_grps_muts_no_dels = load("$(folder_name)/$(folder_name)__rep_seq_grps_muts_no_dels.jld2", "rep_seq_grps_muts_no_dels")
    global rep_seq_grps_AA_pos_only
        rep_seq_grps_AA_pos_only = load("$(folder_name)/$(folder_name)__rep_seq_grps_AA_pos_only.jld2", "rep_seq_grps_AA_pos_only")
    global rep_seq_grps_AA_no_dels
        rep_seq_grps_AA_no_dels = load("$(folder_name)/$(folder_name)__rep_seq_grps_AA_no_dels.jld2", "rep_seq_grps_AA_no_dels")
    global rep_seq_grps_AA_dels
        rep_seq_grps_AA_dels = load("$(folder_name)/$(folder_name)__rep_seq_grps_AA_dels.jld2", "rep_seq_grps_AA_dels")
    global rep_seq_grps_dels
        rep_seq_grps_dels = load("$(folder_name)/$(folder_name)__rep_seq_grps_dels.jld2", "rep_seq_grps_dels")
    global rep_seq_grps_AA_pos_only_no_dels
        rep_seq_grps_AA_pos_only_no_dels = load("$(folder_name)/$(folder_name)__rep_seq_grps_AA_pos_only_no_dels.jld2", "rep_seq_grps_AA_pos_only_no_dels")
    global rep_seq_grps_del_ranges_ct
        rep_seq_grps_del_ranges_ct = load("$(folder_name)/$(folder_name)__rep_seq_grps_del_ranges_ct.jld2", "rep_seq_grps_del_ranges_ct")
    global rep_seq_grps_AA
        rep_seq_grps_AA = load("$(folder_name)/$(folder_name)__rep_seq_grps_AA.jld2", "rep_seq_grps_AA")
    global rep_seq_grps_nuc_dropout
        rep_seq_grps_nuc_dropout = load("$(folder_name)/$(folder_name)__rep_seq_grps_nuc_dropout.jld2", "rep_seq_grps_nuc_dropout")
#    global rep_seq_grps_unknown_AA
#        rep_seq_grps_unknown_AA = load("$(folder_name)/$(folder_name)__rep_seq_grps_unknown_AA.jld2", "rep_seq_grps_unknown_AA")
#    global rep_seq_grps_mixed_AA_muts
#        rep_seq_grps_mixed_AA_muts = load("$(folder_name)/$(folder_name)__rep_seq_grps_mixed_AA_muts.jld2", "rep_seq_grps_mixed_AA_muts")
    global AA_dels_rep_seq_grps
        AA_dels_rep_seq_grps = load("$(folder_name)/$(folder_name)__AA_dels_rep_seq_grps.jld2", "AA_dels_rep_seq_grps")
    global AA_muts_rep_seq_grps
        AA_muts_rep_seq_grps = load("$(folder_name)/$(folder_name)__AA_muts_rep_seq_grps.jld2", "AA_muts_rep_seq_grps")
    global AA_muts_rep_seq_grps_pos_only
        AA_muts_rep_seq_grps_pos_only = load("$(folder_name)/$(folder_name)__AA_muts_rep_seq_grps_pos_only.jld2", "AA_muts_rep_seq_grps_pos_only")
    global AA_muts_rep_seq_grps_no_dels
        AA_muts_rep_seq_grps_no_dels = load("$(folder_name)/$(folder_name)__AA_muts_rep_seq_grps_no_dels.jld2", "AA_muts_rep_seq_grps_no_dels")
    global nuc_muts_rep_seq_grps
        nuc_muts_rep_seq_grps = load("$(folder_name)/$(folder_name)__nuc_muts_rep_seq_grps.jld2", "nuc_muts_rep_seq_grps")
    global nuc_muts_rep_seq_grps_no_dels
        nuc_muts_rep_seq_grps_no_dels = load("$(folder_name)/$(folder_name)__nuc_muts_rep_seq_grps_no_dels.jld2", "nuc_muts_rep_seq_grps_no_dels")
    global nuc_dels_rep_seq_grps
        nuc_dels_rep_seq_grps = load("$(folder_name)/$(folder_name)__nuc_dels_rep_seq_grps.jld2", "nuc_dels_rep_seq_grps")    
##########################################################################################################################################################################
##########################################################################################################################################################################
################################################ Below: Used to be in ungodly @save form, now changed ####################################################################
##########################################################################################################################################################################
##########################################################################################################################################################################
    global AA_muts_ct_pos_only_adj_sort_by_site
        AA_muts_ct_pos_only_adj_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_sort_by_site.jld2", "AA_muts_ct_pos_only_adj_sort_by_site")
    global nuc_muts_ct_adj_score_no_dels_sort_by_site
        nuc_muts_ct_adj_score_no_dels_sort_by_site = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_score_no_dels_sort_by_site.jld2", "nuc_muts_ct_adj_score_no_dels_sort_by_site")
    global nuc_muts_ct_adj_no_dels_sort_by_site
        nuc_muts_ct_adj_no_dels_sort_by_site = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_no_dels_sort_by_site.jld2", "nuc_muts_ct_adj_no_dels_sort_by_site")
    global nuc_muts_ct_adj_sort_by_seq_ct
        nuc_muts_ct_adj_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_sort_by_seq_ct.jld2", "nuc_muts_ct_adj_sort_by_seq_ct")
    global nuc_muts_ct_adj_score_sort_by_score
        nuc_muts_ct_adj_score_sort_by_score = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_score_sort_by_score.jld2", "nuc_muts_ct_adj_score_sort_by_score")
    global AA_muts_ct_adj_score_sort_by_site
        AA_muts_ct_adj_score_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_score_sort_by_site.jld2", "AA_muts_ct_adj_score_sort_by_site")
    global AA_muts_ct_pos_only_adj_no_dels_sort_by_seq_ct
        AA_muts_ct_pos_only_adj_no_dels_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_no_dels_sort_by_seq_ct.jld2", "AA_muts_ct_pos_only_adj_no_dels_sort_by_seq_ct")
    global AA_muts_ct_pos_only_adj_score_sort_by_site
        AA_muts_ct_pos_only_adj_score_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_score_sort_by_site.jld2", "AA_muts_ct_pos_only_adj_score_sort_by_site")
    global AA_muts_ct_pos_only_adj_sort_by_seq_ct
        AA_muts_ct_pos_only_adj_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_sort_by_seq_ct.jld2", "AA_muts_ct_pos_only_adj_sort_by_seq_ct")
    global AA_muts_ct_adj_score_no_dels_sort_by_site
        AA_muts_ct_adj_score_no_dels_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_score_no_dels_sort_by_site.jld2", "AA_muts_ct_adj_score_no_dels_sort_by_site")
    global AA_muts_ct_adj_sort_by_seq_ct
        AA_muts_ct_adj_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_sort_by_seq_ct.jld2", "AA_muts_ct_adj_sort_by_seq_ct")
    global AA_muts_ct_adj_no_dels_sort_by_site
        AA_muts_ct_adj_no_dels_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_no_dels_sort_by_site.jld2", "AA_muts_ct_adj_no_dels_sort_by_site")
    global AA_muts_ct_pos_only_adj_score_sort_by_score
        AA_muts_ct_pos_only_adj_score_sort_by_score = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_score_sort_by_score.jld2", "AA_muts_ct_pos_only_adj_score_sort_by_score")
    global AA_muts_ct_pos_only_adj_score_no_dels_sort_by_score
        AA_muts_ct_pos_only_adj_score_no_dels_sort_by_score = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_score_no_dels_sort_by_score.jld2", "AA_muts_ct_pos_only_adj_score_no_dels_sort_by_score")
    global nuc_muts_ct_adj_no_dels_sort_by_seq_ct
        nuc_muts_ct_adj_no_dels_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_no_dels_sort_by_seq_ct.jld2", "nuc_muts_ct_adj_no_dels_sort_by_seq_ct")
    global AA_muts_ct_adj_sort_by_site
        AA_muts_ct_adj_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_sort_by_site.jld2", "AA_muts_ct_adj_sort_by_site")
    global AA_muts_ct_pos_only_adj_no_dels_sort_by_site
        AA_muts_ct_pos_only_adj_no_dels_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_no_dels_sort_by_site.jld2", "AA_muts_ct_pos_only_adj_no_dels_sort_by_site")
    global AA_muts_ct_adj_score_sort_by_score
        AA_muts_ct_adj_score_sort_by_score = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_score_sort_by_score.jld2", "AA_muts_ct_adj_score_sort_by_score")
    global nuc_muts_ct_adj_score_no_dels_sort_by_score
        nuc_muts_ct_adj_score_no_dels_sort_by_score = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_score_no_dels_sort_by_score.jld2", "nuc_muts_ct_adj_score_no_dels_sort_by_score")
    global nuc_muts_ct_adj_sort_by_site
        nuc_muts_ct_adj_sort_by_site = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_sort_by_site.jld2", "nuc_muts_ct_adj_sort_by_site")
    global AA_muts_ct_adj_score_no_dels_sort_by_score
        AA_muts_ct_adj_score_no_dels_sort_by_score = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_score_no_dels_sort_by_score.jld2", "AA_muts_ct_adj_score_no_dels_sort_by_score")
    global AA_muts_ct_adj_no_dels_sort_by_seq_ct
        AA_muts_ct_adj_no_dels_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_no_dels_sort_by_seq_ct.jld2", "AA_muts_ct_adj_no_dels_sort_by_seq_ct")
    global AA_muts_ct_pos_only_adj_score_no_dels_sort_by_site
        AA_muts_ct_pos_only_adj_score_no_dels_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_score_no_dels_sort_by_site.jld2", "AA_muts_ct_pos_only_adj_score_no_dels_sort_by_site")
    global nuc_muts_ct_adj_score_sort_by_site
        nuc_muts_ct_adj_score_sort_by_site = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_score_sort_by_site.jld2", "nuc_muts_ct_adj_score_sort_by_site")
###########################################################################################################################################################################
###########################################################################################################################################################################
    global AA_muts_ct_no_dels_sort_by_site
        AA_muts_ct_no_dels_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_no_dels_sort_by_site.jld2", "AA_muts_ct_no_dels_sort_by_site")
    global AA_muts_ct_pos_only_sort_by_seq_ct
        AA_muts_ct_pos_only_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_sort_by_seq_ct.jld2", "AA_muts_ct_pos_only_sort_by_seq_ct")
    global nuc_muts_ct_sort_by_site
        nuc_muts_ct_sort_by_site = load("$(folder_name)/$(folder_name)__nuc_muts_ct_sort_by_site.jld2", "nuc_muts_ct_sort_by_site")
    global excluded_AA
        excluded_AA = load("$(folder_name)/$(folder_name)__excluded_AA.jld2", "excluded_AA")
    global chronic_search_muts
        chronic_search_muts = load("$(folder_name)/$(folder_name)__chronic_search_muts.jld2", "chronic_search_muts")
    global excluded_pos
        excluded_pos = load("$(folder_name)/$(folder_name)__excluded_pos.jld2", "excluded_pos")
    global rep_seqs
        rep_seqs = load("$(folder_name)/$(folder_name)__rep_seqs.jld2", "rep_seqs")
    global non_rep_seqs
        non_rep_seqs = load("$(folder_name)/$(folder_name)__non_rep_seqs.jld2", "non_rep_seqs")
    global all_seqs
        all_seqs = load("$(folder_name)/$(folder_name)__all_seqs.jld2", "all_seqs")
    global domain_mut_density_sort_by_gene
        domain_mut_density_sort_by_gene = load("$(folder_name)/$(folder_name)__domain_mut_density_sort_by_gene.jld2", "domain_mut_density_sort_by_gene")
    global gene_mut_density_sort_by_density
        gene_mut_density_sort_by_density = load("$(folder_name)/$(folder_name)__gene_mut_density_sort_by_density.jld2", "gene_mut_density_sort_by_density")
    global nuc_muts_ct_sort_by_seq_ct
        nuc_muts_ct_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__nuc_muts_ct_sort_by_seq_ct.jld2", "nuc_muts_ct_sort_by_seq_ct")
    global AA_muts_ct_sort_by_seq_ct
        AA_muts_ct_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__AA_muts_ct_sort_by_seq_ct.jld2", "AA_muts_ct_sort_by_seq_ct")
    global AA_muts_ct_no_dels_sort_by_seq_ct
        AA_muts_ct_no_dels_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__AA_muts_ct_no_dels_sort_by_seq_ct.jld2", "AA_muts_ct_no_dels_sort_by_seq_ct")
    global AA_muts_ct_pos_only_no_dels_sort_by_site
        AA_muts_ct_pos_only_no_dels_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_no_dels_sort_by_site.jld2", "AA_muts_ct_pos_only_no_dels_sort_by_site")
    global gene_mut_density_sort_by_gene
        gene_mut_density_sort_by_gene = load("$(folder_name)/$(folder_name)__gene_mut_density_sort_by_gene.jld2", "gene_mut_density_sort_by_gene")
    global AA_muts_ct_sort_by_site
        AA_muts_ct_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_sort_by_site.jld2", "AA_muts_ct_sort_by_site")
    global AA_muts_ct_pos_only_sort_by_site
        AA_muts_ct_pos_only_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_sort_by_site.jld2", "AA_muts_ct_pos_only_sort_by_site")
    global domain_mut_density_sort_by_density
        domain_mut_density_sort_by_density = load("$(folder_name)/$(folder_name)__domain_mut_density_sort_by_density.jld2", "domain_mut_density_sort_by_density")
    global too_many_reversions
        too_many_reversions = load("$(folder_name)/$(folder_name)__too_many_reversions.jld2", "too_many_reversions")
    global AA_muts_ct_pos_only_no_dels_sort_by_seq_ct
        AA_muts_ct_pos_only_no_dels_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_no_dels_sort_by_seq_ct.jld2", "AA_muts_ct_pos_only_no_dels_sort_by_seq_ct")

    global AA_muts_ct_chr_all_ratio_ct_sort
        AA_muts_ct_chr_all_ratio_ct_sort = load("$(folder_name)/$(folder_name)__AA_muts_ct_chr_all_ratio_ct_sort.jld2", "AA_muts_ct_chr_all_ratio_ct_sort")
    global AA_muts_ct_chr_all_ratio_pos_sort
        AA_muts_ct_chr_all_ratio_pos_sort = load("$(folder_name)/$(folder_name)__AA_muts_ct_chr_all_ratio_pos_sort.jld2", "AA_muts_ct_chr_all_ratio_pos_sort")    

    global AA_muts_ct_no_dels_chr_all_ratio_ct_sort
        AA_muts_ct_no_dels_chr_all_ratio_ct_sort = load("$(folder_name)/$(folder_name)__AA_muts_ct_no_dels_chr_all_ratio_ct_sort.jld2", "AA_muts_ct_no_dels_chr_all_ratio_ct_sort")
    global AA_muts_ct_no_dels_chr_all_ratio_pos_sort
        AA_muts_ct_no_dels_chr_all_ratio_pos_sort = load("$(folder_name)/$(folder_name)__AA_muts_ct_no_dels_chr_all_ratio_pos_sort.jld2", "AA_muts_ct_no_dels_chr_all_ratio_pos_sort")
    
    global AA_muts_ct_pos_only_no_dels_chr_all_ratio_pos_sort
        AA_muts_ct_pos_only_no_dels_chr_all_ratio_pos_sort = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_no_dels_chr_all_ratio_pos_sort.jld2", "AA_muts_ct_pos_only_no_dels_chr_all_ratio_pos_sort")
    global AA_muts_ct_pos_only_no_dels_chr_all_ratio_ct_sort
        AA_muts_ct_pos_only_no_dels_chr_all_ratio_ct_sort = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_no_dels_chr_all_ratio_ct_sort.jld2", "AA_muts_ct_pos_only_no_dels_chr_all_ratio_ct_sort")
    
    global avg_AA_subs_per_chr_seq
        avg_AA_subs_per_chr_seq = load("$(folder_name)/$(folder_name)__avg_AA_subs_per_chr_seq.jld2", "avg_AA_subs_per_chr_seq")
    global avg_AA_subs_per_circ_seq
        avg_AA_subs_per_circ_seq = load("$(folder_name)/$(folder_name)__avg_AA_subs_per_circ_seq.jld2", "avg_AA_subs_per_circ_seq")
###########################################################################################################################################################################
###########################################################################################################################################################################
    global NSP_muts_sortByCt_Arr
        NSP_muts_sortByCt_Arr = load("$(folder_name)/$(folder_name)__NSP_muts_sortByCt_Arr.jld2", "NSP_muts_sortByCt_Arr")
    global NSP_muts_sortByPos_Arr
        NSP_muts_sortByPos_Arr = load("$(folder_name)/$(folder_name)__NSP_muts_sortByPos_Arr.jld2", "NSP_muts_sortByPos_Arr")
    global NSP_muts_no_dels_sortByCt_Arr
        NSP_muts_no_dels_sortByCt_Arr = load("$(folder_name)/$(folder_name)__NSP_muts_no_dels_sortByCt_Arr.jld2", "NSP_muts_no_dels_sortByCt_Arr")
    global NSP_muts_no_dels_sortByPos_Arr
        NSP_muts_no_dels_sortByPos_Arr = load("$(folder_name)/$(folder_name)__NSP_muts_no_dels_sortByPos_Arr.jld2", "NSP_muts_no_dels_sortByPos_Arr")
    global NSP1_muts_sortByCt
        NSP1_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP1_muts_sortByCt.jld2", "NSP1_muts_sortByCt")
    global NSP1_muts_sortByPos
        NSP1_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP1_muts_sortByPos.jld2", "NSP1_muts_sortByPos")
    global NSP2_muts_sortByCt
        NSP2_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP2_muts_sortByCt.jld2", "NSP2_muts_sortByCt")
    global NSP2_muts_sortByPos
        NSP2_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP2_muts_sortByPos.jld2", "NSP2_muts_sortByPos")
    global NSP3_muts_sortByCt
        NSP3_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP3_muts_sortByCt.jld2", "NSP3_muts_sortByCt")
    global NSP3_muts_sortByPos
        NSP3_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP3_muts_sortByPos.jld2", "NSP3_muts_sortByPos")
    global NSP4_muts_sortByCt
        NSP4_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP4_muts_sortByCt.jld2", "NSP4_muts_sortByCt")
    global NSP4_muts_sortByPos
        NSP4_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP4_muts_sortByPos.jld2", "NSP4_muts_sortByPos")
    global NSP5_muts_sortByCt
        NSP5_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP5_muts_sortByCt.jld2", "NSP5_muts_sortByCt")
    global NSP5_muts_sortByPos
        NSP5_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP5_muts_sortByPos.jld2", "NSP5_muts_sortByPos")
    global NSP6_muts_sortByCt
        NSP6_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP6_muts_sortByCt.jld2", "NSP6_muts_sortByCt")
    global NSP6_muts_sortByPos
        NSP6_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP6_muts_sortByPos.jld2", "NSP6_muts_sortByPos")
    global NSP7_muts_sortByCt
        NSP7_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP7_muts_sortByCt.jld2", "NSP7_muts_sortByCt")
    global NSP7_muts_sortByPos
        NSP7_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP7_muts_sortByPos.jld2", "NSP7_muts_sortByPos")
    global NSP8_muts_sortByCt
        NSP8_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP8_muts_sortByCt.jld2", "NSP8_muts_sortByCt")
    global NSP8_muts_sortByPos
        NSP8_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP8_muts_sortByPos.jld2", "NSP8_muts_sortByPos")
    global NSP9_muts_sortByCt
        NSP9_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP9_muts_sortByCt.jld2", "NSP9_muts_sortByCt")
    global NSP9_muts_sortByPos
        NSP9_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP9_muts_sortByPos.jld2", "NSP9_muts_sortByPos")
    global NSP10_muts_sortByCt
        NSP10_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP10_muts_sortByCt.jld2", "NSP10_muts_sortByCt")
    global NSP10_muts_sortByPos
        NSP10_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP10_muts_sortByPos.jld2", "NSP10_muts_sortByPos")
    global NSP11_muts_sortByCt
        NSP11_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP11_muts_sortByCt.jld2", "NSP11_muts_sortByCt")
    global NSP11_muts_sortByPos
        NSP11_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP11_muts_sortByPos.jld2", "NSP11_muts_sortByPos")
    global NSP12_muts_sortByCt
        NSP12_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP12_muts_sortByCt.jld2", "NSP12_muts_sortByCt")
    global NSP12_muts_sortByPos
        NSP12_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP12_muts_sortByPos.jld2", "NSP12_muts_sortByPos")
    global NSP13_muts_sortByCt
        NSP13_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP13_muts_sortByCt.jld2", "NSP13_muts_sortByCt")
    global NSP13_muts_sortByPos
        NSP13_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP13_muts_sortByPos.jld2", "NSP13_muts_sortByPos")
    global NSP14_muts_sortByCt
        NSP14_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP14_muts_sortByCt.jld2", "NSP14_muts_sortByCt")
    global NSP14_muts_sortByPos
        NSP14_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP14_muts_sortByPos.jld2", "NSP14_muts_sortByPos")
    global NSP15_muts_sortByCt
        NSP15_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP15_muts_sortByCt.jld2", "NSP15_muts_sortByCt")
    global NSP15_muts_sortByPos
        NSP15_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP15_muts_sortByPos.jld2", "NSP15_muts_sortByPos")
    global NSP16_muts_sortByCt
        NSP16_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP16_muts_sortByCt.jld2", "NSP16_muts_sortByCt")
    global NSP16_muts_sortByPos
        NSP16_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP16_muts_sortByPos.jld2", "NSP16_muts_sortByPos")
    global NSP1_muts_no_dels_sortByCt
        NSP1_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP1_muts_no_dels_sortByCt.jld2", "NSP1_muts_no_dels_sortByCt")
    global NSP1_muts_no_dels_sortByPos
        NSP1_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP1_muts_no_dels_sortByPos.jld2", "NSP1_muts_no_dels_sortByPos")
    global NSP2_muts_no_dels_sortByCt
        NSP2_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP2_muts_no_dels_sortByCt.jld2", "NSP2_muts_no_dels_sortByCt")
    global NSP2_muts_no_dels_sortByPos
        NSP2_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP2_muts_no_dels_sortByPos.jld2", "NSP2_muts_no_dels_sortByPos")
    global NSP3_muts_no_dels_sortByCt
        NSP3_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP3_muts_no_dels_sortByCt.jld2", "NSP3_muts_no_dels_sortByCt")
    global NSP3_muts_no_dels_sortByPos
        NSP3_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP3_muts_no_dels_sortByPos.jld2", "NSP3_muts_no_dels_sortByPos")
    global NSP4_muts_no_dels_sortByCt
        NSP4_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP4_muts_no_dels_sortByCt.jld2", "NSP4_muts_no_dels_sortByCt")
    global NSP4_muts_no_dels_sortByPos
        NSP4_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP4_muts_no_dels_sortByPos.jld2", "NSP4_muts_no_dels_sortByPos")
    global NSP5_muts_no_dels_sortByCt
        NSP5_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP5_muts_no_dels_sortByCt.jld2", "NSP5_muts_no_dels_sortByCt")
    global NSP5_muts_no_dels_sortByPos
        NSP5_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP5_muts_no_dels_sortByPos.jld2", "NSP5_muts_no_dels_sortByPos")
    global NSP6_muts_no_dels_sortByCt
        NSP6_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP6_muts_no_dels_sortByCt.jld2", "NSP6_muts_no_dels_sortByCt")
    global NSP6_muts_no_dels_sortByPos
        NSP6_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP6_muts_no_dels_sortByPos.jld2", "NSP6_muts_no_dels_sortByPos")
    global NSP7_muts_no_dels_sortByCt
        NSP7_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP7_muts_no_dels_sortByCt.jld2", "NSP7_muts_no_dels_sortByCt")
    global NSP7_muts_no_dels_sortByPos
        NSP7_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP7_muts_no_dels_sortByPos.jld2", "NSP7_muts_no_dels_sortByPos")
    global NSP8_muts_no_dels_sortByCt
        NSP8_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP8_muts_no_dels_sortByCt.jld2", "NSP8_muts_no_dels_sortByCt")
    global NSP8_muts_no_dels_sortByPos
        NSP8_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP8_muts_no_dels_sortByPos.jld2", "NSP8_muts_no_dels_sortByPos")
    global NSP9_muts_no_dels_sortByCt
        NSP9_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP9_muts_no_dels_sortByCt.jld2", "NSP9_muts_no_dels_sortByCt")
    global NSP9_muts_no_dels_sortByPos
        NSP9_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP9_muts_no_dels_sortByPos.jld2", "NSP9_muts_no_dels_sortByPos")
    global NSP10_muts_no_dels_sortByCt
        NSP10_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP10_muts_no_dels_sortByCt.jld2", "NSP10_muts_no_dels_sortByCt")
    global NSP10_muts_no_dels_sortByPos
        NSP10_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP10_muts_no_dels_sortByPos.jld2", "NSP10_muts_no_dels_sortByPos")
    global NSP11_muts_no_dels_sortByCt
        NSP11_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP11_muts_no_dels_sortByCt.jld2", "NSP11_muts_no_dels_sortByCt")
    global NSP11_muts_no_dels_sortByPos
        NSP11_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP11_muts_no_dels_sortByPos.jld2", "NSP11_muts_no_dels_sortByPos")
    global NSP12_muts_no_dels_sortByCt
        NSP12_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP12_muts_no_dels_sortByCt.jld2", "NSP12_muts_no_dels_sortByCt")
    global NSP12_muts_no_dels_sortByPos
        NSP12_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP12_muts_no_dels_sortByPos.jld2", "NSP12_muts_no_dels_sortByPos")
    global NSP13_muts_no_dels_sortByCt
        NSP13_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP13_muts_no_dels_sortByCt.jld2", "NSP13_muts_no_dels_sortByCt")
    global NSP13_muts_no_dels_sortByPos
        NSP13_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP13_muts_no_dels_sortByPos.jld2", "NSP13_muts_no_dels_sortByPos")
    global NSP14_muts_no_dels_sortByCt
        NSP14_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP14_muts_no_dels_sortByCt.jld2", "NSP14_muts_no_dels_sortByCt")
    global NSP14_muts_no_dels_sortByPos
        NSP14_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP14_muts_no_dels_sortByPos.jld2", "NSP14_muts_no_dels_sortByPos")
    global NSP15_muts_no_dels_sortByCt
        NSP15_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP15_muts_no_dels_sortByCt.jld2", "NSP15_muts_no_dels_sortByCt")
    global NSP15_muts_no_dels_sortByPos
        NSP15_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP15_muts_no_dels_sortByPos.jld2", "NSP15_muts_no_dels_sortByPos")
    global NSP16_muts_no_dels_sortByCt
        NSP16_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP16_muts_no_dels_sortByCt.jld2", "NSP16_muts_no_dels_sortByCt")
    global NSP16_muts_no_dels_sortByPos
        NSP16_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP16_muts_no_dels_sortByPos.jld2", "NSP16_muts_no_dels_sortByPos")
####################################################################################################
    global all_seqs_set
        all_seqs_set = load("$(folder_name)/$(folder_name)__all_seqs_set.jld2", "all_seqs_set")
    global all_qualifying_seqs
        all_qualifying_seqs = load("$(folder_name)/$(folder_name)__all_qualifying_seqs.jld2", "all_qualifying_seqs")
    global all_qualifying_seqs_set
        all_qualifying_seqs_set = load("$(folder_name)/$(folder_name)__all_qualifying_seqs_set.jld2", "all_qualifying_seqs_set")
    global all_nonqualifying_seqs
        all_nonqualifying_seqs = load("$(folder_name)/$(folder_name)__all_nonqualifying_seqs.jld2", "all_nonqualifying_seqs")
    global all_nonqualifying_seqs_set
        all_nonqualifying_seqs_set = load("$(folder_name)/$(folder_name)__all_nonqualifying_seqs_set.jld2", "all_nonqualifying_seqs_set")
###########################################################################################################################################################################
    global AA_muts_ct_no_dels_no_revs_sort_by_site
        AA_muts_ct_no_dels_no_revs_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_no_dels_no_revs_sort_by_site.jld2", "AA_muts_ct_no_dels_no_revs_sort_by_site")
    global AA_muts_ct_no_dels_no_revs_sort_by_seq_ct
        AA_muts_ct_no_dels_no_revs_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__AA_muts_ct_no_dels_no_revs_sort_by_seq_ct.jld2", "AA_muts_ct_no_dels_no_revs_sort_by_seq_ct")
    global chronic_search_muts_v2
        chronic_search_muts_v2 = load("$(folder_name)/$(folder_name)__chronic_search_muts_v2.jld2", "chronic_search_muts_v2")
    global avg_AA_subs_per_chr_seq_no_revs
        avg_AA_subs_per_chr_seq_no_revs = load("$(folder_name)/$(folder_name)__avg_AA_subs_per_chr_seq_no_revs.jld2", "avg_AA_subs_per_chr_seq_no_revs")
###########################################################################################################################################################################
    global nuc_muts_ct_no_dels_chr_all_ratio_ct_sort
        nuc_muts_ct_no_dels_chr_all_ratio_ct_sort = load("$(folder_name)/$(folder_name)__nuc_muts_ct_no_dels_chr_all_ratio_ct_sort.jld2", "nuc_muts_ct_no_dels_chr_all_ratio_ct_sort")
    global nuc_muts_ct_no_dels_chr_all_ratio_pos_sort
        nuc_muts_ct_no_dels_chr_all_ratio_pos_sort = load("$(folder_name)/$(folder_name)__nuc_muts_ct_no_dels_chr_all_ratio_pos_sort.jld2", "nuc_muts_ct_no_dels_chr_all_ratio_pos_sort")
    global nuc_muts_ct_pos_only_no_dels_chr_all_ratio_pos_sort
        nuc_muts_ct_pos_only_no_dels_chr_all_ratio_pos_sort = load("$(folder_name)/$(folder_name)__nuc_muts_ct_pos_only_no_dels_chr_all_ratio_pos_sort.jld2", "nuc_muts_ct_pos_only_no_dels_chr_all_ratio_pos_sort")
    global nuc_muts_ct_pos_only_no_dels_chr_all_ratio_ct_sort
        nuc_muts_ct_pos_only_no_dels_chr_all_ratio_ct_sort = load("$(folder_name)/$(folder_name)__nuc_muts_ct_pos_only_no_dels_chr_all_ratio_ct_sort.jld2", "nuc_muts_ct_pos_only_no_dels_chr_all_ratio_ct_sort")
    global avg_nuc_subs_per_chr_seq
        avg_nuc_subs_per_chr_seq = load("$(folder_name)/$(folder_name)__avg_nuc_subs_per_chr_seq.jld2", "avg_nuc_subs_per_chr_seq")
    global avg_nuc_subs_per_circ_seq
        avg_nuc_subs_per_circ_seq = load("$(folder_name)/$(folder_name)__avg_nuc_subs_per_circ_seq.jld2", "avg_nuc_subs_per_circ_seq")
###########################################################################################################################################################################
###########################################################################################################################################################################
    return date_nuc_mut_ct, date_nuc_mut_ct_no_dels, date_AA_mut_ct, date_AA_mut_ct_no_dels, date_AA_mut_ct_pos_only_no_dels, 
    seq_ct_by_year, seq_ct_by_year_month, seq_ct_by_year_month_day, 
    seq_collection_date, seq_date_index, seq_date_tuple, 
###################################################################################################################################
    seq_clade, seq_clade_display, seq_pango, 
    seq_pango_unaliased, seq_clade_ct, seq_pango_ct, seq_pango_unaliased_ct, 
###################################################################################################################################
    too_many_reversions, 
###################################################################################################################################
    seq_nuc_muts, seq_nuc_del_ranges, seq_nuc_dropout, seq_nuc_muts_WT, seq_nuc_del_ranges_WT, 
################################
    seq_AA_insertions_WT, seq_nuc_insertions_WT,
################################
    seq_AA_muts, seq_AA_muts_no_dels, seq_AA_muts_pos_only, seq_AA_del_ranges, seq_mixed_AA_muts, 
    seq_AA_muts_WT, seq_AA_muts_WT_pos_only, seq_AA_del_ranges_WT, seq_AA_muts_pos_only_no_dels, 
########################################################
    nuc_muts_seq, nuc_muts_seq_WT, 
    nuc_dels_seq, nuc_dels_seq_WT, 
    AA_muts_seq, AA_muts_seq_pos_only, AA_muts_seq_WT, AA_muts_seq_WT_pos_only, AA_muts_seq_pos_only_no_dels, 
    AA_dels_seq, AA_dels_seq_WT, 
######################################################## 
    seq_unknown_AA, seq_unknown_AA_ranges, seq_mixed_nucs, 
############################
    nuc_dels_ct, nuc_muts_ct, nuc_muts_ct_no_dels, 
###############
    AA_dels_ct, AA_muts_ct, AA_muts_ct_no_dels, AA_muts_ct_pos_only, AA_muts_ct_pos_only_no_dels, AA_muts_ct_no_dels_no_revs, 
############################
    nuc_muts_ct_sort_by_site, nuc_muts_ct_sort_by_seq_ct, 
############################ 
    AA_muts_ct_sort_by_site, AA_muts_ct_sort_by_seq_ct, AA_muts_ct_no_dels_sort_by_site, AA_muts_ct_no_dels_no_revs_sort_by_site, 
    AA_muts_ct_no_dels_no_revs_sort_by_seq_ct, AA_muts_ct_no_dels_sort_by_seq_ct,  AA_muts_ct_pos_only_sort_by_site, 
    AA_muts_ct_pos_only_sort_by_seq_ct, AA_muts_ct_pos_only_no_dels_sort_by_site, AA_muts_ct_pos_only_no_dels_sort_by_seq_ct, 
###################################################################################################################################
################################################################################################################################### 
    nuc_muts_ct_adj, nuc_muts_ct_adj_no_dels, nuc_muts_ct_adj_score, nuc_muts_ct_adj_score_no_dels,   
    nuc_muts_ct_adj_sort_by_seq_ct, nuc_muts_ct_adj_no_dels_sort_by_site, nuc_muts_ct_adj_no_dels_sort_by_seq_ct,
    nuc_muts_ct_adj_sort_by_site, nuc_muts_ct_adj_score_no_dels_sort_by_score, AA_muts_ct_pos_only_adj_score_no_dels_sort_by_score, 
    nuc_muts_ct_adj_score_sort_by_site, nuc_muts_ct_adj_score_sort_by_score, nuc_muts_ct_adj_score_no_dels_sort_by_site, 
    AA_muts_ct_adj, AA_muts_ct_adj_no_dels, AA_muts_ct_pos_only_adj, AA_muts_ct_pos_only_adj_no_dels, 
    AA_muts_ct_adj_score, AA_muts_ct_adj_score_no_dels, AA_muts_ct_pos_only_adj_score, AA_muts_ct_pos_only_adj_score_no_dels,
    AA_muts_ct_adj_sort_by_site, AA_muts_ct_adj_sort_by_seq_ct, AA_muts_ct_adj_no_dels_sort_by_site,
    AA_muts_ct_adj_no_dels_sort_by_seq_ct, AA_muts_ct_adj_score_sort_by_score, AA_muts_ct_adj_score_sort_by_site, 
    AA_muts_ct_adj_score_no_dels_sort_by_score, AA_muts_ct_adj_score_no_dels_sort_by_site, 
    AA_muts_ct_pos_only_adj_sort_by_site, AA_muts_ct_pos_only_adj_sort_by_seq_ct, AA_muts_ct_pos_only_adj_no_dels_sort_by_site, 
    AA_muts_ct_pos_only_adj_no_dels_sort_by_seq_ct, AA_muts_ct_pos_only_adj_score_sort_by_site, 
    AA_muts_ct_pos_only_adj_score_sort_by_score, AA_muts_ct_pos_only_adj_score_no_dels_sort_by_site, 
####################################################################################################################################
    AA_muts_ct_chr_all_ratio, AA_muts_ct_no_dels_chr_all_ratio, AA_muts_ct_no_dels_no_revs_chr_all_ratio, AA_muts_ct_pos_only_no_dels_no_revs_chr_all_ratio, 
########################
    AA_muts_ct_chr_all_ratio_ct_sort, AA_muts_ct_chr_all_ratio_pos_sort, 
    AA_muts_ct_no_dels_chr_all_ratio_ct_sort, AA_muts_ct_no_dels_chr_all_ratio_pos_sort, 
    AA_muts_ct_pos_only_no_dels_chr_all_ratio_pos_sort, AA_muts_ct_pos_only_no_dels_chr_all_ratio_ct_sort, 
###################################################### 
    nuc_muts_ct_no_dels_chr_all_ratio, nuc_muts_ct_no_dels_no_revs_chr_all_ratio, nuc_muts_ct_pos_only_no_dels_chr_all_ratio, nuc_muts_ct_pos_only_no_dels_no_revs_chr_all_ratio, 
#######################
    nuc_muts_ct_no_dels_chr_all_ratio_ct_sort, nuc_muts_ct_no_dels_chr_all_ratio_pos_sort, 
    nuc_muts_ct_pos_only_no_dels_chr_all_ratio_pos_sort, nuc_muts_ct_pos_only_no_dels_chr_all_ratio_ct_sort, 
####################################################################################################################################
    NSP_muts, NSP_muts_sortByCt_Arr, NSP_muts_sortByPos_Arr, NSP_muts_no_dels_sortByCt_Arr, NSP_muts_no_dels_sortByPos_Arr, 
    NSP1_muts_sortByCt, NSP2_muts_sortByCt, NSP3_muts_sortByCt, NSP4_muts_sortByCt, NSP5_muts_sortByCt, NSP6_muts_sortByCt,
    NSP7_muts_sortByCt, NSP8_muts_sortByCt, NSP9_muts_sortByCt, NSP10_muts_sortByCt, NSP11_muts_sortByCt, NSP12_muts_sortByCt, 
    NSP13_muts_sortByCt, NSP14_muts_sortByCt, NSP15_muts_sortByCt, NSP16_muts_sortByCt, NSP1_muts_sortByPos, NSP2_muts_sortByPos, 
    NSP3_muts_sortByPos, NSP4_muts_sortByPos, NSP5_muts_sortByPos, NSP6_muts_sortByPos, NSP7_muts_sortByPos, NSP8_muts_sortByPos, 
    NSP9_muts_sortByPos, NSP10_muts_sortByPos, NSP11_muts_sortByPos, NSP12_muts_sortByPos, NSP13_muts_sortByPos, NSP14_muts_sortByPos, 
    NSP15_muts_sortByPos, NSP16_muts_sortByPos, NSP1_muts_no_dels_sortByCt, NSP2_muts_no_dels_sortByCt, NSP3_muts_no_dels_sortByCt, 
    NSP4_muts_no_dels_sortByCt, NSP5_muts_no_dels_sortByCt, NSP6_muts_no_dels_sortByCt, NSP7_muts_no_dels_sortByCt, 
    NSP8_muts_no_dels_sortByCt, NSP9_muts_no_dels_sortByCt, NSP10_muts_no_dels_sortByCt, NSP11_muts_no_dels_sortByCt, 
    NSP12_muts_no_dels_sortByCt, NSP13_muts_no_dels_sortByCt, NSP14_muts_no_dels_sortByCt, NSP15_muts_no_dels_sortByCt, 
    NSP16_muts_no_dels_sortByCt, NSP1_muts_no_dels_sortByPos, NSP2_muts_no_dels_sortByPos, NSP3_muts_no_dels_sortByPos, 
    NSP4_muts_no_dels_sortByPos, NSP5_muts_no_dels_sortByPos, NSP6_muts_no_dels_sortByPos, NSP7_muts_no_dels_sortByPos, 
    NSP8_muts_no_dels_sortByPos, NSP9_muts_no_dels_sortByPos, NSP10_muts_no_dels_sortByPos, NSP11_muts_no_dels_sortByPos, 
    NSP12_muts_no_dels_sortByPos, NSP13_muts_no_dels_sortByPos, NSP14_muts_no_dels_sortByPos, NSP15_muts_no_dels_sortByPos, 
    NSP16_muts_no_dels_sortByPos,
####################################################################################################################################
    chronic_search_muts, chronic_search_muts_v2, 
####################################################################################################################################
    avg_AA_subs_per_circ_seq, avg_nuc_subs_per_circ_seq, avg_AA_subs_per_chr_seq, avg_nuc_subs_per_chr_seq,
####################################################################################################################################
    nuc_muts_rep_seq_grps, nuc_dels_rep_seq_grps, 
    nuc_muts_rep_seq_grps_no_dels, rep_seq_grps_dels, rep_seq_grps_muts_no_dels, rep_seq_grps_del_ranges_ct, rep_seq_grps_nuc_dropout, 
    rep_seq_grps_mixed_nucs, rep_seq_grps_AA_dels, rep_seq_grps_AA_no_dels, AA_muts_rep_seq_grps, AA_dels_rep_seq_grps, 
    AA_muts_rep_seq_grps_no_dels, AA_muts_rep_seq_grps_pos_only,  
    rep_seq_grps_nuc_muts_WT, rep_seq_grps_nuc_dels_WT, rep_seq_grps_AA_muts_WT, rep_seq_grps_AA_dels_WT, nuc_muts_rep_seq_grps_WT, 
    nuc_dels_rep_seq_grps_WT, AA_muts_rep_seq_grps_WT, AA_dels_rep_seq_grps_WT, rep_seq_grps_AA_muts_WT_pos_only, 
    AA_muts_rep_seq_grps_WT_pos_only, rep_seq_grps_pango, rep_seq_grps_pango_unaliased, excluded_pos, excluded_AA, all_seqs, rep_seqs, 
    non_rep_seqs, rep_seq_grps_seqs, rep_seq_grps_muts, rep_seq_grps_clade, rep_seq_grps_AA, rep_seq_grps_AA_pos_only, 
    rep_seq_grps_AA_pos_only_no_dels, non_rep_seqs_AA, non_rep_seqs_AA_pos_only, non_rep_seqs_AA_pos_only_no_dels, 
####################################################################################################################################
    rep_seq_grps_maxmut_nuc_muts_WT, rep_seq_grps_maxmut_nuc_dels_WT, 
    rep_seq_grps_maxmut_AA_muts_WT, rep_seq_grps_maxmut_AA_dels_WT, rep_seq_grps_maxmut_AA_muts_WT_pos_only,
    rep_seq_grps_maxmut_seqs, rep_seq_grps_maxmut_nuc, rep_seq_grps_maxmut_nuc_no_dels, rep_seq_grps_maxmut_dels, 
    rep_seq_grps_maxmut_nuc_dropout, rep_seq_grps_maxmut_mixed_nucs, rep_seq_grps_maxmut_AA, rep_seq_grps_maxmut_AA_no_dels, 
    rep_seq_grps_maxmut_AA_dels, rep_seq_grps_maxmut_AA_pos_only, rep_seq_grps_maxmut_AA_pos_only_no_dels, 
    rep_seq_grps_maxmut_unknown_AA, rep_seq_grps_maxmut_unknown_AA_ranges, rep_seq_grps_maxmut_mixed_AA_muts, rep_seq_grps_maxmut_del_ranges_ct,
####################################################################################################################################
    gene_mut_density_sort_by_gene, gene_mut_density_sort_by_density, domain_mut_density_sort_by_gene, domain_mut_density_sort_by_density, 
    gene_mut_density, domain_mut_density, 
####################################################################################################################################
    all_seqs_set, all_qualifying_seqs, all_qualifying_seqs_set, all_nonqualifying_seqs, all_nonqualifying_seqs_set
####################################################################################################################################
    total_chr_AA_subs, total_nuc_revs_seq, seq_nuc_total_revs, total_AA_revs_seq, seq_AA_total_revs, seq_AA_revs
end
# REMOVED: rep_seq_grps_unknown_AA, rep_seq_grps_mixed_AA_muts,
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
function gene_sub_pos_ranks_chr(gene_array::Vector{String}, gene_AA_lengths::Dict{String, Int}, AA_muts_ct_pos_only_no_dels::Dict{String, Int} )
    ORF1a_pos_ct = Dict{String, Int}()
    ORF1b_pos_ct = Dict{String, Int}()
    S_pos_ct = Dict{String, Int}()
    ORF3a_pos_ct = Dict{String, Int}()
    E_pos_ct = Dict{String, Int}()
    M_pos_ct = Dict{String, Int}()
    ORF6_pos_ct = Dict{String, Int}()
    ORF7a_pos_ct = Dict{String, Int}()
    ORF7b_pos_ct = Dict{String, Int}()
    ORF8_pos_ct = Dict{String, Int}()
    N_pos_ct = Dict{String, Int}()
    ORF9b_pos_ct = Dict{String, Int}()
    gene_pos_ct_dict_arr = [ORF1a_pos_ct, ORF1b_pos_ct, S_pos_ct, ORF3a_pos_ct, E_pos_ct, M_pos_ct, ORF6_pos_ct, ORF7a_pos_ct, ORF7b_pos_ct, ORF8_pos_ct, N_pos_ct, ORF9b_pos_ct]
######################################################
    ORF1a_pos_ct_v1 = Dict{Int, Int}()
    ORF1b_pos_ct_v1 = Dict{Int, Int}()
    S_pos_ct_v1 = Dict{Int, Int}()
    ORF3a_pos_ct_v1 = Dict{Int, Int}()
    E_pos_ct_v1 = Dict{Int, Int}()
    M_pos_ct_v1 = Dict{Int, Int}()
    ORF6_pos_ct_v1 = Dict{Int, Int}()
    ORF7a_pos_ct_v1 = Dict{Int, Int}()
    ORF7b_pos_ct_v1 = Dict{Int, Int}()
    ORF8_pos_ct_v1 = Dict{Int, Int}()
    N_pos_ct_v1 = Dict{Int, Int}()
    ORF9b_pos_ct_v1 = Dict{Int, Int}()
    gene_pos_ct_v1_dict_arr = [ORF1a_pos_ct_v1, ORF1b_pos_ct_v1, S_pos_ct_v1, ORF3a_pos_ct_v1, E_pos_ct_v1, M_pos_ct_v1, ORF6_pos_ct_v1, ORF7a_pos_ct_v1, ORF7b_pos_ct_v1, ORF8_pos_ct_v1, N_pos_ct_v1, ORF9b_pos_ct_v1]
######################################################
    for i in 1:length(gene_pos_ct_dict_arr)
        dict = gene_pos_ct_dict_arr[i]
        dict_v1 = gene_pos_ct_v1_dict_arr[i]
        gene = gene_array[i]
        gene_len = gene_AA_lengths[gene]
        for j in 1:gene_len
            site = gene*":"*"$(j)"
            dict[site] = 0
            dict_v1[j] = 0
        end
    end
#############################################################
#############################################################
    for (mut, ct) in AA_muts_ct_pos_only_no_dels
        pos = aa_pos_comprehensive_dict[mut]
        if aa_gene_comprehensive_dict[mut] == "ORF1a"
            ORF1a_pos_ct[mut] = ct
            ORF1a_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF1b"
            ORF1b_pos_ct[mut] = ct
            ORF1b_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "S"
            S_pos_ct[mut] = ct
            S_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF3a"
            ORF3a_pos_ct[mut] = ct
            ORF3a_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "E"
            E_pos_ct[mut] = ct
            E_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "M"
            M_pos_ct[mut] = ct
            M_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF6"
            ORF6_pos_ct[mut] = ct
            ORF6_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF7a"
            ORF7a_pos_ct[mut] = ct
            ORF7a_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF7b"
            ORF7b_pos_ct[mut] = ct
            ORF7b_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF8"
            ORF8_pos_ct[mut] = ct
            ORF8_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "N"
            N_pos_ct[mut] = ct
            N_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF9b"
            ORF9b_pos_ct[mut] = ct
            ORF9b_pos_ct_v1[pos] = ct
        end
    end
######################### 
    global ORF1a_pos_ct_sort_by_pos = sort(collect(ORF1a_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF1b_pos_ct_sort_by_pos = sort(collect(ORF1b_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global S_pos_ct_sort_by_pos = sort(collect(S_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF3a_pos_ct_sort_by_pos = sort(collect(ORF3a_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global E_pos_ct_sort_by_pos = sort(collect(E_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global M_pos_ct_sort_by_pos = sort(collect(M_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF6_pos_ct_sort_by_pos = sort(collect(ORF6_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF7a_pos_ct_sort_by_pos = sort(collect(ORF7a_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF7b_pos_ct_sort_by_pos = sort(collect(ORF7b_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF8_pos_ct_sort_by_pos = sort(collect(ORF8_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global N_pos_ct_sort_by_pos = sort(collect(N_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF9b_pos_ct_sort_by_pos = sort(collect(ORF9b_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
#########################
    global ORF1a_pos_ct_sort_by_ct = sort(collect(ORF1a_pos_ct), by = x -> x[2], rev=true)
    global ORF1b_pos_ct_sort_by_ct = sort(collect(ORF1b_pos_ct), by = x -> x[2], rev=true)
    global S_pos_ct_sort_by_ct = sort(collect(S_pos_ct), by = x -> x[2], rev=true)
    global ORF3a_pos_ct_sort_by_ct = sort(collect(ORF3a_pos_ct), by = x -> x[2], rev=true)
    global E_pos_ct_sort_by_ct = sort(collect(E_pos_ct), by = x -> x[2], rev=true)
    global M_pos_ct_sort_by_ct = sort(collect(M_pos_ct), by = x -> x[2], rev=true)
    global ORF6_pos_ct_sort_by_ct = sort(collect(ORF6_pos_ct), by = x -> x[2], rev=true)
    global ORF7a_pos_ct_sort_by_ct = sort(collect(ORF7a_pos_ct), by = x -> x[2], rev=true)
    global ORF7b_pos_ct_sort_by_ct = sort(collect(ORF7b_pos_ct), by = x -> x[2], rev=true)
    global ORF8_pos_ct_sort_by_ct = sort(collect(ORF8_pos_ct), by = x -> x[2], rev=true)
    global N_pos_ct_sort_by_ct = sort(collect(N_pos_ct), by = x -> x[2], rev=true)
    global ORF9b_pos_ct_sort_by_ct = sort(collect(ORF9b_pos_ct), by = x -> x[2], rev=true)
#########################
    global ORF1a_pos_ct_sort_by_pos_v1 = sort(collect(ORF1a_pos_ct_v1), by = x -> x[1])
    global ORF1b_pos_ct_sort_by_pos_v1 = sort(collect(ORF1b_pos_ct_v1), by = x -> x[1])
    global S_pos_ct_sort_by_pos_v1 = sort(collect(S_pos_ct_v1), by = x -> x[1])
    global ORF3a_pos_ct_sort_by_pos_v1 = sort(collect(ORF3a_pos_ct_v1), by = x -> x[1])
    global E_pos_ct_sort_by_pos_v1 = sort(collect(E_pos_ct_v1), by = x -> x[1])
    global M_pos_ct_sort_by_pos_v1 = sort(collect(M_pos_ct_v1), by = x -> x[1])
    global ORF6_pos_ct_sort_by_pos_v1 = sort(collect(ORF6_pos_ct_v1), by = x -> x[1])
    global ORF7a_pos_ct_sort_by_pos_v1 = sort(collect(ORF7a_pos_ct_v1), by = x -> x[1])
    global ORF7b_pos_ct_sort_by_pos_v1 = sort(collect(ORF7b_pos_ct_v1), by = x -> x[1])
    global ORF8_pos_ct_sort_by_pos_v1 = sort(collect(ORF8_pos_ct_v1), by = x -> x[1])
    global N_pos_ct_sort_by_pos_v1 = sort(collect(N_pos_ct_v1), by = x -> x[1])
    global ORF9b_pos_ct_sort_by_pos_v1 = sort(collect(ORF9b_pos_ct_v1), by = x -> x[1])
#########################
    global ORF1a_pos_ct_sort_by_ct_v1 = sort(collect(ORF1a_pos_ct_v1), by = x -> x[2], rev=true)
    global ORF1b_pos_ct_sort_by_ct_v1 = sort(collect(ORF1b_pos_ct_v1), by = x -> x[2], rev=true)
    global S_pos_ct_sort_by_ct_v1 = sort(collect(S_pos_ct_v1), by = x -> x[2], rev=true)
    global ORF3a_pos_ct_sort_by_ct_v1 = sort(collect(ORF3a_pos_ct_v1), by = x -> x[2], rev=true)
    global E_pos_ct_sort_by_ct_v1 = sort(collect(E_pos_ct_v1), by = x -> x[2], rev=true)
    global M_pos_ct_sort_by_ct_v1 = sort(collect(M_pos_ct_v1), by = x -> x[2], rev=true)
    global ORF6_pos_ct_sort_by_ct_v1 = sort(collect(ORF6_pos_ct_v1), by = x -> x[2], rev=true)
    global ORF7a_pos_ct_sort_by_ct_v1 = sort(collect(ORF7a_pos_ct_v1), by = x -> x[2], rev=true)
    global ORF7b_pos_ct_sort_by_ct_v1 = sort(collect(ORF7b_pos_ct_v1), by = x -> x[2], rev=true)
    global ORF8_pos_ct_sort_by_ct_v1 = sort(collect(ORF8_pos_ct_v1), by = x -> x[2], rev=true)
    global N_pos_ct_sort_by_ct_v1 = sort(collect(N_pos_ct_v1), by = x -> x[2], rev=true)
    global ORF9b_pos_ct_sort_by_ct_v1 = sort(collect(ORF9b_pos_ct_v1), by = x -> x[2], rev=true)
end
#####################################################################################################################################
#####################################################################################################################################
#####################################################################################################################################
function gene_sub_ranks_chr(gene_array::Vector{String}, sub_types_at_every_site_combined::Dict{String, Dict{Int, Set{String}}}, AA_muts_ct_no_dels::Dict{String, Int} )
    ORF1a_ct = Dict{String, Int}()
    ORF1b_ct = Dict{String, Int}()
    S_ct = Dict{String, Int}()
    ORF3a_ct = Dict{String, Int}()
    E_ct = Dict{String, Int}()
    M_ct = Dict{String, Int}()
    ORF6_ct = Dict{String, Int}()
    ORF7a_ct = Dict{String, Int}()
    ORF7b_ct = Dict{String, Int}()
    ORF8_ct = Dict{String, Int}()
    N_ct = Dict{String, Int}()
    ORF9b_ct = Dict{String, Int}()
#############################################################
    gene_ct_dict_arr = [ORF1a_ct, ORF1b_ct, S_ct, ORF3a_ct, E_ct, M_ct, ORF6_ct, ORF7a_ct, ORF7b_ct, ORF8_ct, N_ct, ORF9b_ct] 
#############################################################
#                                        Gene      AAsite  all_sub_types_at_AAsite (e.g., AV, AT, TA, etc)
# sub_types_at_every_site_combined = Dict{String, Dict{Int, Set{String}}}()
    for i in 1:length(gene_array)
        gene = gene_array[i]
        gene_ct_dict = gene_ct_dict_arr[i]
        for (AAsite, mut_type_set) in sub_types_at_every_site_combined[gene]
            for mut_type in mut_type_set
                sub = gene*":"*string(mut_type[1])*"$(AAsite)"*string(mut_type[end])
                gene_ct_dict[sub] = 0
            end
        end
    end
#############################################################
    for (mut, ct) in AA_muts_ct_no_dels
        if aa_gene_comprehensive_dict[mut] == "ORF1a"
            ORF1a_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF1b"
            ORF1b_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "S"
            S_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF3a"
            ORF3a_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "E"
            E_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "M"
            M_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF6"
            ORF6_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF7a"
            ORF7a_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF7b"
            ORF7b_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF8"
            ORF8_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "N"
            N_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF9b"
            ORF9b_ct[mut] = ct
        end
    end
    fin_sortkey(m) = (aa_pos_comprehensive_dict[m], refAA_comprehensive_dict[m]*qryAA_comprehensive_dict[m])
########################################## 
    global ORF1a_ct_sort_by_pos = sort(collect(ORF1a_ct), by = x -> fin_sortkey(x[1]))
    global ORF1b_ct_sort_by_pos = sort(collect(ORF1b_ct), by = x -> fin_sortkey(x[1]))
    global S_ct_sort_by_pos = sort(collect(S_ct), by = x -> fin_sortkey(x[1]))
    global ORF3a_ct_sort_by_pos = sort(collect(ORF3a_ct), by = x -> fin_sortkey(x[1]))
    global E_ct_sort_by_pos = sort(collect(E_ct), by = x -> fin_sortkey(x[1]))
    global M_ct_sort_by_pos = sort(collect(M_ct), by = x -> fin_sortkey(x[1]))
    global ORF6_ct_sort_by_pos = sort(collect(ORF6_ct), by = x -> fin_sortkey(x[1]))
    global ORF7a_ct_sort_by_pos = sort(collect(ORF7a_ct), by = x -> fin_sortkey(x[1]))
    global ORF7b_ct_sort_by_pos = sort(collect(ORF7b_ct), by = x -> fin_sortkey(x[1]))
    global ORF8_ct_sort_by_pos = sort(collect(ORF8_ct), by = x -> fin_sortkey(x[1]))
    global N_ct_sort_by_pos = sort(collect(N_ct), by = x -> fin_sortkey(x[1]))
    global ORF9b_ct_sort_by_pos = sort(collect(ORF9b_ct), by = x -> fin_sortkey(x[1]))
##########################################
    global ORF1a_ct_sort_by_ct = sort(collect(ORF1a_ct), by = x -> x[2], rev=true)
    global ORF1b_ct_sort_by_ct = sort(collect(ORF1b_ct), by = x -> x[2], rev=true)
    global S_ct_sort_by_ct = sort(collect(S_ct), by = x -> x[2], rev=true)
    global ORF3a_ct_sort_by_ct = sort(collect(ORF3a_ct), by = x -> x[2], rev=true)
    global E_ct_sort_by_ct = sort(collect(E_ct), by = x -> x[2], rev=true)
    global M_ct_sort_by_ct = sort(collect(M_ct), by = x -> x[2], rev=true)
    global ORF6_ct_sort_by_ct = sort(collect(ORF6_ct), by = x -> x[2], rev=true)
    global ORF7a_ct_sort_by_ct = sort(collect(ORF7a_ct), by = x -> x[2], rev=true)
    global ORF7b_ct_sort_by_ct = sort(collect(ORF7b_ct), by = x -> x[2], rev=true)
    global ORF8_ct_sort_by_ct = sort(collect(ORF8_ct), by = x -> x[2], rev=true)
    global N_ct_sort_by_ct = sort(collect(N_ct), by = x -> x[2], rev=true)
    global ORF9b_ct_sort_by_ct = sort(collect(ORF9b_ct), by = x -> x[2], rev=true)
end
#####################################################################################################################################
function NSP_sub_ranks_chr(NSP_array::Vector{String}, NSP_sub_types_at_every_site_combined::Dict{String, Dict{Int, Set{String}}}, AA_muts_ct_no_dels::Dict{String, Int} )
    NSP1_ct = Dict{String, Int}()
    NSP2_ct = Dict{String, Int}()
    NSP3_ct = Dict{String, Int}()
    NSP4_ct = Dict{String, Int}()
    NSP5_ct = Dict{String, Int}()
    NSP6_ct = Dict{String, Int}()
    NSP7_ct = Dict{String, Int}()
    NSP8_ct = Dict{String, Int}()
    NSP9_ct = Dict{String, Int}()
    NSP10_ct = Dict{String, Int}()
    NSP11_ct = Dict{String, Int}()
    NSP12_ct = Dict{String, Int}()
    NSP13_ct = Dict{String, Int}()
    NSP14_ct = Dict{String, Int}()
    NSP15_ct = Dict{String, Int}()
    NSP16_ct = Dict{String, Int}()
#############################################################
    NSP_ct_dict_arr = [NSP1_ct, NSP2_ct, NSP3_ct, NSP4_ct, NSP5_ct, NSP6_ct, NSP7_ct, NSP8_ct, NSP9_ct, NSP10_ct, NSP11_ct, NSP12_ct, NSP13_ct, NSP14_ct, NSP15_ct, NSP16_ct]
#                                              NSP       AAsite  all_sub_types_at_AAsite (e.g., AV, AT, TA, etc)
# NSP_sub_types_at_every_site_combined = Dict{String, Dict{Int, Set{String}}}()   
    for i in 1:length(NSP_array)
        NSP = NSP_array[i]
        NSP_ct_dict = NSP_ct_dict_arr[i]
        for (AAsite, mut_type_set) in NSP_sub_types_at_every_site_combined[NSP]
            for mut_type in mut_type_set
                sub = NSP*":"*string(mut_type[1])*"$(AAsite)"*string(mut_type[end])
                NSP_ct_dict[sub] = 0
            end
        end
    end   
    for (sub, ct) in AA_muts_ct_no_dels
        gene = aa_gene_comprehensive_dict[sub]
        if gene == "ORF1a" || gene == "ORF1b"
            AA_site = aa_pos_comprehensive_dict[sub]
            if AA_site < 4402
                NSP_sub = ORF1abMut_to_NSP(sub)
                NSP = NSP_muts_gene_dict[NSP_sub]
                NSP_pos = NSP_muts_pos_dict[NSP_sub]
                NSP_num = parse(Int, split(NSP, "P")[2])
                NSP_dict = NSP_ct_dict_arr[NSP_num]
                NSP_dict[NSP_sub] = ct
            end
        end
    end
    fin_sortkey(m) = (NSP_muts_pos_dict[m], NSP_ref_AA_dict[m]*NSP_qry_AA_dict[m])
#############################
    global NSP1_ct_sort_by_pos = sort(collect(NSP1_ct), by = x -> fin_sortkey(x[1]))
    global NSP2_ct_sort_by_pos = sort(collect(NSP2_ct), by = x -> fin_sortkey(x[1]))
    global NSP3_ct_sort_by_pos = sort(collect(NSP3_ct), by = x -> fin_sortkey(x[1]))
    global NSP4_ct_sort_by_pos = sort(collect(NSP4_ct), by = x -> fin_sortkey(x[1]))
    global NSP5_ct_sort_by_pos = sort(collect(NSP5_ct), by = x -> fin_sortkey(x[1]))
    global NSP6_ct_sort_by_pos = sort(collect(NSP6_ct), by = x -> fin_sortkey(x[1]))
    global NSP7_ct_sort_by_pos = sort(collect(NSP7_ct), by = x -> fin_sortkey(x[1]))
    global NSP8_ct_sort_by_pos = sort(collect(NSP8_ct), by = x -> fin_sortkey(x[1]))
    global NSP9_ct_sort_by_pos = sort(collect(NSP9_ct), by = x -> fin_sortkey(x[1]))
    global NSP10_ct_sort_by_pos = sort(collect(NSP10_ct), by = x -> fin_sortkey(x[1]))
    global NSP11_ct_sort_by_pos = sort(collect(NSP11_ct), by = x -> fin_sortkey(x[1]))
    global NSP12_ct_sort_by_pos = sort(collect(NSP12_ct), by = x -> fin_sortkey(x[1]))
    global NSP13_ct_sort_by_pos = sort(collect(NSP13_ct), by = x -> fin_sortkey(x[1]))
    global NSP14_ct_sort_by_pos = sort(collect(NSP14_ct), by = x -> fin_sortkey(x[1]))
    global NSP15_ct_sort_by_pos = sort(collect(NSP15_ct), by = x -> fin_sortkey(x[1]))
    global NSP16_ct_sort_by_pos = sort(collect(NSP16_ct), by = x -> fin_sortkey(x[1]))
#############################
    global NSP1_ct_sort_by_ct = sort(collect(NSP1_ct), by = x -> x[2], rev=true)
    global NSP2_ct_sort_by_ct = sort(collect(NSP2_ct), by = x -> x[2], rev=true)
    global NSP3_ct_sort_by_ct = sort(collect(NSP3_ct), by = x -> x[2], rev=true)
    global NSP4_ct_sort_by_ct = sort(collect(NSP4_ct), by = x -> x[2], rev=true)
    global NSP5_ct_sort_by_ct = sort(collect(NSP5_ct), by = x -> x[2], rev=true)
    global NSP6_ct_sort_by_ct = sort(collect(NSP6_ct), by = x -> x[2], rev=true)
    global NSP7_ct_sort_by_ct = sort(collect(NSP7_ct), by = x -> x[2], rev=true)
    global NSP8_ct_sort_by_ct = sort(collect(NSP8_ct), by = x -> x[2], rev=true)
    global NSP9_ct_sort_by_ct = sort(collect(NSP9_ct), by = x -> x[2], rev=true)
    global NSP10_ct_sort_by_ct = sort(collect(NSP10_ct), by = x -> x[2], rev=true)
    global NSP11_ct_sort_by_ct = sort(collect(NSP11_ct), by = x -> x[2], rev=true)
    global NSP12_ct_sort_by_ct = sort(collect(NSP12_ct), by = x -> x[2], rev=true)
    global NSP13_ct_sort_by_ct = sort(collect(NSP13_ct), by = x -> x[2], rev=true)
    global NSP14_ct_sort_by_ct = sort(collect(NSP14_ct), by = x -> x[2], rev=true)
    global NSP15_ct_sort_by_ct = sort(collect(NSP15_ct), by = x -> x[2], rev=true)
    global NSP16_ct_sort_by_ct = sort(collect(NSP16_ct), by = x -> x[2], rev=true)
end
#####################################################################################################################################
function NSP_sub_pos_ranks_chr(NSP_array::Vector{String}, NSP_sub_types_at_every_site_combined::Dict{String, Dict{Int, Set{String}}}, AA_muts_ct_pos_only_no_dels::Dict{String, Int} )
    NSP1_pos_ct = Dict{String, Int}()
    NSP2_pos_ct = Dict{String, Int}()
    NSP3_pos_ct = Dict{String, Int}()
    NSP4_pos_ct = Dict{String, Int}()
    NSP5_pos_ct = Dict{String, Int}()
    NSP6_pos_ct = Dict{String, Int}()
    NSP7_pos_ct = Dict{String, Int}()
    NSP8_pos_ct = Dict{String, Int}()
    NSP9_pos_ct = Dict{String, Int}()
    NSP10_pos_ct = Dict{String, Int}()
    NSP11_pos_ct = Dict{String, Int}()
    NSP12_pos_ct = Dict{String, Int}()
    NSP13_pos_ct = Dict{String, Int}()
    NSP14_pos_ct = Dict{String, Int}()
    NSP15_pos_ct = Dict{String, Int}()
    NSP16_pos_ct = Dict{String, Int}()
    NSP_pos_ct_array = [NSP1_pos_ct, NSP2_pos_ct, NSP3_pos_ct, NSP4_pos_ct, NSP5_pos_ct, NSP6_pos_ct, NSP7_pos_ct, NSP8_pos_ct, NSP9_pos_ct, NSP10_pos_ct, NSP11_pos_ct, NSP12_pos_ct, NSP13_pos_ct, NSP14_pos_ct, NSP15_pos_ct, NSP16_pos_ct]
##########################################
    NSP1_pos_ct_v1 = Dict{Int, Int}()
    NSP2_pos_ct_v1 = Dict{Int, Int}()
    NSP3_pos_ct_v1 = Dict{Int, Int}()
    NSP4_pos_ct_v1 = Dict{Int, Int}()
    NSP5_pos_ct_v1 = Dict{Int, Int}()
    NSP6_pos_ct_v1 = Dict{Int, Int}()
    NSP7_pos_ct_v1 = Dict{Int, Int}()
    NSP8_pos_ct_v1 = Dict{Int, Int}()
    NSP9_pos_ct_v1 = Dict{Int, Int}()
    NSP10_pos_ct_v1 = Dict{Int, Int}()
    NSP11_pos_ct_v1 = Dict{Int, Int}()
    NSP12_pos_ct_v1 = Dict{Int, Int}()
    NSP13_pos_ct_v1 = Dict{Int, Int}()
    NSP14_pos_ct_v1 = Dict{Int, Int}()
    NSP15_pos_ct_v1 = Dict{Int, Int}()
    NSP16_pos_ct_v1 = Dict{Int, Int}()
    NSP_pos_ct_v1_array = [NSP1_pos_ct_v1, NSP2_pos_ct_v1, NSP3_pos_ct_v1, NSP4_pos_ct_v1, NSP5_pos_ct_v1, NSP6_pos_ct_v1, NSP7_pos_ct_v1, NSP8_pos_ct_v1, NSP9_pos_ct_v1, NSP10_pos_ct_v1, NSP11_pos_ct_v1, NSP12_pos_ct_v1, NSP13_pos_ct_v1, NSP14_pos_ct_v1, NSP15_pos_ct_v1, NSP16_pos_ct_v1]
#######################################################
    NSP_pos_ct = Dict{String, Int}()
    for i in 1:length(NSP_pos_ct_array)
        NSP_dict = NSP_pos_ct_array[i]
        NSP_dict_v1 = NSP_pos_ct_v1_array[i]
        NSP = NSP_array[i]
        NSP_len = NSP_AA_size[NSP]
        for j in 1:NSP_len
            NSP_pos = NSP*":"*"$(j)"
            NSP_dict[NSP_pos] = 0
            NSP_dict_v1[j] = 0
            NSP_pos_ct[NSP_pos] = 0
        end
    end
##########################################################################################################################
##########################################################################################################################
    NSP1_ct = Dict{String, Int}()
    NSP2_ct = Dict{String, Int}()
    NSP3_ct = Dict{String, Int}()
    NSP4_ct = Dict{String, Int}()
    NSP5_ct = Dict{String, Int}()
    NSP6_ct = Dict{String, Int}()
    NSP7_ct = Dict{String, Int}()
    NSP8_ct = Dict{String, Int}()
    NSP9_ct = Dict{String, Int}()
    NSP10_ct = Dict{String, Int}()
    NSP11_ct = Dict{String, Int}()
    NSP12_ct = Dict{String, Int}()
    NSP13_ct = Dict{String, Int}()
    NSP14_ct = Dict{String, Int}()
    NSP15_ct = Dict{String, Int}()
    NSP16_ct = Dict{String, Int}()
#############################################################
    NSP_ct_dict_arr = [NSP1_ct, NSP2_ct, NSP3_ct, NSP4_ct, NSP5_ct, NSP6_ct, NSP7_ct, NSP8_ct, NSP9_ct, NSP10_ct, NSP11_ct, NSP12_ct, NSP13_ct, NSP14_ct, NSP15_ct, NSP16_ct]
    for i in 1:length(NSP_array)
        NSP = NSP_array[i]
        NSP_ct_dict = NSP_ct_dict_arr[i]
        for (AAsite, mut_type_set) in NSP_sub_types_at_every_site_combined[NSP]
            for mut_type in mut_type_set
                sub = NSP*":"*string(mut_type[1])*"$(AAsite)"*string(mut_type[end])
                NSP_ct_dict[sub] = 0
            end
        end
    end   
    for (sub, ct) in AA_muts_ct_no_dels
        gene = aa_gene_comprehensive_dict[sub]
        if gene == "ORF1a" || gene == "ORF1b"
            NSP_sub = ORF1abMut_to_NSP(sub)
            NSP = NSP_muts_gene_dict[NSP_sub]
            NSP_pos = NSP_muts_pos_dict[NSP_sub]
            NSP_num = parse(Int, split(NSP, "P")[2])
            NSP_dict = NSP_ct_dict_arr[NSP_num]
            NSP_dict[NSP_sub] = ct
        end
    end
##########################################################################################################################
##########################################################################################################################
    for i in 1:length(NSP_ct_dict_arr)
        NSP = NSP_array[i]
        NSP_dict = NSP_ct_dict_arr[i]
        NSP_pos_dict = NSP_pos_ct_array[i]
        NSP_pos_dict_v1 = NSP_pos_ct_v1_array[i]
        for (sub, ct) in NSP_dict
            pos = NSP_muts_pos_dict[sub]
            sub_pos = NSP*":"*"$(pos)"
            NSP_pos_dict[sub_pos] += ct
            NSP_pos_dict_v1[pos] += ct
            NSP_pos_ct[sub_pos] += ct
        end
    end
    global NSP1_pos_ct_sort_by_pos = sort(collect(NSP1_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP2_pos_ct_sort_by_pos = sort(collect(NSP2_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP3_pos_ct_sort_by_pos = sort(collect(NSP3_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP4_pos_ct_sort_by_pos = sort(collect(NSP4_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP5_pos_ct_sort_by_pos = sort(collect(NSP5_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP6_pos_ct_sort_by_pos = sort(collect(NSP6_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP7_pos_ct_sort_by_pos = sort(collect(NSP7_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP8_pos_ct_sort_by_pos = sort(collect(NSP8_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP9_pos_ct_sort_by_pos = sort(collect(NSP9_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP10_pos_ct_sort_by_pos = sort(collect(NSP10_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP11_pos_ct_sort_by_pos = sort(collect(NSP11_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP12_pos_ct_sort_by_pos = sort(collect(NSP12_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP13_pos_ct_sort_by_pos = sort(collect(NSP13_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP14_pos_ct_sort_by_pos = sort(collect(NSP14_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP15_pos_ct_sort_by_pos = sort(collect(NSP15_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP16_pos_ct_sort_by_pos = sort(collect(NSP16_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
#########################
    global NSP1_pos_ct_sort_by_ct = sort(collect(NSP1_pos_ct), by = x -> x[2], rev=true)
    global NSP2_pos_ct_sort_by_ct = sort(collect(NSP2_pos_ct), by = x -> x[2], rev=true)
    global NSP3_pos_ct_sort_by_ct = sort(collect(NSP3_pos_ct), by = x -> x[2], rev=true)
    global NSP4_pos_ct_sort_by_ct = sort(collect(NSP4_pos_ct), by = x -> x[2], rev=true)
    global NSP5_pos_ct_sort_by_ct = sort(collect(NSP5_pos_ct), by = x -> x[2], rev=true)
    global NSP6_pos_ct_sort_by_ct = sort(collect(NSP6_pos_ct), by = x -> x[2], rev=true)
    global NSP7_pos_ct_sort_by_ct = sort(collect(NSP7_pos_ct), by = x -> x[2], rev=true)
    global NSP8_pos_ct_sort_by_ct = sort(collect(NSP8_pos_ct), by = x -> x[2], rev=true)
    global NSP9_pos_ct_sort_by_ct = sort(collect(NSP9_pos_ct), by = x -> x[2], rev=true)
    global NSP10_pos_ct_sort_by_ct = sort(collect(NSP10_pos_ct), by = x -> x[2], rev=true)
    global NSP11_pos_ct_sort_by_ct = sort(collect(NSP11_pos_ct), by = x -> x[2], rev=true)
    global NSP12_pos_ct_sort_by_ct = sort(collect(NSP12_pos_ct), by = x -> x[2], rev=true)
    global NSP13_pos_ct_sort_by_ct = sort(collect(NSP13_pos_ct), by = x -> x[2], rev=true)
    global NSP14_pos_ct_sort_by_ct = sort(collect(NSP14_pos_ct), by = x -> x[2], rev=true)
    global NSP15_pos_ct_sort_by_ct = sort(collect(NSP15_pos_ct), by = x -> x[2], rev=true)
    global NSP16_pos_ct_sort_by_ct = sort(collect(NSP16_pos_ct), by = x -> x[2], rev=true)
#########################
    global NSP1_pos_ct_sort_by_pos_v1 = sort(collect(NSP1_pos_ct_v1), by = x -> x[1])
    global NSP2_pos_ct_sort_by_pos_v1 = sort(collect(NSP2_pos_ct_v1), by = x -> x[1])
    global NSP3_pos_ct_sort_by_pos_v1 = sort(collect(NSP3_pos_ct_v1), by = x -> x[1])
    global NSP4_pos_ct_sort_by_pos_v1 = sort(collect(NSP4_pos_ct_v1), by = x -> x[1])
    global NSP5_pos_ct_sort_by_pos_v1 = sort(collect(NSP5_pos_ct_v1), by = x -> x[1])
    global NSP6_pos_ct_sort_by_pos_v1 = sort(collect(NSP6_pos_ct_v1), by = x -> x[1])
    global NSP7_pos_ct_sort_by_pos_v1 = sort(collect(NSP7_pos_ct_v1), by = x -> x[1])
    global NSP8_pos_ct_sort_by_pos_v1 = sort(collect(NSP8_pos_ct_v1), by = x -> x[1])
    global NSP9_pos_ct_sort_by_pos_v1 = sort(collect(NSP9_pos_ct_v1), by = x -> x[1])
    global NSP10_pos_ct_sort_by_pos_v1 = sort(collect(NSP10_pos_ct_v1), by = x -> x[1])
    global NSP11_pos_ct_sort_by_pos_v1 = sort(collect(NSP11_pos_ct_v1), by = x -> x[1])
    global NSP12_pos_ct_sort_by_pos_v1 = sort(collect(NSP12_pos_ct_v1), by = x -> x[1])
    global NSP13_pos_ct_sort_by_pos_v1 = sort(collect(NSP13_pos_ct_v1), by = x -> x[1])
    global NSP14_pos_ct_sort_by_pos_v1 = sort(collect(NSP14_pos_ct_v1), by = x -> x[1])
    global NSP15_pos_ct_sort_by_pos_v1 = sort(collect(NSP15_pos_ct_v1), by = x -> x[1])
    global NSP16_pos_ct_sort_by_pos_v1 = sort(collect(NSP16_pos_ct_v1), by = x -> x[1])
end
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
function gene_sub_ranks_all(gene_array::Vector{String}, sub_types_at_every_site_combined::Dict{String, Dict{Int, Set{String}}}, AA_muts_ct_no_dels_all::Dict{String, Int} )
    ORF1a_ct_all = Dict{String, Int}()
    ORF1b_ct_all = Dict{String, Int}()
    S_ct_all = Dict{String, Int}()
    ORF3a_ct_all = Dict{String, Int}()
    E_ct_all = Dict{String, Int}()
    M_ct_all = Dict{String, Int}()
    ORF6_ct_all = Dict{String, Int}()
    ORF7a_ct_all = Dict{String, Int}()
    ORF7b_ct_all = Dict{String, Int}()
    ORF8_ct_all = Dict{String, Int}()
    N_ct_all = Dict{String, Int}()
    ORF9b_ct_all = Dict{String, Int}()
#############################################################
    #                                        Gene      AAsite  all_sub_types_at_AAsite (e.g., AV, AT, TA, etc)
# sub_types_at_every_site_combined = Dict{String, Dict{Int, Set{String}}}()
    gene_ct_dict_all_arr = [ORF1a_ct_all, ORF1b_ct_all, S_ct_all, ORF3a_ct_all, E_ct_all, M_ct_all, ORF6_ct_all, ORF7a_ct_all, ORF7b_ct_all, ORF8_ct_all, N_ct_all, ORF9b_ct_all]
    for i in 1:length(gene_array)
        gene = gene_array[i]
        gene_ct_dict = gene_ct_dict_all_arr[i]
        for (AAsite, mut_type_set) in sub_types_at_every_site_combined[gene]
            for mut_type in mut_type_set
                sub = gene*":"*string(mut_type[1])*"$(AAsite)"*string(mut_type[end])
                gene_ct_dict[sub] = 0
            end
        end
    end
#############################################################
    for (mut, ct) in AA_muts_ct_no_dels_all
        if aa_gene_comprehensive_dict[mut] == "ORF1a"
            ORF1a_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF1b"
            ORF1b_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "S"
            S_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF3a"
            ORF3a_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "E"
            E_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "M"
            M_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF6"
            ORF6_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF7a"
            ORF7a_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF7b"
            ORF7b_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF8"
            ORF8_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "N"
            N_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF9b"
            ORF9b_ct_all[mut] = ct
        end
    end
    fin_sortkey(m) = (aa_pos_comprehensive_dict[m], refAA_comprehensive_dict[m]*qryAA_comprehensive_dict[m])
    global ORF1a_ct_all_sort_by_pos = sort(collect(ORF1a_ct_all), by = x -> fin_sortkey(x[1]))
    global ORF1b_ct_all_sort_by_pos = sort(collect(ORF1b_ct_all), by = x -> fin_sortkey(x[1]))
    global S_ct_all_sort_by_pos = sort(collect(S_ct_all), by = x -> fin_sortkey(x[1]))
    global ORF3a_ct_all_sort_by_pos = sort(collect(ORF3a_ct_all), by = x -> fin_sortkey(x[1]))
    global E_ct_all_sort_by_pos = sort(collect(E_ct_all), by = x -> fin_sortkey(x[1]))
    global M_ct_all_sort_by_pos = sort(collect(M_ct_all), by = x -> fin_sortkey(x[1]))
    global ORF6_ct_all_sort_by_pos = sort(collect(ORF6_ct_all), by = x -> fin_sortkey(x[1]))
    global ORF7a_ct_all_sort_by_pos = sort(collect(ORF7a_ct_all), by = x -> fin_sortkey(x[1]))
    global ORF7b_ct_all_sort_by_pos = sort(collect(ORF7b_ct_all), by = x -> fin_sortkey(x[1]))
    global ORF8_ct_all_sort_by_pos = sort(collect(ORF8_ct_all), by = x -> fin_sortkey(x[1]))
    global N_ct_all_sort_by_pos = sort(collect(N_ct_all), by = x -> fin_sortkey(x[1]))
    global ORF9b_ct_all_sort_by_pos = sort(collect(ORF9b_ct_all), by = x -> fin_sortkey(x[1]))

    global ORF1b_ct_all_sort_by_ct = sort(collect(ORF1b_ct_all), by = x -> x[2], rev=true)
    global ORF1a_ct_all_sort_by_ct = sort(collect(ORF1a_ct_all), by = x -> x[2], rev=true)
    global S_ct_all_sort_by_ct = sort(collect(S_ct_all), by = x -> x[2], rev=true)
    global ORF3a_ct_all_sort_by_ct = sort(collect(ORF3a_ct_all), by = x -> x[2], rev=true)
    global E_ct_all_sort_by_ct = sort(collect(E_ct_all), by = x -> x[2], rev=true)
    global M_ct_all_sort_by_ct = sort(collect(M_ct_all), by = x -> x[2], rev=true)
    global ORF6_ct_all_sort_by_ct = sort(collect(ORF6_ct_all), by = x -> x[2], rev=true)
    global ORF7a_ct_all_sort_by_ct = sort(collect(ORF7a_ct_all), by = x -> x[2], rev=true)
    global ORF7b_ct_all_sort_by_ct = sort(collect(ORF7b_ct_all), by = x -> x[2], rev=true)
    global ORF8_ct_all_sort_by_ct = sort(collect(ORF8_ct_all), by = x -> x[2], rev=true)
    global N_ct_all_sort_by_ct = sort(collect(N_ct_all), by = x -> x[2], rev=true)
    global ORF9b_ct_all_sort_by_ct = sort(collect(ORF9b_ct_all), by = x -> x[2], rev=true)
end
###########################################################################################################################################################################
###########################################################################################################################################################################
function gene_sub_pos_ranks_all(gene_array::Vector{String}, gene_AA_lengths::Dict{String, Int},  AA_muts_ct_pos_only_no_dels_all::Dict{String, Int})
    ORF1a_pos_ct_all = Dict{String, Int}()
    ORF1b_pos_ct_all = Dict{String, Int}()
    S_pos_ct_all = Dict{String, Int}()
    ORF3a_pos_ct_all = Dict{String, Int}()
    E_pos_ct_all = Dict{String, Int}()
    M_pos_ct_all = Dict{String, Int}()
    ORF6_pos_ct_all = Dict{String, Int}()
    ORF7a_pos_ct_all = Dict{String, Int}()
    ORF7b_pos_ct_all = Dict{String, Int}()
    ORF8_pos_ct_all = Dict{String, Int}()
    N_pos_ct_all = Dict{String, Int}()
    ORF9b_pos_ct_all = Dict{String, Int}()
    gene_ct_pos_all_dict_arr = [ORF1a_pos_ct_all, ORF1b_pos_ct_all, S_pos_ct_all, ORF3a_pos_ct_all, E_pos_ct_all, M_pos_ct_all, ORF6_pos_ct_all, ORF7a_pos_ct_all, ORF7b_pos_ct_all, ORF8_pos_ct_all, N_pos_ct_all, ORF9b_pos_ct_all] 
###################################################### 
    ORF1a_pos_ct_all_v1 = Dict{Int, Int}()
    ORF1b_pos_ct_all_v1 = Dict{Int, Int}()
    S_pos_ct_all_v1 = Dict{Int, Int}()
    ORF3a_pos_ct_all_v1 = Dict{Int, Int}()
    E_pos_ct_all_v1 = Dict{Int, Int}()
    M_pos_ct_all_v1 = Dict{Int, Int}()
    ORF6_pos_ct_all_v1 = Dict{Int, Int}()
    ORF7a_pos_ct_all_v1 = Dict{Int, Int}()
    ORF7b_pos_ct_all_v1 = Dict{Int, Int}()
    ORF8_pos_ct_all_v1 = Dict{Int, Int}()
    N_pos_ct_all_v1 = Dict{Int, Int}()
    ORF9b_pos_ct_all_v1 = Dict{Int, Int}()
    gene_pos_ct_all_v1_dict_arr = [ORF1a_pos_ct_all_v1, ORF1b_pos_ct_all_v1, S_pos_ct_all_v1, ORF3a_pos_ct_all_v1, E_pos_ct_all_v1, M_pos_ct_all_v1, ORF6_pos_ct_all_v1, ORF7a_pos_ct_all_v1, ORF7b_pos_ct_all_v1, ORF8_pos_ct_all_v1, N_pos_ct_all_v1, ORF9b_pos_ct_all_v1]
###################################################### 
    for i in 1:length(gene_ct_pos_all_dict_arr)
        dict = gene_ct_pos_all_dict_arr[i]
        dict_v1 = gene_pos_ct_all_v1_dict_arr[i]
        gene = gene_array[i]
        gene_len = gene_AA_lengths[gene]
        for j in 1:gene_len
            site = gene*":"*"$(j)"
            dict[site] = 0
            dict_v1[j] = 0
        end
    end
#############################################################
    for (mut, ct) in AA_muts_ct_pos_only_no_dels_all
        pos = aa_pos_comprehensive_dict[mut]
        if aa_gene_comprehensive_dict[mut] == "ORF1a"
            ORF1a_pos_ct_all[mut] = ct
            ORF1a_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF1b"
            ORF1b_pos_ct_all[mut] = ct
            ORF1b_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "S"
            S_pos_ct_all[mut] = ct
            S_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF3a"
            ORF3a_pos_ct_all[mut] = ct
            ORF3a_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "E"
            E_pos_ct_all[mut] = ct
            E_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "M"
            M_pos_ct_all[mut] = ct
            M_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF6"
            ORF6_pos_ct_all[mut] = ct
            ORF6_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF7a"
            ORF7a_pos_ct_all[mut] = ct
            ORF7a_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF7b"
            ORF7b_pos_ct_all[mut] = ct
            ORF7b_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF8"
            ORF8_pos_ct_all[mut] = ct
            ORF8_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "N"
            N_pos_ct_all[mut] = ct
            N_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF9b"
            ORF9b_pos_ct_all[mut] = ct
            ORF9b_pos_ct_all_v1[pos] = ct
        end
    end
    global ORF1a_pos_ct_all_sort_by_pos = sort(collect(ORF1a_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF1b_pos_ct_all_sort_by_pos = sort(collect(ORF1b_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])
    global S_pos_ct_all_sort_by_pos = sort(collect(S_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF3a_pos_ct_all_sort_by_pos = sort(collect(ORF3a_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])
    global E_pos_ct_all_sort_by_pos = sort(collect(E_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]]) 
    global M_pos_ct_all_sort_by_pos = sort(collect(M_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF6_pos_ct_all_sort_by_pos = sort(collect(ORF6_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF7a_pos_ct_all_sort_by_pos = sort(collect(ORF7a_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF7b_pos_ct_all_sort_by_pos = sort(collect(ORF7b_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF8_pos_ct_all_sort_by_pos = sort(collect(ORF8_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])
    global N_pos_ct_all_sort_by_pos = sort(collect(N_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])   
    global ORF9b_pos_ct_all_sort_by_pos = sort(collect(ORF9b_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])
################################## 
    global ORF1a_pos_ct_all_sort_by_ct = sort(collect(ORF1a_pos_ct_all), by = x -> x[2], rev=true)
    global ORF1b_pos_ct_all_sort_by_ct = sort(collect(ORF1b_pos_ct_all), by = x -> x[2], rev=true)
    global S_pos_ct_all_sort_by_ct = sort(collect(S_pos_ct_all), by = x -> x[2], rev=true)
    global ORF3a_pos_ct_all_sort_by_ct = sort(collect(ORF3a_pos_ct_all), by = x -> x[2], rev=true)
    global E_pos_ct_all_sort_by_ct = sort(collect(E_pos_ct_all), by = x -> x[2], rev=true)
    global M_pos_ct_all_sort_by_ct = sort(collect(M_pos_ct_all), by = x -> x[2], rev=true)
    global ORF6_pos_ct_all_sort_by_ct = sort(collect(ORF6_pos_ct_all), by = x -> x[2], rev=true)
    global ORF7a_pos_ct_all_sort_by_ct = sort(collect(ORF7a_pos_ct_all), by = x -> x[2], rev=true)
    global ORF7b_pos_ct_all_sort_by_ct = sort(collect(ORF7b_pos_ct_all), by = x -> x[2], rev=true)
    global ORF8_pos_ct_all_sort_by_ct = sort(collect(ORF8_pos_ct_all), by = x -> x[2], rev=true)
    global N_pos_ct_all_sort_by_ct = sort(collect(N_pos_ct_all), by = x -> x[2], rev=true)
    global ORF9b_pos_ct_all_sort_by_ct = sort(collect(ORF9b_pos_ct_all), by = x -> x[2], rev=true)
##################################
    global ORF1a_pos_ct_all_sort_by_pos_v1 = sort(collect(ORF1a_pos_ct_all_v1), by = x -> x[1])
    global ORF1b_pos_ct_all_sort_by_pos_v1 = sort(collect(ORF1b_pos_ct_all_v1), by = x -> x[1])
    global S_pos_ct_all_sort_by_pos_v1 = sort(collect(S_pos_ct_all_v1), by = x -> x[1])
    global ORF3a_pos_ct_all_sort_by_pos_v1 = sort(collect(ORF3a_pos_ct_all_v1), by = x -> x[1])
    global E_pos_ct_all_sort_by_pos_v1 = sort(collect(E_pos_ct_all_v1), by = x -> x[1]) 
    global M_pos_ct_all_sort_by_pos_v1 = sort(collect(M_pos_ct_all_v1), by = x -> x[1])
    global ORF6_pos_ct_all_sort_by_pos_v1 = sort(collect(ORF6_pos_ct_all_v1), by = x -> x[1])
    global ORF7a_pos_ct_all_sort_by_pos_v1 = sort(collect(ORF7a_pos_ct_all_v1), by = x -> x[1])
    global ORF7b_pos_ct_all_sort_by_pos_v1 = sort(collect(ORF7b_pos_ct_all_v1), by = x -> x[1])
    global ORF8_pos_ct_all_sort_by_pos_v1 = sort(collect(ORF8_pos_ct_all_v1), by = x -> x[1])
    global N_pos_ct_all_sort_by_pos_v1 = sort(collect(N_pos_ct_all_v1), by = x -> x[1])   
    global ORF9b_pos_ct_all_sort_by_pos_v1 = sort(collect(ORF9b_pos_ct_all_v1), by = x -> x[1])
################################## 
    global ORF1a_pos_ct_all_sort_by_ct_v1 = sort(collect(ORF1a_pos_ct_all_v1), by = x -> x[2], rev=true)
    global ORF1b_pos_ct_all_sort_by_ct_v1 = sort(collect(ORF1b_pos_ct_all_v1), by = x -> x[2], rev=true)
    global S_pos_ct_all_sort_by_ct_v1 = sort(collect(S_pos_ct_all_v1), by = x -> x[2], rev=true)
    global ORF3a_pos_ct_all_sort_by_ct_v1 = sort(collect(ORF3a_pos_ct_all_v1), by = x -> x[2], rev=true)
    global E_pos_ct_all_sort_by_ct_v1 = sort(collect(E_pos_ct_all_v1), by = x -> x[2], rev=true)
    global M_pos_ct_all_sort_by_ct_v1 = sort(collect(M_pos_ct_all_v1), by = x -> x[2], rev=true)
    global ORF6_pos_ct_all_sort_by_ct_v1 = sort(collect(ORF6_pos_ct_all_v1), by = x -> x[2], rev=true)
    global ORF7a_pos_ct_all_sort_by_ct_v1 = sort(collect(ORF7a_pos_ct_all_v1), by = x -> x[2], rev=true)
    global ORF7b_pos_ct_all_sort_by_ct_v1 = sort(collect(ORF7b_pos_ct_all_v1), by = x -> x[2], rev=true)
    global ORF8_pos_ct_all_sort_by_ct_v1 = sort(collect(ORF8_pos_ct_all_v1), by = x -> x[2], rev=true)
    global N_pos_ct_all_sort_by_ct_v1 = sort(collect(N_pos_ct_all_v1), by = x -> x[2], rev=true)
    global ORF9b_pos_ct_all_sort_by_ct_v1 = sort(collect(ORF9b_pos_ct_all_v1), by = x -> x[2], rev=true)
end
###########################################################################################################################################################################
###########################################################################################################################################################################
function NSP_sub_ranks_all(NSP_array::Vector{String}, NSP_sub_types_at_every_site_combined::Dict{String, Dict{Int, Set{String}}}, AA_muts_ct_no_dels_all::Dict{String, Int} )
    NSP1_ct_all = Dict{String, Int}()
    NSP2_ct_all = Dict{String, Int}()
    NSP3_ct_all = Dict{String, Int}()
    NSP4_ct_all = Dict{String, Int}()
    NSP5_ct_all = Dict{String, Int}()
    NSP6_ct_all = Dict{String, Int}()
    NSP7_ct_all = Dict{String, Int}()
    NSP8_ct_all = Dict{String, Int}()
    NSP9_ct_all = Dict{String, Int}()
    NSP10_ct_all = Dict{String, Int}()
    NSP11_ct_all = Dict{String, Int}()
    NSP12_ct_all = Dict{String, Int}()
    NSP13_ct_all = Dict{String, Int}()
    NSP14_ct_all = Dict{String, Int}()
    NSP15_ct_all = Dict{String, Int}()
    NSP16_ct_all = Dict{String, Int}()
#############################################################
    NSP_ct_all_dict_arr = [NSP1_ct_all, NSP2_ct_all, NSP3_ct_all, NSP4_ct_all, NSP5_ct_all, NSP6_ct_all, NSP7_ct_all, NSP8_ct_all, NSP9_ct_all, NSP10_ct_all, NSP11_ct_all, NSP12_ct_all, NSP13_ct_all, NSP14_ct_all, NSP15_ct_all, NSP16_ct_all]
#                                              NSP       AAsite  all_sub_types_at_AAsite (e.g., AV, AT, TA, etc)
# NSPsub_types_at_every_site_combined = Dict{String, Dict{Int, Set{String}}}()   
    for i in 1:length(NSP_array)
        NSP = NSP_array[i]
        NSP_ct_dict = NSP_ct_all_dict_arr[i]
        for (AAsite, mut_type_set) in NSP_sub_types_at_every_site_combined[NSP]
            for mut_type in mut_type_set
                sub = NSP*":"*string(mut_type[1])*"$(AAsite)"*string(mut_type[end])
                NSP_ct_dict[sub] = 0
            end
        end
    end   
    for (sub, ct) in AA_muts_ct_no_dels_all
        AA_site = aa_pos_comprehensive_dict[sub]
        if AA_site < 4402
            gene = aa_gene_comprehensive_dict[sub]
            if gene == "ORF1a" || gene == "ORF1b"
                NSP_sub = ORF1abMut_to_NSP(sub)
                NSP = NSP_muts_gene_dict[NSP_sub]
                NSP_pos = NSP_muts_pos_dict[NSP_sub]
                NSP_num = parse(Int, split(NSP, "P")[2])
                NSP_dict = NSP_ct_all_dict_arr[NSP_num]
                NSP_dict[NSP_sub] = ct
            end
        end
    end
#############################
    fin_sortkey(m) = (NSP_muts_pos_dict[m], NSP_ref_AA_dict[m]*NSP_qry_AA_dict[m])
    global NSP1_ct_all_sort_by_pos = sort(collect(NSP1_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP2_ct_all_sort_by_pos = sort(collect(NSP2_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP3_ct_all_sort_by_pos = sort(collect(NSP3_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP4_ct_all_sort_by_pos = sort(collect(NSP4_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP5_ct_all_sort_by_pos = sort(collect(NSP5_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP6_ct_all_sort_by_pos = sort(collect(NSP6_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP7_ct_all_sort_by_pos = sort(collect(NSP7_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP8_ct_all_sort_by_pos = sort(collect(NSP8_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP9_ct_all_sort_by_pos = sort(collect(NSP9_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP10_ct_all_sort_by_pos = sort(collect(NSP10_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP11_ct_all_sort_by_pos = sort(collect(NSP11_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP12_ct_all_sort_by_pos = sort(collect(NSP12_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP13_ct_all_sort_by_pos = sort(collect(NSP13_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP14_ct_all_sort_by_pos = sort(collect(NSP14_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP15_ct_all_sort_by_pos = sort(collect(NSP15_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP16_ct_all_sort_by_pos = sort(collect(NSP16_ct_all), by = x -> fin_sortkey(x[1]))
#############################
    global NSP1_ct_all_sort_by_ct = sort(collect(NSP1_ct_all), by = x -> x[2], rev=true)
    global NSP2_ct_all_sort_by_ct = sort(collect(NSP2_ct_all), by = x -> x[2], rev=true)
    global NSP3_ct_all_sort_by_ct = sort(collect(NSP3_ct_all), by = x -> x[2], rev=true)
    global NSP4_ct_all_sort_by_ct = sort(collect(NSP4_ct_all), by = x -> x[2], rev=true)
    global NSP5_ct_all_sort_by_ct = sort(collect(NSP5_ct_all), by = x -> x[2], rev=true)
    global NSP6_ct_all_sort_by_ct = sort(collect(NSP6_ct_all), by = x -> x[2], rev=true)
    global NSP7_ct_all_sort_by_ct = sort(collect(NSP7_ct_all), by = x -> x[2], rev=true)
    global NSP8_ct_all_sort_by_ct = sort(collect(NSP8_ct_all), by = x -> x[2], rev=true)
    global NSP9_ct_all_sort_by_ct = sort(collect(NSP9_ct_all), by = x -> x[2], rev=true)
    global NSP10_ct_all_sort_by_ct = sort(collect(NSP10_ct_all), by = x -> x[2], rev=true)
    global NSP11_ct_all_sort_by_ct = sort(collect(NSP11_ct_all), by = x -> x[2], rev=true)
    global NSP12_ct_all_sort_by_ct = sort(collect(NSP12_ct_all), by = x -> x[2], rev=true)
    global NSP13_ct_all_sort_by_ct = sort(collect(NSP13_ct_all), by = x -> x[2], rev=true)
    global NSP14_ct_all_sort_by_ct = sort(collect(NSP14_ct_all), by = x -> x[2], rev=true)
    global NSP15_ct_all_sort_by_ct = sort(collect(NSP15_ct_all), by = x -> x[2], rev=true)
    global NSP16_ct_all_sort_by_ct = sort(collect(NSP16_ct_all), by = x -> x[2], rev=true)
end
###########################################################################################################################################################################
###########################################################################################################################################################################
# Removed from parameters: NSP_AA_size::Dict{String, Int}
function NSP_sub_pos_ranks_all(NSP_array::Vector{String}, NSP_sub_types_at_every_site_combined::Dict{String, Dict{Int, Set{String}}}, AA_muts_ct_no_dels_all::Dict{String, Int} )
    NSP1_pos_ct_all = Dict{String, Int}()
    NSP2_pos_ct_all = Dict{String, Int}()
    NSP3_pos_ct_all = Dict{String, Int}()
    NSP4_pos_ct_all = Dict{String, Int}()
    NSP5_pos_ct_all = Dict{String, Int}()
    NSP6_pos_ct_all = Dict{String, Int}()
    NSP7_pos_ct_all = Dict{String, Int}()
    NSP8_pos_ct_all = Dict{String, Int}()
    NSP9_pos_ct_all = Dict{String, Int}()
    NSP10_pos_ct_all = Dict{String, Int}()
    NSP11_pos_ct_all = Dict{String, Int}()
    NSP12_pos_ct_all = Dict{String, Int}()
    NSP13_pos_ct_all = Dict{String, Int}()
    NSP14_pos_ct_all = Dict{String, Int}()
    NSP15_pos_ct_all = Dict{String, Int}()
    NSP16_pos_ct_all = Dict{String, Int}()
    NSP_pos_ct_all_array = [NSP1_pos_ct_all, NSP2_pos_ct_all, NSP3_pos_ct_all, NSP4_pos_ct_all, NSP5_pos_ct_all, NSP6_pos_ct_all, NSP7_pos_ct_all, NSP8_pos_ct_all, NSP9_pos_ct_all, NSP10_pos_ct_all, NSP11_pos_ct_all, NSP12_pos_ct_all, NSP13_pos_ct_all, NSP14_pos_ct_all, NSP15_pos_ct_all, NSP16_pos_ct_all]
#######################################################
    NSP1_pos_ct_all_v1 = Dict{Int, Int}()
    NSP2_pos_ct_all_v1 = Dict{Int, Int}()
    NSP3_pos_ct_all_v1 = Dict{Int, Int}()
    NSP4_pos_ct_all_v1 = Dict{Int, Int}()
    NSP5_pos_ct_all_v1 = Dict{Int, Int}()
    NSP6_pos_ct_all_v1 = Dict{Int, Int}()
    NSP7_pos_ct_all_v1 = Dict{Int, Int}()
    NSP8_pos_ct_all_v1 = Dict{Int, Int}()
    NSP9_pos_ct_all_v1 = Dict{Int, Int}()
    NSP10_pos_ct_all_v1 = Dict{Int, Int}()
    NSP11_pos_ct_all_v1 = Dict{Int, Int}()
    NSP12_pos_ct_all_v1 = Dict{Int, Int}()
    NSP13_pos_ct_all_v1 = Dict{Int, Int}()
    NSP14_pos_ct_all_v1 = Dict{Int, Int}()
    NSP15_pos_ct_all_v1 = Dict{Int, Int}()
    NSP16_pos_ct_all_v1 = Dict{Int, Int}()
    NSP_pos_ct_all_v1_array = [NSP1_pos_ct_all_v1, NSP2_pos_ct_all_v1, NSP3_pos_ct_all_v1, NSP4_pos_ct_all_v1, NSP5_pos_ct_all_v1, NSP6_pos_ct_all_v1, NSP7_pos_ct_all_v1, NSP8_pos_ct_all_v1, NSP9_pos_ct_all_v1, NSP10_pos_ct_all_v1, NSP11_pos_ct_all_v1, NSP12_pos_ct_all_v1, NSP13_pos_ct_all_v1, NSP14_pos_ct_all_v1, NSP15_pos_ct_all_v1, NSP16_pos_ct_all_v1]
#######################################################
    NSP_pos_ct_all = Dict{String, Int}()
    for i in 1:length(NSP_pos_ct_all_array)
        NSP_all_dict = NSP_pos_ct_all_array[i]
        NSP_all_dict_v1 = NSP_pos_ct_all_v1_array[i]
        NSP = NSP_array[i]
        NSP_len = NSP_AA_size[NSP]
        for j in 1:NSP_len
            NSP_pos = NSP*":"*"$(j)"
            NSP_all_dict[NSP_pos] = 0
            NSP_pos_ct_all[NSP_pos] = 0
            NSP_all_dict_v1[j] = 0
        end
    end
############################################################
    NSP1_ct_all = Dict{String, Int}()
    NSP2_ct_all = Dict{String, Int}()
    NSP3_ct_all = Dict{String, Int}()
    NSP4_ct_all = Dict{String, Int}()
    NSP5_ct_all = Dict{String, Int}()
    NSP6_ct_all = Dict{String, Int}()
    NSP7_ct_all = Dict{String, Int}()
    NSP8_ct_all = Dict{String, Int}()
    NSP9_ct_all = Dict{String, Int}()
    NSP10_ct_all = Dict{String, Int}()
    NSP11_ct_all = Dict{String, Int}()
    NSP12_ct_all = Dict{String, Int}()
    NSP13_ct_all = Dict{String, Int}()
    NSP14_ct_all = Dict{String, Int}()
    NSP15_ct_all = Dict{String, Int}()
    NSP16_ct_all = Dict{String, Int}()
##################################################################################################################################
    NSP_ct_all_dict_arr = [NSP1_ct_all, NSP2_ct_all, NSP3_ct_all, NSP4_ct_all, NSP5_ct_all, NSP6_ct_all, NSP7_ct_all, NSP8_ct_all, NSP9_ct_all, NSP10_ct_all, NSP11_ct_all, NSP12_ct_all, NSP13_ct_all, NSP14_ct_all, NSP15_ct_all, NSP16_ct_all]
#                                              NSP       AAsite  all_sub_types_at_AAsite (e.g., AV, AT, TA, etc)
# NSPsub_types_at_every_site_combined = Dict{String, Dict{Int, Set{String}}}()   
    for i in 1:length(NSP_array)
        NSP = NSP_array[i]
        NSP_ct_dict = NSP_ct_all_dict_arr[i]
        for (AAsite, mut_type_set) in NSP_sub_types_at_every_site_combined[NSP]
            for mut_type in mut_type_set
                sub = NSP*":"*string(mut_type[1])*"$(AAsite)"*string(mut_type[end])
                NSP_ct_dict[sub] = 0
            end
        end
    end   
    for (sub, ct) in AA_muts_ct_no_dels_all
        gene = aa_gene_comprehensive_dict[sub]
        if gene == "ORF1a" || gene == "ORF1b"
            AA_site = parse(Int, split(sub, ":")[2][2:end-1])
            if AA_site < 4402
                NSP_sub = ORF1abMut_to_NSP(sub)
                NSP = NSP_muts_gene_dict[NSP_sub]
                NSP_pos = NSP_muts_pos_dict[NSP_sub]
                NSP_num = parse(Int, split(NSP, "P")[2])
                NSP_dict = NSP_ct_all_dict_arr[NSP_num]
                NSP_dict[NSP_sub] = ct
            end
        end
    end
###################################################################################################################################
    for i in 1:length(NSP_pos_ct_all_array)
        NSP = NSP_array[i]
        NSP_dict = NSP_ct_all_dict_arr[i]
        NSP_pos_dict = NSP_pos_ct_all_array[i]
        NSP_pos_dict_v1 = NSP_pos_ct_all_v1_array[i]
        for (sub, ct) in NSP_dict
            pos = parse(Int, split(sub, ":")[2][2:end-1])
            sub_pos = NSP*":"*"$(pos)"
            NSP_pos_dict[sub_pos] += ct
            NSP_pos_dict_v1[pos] += ct
            NSP_pos_ct_all[sub_pos] += ct
        end
    end
##############################################################
    global NSP1_pos_ct_all_sort_by_pos = sort(collect(NSP1_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP2_pos_ct_all_sort_by_pos = sort(collect(NSP2_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP3_pos_ct_all_sort_by_pos = sort(collect(NSP3_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP4_pos_ct_all_sort_by_pos = sort(collect(NSP4_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP5_pos_ct_all_sort_by_pos = sort(collect(NSP5_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP6_pos_ct_all_sort_by_pos = sort(collect(NSP6_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP7_pos_ct_all_sort_by_pos = sort(collect(NSP7_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP8_pos_ct_all_sort_by_pos = sort(collect(NSP8_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP9_pos_ct_all_sort_by_pos = sort(collect(NSP9_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP10_pos_ct_all_sort_by_pos = sort(collect(NSP10_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP11_pos_ct_all_sort_by_pos = sort(collect(NSP11_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP12_pos_ct_all_sort_by_pos = sort(collect(NSP12_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP13_pos_ct_all_sort_by_pos = sort(collect(NSP13_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP14_pos_ct_all_sort_by_pos = sort(collect(NSP14_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP15_pos_ct_all_sort_by_pos = sort(collect(NSP15_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP16_pos_ct_all_sort_by_pos = sort(collect(NSP16_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
#############################
    global NSP1_pos_ct_all_sort_by_ct = sort(collect(NSP1_pos_ct_all), by = x -> x[2], rev=true)
    global NSP2_pos_ct_all_sort_by_ct = sort(collect(NSP2_pos_ct_all), by = x -> x[2], rev=true)
    global NSP3_pos_ct_all_sort_by_ct = sort(collect(NSP3_pos_ct_all), by = x -> x[2], rev=true)
    global NSP4_pos_ct_all_sort_by_ct = sort(collect(NSP4_pos_ct_all), by = x -> x[2], rev=true)
    global NSP5_pos_ct_all_sort_by_ct = sort(collect(NSP5_pos_ct_all), by = x -> x[2], rev=true)
    global NSP6_pos_ct_all_sort_by_ct = sort(collect(NSP6_pos_ct_all), by = x -> x[2], rev=true)
    global NSP7_pos_ct_all_sort_by_ct = sort(collect(NSP7_pos_ct_all), by = x -> x[2], rev=true)
    global NSP8_pos_ct_all_sort_by_ct = sort(collect(NSP8_pos_ct_all), by = x -> x[2], rev=true)
    global NSP9_pos_ct_all_sort_by_ct = sort(collect(NSP9_pos_ct_all), by = x -> x[2], rev=true)
    global NSP10_pos_ct_all_sort_by_ct = sort(collect(NSP10_pos_ct_all), by = x -> x[2], rev=true)
    global NSP11_pos_ct_all_sort_by_ct = sort(collect(NSP11_pos_ct_all), by = x -> x[2], rev=true)
    global NSP12_pos_ct_all_sort_by_ct = sort(collect(NSP12_pos_ct_all), by = x -> x[2], rev=true)
    global NSP13_pos_ct_all_sort_by_ct = sort(collect(NSP13_pos_ct_all), by = x -> x[2], rev=true)
    global NSP14_pos_ct_all_sort_by_ct = sort(collect(NSP14_pos_ct_all), by = x -> x[2], rev=true)
    global NSP15_pos_ct_all_sort_by_ct = sort(collect(NSP15_pos_ct_all), by = x -> x[2], rev=true)
    global NSP16_pos_ct_all_sort_by_ct = sort(collect(NSP16_pos_ct_all), by = x -> x[2], rev=true)
###############################
    global NSP1_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP1_pos_ct_all_v1), by = x -> x[1])
    global NSP2_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP2_pos_ct_all_v1), by = x -> x[1])
    global NSP3_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP3_pos_ct_all_v1), by = x -> x[1])
    global NSP4_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP4_pos_ct_all_v1), by = x -> x[1])
    global NSP5_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP5_pos_ct_all_v1), by = x -> x[1])
    global NSP6_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP6_pos_ct_all_v1), by = x -> x[1])
    global NSP7_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP7_pos_ct_all_v1), by = x -> x[1])
    global NSP8_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP8_pos_ct_all_v1), by = x -> x[1])
    global NSP9_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP9_pos_ct_all_v1), by = x -> x[1])
    global NSP10_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP10_pos_ct_all_v1), by = x -> x[1])
    global NSP11_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP11_pos_ct_all_v1), by = x -> x[1])
    global NSP12_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP12_pos_ct_all_v1), by = x -> x[1])
    global NSP13_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP13_pos_ct_all_v1), by = x -> x[1])
    global NSP14_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP14_pos_ct_all_v1), by = x -> x[1])
    global NSP15_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP15_pos_ct_all_v1), by = x -> x[1])
    global NSP16_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP16_pos_ct_all_v1), by = x -> x[1])
end
######################################################################################################################################
runtime = time() - start_chr_load_fx
runtime_rd = round(digits=2, runtime)
runtime1, runtime2 = seconds_to_hrs_min_sec(runtime); print("\n"^1)
println("Runtime v0 = $(runtime) seconds")
println("Runtime v2 = $(runtime2)"); print("\n"^2)
######################################################################################################################################
######################################################################################################################################


2026_05_05__2356PM
11:56.24_PM


Runtime v0 = 0.9410078525543213 seconds
Runtime v2 = 0 hr, 0 min, 00.94 sec




In [134]:
### Execute Load Chronic Dicts DQ Fx | 2026_04_08 version | chronics_2026_04_08__6186seq | Runtime: 16.43 sec | 
print("\n"^1); date_now = Dates.format(now(), "yyyy_mm_dd__HMMp"); println(date_now); nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime); print("\n"^1)
chr_dict_load_start = time()

abs_min_mut_thresh = 15
DQ_mut_thresh = 20
date_pct_DQ_thresh = 95

rep_thresh = 5
revs_thresh = 6
EPCI_qc_str = "$(abs_min_mut_thresh)_$(DQ_mut_thresh)_$(date_pct_DQ_thresh)"
HQCS_qc_string = "5_1_5"

print_ct_thresh = 1
date = "2026_04_08"
ndjson_name = "chronics_2026_03_22_6186seq_v5"
folder_name = "$(date)__$(ndjson_name)__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)"
println(); println("EPCI_qc_str = $(EPCI_qc_str) | HQCS_qc_string = $(HQCS_qc_string)"); println()

chronic_load_dicts2_DQ(ndjson_name, folder_name, date, rep_thresh, revs_thresh, print_ct_thresh, DQ_mut_thresh, date_pct_DQ_thresh, abs_min_mut_thresh)
# ndjson_name::String, folder_name::String, date::String, rep_thresh::Int, revs_thresh::Int, print_ct_thresh::Int, DQ_mut_thresh::Int, DatePctDQThresh::Float64, abs_min_mut_thresh::Int
######################################################################################################################################
seq_AA_muts["EPI_ISL_8725398"] = Set{String}();             seq_AA_muts["EPI_ISL_949208"] = Set{String}();   seq_pango["EPI_ISL_6281381"] = "AY.4"
######################################################################################################################################
#seq_AA_muts["EPI_ISL_9172208"]
######################################################################################################################################
### Create  all_unique_chr_seqs  &  EPCI_set 
rep_seq_grps_maxmut_seqs_arr = collect(values(rep_seq_grps_maxmut_seqs))
all_unique_chr_seqs = union(rep_seq_grps_maxmut_seqs_arr, non_rep_seqs)
all_unique_chr_seqs_ct = length(all_unique_chr_seqs)
EPCI_set = Set{String}()
for seq in all_unique_chr_seqs
    push!(EPCI_set, seq)
end
println("length(all_unique_chr_seqs) = $(length(all_unique_chr_seqs))")
println("length(EPCI_set)  $(length(EPCI_set))"); print("\n"^2)

SF375A = get(AA_muts_ct, "S:S375A", 0) + get(AA_muts_ct, "S:F375A", 0)
A376P = get(AA_muts_ct, "S:A376P", 0) + get(AA_muts_ct, "S:T376P", 0); println()
F377P = get(AA_muts_ct, "S:F377P", 0) + get(AA_muts_ct, "S:F377P", 0); println()
println("Total S:S375A seqs = $(SF375A)"); println("Total S:A376P seqs = $(A376P)"); println(); println("Total S:F377P seqs = $(F377P)"); println()
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
################### Below: Fixing S:F375P, S:F375A, S:A376P, S:371-373 Nextclade errors & replacing with the correct S:∆374-376 or ∆374-375 or ∆375-376 ####################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
#seq_AA_del_ranges = seq_AA_dels
#seq_AA_del_ranges_WT = seq_AA_dels_WT
seq_AA_dels = Dict{String, Set{String}}()
seq_AA_dels_WT = Dict{String, Set{String}}()
############################################################################################################################################################################
############################################################################################################################################################################
seq_privAA_len = Dict{String, Int}()
for (seq, AAsubvec) in seq_AA_muts_no_dels
    seqAAlen = length(AAsubvec)
    aa_del_range_len = length(seq_AA_del_ranges[seq])
    seq_privAA_len[seq] = seqAAlen + aa_del_range_len
end
############################################################################################################################################################################
############################################################################################################################################################################
########################################################################################################################################################################
for (seq, delrangeset) in seq_AA_del_ranges
    if seq ≠ "EPI_ISL_19459469"
        if "S:371-373" in delrangeset
            delete!(seq_AA_del_ranges[seq], "S:371-373")
            delete!(seq_AA_del_ranges_WT[seq], "S:371-373")
            push!(seq_AA_del_ranges[seq], "S:374-376")
            push!(seq_AA_del_ranges_WT[seq], "S:374-376")
        end
    end
end
########################################################################################################################################################################
for seq in keys(seq_AA_muts)
    if "S:F375A" in seq_AA_muts[seq]
        delete!(seq_AA_muts[seq], "S:F375A")
        delete!(seq_AA_muts_no_dels[seq], "S:F375A")
        delete!(seq_AA_muts_WT[seq], "S:S375A")
        delete!(seq_AA_muts[seq], "S:F375A")
        delete!(seq_AA_muts_pos_only[seq], "S:371")
        delete!(seq_AA_muts_pos_only[seq], "S:372")
        delete!(seq_AA_muts_pos_only[seq], "S:373")
        delete!(seq_AA_muts_WT_pos_only[seq], "S:372")
##################
        delete!(seq_AA_muts[seq], "S:F371-")
        delete!(seq_AA_muts[seq], "S:L371-")
        delete!(seq_AA_muts[seq], "S:S371-")
        delete!(seq_AA_muts[seq], "S:A372-")
        delete!(seq_AA_muts[seq], "S:P373-")
##################
        delete!(seq_AA_muts_no_dels[seq], "S:F371-")
        delete!(seq_AA_muts_no_dels[seq], "S:L371-")
        delete!(seq_AA_muts_no_dels[seq], "S:S371-")
        delete!(seq_AA_muts_no_dels[seq], "S:A372-")
        delete!(seq_AA_muts_no_dels[seq], "S:P373-")
##################
        delete!(seq_AA_muts_WT[seq], "S:S371-")
        delete!(seq_AA_muts_WT[seq], "S:A372-")
        delete!(seq_AA_muts_WT[seq], "S:S373-")
##################
        push!(seq_AA_muts[seq], "S:F374-")
        push!(seq_AA_muts[seq], "S:F375-")
        push!(seq_AA_muts[seq], "S:A376-")
        push!(seq_AA_muts_WT[seq], "S:F374-")
        push!(seq_AA_muts_WT[seq], "S:S375-")
        push!(seq_AA_muts_WT[seq], "S:T376-")
        push!(seq_AA_muts_pos_only[seq], "S:374")
        push!(seq_AA_muts_pos_only[seq], "S:375")
        push!(seq_AA_muts_pos_only[seq], "S:376")
        push!(seq_AA_muts_WT_pos_only[seq], "S:374")
    end
    if "S:A376P" in seq_AA_muts[seq]
        delete!(seq_AA_muts[seq], "S:A376P")
        delete!(seq_AA_muts_no_dels[seq], "S:A376P")
        delete!(seq_AA_muts_WT[seq], "S:T376P")
    end
    if "S:T376P" in seq_AA_muts[seq]
        delete!(seq_AA_muts[seq], "S:T376P")
        delete!(seq_AA_muts_no_dels[seq], "S:T376P")
        delete!(seq_AA_muts_WT[seq], "S:T376P")
    end
end
########################################################################################################################################################################
for seq in keys(seq_AA_muts)
    if "S:F375P" in seq_AA_muts[seq] && "S:A376T" in seq_AA_muts[seq]
############
        delete!(seq_AA_del_ranges[seq], "S:373-374")
        push!(seq_AA_del_ranges[seq], "S:374-375")
############
        delete!(seq_AA_del_ranges_WT[seq], "S:373-374")
        push!(seq_AA_del_ranges_WT[seq], "S:374-375")
############
        delete!(seq_AA_muts[seq], "S:P373F")
        delete!(seq_AA_muts_no_dels[seq], "S:P373F")
        delete!(seq_AA_muts_WT[seq], "S:P373F")
        delete!(AA_muts_seq["S:P373F"], seq)
        delete!(AA_muts_seq_WT["S:S373F"], seq)
        ######
        delete!(seq_AA_muts[seq], "S:F375P")
        delete!(seq_AA_muts_no_dels[seq], "S:F375P")
        delete!(seq_AA_muts_WT[seq], "S:S375P")
        delete!(AA_muts_seq["S:F375P"], seq)
        delete!(AA_muts_seq_WT["S:S375P"], seq)
############
        delete!(seq_AA_muts[seq], "S:F371-")
        delete!(seq_AA_muts_WT[seq], "S:F371-")
        delete!(AA_muts_seq["S:F371-"], seq)
        delete!(AA_muts_seq_WT["S:S371-"], seq)
        #######
        delete!(seq_AA_muts[seq], "S:A372-")
        delete!(seq_AA_muts_WT[seq], "S:A372-")
        delete!(AA_muts_seq["S:A372-"], seq)
        delete!(AA_muts_seq_WT["S:A372-"], seq)
        #######
        delete!(seq_AA_muts[seq], "S:P373-")
        delete!(seq_AA_muts_WT[seq], "S:S373-")
        delete!(AA_muts_seq["S:P373-"], seq)
        delete!(AA_muts_seq_WT["S:S373-"], seq)
############
        push!(seq_AA_muts[seq], "S:F375-")
        push!(seq_AA_muts_no_dels[seq], "S:F375-")
        push!(seq_AA_muts[seq], "S:F375-")
        push!(seq_AA_muts_WT[seq], "S:S375-")
        push!(AA_muts_seq["S:F375-"], seq)
        push!(AA_muts_seq_WT["S:S375-"], seq)
############
        push!(seq_AA_muts_WT[seq], "S:S373P")
        push!(AA_muts_seq_WT["S:S373P"], seq)
    end
#######################################################################
    if "S:F375P" in seq_AA_muts[seq] && "S:A376I" in seq_AA_muts[seq]
############
        delete!(seq_AA_del_ranges[seq], "S:373-374")
        push!(seq_AA_del_ranges[seq], "S:375-376")
############
        delete!(seq_AA_del_ranges_WT[seq], "S:373-374")
        push!(seq_AA_del_ranges_WT[seq], "S:375-376")
############
        delete!(seq_AA_muts[seq], "S:F375P")
        delete!(seq_AA_muts_no_dels[seq], "S:F375P")
        delete!(seq_AA_muts_WT[seq], "S:S375P")
        delete!(AA_muts_seq["S:F375P"], seq)
        delete!(AA_muts_seq_WT["S:S375P"], seq)
############
        delete!(seq_AA_muts[seq], "S:F371-")
        delete!(seq_AA_muts_WT[seq], "S:S371-")
        delete!(AA_muts_seq["S:F371-"], seq)
        delete!(AA_muts_seq_WT["S:S371-"], seq)
        #######
        delete!(seq_AA_muts[seq], "S:A372-")
        delete!(seq_AA_muts_WT[seq], "S:A372-")
        delete!(AA_muts_seq["S:A372-"], seq)
        delete!(AA_muts_seq_WT["S:A372-"], seq)
        #######
        delete!(seq_AA_muts[seq], "S:P373-")
        delete!(seq_AA_muts_WT[seq], "S:S373-")
        delete!(AA_muts_seq["S:P373-"], seq)
        delete!(AA_muts_seq_WT["S:S373-"], seq)
############
        delete!(seq_AA_muts[seq], "S:F374-")
        delete!(seq_AA_muts_WT[seq], "S:F374-")
        delete!(AA_muts_seq["S:F374-"], seq)
        delete!(AA_muts_seq_WT["S:F374-"], seq)
############
        push!(seq_AA_muts[seq], "S:F375-")
        push!(seq_AA_muts[seq], "S:F375-")
        push!(seq_AA_muts_WT[seq], "S:S375-")
        push!(AA_muts_seq["S:F375-"], seq)
        push!(AA_muts_seq_WT["S:S375-"], seq)
############
        push!(seq_AA_muts[seq], "S:A376-")
        push!(seq_AA_muts_no_dels[seq], "S:A376-")
        push!(seq_AA_muts_WT[seq], "S:T376-")
############
        push!(seq_AA_muts[seq], "S:F374I")
        push!(seq_AA_muts_no_dels[seq], "S:F374I")
        push!(seq_AA_muts[seq], "S:F374I")
        push!(seq_AA_muts_WT[seq], "S:F374I")
        push!(AA_muts_seq["S:F374L"], seq)
        push!(AA_muts_seq_WT["S:F374L"], seq)
############
        push!(seq_AA_muts_WT[seq], "S:S373P")
        push!(AA_muts_seq_WT["S:S373P"], seq)
    end
#######################################################################
    if "S:F375P" in seq_AA_muts[seq] && "S:A376L" in seq_AA_muts[seq]
############
        delete!(seq_AA_del_ranges[seq], "S:373-374")
        push!(seq_AA_del_ranges[seq], "S:375-376")
############
        delete!(seq_AA_del_ranges_WT[seq], "S:373-374")
        push!(seq_AA_del_ranges_WT[seq], "S:375-376")
############
        delete!(seq_AA_muts[seq], "S:F375P")
        delete!(seq_AA_muts_no_dels[seq], "S:F375P")
        delete!(seq_AA_muts_WT[seq], "S:S375P")
        delete!(AA_muts_seq["S:F375P"], seq)
        delete!(AA_muts_seq_WT["S:S375P"], seq)
############
        delete!(seq_AA_muts[seq], "S:F371-")
        delete!(seq_AA_muts_WT[seq], "S:S371-")
        delete!(AA_muts_seq["S:F371-"], seq)
        delete!(AA_muts_seq_WT["S:S371-"], seq)
        #######
        delete!(seq_AA_muts[seq], "S:A372-")
        delete!(seq_AA_muts_WT[seq], "S:A372-")
        delete!(AA_muts_seq["S:A372-"], seq)
        delete!(AA_muts_seq_WT["S:A372-"], seq)
        #######
        delete!(seq_AA_muts[seq], "S:P373-")
        delete!(seq_AA_muts_WT[seq], "S:S373-")
        delete!(AA_muts_seq["S:P373-"], seq)
        delete!(AA_muts_seq_WT["S:S373-"], seq)
############
        delete!(seq_AA_muts[seq], "S:F374-")
        delete!(seq_AA_muts_no_dels[seq], "S:F374-")
        delete!(seq_AA_muts_WT[seq], "S:F374-")
        delete!(AA_muts_seq["S:F374-"], seq)
        delete!(AA_muts_seq_WT["S:F374-"], seq)
############
        push!(seq_AA_muts[seq], "S:F375-")
        push!(seq_AA_muts_no_dels[seq], "S:F375-")
        push!(seq_AA_muts[seq], "S:F375-")
        push!(seq_AA_muts_WT[seq], "S:S375-")
        push!(AA_muts_seq["S:F375-"], seq)
        push!(AA_muts_seq_WT["S:S375-"], seq)
############
        push!(seq_AA_muts[seq], "S:F374L")
        push!(seq_AA_muts_no_dels[seq], "S:F374L")
        push!(seq_AA_muts[seq], "S:F374L")
        push!(seq_AA_muts_WT[seq], "S:F374L")
        push!(AA_muts_seq["S:F374L"], seq)
        push!(AA_muts_seq_WT["S:F374L"], seq)
############
        push!(seq_AA_muts_WT[seq], "S:S373P")
    end
end
##################################################
if "S:F377P" in seq_AA_muts["EPI_ISL_15072543"]
    seq = "EPI_ISL_15072543"
    delete!(seq_AA_muts[seq], "S:F377P")
    delete!(seq_AA_muts_no_dels[seq], "S:F377P")
    delete!(seq_AA_muts_WT[seq], "S:F377P")
    delete!(AA_muts_seq["S:F377P"], seq)
    delete!(AA_muts_seq_WT["S:F377P"], seq)
    ########
    delete!(seq_AA_muts[seq], "S:P373-")
    delete!(seq_AA_muts_WT[seq], "S:S373-")
    delete!(AA_muts_seq["S:P373-"], seq)
    delete!(AA_muts_seq_WT["S:S373-"], seq)
##########
    push!(seq_AA_muts[seq], "S:F377-")
    push!(seq_AA_muts_WT[seq], "S:F377-")
    push!(AA_muts_seq["S:F377-"], seq)
    push!(AA_muts_seq_WT["S:F377-"], seq)
    ######
    push!(AA_muts_seq_WT["S:S373P"], seq)
end
########################################################################################################################################################################
seq_AA_dels = Dict{String, Set{String}}()
seq_AA_dels_WT = Dict{String, Set{String}}()
for seq in all_seqs_set
    seq_AA_dels[seq] = Set{String}()
    for mut in seq_AA_muts[seq]
        if mut[end] == '-'
            push!(seq_AA_dels[seq], mut)
        end
    end
    seq_AA_dels_WT[seq] = Set{String}()
    for mut in seq_AA_muts_WT[seq]
        if mut[end] == '-'
            push!(seq_AA_dels_WT[seq], mut)
        end
    end
end
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
################################################### Below: Fixing AA_muts_ct & similar (2026_04_07) ########################################################################
########################################## Should be temporary——only needed until main chronic fx fixed & run ##############################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
AA_muts_ct = Dict{String, Int}()
AA_muts_ct_no_dels_no_revs = Dict{String, Int}()
AA_muts_ct_pos_only = Dict{String, Int}()
AA_muts_ct_pos_only_no_dels = Dict{String, Int}()
AA_muts_ct_no_dels = Dict{String, Int}()
AA_dels_ct = Dict{String, Int}()
AA_dels_ct_pos_only = Dict{String, Int}()
for seq in EPCI_set
    for mut in seq_AA_muts[seq]
        aapos = aa_pos_comprehensive_dict[mut]
        aagene = aa_gene_comprehensive_dict[mut]
        aagene_and_pos = aa_gene_and_pos_comprehensive_dict[mut]
        AA_muts_ct[mut] = get(AA_muts_ct, mut, 0) + 1
        AA_muts_ct_pos_only[aagene_and_pos] = get(AA_muts_ct_pos_only, aagene_and_pos, 0) + 1
        if mut[end] ≠ '-'
            AA_muts_ct_no_dels[mut] = get(AA_muts_ct_no_dels, mut, 0) + 1
            AA_muts_ct_pos_only_no_dels[aagene_and_pos] = get(AA_muts_ct_pos_only_no_dels, aagene_and_pos, 0) + 1
            if qryAA_comprehensive_dict[mut] ≠ string(gene_AA_dict[aagene][aapos])
                AA_muts_ct_no_dels_no_revs[mut] = get(AA_muts_ct_no_dels_no_revs, mut, 0) + 1
            end
        end
        if mut[end] == '-'
            AA_dels_ct[mut] = get(AA_dels_ct, mut, 0) + 1
            AA_dels_ct_pos_only[aagene_and_pos] = get(AA_dels_ct_pos_only, aagene_and_pos, 0) + 1
        end
    end
end
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
print("\n"^2)

SF375A = get(AA_muts_ct, "S:S375A", 0) + get(AA_muts_ct, "S:F375A", 0)
A376P = get(AA_muts_ct, "S:A376P", 0) + get(AA_muts_ct, "S:T376P", 0); println()
F377P = get(AA_muts_ct, "S:F377P", 0) + get(AA_muts_ct, "S:F377P", 0); println()
println("Total S:S375A seqs = $(SF375A)"); println("Total S:A376P seqs = $(A376P)"); println(); println("Total S:F377P seqs = $(F377P)"); println()
############################################################################################################################################################################master

chronic_pango_set = Set{String}()
for seq in all_seqs_set
    push!(chronic_pango_set, seq_pango[seq])
end
######################################################################################################################################
#for seq in EPCI_set
#    for del in seq_AA_dels[seq]
#        aa_gene_comprehensive_dict[del] = string(split(del, ":")[1])
#        firstdel = string(split(del, "-")[1])
#        aa_pos_comprehensive_dict[del] = aa_pos_comprehensive_dict[firstdel]
#    end
#end
############################################################################################################################################################################
#### Removing clearly artifactual reversions that occur directly next to dropout (and are almost always from labs that frequently have this problem)
FCS_fake_revs = Set(["S:K679N", "S:H681P", "S:R681P"])
FCS_fake_revs_pos = Set(["S:679", "S:681"])
for seq in all_qualifying_seqs_set
    if "S:683" in seq_unknown_AA[seq] || "S:676" in seq_unknown_AA[seq]
        if seq in EPCI_set
            for mut in FCS_fake_revs
                if mut in seq_AA_muts[seq]
                    mut_pos = aa_gene_and_pos_comprehensive_dict[mut]
                    AA_muts_ct[mut] -= 1
                    AA_muts_ct_no_dels[mut] -= 1
                    AA_muts_ct_pos_only[mut_pos] -= 1
                    AA_muts_ct_pos_only_no_dels[mut_pos] -= 1
                end
            end
        end
        setdiff!(seq_AA_muts[seq], FCS_fake_revs)
        setdiff!(seq_AA_muts_no_dels[seq], FCS_fake_revs)
        setdiff!(seq_AA_muts_pos_only[seq], FCS_fake_revs_pos)
        setdiff!(seq_AA_muts_pos_only_no_dels[seq], FCS_fake_revs_pos)
    end
end
######################################################################################################################################
if haskey(AA_muts_ct_pos_only_no_dels, "")
    delete!(AA_muts_ct_pos_only_no_dels, "")
end
if haskey(AA_muts_ct_no_dels, "")
    delete!(AA_muts_ct_no_dels, "")
end
######################################################################################################################################
## Deletions royally screw up any attempt to find correlated mutations since they frequently occur in bunches, which are, of course, highly correlated, but only in the most trivial way. 
## The code below is a preliminary attempt to include common deletions by only allowing the inclusion of a single AA deletion, though most contain multiple consecutive 
## AA deletions. Also included are mutations that only occur as 1-AA deletions, such as E:I13- and E:V14-. It can be removed or inserted without substantially changing the results 
deletion_exceptions = list_to_set("E:I13-, E:V14-, M:G6-, S:C15-, S:C136-, S:D138-, S:A243-, S:F371-, S:F374-, S:F375-, S:V483-, S:A484-, S:E484-, S:D1257-, ORF3a:T14-, ORF3a:V255-, ORF3a:V256-, ORF3a:N257-, ORF7a:I103-, ORF7a:*122-, ORF1a:N2081-, ORF1a:D2136-, ORF1a:S4398-")
pos_only_deletion_exception_ct_dict = Dict{String, Int}("E:I13-"=>0, "E:V14-"=>0, "M:G6-"=>0, "S:C15-"=>0, "S:C136-"=>0, "S:D138-"=>0, "S:A243-"=>0, "S:F371-"=>0, "S:F374-"=>0, "S:F375-"=>0, "S:V483-"=>0, "S:A484-"=>0, "S:E484-"=>0, "S:D1257-"=>0, "ORF3a:T14-"=>0, "ORF3a:V255-"=>0, "ORF3a:V256-"=>0, "ORF3a:N257-"=>0, "ORF7a:I103-"=>0, "ORF7a:*122-"=>0, "ORF1a:N2081-"=>0, "ORF1a:D2136-"=>0, "ORF1a:S4398-"=>0)    
del_to_del_pos = Dict{String, String}("E:I13-"=>"E:13", "E:V14-"=>"E:14", "M:G6-"=>"M:6", "S:C15-"=>"S:15", "S:C136-"=>"S:136", "S:D138-"=>"S:138", "S:A243-"=>"S:243", "S:F371-"=>"S:371", "S:F374-"=>"S:374", "S:F375-"=>"S:375", "S:V483-"=>"S:483", "S:A484-"=>"S:484", "S:E484-"=>"S:484", "S:D1257-"=>"S:1257", "ORF3a:T14-"=>"ORF3a:14", "ORF3a:V255-"=>"ORF3a:255", "ORF3a:V256-"=>"ORF3a:256", "ORF3a:N257-"=>"ORF3a:257", "ORF7a:I103-"=>"ORF7a:103", "ORF7a:*122-"=>"ORF7a:122", "ORF1a:N2081-"=>"ORF1a:2081", "ORF1a:D2136-"=>"ORF1a:2136", "ORF1a:S4398-"=>"ORF1a:4398")
pos_only_exception_count::Int = 0
for seq in all_seqs_set
    for del in deletion_exceptions
        if del in seq_AA_muts[seq]
            del_pos = del_to_del_pos[del]
            push!(seq_AA_muts_pos_only_no_dels[seq], del_pos)
            pos_only_exception_count += 1
            pos_only_deletion_exception_ct_dict[del] += 1
        end
    end
end
######################################################################################################################################
for del in deletion_exceptions
    del_pos = del_to_del_pos[del]
    if !haskey(AA_muts_ct_pos_only_no_dels, del_pos)
        AA_muts_ct_pos_only_no_dels[del_pos] = 0
    end
    AA_muts_ct_pos_only_no_dels[del_pos] += pos_only_deletion_exception_ct_dict[del]
    AA_muts_ct_pos_only_no_dels_chr_all_ratio[del_pos] = 999
end
######################################################################################################################################
deletion_exception_ct_dict = Dict{String, Int}("E:I13-"=>0, "E:V14-"=>0, "M:G6-"=>0, "S:C15-"=>0, "S:C136-"=>0, "S:D138-"=>0, "S:A243-"=>0, "S:F371-"=>0, "S:F374-"=>0, "S:F375-"=>0, "S:V483-"=>0, "S:A484-"=>0, "S:E484-"=>0, "S:D1257-"=>0, "ORF3a:T14-"=>0, "ORF3a:V255-"=>0, "ORF3a:V256-"=>0, "ORF3a:N257-"=>0, "ORF7a:I103-"=>0, "ORF7a:*122-"=>0, "ORF1a:N2081-"=>0, "ORF1a:D2136-"=>0, "ORF1a:S4398-"=>0)    
exception_count::Int = 0
for seq in EPCI_set
    for del in deletion_exceptions
        if del in seq_AA_muts[seq]
            push!(seq_AA_muts_no_dels[seq], del)
            exception_count += 1
            deletion_exception_ct_dict[del] += 1
        end
    end
end
######################################################################################################################################
for del in deletion_exceptions
    AA_muts_ct_no_dels[del] = get(AA_muts_ct_no_dels, del, 0)
    AA_muts_ct_no_dels[del] += deletion_exception_ct_dict[del]
    AA_muts_ct_chr_all_ratio[del] = 999
end
######################################################################################################################################
blank_mutstrings = Set(["", " ", "  ", "   ", "    ", "     ", "      ", "       ", "        ", "         ", "          ", "           ", "            "])
for blnk in blank_mutstrings
    AA_muts_ct_chr_all_ratio[blnk] = 0
    AA_muts_ct_no_dels_chr_all_ratio[blnk] = 0
    AA_muts_ct_pos_only_no_dels_chr_all_ratio[blnk] = 0
end
#######################################################################################################################################
mut_gene_Dict = Dict{String, Int}("ORF1a"=>1, "ORF1b"=>2, "S"=>3, "E"=>4, "M"=>5, "N"=>6, "ORF3a"=>7, "ORF6"=>8, "ORF7a"=>9, "ORF7b"=>10, "ORF8"=>11, "ORF9b"=>12)
######################################################################################################################################
mp_AA_gene_sortKey(n) = (mut_gene_Dict[aa_gene_comprehensive_dict[n[1]]], aa_pos_comprehensive_dict[n[1]])
mp_AA_gene_sortKey_2(n) = (mut_gene_Dict[aa_gene_comprehensive_dict[n]], aa_pos_comprehensive_dict[n])
mp_AA_ct_sortKey1(n) = (1000÷mut_gene_Dict[aa_gene_comprehensive_dict[n[1]]], aa_pos_comprehensive_dict[n[1]])
mp_AA_ct_sortKey2(n) = (n[2], mp_AA_ct_sortKey1(n))
#########
mp_AA_gene_pos_only_sortKey(n) = (mut_gene_Dict[aa_gene_comprehensive_dict[n[1]]], aa_pos_comprehensive_dict[n[1]])
mp_AA_gene_pos_only_sortKey_2(n) = (mut_gene_Dict[aa_gene_comprehensive_dict[n]], aa_pos_comprehensive_dict[n])
mp_AA_ct_pos_only_sortKey1(n) = (1000÷mut_gene_Dict[aa_gene_comprehensive_dict[n[1]]], aa_pos_comprehensive_dict[n[1]])
mp_AA_ct_pos_only_sortKey2(n) = (n[2], mp_AA_ct_pos_only_sortKey1(n))
#################################
function mp_AApos_sort_key(n)
    if haskey(aa_pos_comprehensive_dict, n)
        return (aa_pos_comprehensive_dict[n], n)
    else
        return (9999, n)
    end
end
function sel_muts_pt1_sort_key(n)
    if n == "" || isempty(n)
        return (0, 0)  # Return a default sort key for empty strings
    else
        return mp_AA_gene_sortKey_2(string(split(n, ", ")[1]))
    end
end
##################################################################
println("Number of Deletion Exceptions Made = $(exception_count)")
deletion_exception_ct_dict_sort = sort(collect(deletion_exception_ct_dict), by = x -> x[2])
for del__ct in deletion_exception_ct_dict_sort
    del = del__ct[1]
    ct = del__ct[2]
    println("$(del) = $(ct)")
end
###########################################################################################
seq_privAA_len = Dict{String, Int}()
for (seq, AAsubvec) in seq_AA_muts_no_dels
    seqAAlen = length(AAsubvec)
    seq_privAA_len[seq] = seqAAlen
end
############################################################################################################################################################################
###########################################################
for (seq, date) in seq_collection_date
    date_arr = string.(collect(date))
    if sum(date_arr .== "-") ≠ 2
        println("Doesn't have two dashes in date = $(seq)")
    else
        year = string(split(date, "-")[1])
        month = string(split(date, "-")[2])
        day = string(split(date, "-")[3])
        if length(month) == 1 && month ≠ "0"
            month = add_leading_zero(month)
        end
        if length(day) == 1 && day ≠ "0"
            day = add_leading_zero(day)
        end
        seq_collection_date[seq] = year*"-"*month*"-"*day
    end
end 
############################################################################################################################################################################
corrected_count = 0
noncorrected_ct = 0
for seq in all_seqs_set
    if seq_date_index[seq] > 4000
#        println("seq_date_tuple = $(seq_date_tuple[seq]); seq_date_tuple[seq][1] = $(seq_date_tuple[seq][1]); seq_date_tuple[seq][2] = $(seq_date_tuple[seq][2]); seq_date_tuple[seq][3] = $(seq_date_tuple[seq][3])")
        if seq_date_tuple[seq][1] ≠ 0 && seq_date_tuple[seq][2] ≠ 0
            new_date_tuple = (seq_date_tuple[seq][1], seq_date_tuple[seq][2], 15)
            seq_date_tuple[seq] = new_date_tuple
            seq_date_index[seq] = tuple_to_index[new_date_tuple]
            corrected_count += 1
        elseif seq_date_tuple[seq][2] == 0
            noncorrected_ct += 1
        end
    end
end
total_baddie_ct = corrected_count + noncorrected_ct
println("seq_date_index corrected = $(corrected_count)")
println("seq_date_index not corrected = $(noncorrected_ct)")
println("total_baddie_ct = $(total_baddie_ct)")
######################################################################################################################################################
######################################################################################################################################  
chr_load_runtime = time() - chr_dict_load_start
chr_load_runtime_rd = round(digits=1, chr_load_runtime)
chr_load_hms1, chr_load_hms2 = seconds_to_hrs_min_sec(chr_load_runtime)
println("Total Time to Load Chronic Dictionaries = $(chr_load_runtime_rd)")
println("Total Time to Load Chronic Dictionaries = $(chr_load_hms1)")
println("Total Time to Load Chronic Dictionaries = $(chr_load_hms2)"); print("\n"^1)
println(); println("EPCI_qc_str = $(EPCI_qc_str) | HQCS_qc_string = $(HQCS_qc_string)"); println()
println("Finished!"); println()
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now); print("\n"^1)
######################################################################################################################################
######################################################################################################################################


2026_05_05__2356PM
11:56.38_PM


EPCI_qc_str = 15_20_95 | HQCS_qc_string = 5_1_5

2026_05_05__2356PM
length(all_unique_chr_seqs) = 2490
length(EPCI_set)  2490




Total S:S375A seqs = 27
Total S:A376P seqs = 28

Total S:F377P seqs = 0





Total S:S375A seqs = 0
Total S:A376P seqs = 0

Total S:F377P seqs = 0

Number of Deletion Exceptions Made = 737
S:E484- = 0
ORF1a:D2136- = 1
S:F371- = 1
E:I13- = 4
S:D1257- = 6
ORF1a:N2081- = 6
S:V483- = 7
ORF1a:S4398- = 7
M:G6- = 9
ORF7a:I103- = 9
ORF7a:*122- = 12
S:A484- = 21
S:F374- = 27
S:C136- = 30
ORF3a:V256- = 33
ORF3a:T14- = 35
ORF3a:N257- = 37
ORF3a:V255- = 44
S:F375- = 44
E:V14- = 59
S:D138- = 74
S:C15- = 86
S:A243- = 185
seq_date_index corrected = 73
seq_date_index not corrected = 53
total_baddie_ct = 126
Total Time to Load Chronic Dictionaries = 23.9
Total Time to Load Chronic Dictionaries = 0:00:23.95
Total Time to Load Chronic Dictionaries = 0 hr, 0 min, 23.95 sec


EPCI_qc_str = 15_20_95 | HQCS_qc_string = 5_1_5

Finished!

2026_05_05

In [135]:
### NEW | 2026_04_02 | Many, many functions (mixed_nuc, nuc_to_AA, etc) | Runtime: 1 min 33 sec 
### Timing Template
print("\n"^1); date_now = Dates.format(now(), "yyyy_mm_dd__HMMp"); println(date_now); nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime); print("\n"^1)
start = time()
######################################################################################################################################
AA_triplets = Dict{String, String}("TTT"=>"F", "TTC"=>"F", "TTA"=>"L", "TTG"=>"L", "TCT"=>"S", "TCC"=>"S", "TCA"=>"S", "TCG"=>"S", "TAT"=>"Y", "TAC"=>"Y", "TAA"=>"*", "TAG"=>"*", "TGT"=>"C", "TGC"=>"C", "TGA"=>"*", "TGG"=>"W", "CTT"=>"L", "CTC"=>"L", "CTA"=>"L", "CTG"=>"L", "CCT"=>"P", "CCC"=>"P", "CCA"=>"P", "CCG"=>"P", "CAT"=>"H", "CAC"=>"H", "CAA"=>"Q", "CAG"=>"Q", "CGT"=>"R", "CGC"=>"R", "CGA"=>"R", "CGG"=>"R", "ATT"=>"I", "ATC"=>"I", "ATA"=>"I", "ATG"=>"M", "ACT"=>"T", "ACC"=>"T", "ACA"=>"T", "ACG"=>"T", "AAT"=>"N", "AAC"=>"N", "AAA"=>"K", "AAG"=>"K", "AGT"=>"S", "AGC"=>"S", "AGA"=>"R", "AGG"=>"R", "GTT"=>"V", "GTC"=>"V", "GTA"=>"V", "GTG"=>"V", "GCT"=>"A", "GCC"=>"A", "GCA"=>"A", "GCG"=>"A", "GAT"=>"D", "GAC"=>"D", "GAA"=>"E", "GAG"=>"E", "GGT"=>"G", "GGC"=>"G", "GGA"=>"G", "GGG"=>"G", "TT-"=>"X", "TC-"=>"X", "TA-"=>"X", "TG-"=>"X", "T-T"=>"X", "T-C"=>"X", "T-A"=>"X", "T-G"=>"X", "T--"=>"X", "CT-"=>"X", "CC-"=>"X", "CA-"=>"X", "CG-"=>"X", "C-T"=>"X", "C-C"=>"X", "C-A"=>"X", "C-G"=>"X", "C--"=>"X", "AT-"=>"X", "AC-"=>"X", "AA-"=>"X", "AG-"=>"X", "A-T"=>"X", "A-C"=>"X", "A-A"=>"X", "A-G"=>"X", "A--"=>"X", "GT-"=>"X", "GC-"=>"X", "GA-"=>"X", "GG-"=>"X", "G-T"=>"X", "G-C"=>"X", "G-A"=>"X", "G-G"=>"X", "G--"=>"X", "-TT"=>"X", "-TC"=>"X", "-TA"=>"X", "-TG"=>"X", "-T-"=>"X", "-CT"=>"X", "-CC"=>"X", "-CA"=>"X", "-CG"=>"X", "-C-"=>"X", "-AT"=>"X", "-AC"=>"X", "-AA"=>"X", "-AG"=>"X", "-A-"=>"X", "-GT"=>"X", "-GC"=>"X", "-GA"=>"X", "-GG"=>"X", "-G-"=>"X", "--T"=>"X", "--C"=>"X", "--A"=>"X", "--G"=>"X", "---"=>"X", "NTT"=>"X", "TNT"=>"X", "TTN"=>"X", "NTC"=>"X", "TNC"=>"X", "TCN"=>"X", "NTA"=>"X", "TNA"=>"X", "TAN"=>"X", "NTG"=>"X", "TNG"=>"X", "TGN"=>"X", "NT-"=>"X", "TN-"=>"X", "T-N"=>"X", "NTN"=>"X", "TNN"=>"X", "NCT"=>"X", "CNT"=>"X", "CTN"=>"X", "NCC"=>"X", "CNC"=>"X", "CCN"=>"X", "NCA"=>"X", "CNA"=>"X", "CAN"=>"X", "NCG"=>"X", "CNG"=>"X", "CGN"=>"X", "NC-"=>"X", "CN-"=>"X", "C-N"=>"X", "NCN"=>"X", "CNN"=>"X", "NAT"=>"X", "ANT"=>"X", "ATN"=>"X", "NAC"=>"X", "ANC"=>"X", "ACN"=>"X", "NAA"=>"X", "ANA"=>"X", "AAN"=>"X", "NAG"=>"X", "ANG"=>"X", "AGN"=>"X", "NA-"=>"X", "AN-"=>"X", "A-N"=>"X", "NAN"=>"X", "ANN"=>"X", "NGT"=>"X", "GNT"=>"X", "GTN"=>"X", "NGC"=>"X", "GNC"=>"X", "GCN"=>"X", "NGA"=>"X", "GNA"=>"X", "GAN"=>"X", "NGG"=>"X", "GNG"=>"X", "GGN"=>"X", "NG-"=>"X", "GN-"=>"X", "G-N"=>"X", "NGN"=>"X", "GNN"=>"X", "N-T"=>"X", "-NT"=>"X", "-TN"=>"X", "N-C"=>"X", "-NC"=>"X", "-CN"=>"X", "N-A"=>"X", "-NA"=>"X", "-AN"=>"X", "N-G"=>"X", "-NG"=>"X", "-GN"=>"X", "N--"=>"X", "-N-"=>"X", "--N"=>"X", "N-N"=>"X", "-NN"=>"X", "NNT"=>"X", "NNC"=>"X", "NNA"=>"X", "NNG"=>"X", "NN-"=>"X", "NNN"=>"X")                 
AA_triplet_dels = Dict{String, String}("TTT"=>"F", "TTC"=>"F", "TTA"=>"L", "TTG"=>"L", "TCT"=>"S", "TCC"=>"S", "TCA"=>"S", "TCG"=>"S", "TAT"=>"Y", "TAC"=>"Y", "TAA"=>"*", "TAG"=>"*", "TGT"=>"C", "TGC"=>"C", "TGA"=>"*", "TGG"=>"W", "CTT"=>"L", "CTC"=>"L", "CTA"=>"L", "CTG"=>"L", "CCT"=>"P", "CCC"=>"P", "CCA"=>"P", "CCG"=>"P", "CAT"=>"H", "CAC"=>"H", "CAA"=>"Q", "CAG"=>"Q", "CGT"=>"R", "CGC"=>"R", "CGA"=>"R", "CGG"=>"R", "ATT"=>"I", "ATC"=>"I", "ATA"=>"I", "ATG"=>"M", "ACT"=>"T", "ACC"=>"T", "ACA"=>"T", "ACG"=>"T", "AAT"=>"N", "AAC"=>"N", "AAA"=>"K", "AAG"=>"K", "AGT"=>"S", "AGC"=>"S", "AGA"=>"R", "AGG"=>"R", "GTT"=>"V", "GTC"=>"V", "GTA"=>"V", "GTG"=>"V", "GCT"=>"A", "GCC"=>"A", "GCA"=>"A", "GCG"=>"A", "GAT"=>"D", "GAC"=>"D", "GAA"=>"E", "GAG"=>"E", "GGT"=>"G", "GGC"=>"G", "GGA"=>"G", "GGG"=>"G", "TT-"=>"X", "TC-"=>"X", "TA-"=>"X", "TG-"=>"X", "T-T"=>"X", "T-C"=>"X", "T-A"=>"X", "T-G"=>"X", "T--"=>"-", "CT-"=>"X", "CC-"=>"X", "CA-"=>"X", "CG-"=>"X", "C-T"=>"X", "C-C"=>"X", "C-A"=>"X", "C-G"=>"X", "C--"=>"-", "AT-"=>"X", "AC-"=>"X", "AA-"=>"X", "AG-"=>"X", "A-T"=>"X", "A-C"=>"X", "A-A"=>"X", "A-G"=>"X", "A--"=>"-", "GT-"=>"X", "GC-"=>"X", "GA-"=>"X", "GG-"=>"X", "G-T"=>"X", "G-C"=>"X", "G-A"=>"X", "G-G"=>"X", "G--"=>"-", "-TT"=>"X", "-TC"=>"X", "-TA"=>"X", "-TG"=>"X", "-T-"=>"X", "-CT"=>"X", "-CC"=>"X", "-CA"=>"X", "-CG"=>"X", "-C-"=>"X", "-AT"=>"X", "-AC"=>"X", "-AA"=>"X", "-AG"=>"X", "-A-"=>"X", "-GT"=>"X", "-GC"=>"X", "-GA"=>"X", "-GG"=>"X", "-G-"=>"X", "--T"=>"-", "--C"=>"-", "--A"=>"-", "--G"=>"-", "---"=>"-", "NTT"=>"X", "TNT"=>"X", "TTN"=>"X", "NTC"=>"X", "TNC"=>"X", "TCN"=>"X", "NTA"=>"X", "TNA"=>"X", "TAN"=>"X", "NTG"=>"X", "TNG"=>"X", "TGN"=>"X", "NT-"=>"X", "TN-"=>"X", "T-N"=>"X", "NTN"=>"X", "TNN"=>"X", "NCT"=>"X", "CNT"=>"X", "CTN"=>"X", "NCC"=>"X", "CNC"=>"X", "CCN"=>"X", "NCA"=>"X", "CNA"=>"X", "CAN"=>"X", "NCG"=>"X", "CNG"=>"X", "CGN"=>"X", "NC-"=>"X", "CN-"=>"X", "C-N"=>"X", "NCN"=>"X", "CNN"=>"X", "NAT"=>"X", "ANT"=>"X", "ATN"=>"X", "NAC"=>"X", "ANC"=>"X", "ACN"=>"X", "NAA"=>"X", "ANA"=>"X", "AAN"=>"X", "NAG"=>"X", "ANG"=>"X", "AGN"=>"X", "NA-"=>"X", "AN-"=>"X", "A-N"=>"X", "NAN"=>"X", "ANN"=>"X", "NGT"=>"X", "GNT"=>"X", "GTN"=>"X", "NGC"=>"X", "GNC"=>"X", "GCN"=>"X", "NGA"=>"X", "GNA"=>"X", "GAN"=>"X", "NGG"=>"X", "GNG"=>"X", "GGN"=>"X", "NG-"=>"X", "GN-"=>"X", "G-N"=>"X", "NGN"=>"X", "GNN"=>"X", "N-T"=>"X", "-NT"=>"X", "-TN"=>"X", "N-C"=>"X", "-NC"=>"X", "-CN"=>"X", "N-A"=>"X", "-NA"=>"X", "-AN"=>"X", "N-G"=>"X", "-NG"=>"X", "-GN"=>"X", "N--"=>"X", "-N-"=>"X", "--N"=>"X", "N-N"=>"X", "-NN"=>"X", "NNT"=>"X", "NNC"=>"X", "NNA"=>"X", "NNG"=>"X", "NN-"=>"X", "NNN"=>"X")                 
######################################################################################################################################
function list_to_strings_no_spaces(list::String)
    string_vec = string.(split(list, ","))
    return string_vec
end
######################################################################################################################################
function stringlist_to_strings_nonEPI(txt::String)
    arr_of_strings = Vector{String}()
    no_newlines = replace(txt, "\n" =>" ")
    for seq in split(no_newlines, ", ")
        push!(arr_of_strings, seq)
    end
    sort_arr_of_strings = sort(collect(arr_of_strings), by = x -> nuc_mut_int_comprehensive_dict[x])  
    return sort_arr_of_strings
end
######################################################################################################################################
function list_to_string_array(txt::String) # similar to stringlist_to_strings but not for EPIs
    no_newlines = replace(txt, "\n" =>" ")
    string_array = string.(split(no_newlines, ", "))
    return string_array
end
###########################################################################################################################################################################
###########################################################################################################################################################################
function get_ref_pango_nucseq_and_geneseqs(ref_pango::String)
    ref_seq = "..."
    refAA_ORF1a = "MESLVPGFNEKTHVQLSLPVLQVRDVLVRGFGDSVEEVLSEARQHLKDGTCGLVEVEKGVLPQLEQPYVFIKRSDARTAPHGHVMVELVAELEGIQYGRSGETLGVLVPHVGEIPVAYRKVLLRKNGNKGAGGHSYGADLKSFDLGDELGTDPYEDFQENWNTKHSSGVTRELMRELNGGAYTRYVDNNFCGPDGYPLECIKDLLARAGKASCTLSEQLDFIDTKRGVYCCREHEHEIAWYTERSEKSYELQTPFEIKLAKKFDTFNGECPNFVFPLNSIIKTIQPRVEKKKLDGFMGRIRSVYPVASPNECNQMCLSTLMKCDHCGETSWQTGDFVKATCEFCGTENLTKEGATTCGYLPQNAVVKIYCPACHNSEVGPEHSLAEYHNESGLKTILRKGGRTIAFGGCVFSYVGCHNKCAYWVPRASANIGCNHTGVVGEGSEGLNDNLLEILQKEKVNINIVGDFKLNEEIAIILASFSASTSAFVETVKGLDYKAFKQIVESCGNFKVTKGKAKKGAWNIGEQKSILSPLYAFASEAARVVRSIFSRTLETAQNSVRVLQKAAITILDGISQYSLRLIDAMMFTSDLATNNLVVMAYITGGVVQLTSQWLTNIFGTVYEKLKPVLDWLEEKFKEGVEFLRDGWEIVKFISTCACEIVGGQIVTCAKEIKESVQTFFKLVNKFLALCADSIIIGGAKLKALNLGETFVTHSKGLYRKCVKSREETGLLMPLKAPKEIIFLEGETLPTEVLTEEVVLKTGDLQPLEQPTSEAVEAPLVGTPVCINGLMLLEIKDTEKYCALAPNMMVTNNTFTLKGGAPTKVTFGDDTVIEVQGYKSVNITFELDERIDKVLNEKCSAYTVELGTEVNEFACVVADAVIKTLQPVSELLTPLGIDLDEWSMATYYLFDESGEFKLASHMYCSFYPPDEDEEEGDCEEEEFEPSTQYEYGTEDDYQGKPLEFGATSAALQPEEEQEEDWLDDDSQQTVGQQDGSEDNQTTTIQTIVEVQPQLEMELTPVVQTIEVNSFSGYLKLTDNVYIKNADIVEEAKKVKPTVVVNAANVYLKHGGGVAGALNKATNNAMQVESDDYIATNGPLKVGGSCVLSGHNLAKHCLHVVGPNVNKGEDIQLLKSAYENFNQHEVLLAPLLSAGIFGADPIHSLRVCVDTVRTNVYLAVFDKNLYDKLVSSFLEMKSEKQVEQKIAEIPKEEVKPFITESKPSVEQRKQDDKKIKACVEEVTTTLEETKFLTENLLLYIDINGNLHPDSATLVSDIDITFLKKDAPYIVGDVVQEGVLTAVVIPTKKAGGTTEMLAKALRKVPTDNYITTYPGQGLNGYTVEEAKTVLKKCKSAFYILPSIISNEKQEILGTVSWNLREMLAHAEETRKLMPVCVETKAIVSTIQRKYKGIKIQEGVVDYGARFYFYTSKTTVASLINTLNDLNETLVTMPLGYVTHGLNLEEAARYMRSLKVPATVSVSSPDAVTAYNGYLTSSSKTPEEHFIETISLAGSYKDWSYSGQSTQLGIEFLKRGDKSVYYTSNPTTFHLDGEVITFDNLKTLLSLREVRTIKVFTTVDNINLHTQVVDMSMTYGQQFGPTYLDGADVTKIKPHNSHEGKTFYVLPNDDTLRVEAFEYYHTTDPSFLGRYMSALNHTKKWKYPQVNGLTSIKWADNNCYLATALLTLQQIELKFNPPALQDAYYRARAGEAANFCALILAYCNKTVGELGDVRETMSYLFQHANLDSCKRVLNVVCKTCGQQQTTLKGVEAVMYMGTLSYEQFKKGVQIPCTCGKQATKYLVQQESPFVMMSAPPAQYELKHGTFTCASEYTGNYQCGHYKHITSKETLYCIDGALLTKSSEYKGPITDVFYKENSYTTTIKPVTYKLDGVVCTEIDPKLDNYYKKDNSYFTEQPIDLVPNQPYPNASFDNFKFVCDNIKFADDLNQLTGYKKPASRELKVTFFPDLNGDVVAIDYKHYTPSFKKGAKLLHKPIVWHVNNATNKATYKPNTWCIRCLWSTKPVETSNSFDVLKSEDAQGMDNLACEDLKPVSEEVVENPTIQKDVLECNVKTTEVVGDIILKPANNSLKITEEVGHTDLMAAYVDNSSLTIKKPNELSRVLGLKTLATHGLAAVNSVPWDTIANYAKPFLNKVVSTTTNIVTRCLNRVCTNYMPYFFTLLLQLCTFTRSTNSRIKASMPTTIAKNTVKSVGKFCLEASFNYLKSPNFSKLINIIIWFLLLSVCLGSLIYSTAALGVLMSNLGMPSYCTGYREGYLNSTNVTIATYCTGSIPCSVCLSGLDSLDTYPSLETIQITISSFKWDLTAFGLVAEWFLAYILFTRFFYVLGLAAIMQLFFSYFAVHFISNSWLMWLIINLVQMAPISAMVRMYIFFASFYYVWKSYVHVVDGCNSSTCMMCYKRNRATRVECTTIVNGVRRSFYVYANGGKGFCKLHNWNCVNCDTFCAGSTFISDEVARDLSLQFKRPINPTDQSSYIVDSVTVKNGSIHLYFDKAGQKTYERHSLSHFVNLDNLRANNTKGSLPINVIVFDGKSKCEESSAKSASVYYSQLMCQPILLLDQALVSDVGDSAEVAVKMFDAYVNTFSSTFNVPMEKLKTLVATAEAELAKNVSLDNVLSTFISAARQGFVDSDVETKDVVECLKLSHQSDIEVTGDSCNNYMLTYNKVENMTPRDLGACIDCSARHINAQVAKSHNIALIWNVKDFMSLSEQLRKQIRSAAKKNNLPFKLTCATTRQVVNVVTTKIALKGGKIVNNWLKQLIKVTLVFLFVAAIFYLITPVHVMSKHTDFSSEIIGYKAIDGGVTRDIASTDTCFANKHADFDTWFSQRGGSYTNDKACPLIAAVITREVGFVVPGLPGTILRTTNGDFLHFLPRVFSAVGNICYTPSKLIEYTDFATSACVLAAECTIFKDASGKPVPYCYDTNVLEGSVAYESLRPDTRYVLMDGSIIQFPNTYLEGSVRVVTTFDSEYCRHGTCERSEAGVCVSTSGRWVLNNDYYRSLPGVFCGVDAVNLLTNMFTPLIQPIGALDISASIVAGGIVAIVVTCLAYYFMRFRRAFGEYSHVVAFNTLLFLMSFTVLCLTPVYSFLPGVYSVIYLYLTFYLTNDVSFLAHIQWMVMFTPLVPFWITIAYIICISTKHFYWFFSNYLKRRVVFNGVSFSTFEEAALCTFLLNKEMYLKLRSDVLLPLTQYNRYLALYNKYKYFSGAMDTTSYREAACCHLAKALNDFSNSGSDVLYQPPQTSITSAVLQSGFRKMAFPSGKVEGCMVQVTCGTTTLNGLWLDDVVYCPRHVICTSEDMLNPNYEDLLIRKSNHNFLVQAGNVQLRVIGHSMQNCVLKLKVDTANPKTPKYKFVRIQPGQTFSVLACYNGSPSGVYQCAMRPNFTIKGSFLNGSCGSVGFNIDYDCVSFCYMHHMELPTGVHAGTDLEGNFYGPFVDRQTAQAAGTDTTITVNVLAWLYAAVINGDRWFLNRFTTTLNDFNLVAMKYNYEPLTQDHVDILGPLSAQTGIAVLDMCASLKELLQNGMNGRTILGSALLEDEFTPFDVVRQCSGVTFQSAVKRTIKGTHHWLLLTILTSLLVLVQSTQWSLFFFLYENAFLPFAMGIIAMSAFAMMFVKHKHAFLCLFLLPSLATVAYFNMVYMPASWVMRIMTWLDMVDTSLSGFKLKDCVMYASAVVLLILMTARTVYDDGARRVWTLMNVLTLVYKVYYGNALDQAISMWALIISVTSNYSGVVTTVMFLARGIVFMCVEYCPIFFITGNTLQCIMLVYCFLGYFCTCYFGLFCLLNRYFRLTLGVYDYLVSTQEFRYMNSQGLLPPKNSIDAFKLNIKLLGVGGKPCIKVATVQSKMSDVKCTSVVLLSVLQQLRVESSSKLWAQCVQLHNDILLAKDTTEAFEKMVSLLSVLLSMQGAVDINKLCEEMLDNRATLQAIASEFSSLPSYAAFATAQEAYEQAVANGDSEVVLKKLKKSLNVAKSEFDRDAAMQRKLEKMADQAMTQMYKQARSEDKRAKVTSAMQTMLFTMLRKLDNDALNNIINNARDGCVPLNIIPLTTAAKLMVVIPDYNTYKNTCDGTTFTYASALWEIQQVVDADSKIVQLSEISMDNSPNLAWPLIVTALRANSAVKLQNNELSPVALRQMSCAAGTTQTACTDDNALAYYNTTKGGRFVLALLSDLQDLKWARFPKSDGTGTIYTELEPPCRFVTDTPKGPKVKYLYFIKGLNNLNRGMVLGSLAATVRLQAGNATEVPANSTVLSFCAFAVDAAKAYKDYLASGGQPITNCVKMLCTHTGTGQAITVTPEANMDQESFGGASCCLYCRCHIDHPNPKGFCDLKGKYVQIPTTCANDPVGFTLKNTVCTVCGMWKGYGCSCDQLREPMLQSADAQSFLNGFAV"
    refAA_ORF1b = "NRVCGVSAARLTPCGTGTSTDVVYRAFDIYNDKVAGFAKFLKTNCCRFQEKDEDDNLIDSYFVVKRHTFSNYQHEETIYNLLKDCPAVAKHDFFKFRIDGDMVPHISRQRLTKYTMADLVYALRHFDEGNCDTLKEILVTYNCCDDDYFNKKDWYDFVENPDILRVYANLGERVRQALLKTVQFCDAMRNAGIVGVLTLDNQDLNGNWYDFGDFIQTTPGSGVPVVDSYYSLLMPILTLTRALTAESHVDTDLTKPYIKWDLLKYDFTEERLKLFDRYFKYWDQTYHPNCVNCLDDRCILHCANFNVLFSTVFPPTSFGPLVRKIFVDGVPFVVSTGYHFRELGVVHNQDVNLHSSRLSFKELLVYAADPAMHAASGNLLLDKRTTCFSVAALTNNVAFQTVKPGNFNKDFYDFAVSKGFFKEGSSVELKHFFFAQDGNAAISDYDYYRYNLPTMCDIRQLLFVVEVVDKYFDCYDGGCINANQVIVNNLDKSAGFPFNKWGKARLYYDSMSYEDQDALFAYTKRNVIPTITQMNLKYAISAKNRARTVAGVSICSTMTNRQFHQKLLKSIAATRGATVVIGTSKFYGGWHNMLKTVYSDVENPHLMGWDYPKCDRAMPNMLRIMASLVLARKHTTCCSLSHRFYRLANECAQVLSEMVMCGGSLYVKPGGTSSGDATTAYANSVFNICQAVTANVNALLSTDGNKIADKYVRNLQHRLYECLYRNRDVDTDFVNEFYAYLRKHFSMMILSDDAVVCFNSTYASQGLVASIKNFKSVLYYQNNVFMSEAKCWTETDLTKGPHEFCSQHTMLVKQGDDYVYLPYPDPSRILGAGCFVDDIVKTDGTLMIERFVSLAIDAYPLTKHPNQEYADVFHLYLQYIRKLHDELTGHMLDMYSVMLTNDNTSRYWEPEFYEAMYTPHTVLQAVGACVLCNSQTSLRCGACIRRPFLCCKCCYDHVISTSHKLVLSVNPYVCNAPGCDVTDVTQLYLGGMSYYCKSHKPPISFPLCANGQVFGLYKNTCVGSDNVTDFNAIATCDWTNAGDYILANTCTERLKLFAAETLKATEETFKLSYGIATVREVLSDRELHLSWEVGKPRPPLNRNYVFTGYRVTKNSKVQIGEYTFEKGDYGDAVVYRGTTTYKLNVGDYFVLTSHTVMPLSAPTLVPQEHYVRITGLYPTLNISDEFSSNVANYQKVGMQKYSTLQGPPGTGKSHFAIGLALYYPSARIVYTACSHAAVDALCEKALKYLPIDKCSRIIPARARVECFDKFKVNSTLEQYVFCTVNALPETTADIVVFDEISMATNYDLSVVNARLRAKHYVYIGDPAQLPAPRTLLTKGTLEPEYFNSVCRLMKTIGPDMFLGTCRRCPAEIVDTVSALVYDNKLKAHKDKSAQCFKMFYKGVITHDVSSAINRPQIGVVREFLTRNPAWRKAVFISPYNSQNAVASKILGLPTQTVDSSQGSEYDYVIFTQTTETAHSCNVNRFNVAITRAKVGILCIMSDRDLYDKLQFTSLEIPRRNVATLQAENVTGLFKDCSKVITGLHPTQAPTHLSVDTKFKTEGLCVDIPGIPKDMTYRRLISMMGFKMNYQVNGYPNMFITREEAIRHVRAWIGFDVEGCHATREAVGTNLPLQLGFSTGVNLVAVPTGYVDTPNNTDFSRVSAKPPPGDQFKHLIPLMYKGLPWNVVRIKIVQMLSDTLKNLSDRVVFVLWAHGFELTSMKYFVKIGPERTCCLCDRRATCFSTASDTYACWHHSIGFDYVYNPFMIDVQQWGFTGNLQSNHDLYCQVHGNAHVASCDAIMTRCLAVHECFVKRVDWTIEYPIIGDELKINAACRKVQHMVVKAALLADKFPVLHDIGNPKAIKCVPQADVEWKFYDAQPCSDKAYKIEELFYSYATHSDKFTDGVCLFWNCNVDRYPANSIVCRFDTRVLSNLNLPGCDGGSLYVNKHAFHTPAFDKSAFVNLKQLPFFYYSDSPCESHGKQVVSDIDYVPLKSATCITRCNLGGAVCRHHANEYRLYLDAYNMMISAGFSLWVYKQFDTYNLWNTFTRLQSLENVAFNVVNKGHFDGQQGEVPVSIINNTVYTKVDGVDVELFENKTTLPVNVAFELWAKRNIKPVPEVKILNNLGVDIAANTVIWDYKRDAPAHISTIGVCSMTDIAKKPTETICAPLTVFFDGRVDGQVDLFRNARNGVLITEGSVKGLQPSVGPKQASLNGVTLIGEAVKTQFNYYKKVDGVVQQLPETYFTQSRNLQEFKPRSQMEIDFLELAMDEFIERYKLEGYAFEHIVYGDFSHSQLGGLHLLIGLAKRFKESPFELEDFIPMDSTVKNYFITDAQTGSSKCVCSVIDLLLDDFVEIIKSQDLSVVSKVVKVTIDYTEISFMLWCKDGHVETFYPKLQSSQAWQPGVAMPNLYKMQRMLLEKCDLQNYGDSATLPKGIMMNVAKYTQLCQYLNTLTLAVPYNMRVIHFGAGSDKGVAPGTAVLRQWLPTGTLLVDSDLNDFVSDADSTLIGDCATVHTANKWDLIISDMYDPKTKNVTKENDSKEGFFTYICGFIQQKLALGGSVAIKITEHSWNADLYKLMGHFAWWTAFVTNVNASSSEAFLIGCNYLGKPREQIDGYVMHANYIFWRNTNPIQLSSYSLFDMSKFPLKLRGTAVMSLKEGQINDMILSLLSKGRLIIRENNRVVISSDVLVNN*"
    refAA_S = "MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSSVLHSTQDLFLPFFSNVTWFHAIHVSGTNGTKRFDNPVLPFNDGVYFASTEKSNIIRGWIFGTTLDSKTQSLLIVNNATNVVIKVCEFQFCNDPFLGVYYHKNNKSWMESEFRVYSSANNCTFEYVSQPFLMDLEGKQGNFKNLREFVFKNIDGYFKIYSKHTPINLVRDLPQGFSALEPLVDLPIGINITRFQTLLALHRSYLTPGDSSSGWTAGAAAYYVGYLQPRTFLLKYNENGTITDAVDCALDPLSETKCTLKSFTVEKGIYQTSNFRVQPTESIVRFPNITNLCPFGEVFNATRFASVYAWNRKRISNCVADYSVLYNSASFSTFKCYGVSPTKLNDLCFTNVYADSFVIRGDEVRQIAPGQTGKIADYNYKLPDDFTGCVIAWNSNNLDSKVGGNYNYLYRLFRKSNLKPFERDISTEIYQAGSTPCNGVEGFNCYFPLQSYGFQPTNGVGYQPYRVVVLSFELLHAPATVCGPKKSTNLVKNKCVNFNFNGLTGTGVLTESNKKFLPFQQFGRDIADTTDAVRDPQTLEILDITPCSFGGVSVITPGTNTSNQVAVLYQDVNCTEVPVAIHADQLTPTWRVYSTGSNVFQTRAGCLIGAEHVNNSYECDIPIGAGICASYQTQTNSPRRARSVASQSIIAYTMSLGAENSVAYSNNSIAIPTNFTISVTTEILPVSMTKTSVDCTMYICGDSTECSNLLLQYGSFCTQLNRALTGIAVEQDKNTQEVFAQVKQIYKTPPIKDFGGFNFSQILPDPSKPSKRSFIEDLLFNKVTLADAGFIKQYGDCLGDIAARDLICAQKFNGLTVLPPLLTDEMIAQYTSALLAGTITSGWTFGAGAALQIPFAMQMAYRFNGIGVTQNVLYENQKLIANQFNSAIGKIQDSLSSTASALGKLQDVVNQNAQALNTLVKQLSSNFGAISSVLNDILSRLDKVEAEVQIDRLITGRLQSLQTYVTQQLIRAAEIRASANLAATKMSECVLGQSKRVDFCGKGYHLMSFPQSAPHGVVFLHVTYVPAQEKNFTTAPAICHDGKAHFPREGVFVSNGTHWFVTQRNFYEPQIITTDNTFVSGNCDVVIGIVNNTVYDPLQPELDSFKEELDKYFKNHTSPDVDLGDISGINASVVNIQKEIDRLNEVAKNLNESLIDLQELGKYEQYIKWPWYIWLGFIAGLIAIVMVTIMLCCMTSCCSCLKGCCSCGSCCKFDEDDSEPVLKGVKLHYT*"
    refAA_ORF3a = "MDLFMRIFTIGTVTLKQGEIKDATPSDFVRATATIPIQASLPFGWLIVGVALLAVFQSASKIITLKKRWQLALSKGVHFVCNLLLLFVTVYSHLLLVAAGLEAPFLYLYALVYFLQSINFVRIIMRLWLCWKCRSKNPLLYDANYFLCWHTNCYDYCIPYNSVTSSIVITSGDGTTSPISEHDYQIGGYTEKWESGVKDCVVLHSYFTSDYYQLYSTQLSTDTGVEHVTFFIYNKIVDEPEEHVQIHTIDGSSGVVNPVMEPIYDEPTTTTSVPL*"
    refAA_E = "MYSFVSEETGTLIVNSVLLFLAFVVFLLVTLAILTALRLCAYCCNIVNVSLVKPSFYVYSRVKNLNSSRVPDLLV*"
    refAA_M = "MADSNGTITVEELKKLLEQWNLVIGFLFLTWICLLQFAYANRNRFLYIIKLIFLWLLWPVTLACFVLAAVYRINWITGGIAIAMACLVGLMWLSYFIASFRLFARTRSMWSFNPETNILLNVPLHGTILTRPLLESELVIGAVILRGHLRIAGHHLGRCDIKDLPKEITVATSRTLSYYKLGASQRVAGDSGFAAYSRYRIGNYKLNTDHSSSSDNIALLVQ*"
    refAA_ORF6 = "MFHLVDFQVTIAEILLIIMRTFKVSIWNLDYIINLIIKNLSKSLTENKYSQLDEEQPMEID*"
    refAA_ORF7a = "MKIILFLALITLATCELYHYQECVRGTTVLLKEPCSSGTYEGNSPFHPLADNKFALTCFSTQFAFACPDGVKHVYQLRARSVSPKLFIRQEEVQELYSPIFLIVAAIVFITLCFTLKRKTE*"
    refAA_ORF7b = "MIELSLIDFYLCFLAFLLFLVLIMLIIFWFSLELQDHNETCHA*"
    refAA_ORF8 = "MKFLVFLGIITTVAAFHQECSLQSCTQHQPYVVDDPCPIHFYSKWYIRVGARKSAPLIELCVDEAGSKSPIQYIDIGNYTVSCLPFTINCQEPKLGSLVVRCSFYEDFLEYHDVRVVLDFI*"
    refAA_N = "MSDNGPQNQRNAPRITFGGPSDSTGSNQNGERSGARSKQRRPQGLPNNTASWFTALTQHGKEDLKFPRGQGVPINTNSSPDDQIGYYRRATRRIRGGDGKMKDLSPRWYFYYLGTGPEAGLPYGANKDGIIWVATEGALNTPKDHIGTRNPANNAAIVLQLPQGTTLPKGFYAEGSRGGSQASSRSSSRSRNSSRNSTPGSSRGTSPARMAGNGGDAALALLLLDRLNQLESKMSGKGQQQQGQTVTKKSAAEASKKPRQKRTATKAYNVTQAFGRRGPEQTQGNFGDQELIRQGTDYKHWPQIAQFAPSASAFFGMSRIGMEVTPSGTWLTYTGAIKLDDKDPNFKDQVILLNKHIDAYKTFPPTEPKKDKKKKADETQALPQRQKKQQTVTLLPAADLDDFSKQLQQSMSSADSTQA*"
    refAA_ORF9b = "MDPKISEMHPALRLVDPQIQLAVTRMENAVGRDQNNVGPKVYPIILRLGSPLSLNMARKTLNSLEDKAFQLTPIAVQMTKLATTEELPDEFVVVTVK*"
    if ref_pango ≠ "Wuhan"
        if haskey(nuc_genome_pango_dict, ref_pango)
            ref_seq = nuc_genome_pango_dict[ref_pango]
        elseif haskey(pango_predecessor_meta_dict, ref_pango)
            minus1_pango = ""
            minus2_pango = ""
            minus3_pango = ""
            minus4_pango = ""
            if haskey(pango_predecessor_meta_dict[ref_pango], 1)
                minus1_pango = pango_predecessor_meta_dict[ref_pango][1]
            end
            if haskey(pango_predecessor_meta_dict[ref_pango], 2)
                minus2_pango = pango_predecessor_meta_dict[ref_pango][2]
            end
            if haskey(pango_predecessor_meta_dict[ref_pango], 3)
                minus3_pango = pango_predecessor_meta_dict[ref_pango][3]
            end
            if haskey(pango_predecessor_meta_dict[ref_pango], 4)
                minus4_pango = pango_predecessor_meta_dict[ref_pango][4]
            end
            minus_pango_vec = [minus1_pango, minus2_pango, minus3_pango, minus4_pango]
            for minus_pango_X in minus_pango_vec
                if haskey(nuc_genome_pango_dict, minus_pango_X)
                    ref_seq = nuc_genome_pango_dict[minus_pango_X]
                    break
                end
            end
        else
            for x in 1:5
                minus_pango_X = pango_minus_X_fx(ref_pango, x)
                if haskey(nuc_genome_pango_dict, minus_pango_X)
                    ref_seq = nuc_genome_pango_dict[minus_pango_X]
                    break
                end
            end
        end
##########################################################################################
        if haskey(gene_AA_pango_dict, ref_pango)
            refAA_ORF1a = gene_AA_pango_dict[ref_pango]["ORF1a"]
            refAA_ORF1b = gene_AA_pango_dict[ref_pango]["ORF1b"]
            refAA_S = gene_AA_pango_dict[ref_pango]["S"]
            refAA_ORF3a = gene_AA_pango_dict[ref_pango]["ORF3a"]
            refAA_E = gene_AA_pango_dict[ref_pango]["E"]
            refAA_M = gene_AA_pango_dict[ref_pango]["M"]
            refAA_ORF6 = gene_AA_pango_dict[ref_pango]["ORF6"]
            refAA_ORF7a = gene_AA_pango_dict[ref_pango]["ORF7a"]
            refAA_ORF7b = gene_AA_pango_dict[ref_pango]["ORF7b"]
            refAA_ORF8 = gene_AA_pango_dict[ref_pango]["ORF8"]
            refAA_N = gene_AA_pango_dict[ref_pango]["N"]
            refAA_ORF9b = gene_AA_pango_dict[ref_pango]["ORF9b"]
        elseif haskey(pango_predecessor_meta_dict, ref_pango)
            minus1_pango = ""
            minus2_pango = ""
            minus3_pango = ""
            minus4_pango = ""
            if haskey(pango_predecessor_meta_dict[ref_pango], 1)
                minus1_pango = pango_predecessor_meta_dict[ref_pango][1]
            end
            if haskey(pango_predecessor_meta_dict[ref_pango], 2)
                minus2_pango = pango_predecessor_meta_dict[ref_pango][2]
            end
            if haskey(pango_predecessor_meta_dict[ref_pango], 3)
                minus3_pango = pango_predecessor_meta_dict[ref_pango][3]
            end
            if haskey(pango_predecessor_meta_dict[ref_pango], 4)
                minus4_pango = pango_predecessor_meta_dict[ref_pango][4]
            end
            minus_pango_vec = [minus1_pango, minus2_pango, minus3_pango, minus4_pango]
            for minus_pango_X in minus_pango_vec
                if haskey(gene_AA_pango_dict, minus_pango_X)
                    refAA_ORF1a = gene_AA_pango_dict[minus_pango_X]["ORF1a"]
                    refAA_ORF1b = gene_AA_pango_dict[minus_pango_X]["ORF1b"]
                    refAA_S = gene_AA_pango_dict[minus_pango_X]["S"]
                    refAA_ORF3a = gene_AA_pango_dict[minus_pango_X]["ORF3a"]
                    refAA_E = gene_AA_pango_dict[minus_pango_X]["E"]
                    refAA_M = gene_AA_pango_dict[minus_pango_X]["M"]
                    refAA_ORF6 = gene_AA_pango_dict[minus_pango_X]["ORF6"]
                    refAA_ORF7a = gene_AA_pango_dict[minus_pango_X]["ORF7a"]
                    refAA_ORF7b = gene_AA_pango_dict[minus_pango_X]["ORF7b"]
                    refAA_ORF8 = gene_AA_pango_dict[minus_pango_X]["ORF8"]
                    refAA_N = gene_AA_pango_dict[minus_pango_X]["N"]
                    refAA_ORF9b = gene_AA_pango_dict[minus_pango_X]["ORF9b"]
                    break
                end
            end
        else
            for x in 1:5
                minus_pango_X = pango_minus_X_fx(ref_pango, x)
                if haskey(gene_AA_pango_dict, minus_pango_X)
                    refAA_ORF1a = gene_AA_pango_dict[minus_pango_X]["ORF1a"]
                    refAA_ORF1b = gene_AA_pango_dict[minus_pango_X]["ORF1b"]
                    refAA_S = gene_AA_pango_dict[minus_pango_X]["S"]
                    refAA_ORF3a = gene_AA_pango_dict[minus_pango_X]["ORF3a"]
                    refAA_E = gene_AA_pango_dict[minus_pango_X]["E"]
                    refAA_M = gene_AA_pango_dict[minus_pango_X]["M"]
                    refAA_ORF6 = gene_AA_pango_dict[minus_pango_X]["ORF6"]
                    refAA_ORF7a = gene_AA_pango_dict[minus_pango_X]["ORF7a"]
                    refAA_ORF7b = gene_AA_pango_dict[minus_pango_X]["ORF7b"]
                    refAA_ORF8 = gene_AA_pango_dict[minus_pango_X]["ORF8"]
                    refAA_N = gene_AA_pango_dict[minus_pango_X]["N"]
                    refAA_ORF9b = gene_AA_pango_dict[minus_pango_X]["ORF9b"]
                    break
                end
            end
        end
    end
    return ref_seq, refAA_ORF1a, refAA_ORF1b, refAA_S, refAA_ORF3a, refAA_E, refAA_M, refAA_ORF6, refAA_ORF7a, refAA_ORF7b, refAA_ORF8, refAA_N, refAA_ORF9b
end
###########################################################################################################################################################################
###########################################################################################################################################################################
function mixed_nucs_filter(mut_arr)
    mixed_nuc_arr = Vector{String}()
    mixed_nuc_res_list = Set(["Y", "R", "K", "M", "W", "S"])
    for mut in mut_arr
        if string(mut[end]) in mixed_nuc_res_list
            push!(mixed_nuc_arr, mut)
        end
    end
    return mixed_nuc_arr
end
#####################################################################################################################################
N3_syn = ["TCT", "TCC", "TCA", "TCG", "CTT", "CTC", "CTA", "CTG", "CCT", "CCC", "CCA", "CCG", "CGT", "CGC", "CGA", "CGG", "ACT", "ACC", "ACA", "ACG", "GTT", "GTC", "GTA", "GTG", "GCT", "GCC", "GCA", "GCG", "GGT", "GGC", "GGA", "GGG"]
N3_tv = ["TTT", "TTC", "TTA", "TTG", "TAT", "TAC", "TAA", "TAG", "AAT", "AAC", "AAA", "AAG", "AGT", "AGC", "AGA", "AGG", "GAT", "GAC", "GAA", "GAG"]
#####################################################################################################################################
function muts_to_strings(mut_list_in_string_form::String)
    mut_arr = split(mut_list_in_string_form, ", ")
    mut_array = Vector{String}()
    for mut in mut_arr
        if string(mut[end]) ≠ "-"
            mutstr = string(mut)
            push!(mut_array, mutstr)
        end
    end
    sortKey(n) = (length(n), nuc_mut_int_comprehensive_dict[n])  ## Fx ##
    mut_array_sort = sort(mut_array, by = x -> sortKey(x))
#    mixed_mut_arr = mixed_nucs_filter(mut_arr)
    return mut_array_sort
end
######################################################################################################################################
function mixed_mut_to_regular_mut(mixed_nuc_muts)            ### New, 2025-1-26 (entire function)
    ct = 0
    mixed_regular_muts = Vector{String}()
    for i in 1:length(mixed_nuc_muts)
        mut = mixed_nuc_muts[i]
        qrynuc = qry_nuc_comprehensive_dict[mut]
        refnuc = ref_nuc_comprehensive_dict[mut]
        pos_str = nuc_mut_int_string_comprehensive_dict[mut]
        ref_n_pos = refnuc*pos_str
        if refnuc == "T"
            if qrynuc == "Y"
                new_end = "C"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if qrynuc == "R"
                new_end1 = "A"
                new_end2 = "G"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if qrynuc == "K"
                new_end = "G"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if qrynuc == "M"
                new_end1 = "C"
                new_end2 = "A"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if qrynuc == "W"
                new_end = "A"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if qrynuc == "S"
                new_end1 = "C"
                new_end2 = "G"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
        end
        if refnuc == "C"
            if qrynuc == "Y"
                new_end = "T"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if qrynuc == "R"
                new_end1 = "A"
                new_end2 = "G"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if qrynuc == "K"
                new_end1 = "T"
                new_end2 = "G"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if qrynuc == "M"
                new_end = "A"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if qrynuc == "W"
                new_end1 = "T"
                new_end2 = "A"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if qrynuc == "S"
                new_end = "G"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
        end
        if refnuc == "A"
            if qrynuc == "Y"
                new_end1 = "T"
                new_end2 = "C"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if qrynuc == "R"
                new_end = "G"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if qrynuc == "K"
                new_end1 = "T"
                new_end2 = "G"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if qrynuc == "M"
                new_end = "C"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if qrynuc == "W"
                new_end = "T"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if qrynuc == "S"
                new_end1 = "C"
                new_end2 = "G"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
        end
        if refnuc == "G"
            if qrynuc == "Y"
                new_end1 = "T"
                new_end2 = "C"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if qrynuc == "R"
                new_end = "A"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if qrynuc == "K"
                new_end = "T"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if qrynuc == "M"
                new_end1 = "C"
                new_end2 = "A"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if qrynuc == "W"
                new_end1 = "T"
                new_end2 = "A"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if qrynuc == "S"
                new_end = "C"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
        end
    end
    return mixed_regular_muts
end
######################################################################################################################################
gene_print_array = ["S", "N", "E", "M", "ORF3a", "ORF6", "ORF7a", "ORF7b", "ORF8", "ORF9b", "ORF1a", "ORF1b"]
    coding_ranges = BitSet([266:13467..., 13469:21555..., 21563:25384..., 25393:26220..., 26245:26472..., 26523:27191..., 27202:27387..., 27394:27755..., 27760:27887..., 27894:28259..., 28274:29533..., 28284:28577...])
    noncoding_ranges = BitSet([1:265..., 21556:21562..., 25385:25392..., 26221:26244..., 26473:26522..., 27192:27201..., 27388:27393..., 27888:27893..., 28260:28273..., 29534:29830...])
    coding_range_9b = BitSet([28284:28577...])
    gene_nuc_starts = Dict{Int, Int}(0=>263, 1=>13465, 2=>21560, 3=>25390, 4=>26242, 5=>26520, 6=>27199, 7=>27391, 8=>27753, 9=>27891, 10=>28271, 11=>28281)
    ref_AA_genes = Dict{Int, String}(0=>refAA_ORF1a, 1=>refAA_ORF1b, 2=>refAA_S, 3=>refAA_ORF3a, 4=>refAA_E, 5=>refAA_M, 6=>refAA_ORF6, 7=>refAA_ORF7a, 8=>refAA_ORF7b, 9=>refAA_ORF8, 10=>refAA_N, 11=>refAA_ORF9b)
    gene_strings = Dict{Int, String}(0=>"ORF1a", 1=>"ORF1b", 2=>"S", 3=>"ORF3a", 4=>"E", 5=>"M", 6=>"ORF6", 7=>"ORF7a", 8=>"ORF7b", 9=>"ORF8", 10=>"N", 11=>"ORF9b")
    noncoding_range_dict = Dict{Vector{Int}, String}([1, 265]=>"5' UTR", [21556, 21562]=>"Spike TRS", [25385, 25392]=>"ORF3a TRS", [26221, 26234]=>"ORF3a-E UTR", [26235, 26244]=>"E TRS", [26473, 26522]=>"E-M UTR", [27192, 27201]=>"M-ORF6 UTR", [27388, 27393]=>"ORF7a TRS", [27888, 27893]=>"ORF8 TRS", [28260, 28273]=>"N/ORF9b TRS", [29534, 29830]=>"3' UTR", [29794, 29801]=>"ONM")
    gene_nuc_arr = [[266:13467...], [13469:21555...], [21563:25384...], [25393:26220...], [26245:26472...], [26523:27191...], [27202:27387...], [27394:27755...], [27760:27887...], [27894:28259...], [28274:29533...], [28284:28577...]]
    rem0_gene = [5, 8, 9, 11]
    rem1_gene = [1, 3, 4, 6, 7]
    rem2_gene = [0, 2, 10]
    rem0 = BitSet([26523:27191..., 27760:27887..., 27894:28259..., 28284:28577...])
    rem1 = BitSet([13469:21555..., 25393:26220..., 26245:26472..., 27202:27387..., 27394:27755...])
    rem2 = BitSet([266:13467..., 21563:25384..., 28274:29533...])
    rem9b = BitSet([28284:28577...])
    rem7ab = BitSet([27756:27759...])
    coding_ranges = BitSet([266:13467..., 13468:21555..., 21563:25384..., 25393:26220..., 26245:26472..., 26523:27191..., 27202:27387..., 27394:27755..., 27760:27887..., 27894:28259..., 28274:29533...])
    N_9b_synonymous = Set(["C28379A", "C28394A", "T28406C", "C28475A", "C28535A", "A28547C"])
#######################################################################################################################################
function nuc_to_AA(ref_pango::String, muts::Vector{String})
    muts_filtered = filter(!isempty, muts)
#    all_muts_sort = sort(muts_filtered, by = x -> nuc_mut_int_comprehensive_dict[x])
    ref_seq, refAA_ORF1a, refAA_ORF1b, refAA_S, refAA_ORF3a, refAA_E, refAA_M, refAA_ORF6, refAA_ORF7a, refAA_ORF7b, refAA_ORF8, refAA_N, refAA_ORF9b = get_ref_pango_nucseq_and_geneseqs(ref_pango)
    nuc_gene_num = Dict{Int, Int}()
    nuc_gene_num_9b = Dict{Int, Int}()
    nonsynonymous_nuc_to_AA_dict = Dict{String, String}()
    nonsynonymous_nuc_to_context_dict = Dict{String, String}()
    all_nuc_to_AA_dict = Dict{String, String}()
    all_nuc_to_context_dict = Dict{String, String}()
    synonymous_nuc_to_AA_dict = Dict{String, String}()
    synonymous_nuc_to_context_dict = Dict{String, String}()
    noncoding_nuc_to_context_dict = Dict{String, String}()
    noncoding_to_noncoding_region_dict = Dict{String, String}()
    noncoding_nuc_vector = Vector{String}()
################################################        
    for i in 1:length(gene_nuc_arr)-1
        for nuc_pos in gene_nuc_arr[i]
            nuc_gene_num[nuc_pos] = i-1
        end
    end
    for nuc_pos in gene_nuc_arr[end]
        nuc_gene_num_9b[nuc_pos] = 11
    end
    
    nuc_codon_pos_dict = Dict{Int, Int}()
    for nuc_pos in coding_ranges
        gene_number = nuc_gene_num[nuc_pos]
        gene_start = gene_nuc_starts[gene_number]
        codon_num = (nuc_pos-gene_start)%3 + 1
        nuc_codon_pos_dict[nuc_pos] = codon_num
    end
    nuc_codon_pos_dict_9b = Dict{Int, Int}()
    for nuc_pos in coding_range_9b
        gene_number = 11
        gene_start = gene_nuc_starts[gene_number]
        codon_num = (nuc_pos-gene_start)%3 + 1
        nuc_codon_pos_dict_9b[nuc_pos] = codon_num
    end
#########################################################################################################################################################
#########################################################################################################################################################
    gene_num(nuc_mut) = nuc_gene_num[nuc_mut_int_comprehensive_dict[nuc_mut]]                                          ### FUNCTION ###
    nuc_to_AA_pos(nuc_mut) = string((nuc_mut_int_comprehensive_dict[nuc_mut] - gene_nuc_starts[gene_num(nuc_mut)])÷3)  ### FUNCTION ###
    nuc_to_AA_pos_9b(nuc_mut) = string((nuc_mut_int_comprehensive_dict[nuc_mut] - 28281)÷3)                            ### FUNCTION ###
    nuc2AA_ORF1a(nuc_mut, refAA, qryAA) = gene_strings[gene_num(nuc_mut)]*":"*refAA*nuc_to_AA_pos(nuc_mut)*qryAA       ### FUNCTION ###
    nuc2AA_ORF9b(nuc_mut, refAA, qryAA) = "ORF9b:"*refAA*nuc_to_AA_pos_9b(nuc_mut)*qryAA
#########################################################################################################################################################
#########################################################################################################################################################
    for nuc_mut in muts
        mut = mixed2nuc(nuc_mut)
        if ',' in mut
            mut1 = string(split(mut, ",")[1])
            mut2 = string(split(mut, ",")[2])
            push!(muts, mut1)
            push!(muts, mut2)
            filter!(x -> !(length(x)>6), muts)
        end
    end
    AA_mut_set = Set{String}()
    AA_mut = ""
    for nuc_mut in muts
        pos = nuc_mut_int_comprehensive_dict[nuc_mut]
        if pos in coding_ranges
            mut = mixed2nuc(nuc_mut)  
            gene_number = gene_num(mut)
            ref_triple = ""
            qry_triple = ""
            ref_triple_context = ""
            qry_triple_context = ""
            ref2qry_context = ""
            if nuc_codon_pos_dict[pos] == 1
                ref_triple = ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])  *string(ref_seq[pos+2])
                qry_triple = qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
                ref_triple_context = string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])*string(ref_seq[pos+7])*string(ref_seq[pos+8])
                qry_triple_context = string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])*string(ref_seq[pos+7])*string(ref_seq[pos+8])
            elseif nuc_codon_pos_dict[pos] == 2
                ref_triple = string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]  *string(ref_seq[pos+1])
                qry_triple = string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                ref_triple_context = string(ref_seq[pos-7])*string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*ref_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])*string(ref_seq[pos+7])
                qry_triple_context = string(ref_seq[pos-7])*string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*qry_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])*string(ref_seq[pos+7])
            elseif nuc_codon_pos_dict[pos] == 3
                ref_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]
                qry_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]
                ref_triple_context = string(ref_seq[pos-8])*string(ref_seq[pos-7])*string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*ref_triple*string(ref_seq[pos+1])*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])
                qry_triple_context = string(ref_seq[pos-8])*string(ref_seq[pos-7])*string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*qry_triple*string(ref_seq[pos+1])*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])
            end
            refAA = AA_triplets[ref_triple]
            qryAA = AA_triplets[qry_triple]
            AA_mut = nuc2AA_ORF1a(mut, refAA, qryAA)
            push!(AA_mut_set, AA_mut)
            ref2qry_context = ref_triple_context*"-->"*qry_triple_context
            all_nuc_to_AA_dict[mut] = AA_mut
            if refAA == qryAA && !(pos in rem9b)
                synonymous_nuc_to_AA_dict[mut] = AA_mut
                synonymous_nuc_to_context_dict[mut] = ref2qry_context
            elseif refAA == qryAA && pos in rem9b && mut in N_9b_synonymous
                synonymous_nuc_to_AA_dict[mut] = AA_mut
                synonymous_nuc_to_context_dict[mut] = ref2qry_context
            else
                nonsynonymous_nuc_to_AA_dict[mut] = AA_mut
                nonsynonymous_nuc_to_context_dict[mut] = ref2qry_context
#                push!(nonsynonymous_nuc_muts, mut)
            end
            all_nuc_to_context_dict[mut] = ref2qry_context
###################################
            for nuc_mut2 in muts
                mut2 = mixed2nuc(nuc_mut2)
                pos2 = nuc_mut_int_comprehensive_dict[mut2]
                if pos2 in coding_ranges
                    gene_number2 = gene_num(mut2)
                    if mut ≠ mut2 && gene_number == gene_number2 && nuc_to_AA_pos(mut) == nuc_to_AA_pos(mut2)
                        if nuc_codon_pos_dict[pos] == 1 && nuc_codon_pos_dict[pos2] == 2
                            ref_triple = ref_nuc_comprehensive_dict[mut]*ref_nuc_comprehensive_dict[mut2]*string(ref_seq[pos+2])
                            qry_triple = qry_nuc_comprehensive_dict[mut]*qry_nuc_comprehensive_dict[mut2]*string(ref_seq[pos+2])
                            ref_triple_context = string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])
                            qry_triple_context = string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])
                        elseif nuc_codon_pos_dict[pos] == 1 && nuc_codon_pos_dict[pos2] == 3
                            ref_triple = ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*ref_nuc_comprehensive_dict[mut2]
                            qry_triple = qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*qry_nuc_comprehensive_dict[mut2]
                            ref_triple_context = string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])
                            qry_triple_context = string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])
                        elseif nuc_codon_pos_dict[pos] == 2 && nuc_codon_pos_dict[pos2] == 3
                            ref_triple = string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]*ref_nuc_comprehensive_dict[mut2]
                            qry_triple = string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]*qry_nuc_comprehensive_dict[mut2]
                            ref_triple_context = string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*ref_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])
                            qry_triple_context = string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*qry_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])
                        elseif nuc_codon_pos_dict[pos2] == 1 && nuc_codon_pos_dict[pos] == 2
                            ref_triple = ref_nuc_comprehensive_dict[mut2]*ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                            qry_triple = qry_nuc_comprehensive_dict[mut2]*qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                            ref_triple_context = string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*ref_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])
                            qry_triple_context = string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*qry_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])
                        elseif nuc_codon_pos_dict[pos2] == 1 && nuc_codon_pos_dict[pos] == 3
                            ref_triple = ref_nuc_comprehensive_dict[mut2]*string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]
                            qry_triple = qry_nuc_comprehensive_dict[mut2]*string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]
                            ref_triple_context = string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*ref_triple*string(ref_seq[pos+1])*string(ref_seq[pos+2])*string(ref_seq[pos+3])
                            qry_triple_context = string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*qry_triple*string(ref_seq[pos+1])*string(ref_seq[pos+2])*string(ref_seq[pos+3])
                        elseif nuc_codon_pos_dict[pos2] == 2 && nuc_codon_pos_dict[pos] == 3
                            ref_triple = string(ref_seq[pos-2])*ref_nuc_comprehensive_dict[mut2]*ref_nuc_comprehensive_dict[mut]
                            qry_triple = string(ref_seq[pos-2])*qry_nuc_comprehensive_dict[mut2]*qry_nuc_comprehensive_dict[mut]
                        end
                        refAA2 = AA_triplets[ref_triple]
                        qryAA2 = AA_triplets[qry_triple]
                        AA_mut2 = nuc2AA_ORF1a(mut2, refAA2, qryAA2)
                        push!(AA_mut_set, AA_mut2)
                        ref2qry_context = ref_triple_context*"-->"*qry_triple_context
                        all_nuc_to_AA_dict[mut2] = AA_mut2
                        all_nuc_to_AA_dict[mut] = AA_mut2
                        if refAA2 == qryAA2 && !(pos2 in rem9b)
                            synonymous_nuc_to_AA_dict[mut2] = AA_mut2
                            synonymous_nuc_to_context_dict[mut2] = ref2qry_context 
                        else
                            nonsynonymous_nuc_to_AA_dict[mut2] = AA_mut2
                            nonsynonymous_nuc_to_context_dict[mut2] = ref2qry_context
                            nonsynonymous_nuc_to_AA_dict[mut] = AA_mut2
                            nonsynonymous_nuc_to_context_dict[mut] = ref2qry_context
                        end
                        all_nuc_to_context_dict[mut] = ref2qry_context
                        all_nuc_to_context_dict[mut2] = ref2qry_context
                    end
                end
            end
        else                  
            npos = nuc_mut_int_comprehensive_dict[nuc_mut]
            qry_nuc = qry_nuc_comprehensive_dict[nuc_mut]
            ref_nc_nuc_context = string(ref_seq[npos-8])*string(ref_seq[npos-7])*string(ref_seq[npos-6])*string(ref_seq[npos-5])*string(ref_seq[npos-4])*string(ref_seq[npos-3])*string(ref_seq[npos-2])*string(ref_seq[npos-1])*string(ref_seq[npos])*string(ref_seq[npos+1])*string(ref_seq[npos+2])*string(ref_seq[npos+3])*string(ref_seq[npos+4])*string(ref_seq[npos+5])*string(ref_seq[npos+6])*string(ref_seq[npos+7])*string(ref_seq[npos+8])
            qry_nc_nuc_context = string(ref_seq[npos-8])*string(ref_seq[npos-7])*string(ref_seq[npos-6])*string(ref_seq[npos-5])*string(ref_seq[npos-4])*string(ref_seq[npos-3])*string(ref_seq[npos-2])*string(ref_seq[npos-1])*qry_nuc*string(ref_seq[npos+1])*string(ref_seq[npos+2])*string(ref_seq[npos+3])*string(ref_seq[npos+4])*string(ref_seq[npos+5])*string(ref_seq[npos+6])*string(ref_seq[npos+7])*string(ref_seq[npos+8])
            full_nc_context = ref_nc_nuc_context*"|"*qry_nc_nuc_context
            noncoding_nuc_to_context_dict[nuc_mut] = full_nc_context
            for (start_end, place) in noncoding_range_dict
                frst = start_end[1]
                last = start_end[2]
                if npos ≥ frst && npos ≤ last
                    noncoding_to_noncoding_region_dict[nuc_mut] = place
                    mut_vec = [nuc_mut, place]
                    push!(noncoding_nuc_vector, nuc_mut)
                end
            end
        end
    end
#########################################################################################################
    for nuc_mut in muts
        pos_9b = nuc_mut_int_comprehensive_dict[nuc_mut]
        if pos_9b in rem9b
            mut_9b = mixed2nuc(nuc_mut)
            pos_9b = nuc_mut_int_comprehensive_dict[mut_9b]   
            gene_number_9b = 11
            ref_triple_9b = ""
            qry_triple_9b = ""
            ref_triple_context_9b = ""
            qry_triple_context_9b = ""
            ref2qry_context_9b = ""
            if nuc_codon_pos_dict_9b[pos_9b] == 1
                ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])  *string(ref_seq[pos_9b+2])
                qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])
                ref_triple_context_9b = string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*ref_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])*string(ref_seq[pos_9b+7])*string(ref_seq[pos_9b+8])
                qry_triple_context_9b = string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*qry_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])*string(ref_seq[pos_9b+7])*string(ref_seq[pos_9b+8])
            elseif nuc_codon_pos_dict_9b[pos_9b] == 2
                ref_triple_9b = string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]  *string(ref_seq[pos_9b+1])
                qry_triple_9b = string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                ref_triple_context_9b = string(ref_seq[pos_9b-7])*string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*ref_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])*string(ref_seq[pos_9b+7])
                qry_triple_context_9b = string(ref_seq[pos_9b-7])*string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*qry_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])*string(ref_seq[pos_9b+7])
            elseif nuc_codon_pos_dict_9b[pos_9b] == 3
                ref_triple_9b = string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]
                qry_triple_9b = string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]
                ref_triple_context_9b = string(ref_seq[pos_9b-8])*string(ref_seq[pos_9b-7])*string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*ref_triple_9b*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])
                qry_triple_context_9b = string(ref_seq[pos_9b-8])*string(ref_seq[pos_9b-7])*string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*qry_triple_9b*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])
            end
            refAA_9b = AA_triplets[ref_triple_9b]
            qryAA_9b = AA_triplets[qry_triple_9b]
            AA_mut_9b = nuc2AA_ORF9b(mut_9b, refAA_9b, qryAA_9b)
            push!(AA_mut_set, AA_mut_9b)
            ref2qry_context_9b = ref_triple_context_9b*"-->"*qry_triple_context_9b
            all_nuc_to_AA_dict[mut_9b] = AA_mut_9b
            if refAA_9b == qryAA_9b && nuc_mut in N_9b_synonymous
                synonymous_nuc_to_AA_dict[mut_9b] = AA_mut_9b
                synonymous_nuc_to_context_dict[mut_9b] = ref2qry_context_9b
            end
            if refAA_9b ≠ qryAA_9b
                nonsynonymous_nuc_to_AA_dict[mut_9b] = AA_mut_9b
                nonsynonymous_nuc_to_context_dict[mut_9b] = ref2qry_context_9b
#                push!(nonsynonymous_nuc_muts, mut_9b)
            end
            all_nuc_to_context_dict[mut_9b] = ref2qry_context_9b
###################################
            for nuc_mut2_9b in muts
                mut2_9b = mixed2nuc(nuc_mut2_9b)
                pos2_9b = nuc_mut_int_comprehensive_dict[mut2_9b]
                if pos2_9b in rem9b
                    gene_number2_9b = 11
                    if mut_9b ≠ mut2_9b && nuc_to_AA_pos_9b(mut_9b) == nuc_to_AA_pos_9b(mut2_9b)
                        if nuc_codon_pos_dict_9b[pos_9b] == 1 && nuc_codon_pos_dict_9b[pos2_9b] == 2
                            ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*ref_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b+2])
                            qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*qry_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b+2])
                            ref_triple_context_9b = string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*ref_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])
                            qry_triple_context_9b = string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*qry_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])
                        elseif nuc_codon_pos_dict_9b[pos_9b] == 1 && nuc_codon_pos_dict_9b[pos2_9b] == 3
                            ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*ref_nuc_comprehensive_dict[mut2_9b]
                            qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*qry_nuc_comprehensive_dict[mut2_9b]
                            ref_triple_context_9b = string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*ref_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])
                            qry_triple_context_9b = string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*qry_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])
                        elseif nuc_codon_pos_dict_9b[pos_9b] == 2 && nuc_codon_pos_dict_9b[pos2_9b] == 3
                            ref_triple_9b = string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]*ref_nuc_comprehensive_dict[mut2_9b]
                            qry_triple_9b = string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]*qry_nuc_comprehensive_dict[mut2_9b]
                            ref_triple_context_9b = string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*ref_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])
                            qry_triple_context_9b = string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*qry_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])
                        elseif nuc_codon_pos_dict_9b[pos2_9b] == 1 && nuc_codon_pos_dict_9b[pos_9b] == 2
                            ref_triple_9b = ref_nuc_comprehensive_dict[mut2_9b]*ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                            qry_triple_9b = qry_nuc_comprehensive_dict[mut2_9b]*qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                            ref_triple_context_9b = string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*ref_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])
                            qry_triple_context_9b = string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*qry_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])
                        elseif nuc_codon_pos_dict_9b[pos2_9b] == 1 && nuc_codon_pos_dict_9b[pos_9b] == 3
                            ref_triple_9b = ref_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]
                            qry_triple_9b = qry_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]
                            ref_triple_context_9b = string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*ref_triple_9b*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])
                            qry_triple_context_9b = string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*qry_triple_9b*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])
                        elseif nuc_codon_pos_dict_9b[pos2_9b] == 2 && nuc_codon_pos_dict_9b[pos_9b] == 3
                            ref_triple_9b = string(ref_seq[pos_9b-2])*ref_nuc_comprehensive_dict[mut2_9b]*ref_nuc_comprehensive_dict[mut_9b]
                            qry_triple_9b = string(ref_seq[pos_9b-2])*qry_nuc_comprehensive_dict[mut2_9b]*qry_nuc_comprehensive_dict[mut_9b]
                        end
                        refAA2_9b = AA_triplets[ref_triple_9b]
                        qryAA2_9b = AA_triplets[qry_triple_9b]
                        AA_mut2_9b = nuc2AA_ORF9b(mut2_9b, refAA2_9b, qryAA2_9b)
                        push!(AA_mut_set, AA_mut2_9b)
                        ref2qry_context_9b = ref_triple_context_9b*"-->"*qry_triple_context_9b
                        if refAA2_9b == qryAA2_9b && nuc_mut2_9b in N_9b_synonymous 
                            synonymous_nuc_to_AA_dict[mut2_9b] = AA_mut2_9b
                            synonymous_nuc_to_context_dict[mut2_9b] = ref2qry_context_9b
                            synonymous_nuc_to_AA_dict[mut_9b] = AA_mut2_9b
                            synonymous_nuc_to_context_dict[mut_9b] = ref2qry_context_9b
                        else
                            nonsynonymous_nuc_to_AA_dict[mut2_9b] = AA_mut2_9b
                            nonsynonymous_nuc_to_context_dict[mut2_9b] = ref2qry_context_9b
                            nonsynonymous_nuc_to_AA_dict[mut_9b] = AA_mut2_9b
                            nonsynonymous_nuc_to_context_dict[mut_9b] = ref2qry_context_9b
                        end
                        all_nuc_to_context_dict[mut_9b] = ref2qry_context_9b
                        all_nuc_to_context_dict[mut2_9b] = ref2qry_context_9b
                    end
                end
            end
        end
    end
#################################################################################
    mut_vec_dict = Dict{String, Vector{String}}()
    for gene in gene_print_array
        mut_vec_dict[gene] = Vector{String}()
    end
    for mut in AA_mut_set
        jeen = aa_gene_comprehensive_dict[mut]
        mut_only = string(split(mut, ":")[2])
        push!(mut_vec_dict[jeen], mut_only) 
    end
    for gene in keys(mut_vec_dict)
        if !isempty(mut_vec_dict[gene])
            sort!(mut_vec_dict[gene], by = x -> x[2:end-1])
        end
    end
#####################################################################################################################################
    aasortkeyhere(m) = (gene_order_dict[aa_gene_comprehensive_dict[m]], aa_pos_comprehensive_dict[m])
    AA_sort = sort(collect(AA_mut_set), by = x -> aasortkeyhere(x))
    AA_sort2 = Vector{String}()
    for mut in AA_sort
        refAA = refAA_comprehensive_dict[mut]
        qryAA = qryAA_comprehensive_dict[mut]
        if !(refAA == qryAA)
            push!(AA_sort2, mut)
        end
    end
#    print("\n"^2)
#    println("##################### Amino Acid Mutations ######################")
#    for i in 1:length(AA_sort2)
#        mut = AA_sort2[i]
#        gene = aa_gene_comprehensive_dict[mut]
#        non_gene = string(split(mut, ":")[2])
#        if i == 1
#            print("               ", AA_sort2[i])
#        elseif i > 1
#            lastmut = AA_sort2[i-1]
#            last_gene = aa_gene_comprehensive_dict[lastmut]
#            last_non_gene = string(split(lastmut, ":")[2])
#            if gene == last_gene
#                print(", $(non_gene)")
#            else
#                println()
#                print("               $(mut)")
#            end
#        end
#    end
#    print("\n"^2)
#####################################################################################################################################
    all_nuc_to_AA_dict_sort = sort(collect(all_nuc_to_AA_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
    all_nuc_to_context_dict_sort = sort(collect(all_nuc_to_context_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
    nonsynonymous_nuc_to_AA_dict_sort = sort(collect(nonsynonymous_nuc_to_AA_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
    nonsynonymous_nuc_sort = sort(collect(keys(nonsynonymous_nuc_to_AA_dict)), by = x -> nuc_mut_int_comprehensive_dict[x])
    nonsynonymous_AA_sort = sort(collect(values(nonsynonymous_nuc_to_AA_dict)), by = x -> AA_gene_sortKey_2(x))
    nonsynonymous_nuc_to_context_dict_sort = sort(collect(nonsynonymous_nuc_to_context_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
    synonymous_nuc_to_AA_dict_sort = sort(collect(synonymous_nuc_to_AA_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
    synonymous_nuc_sort = sort(collect(keys(synonymous_nuc_to_AA_dict)), by = x -> nuc_mut_int_comprehensive_dict[x])
    synonymous_AA_sort = sort(collect(values(synonymous_nuc_to_AA_dict)), by = x -> AA_gene_sortKey_2(x))
    synonymous_nuc_to_context_dict_sort = sort(collect(synonymous_nuc_to_context_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
    noncoding_nuc_to_context_dict_sort = sort(collect(noncoding_nuc_to_context_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
    noncoding_nuc_vector_sort = sort(collect(noncoding_nuc_vector), by = x -> nuc_mut_int_comprehensive_dict[x])
    noncoding_to_noncoding_region_dict_sort = sort(collect(noncoding_to_noncoding_region_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
####################################################################################################################################
    nc_len = length(noncoding_to_noncoding_region_dict_sort)
#    println("NONCODING NUC MUTATIONS - Total:$(nc_len)")
    if !isempty(noncoding_to_noncoding_region_dict_sort)
        for i in 1:length(noncoding_to_noncoding_region_dict_sort)
            nc_nuc = noncoding_to_noncoding_region_dict_sort[i][1]
            nc_nuc_pad = rpad(nc_nuc, 7)
            nc_region = noncoding_to_noncoding_region_dict_sort[i][2]
            nc_region_len = length(nc_region)
            ncpad1 = (13 - nc_region_len)÷2
            ncpad12 = " "^ncpad1*nc_region
            nc_region_pad2 = rpad(ncpad12, 13)
            nc_context = noncoding_nuc_to_context_dict_sort[i][2]
            premut_context = ""
            postmut_context = ""
            context = split(nc_context, "-->")
            if !isempty(context)
                if length(context) >1
                    premut_context = split(nc_context, "-->")[1]
                    postmut_context = split(nc_context, "-->")[2]
                end
            end    
            postpad = lpad(postmut_context, 39)
#            println("$(nc_nuc_pad)|$(nc_region_pad2)|$(premut_context)")
#            println(postpad)
#                println(g, "$(nc_nuc_pad)|$(nc_region_pad2)|$(premut_context)")
#                println(g, postpad)
        end
#        println("#"^94); println() 
#            println(g, "#"^94); println(g)
    end
######################################################################################################
    total_syn = length(synonymous_nuc_to_AA_dict_sort)
#    println("SYNONYMOUS NUC MUTATIONS — Total:$(total_syn)")
    for i in 1:length(synonymous_nuc_to_AA_dict_sort)
        synnuc = synonymous_nuc_to_AA_dict_sort[i][1]
        synnuc_pad = rpad(synnuc, 7)
        synAA = synonymous_nuc_to_AA_dict_sort[i][2]
        AAlen = length(synAA)
        pad1 = (14 - AAlen)÷2
        pad12 = " "^pad1*synAA
        synAA_pad = rpad(pad12, 14)
        syncontext = synonymous_nuc_to_context_dict_sort[i][2]
        premut_context = split(syncontext, "-->")[1]
        postmut_context = split(syncontext, "-->")[2]
        postpad = lpad(postmut_context, 38)
#        println("$(synnuc_pad)|$(synAA_pad)|$(premut_context)")
#        println(postpad)
#            println(g, "$(synnuc_pad)|$(synAA_pad)|$(premut_context)")
#            println(g, postpad)
    end
#    println()
#################################################################
    total_syn = length(nonsynonymous_nuc_to_AA_dict_sort)
#    println("NONSYNONYMOUS NUC MUTATIONS — Total:$(total_syn)")
    for i in 1:length(nonsynonymous_nuc_to_AA_dict_sort)
        synnuc = nonsynonymous_nuc_to_AA_dict_sort[i][1]
        synnuc_pad = rpad(synnuc, 7)
        synAA = nonsynonymous_nuc_to_AA_dict_sort[i][2]
        AAlen = length(synAA)
        pad1 = (14 - AAlen)÷2
        pad12 = " "^pad1*synAA
        synAA_pad = rpad(pad12, 14)
        syncontext = nonsynonymous_nuc_to_context_dict_sort[i][2]
        premut_context = split(syncontext, "-->")[1]
        postmut_context = split(syncontext, "-->")[2]
        postpad = lpad(postmut_context, 38)
#        println("$(synnuc_pad)|$(synAA_pad)|$(premut_context)")
#        println(postpad)
#            println(g, "$(synnuc_pad)|$(synAA_pad)|$(premut_context)")
#            println(g, postpad)
    end
#    println()
###################################################################
    nonsynonymous_nuc_total = length(nonsynonymous_nuc_sort)
#    println("        Total Number of Non-synonymous Nuc Muts = $(nonsynonymous_nuc_total)")
    nonsynonymous_nuc_sort_join = join(nonsynonymous_nuc_sort, ", ")
#    println("################ Nonsynonymous Nuc Mutations ################")
#    println(nonsynonymous_nuc_sort_join)
#    print("\n"^1)
    for i in 1:length(nonsynonymous_nuc_sort)
#        println("               $(nonsynonymous_nuc_to_AA_dict_sort[i][1]) | $(nonsynonymous_nuc_to_AA_dict_sort[i][2])")
        premut_nucseq = split(nonsynonymous_nuc_to_context_dict_sort[i][2], "-->")[1]
        postmut_nucseq = split(nonsynonymous_nuc_to_context_dict_sort[i][2], "-->")[2]
#        println("                 $(premut_nucseq)")
#        println("                 $(postmut_nucseq)")
    end
#synonymous_nuc_to_context_dict[mut] = ref_triple_context*"-->"*qry_triple_context
    return synonymous_nuc_to_AA_dict_sort, nonsynonymous_nuc_to_AA_dict_sort, all_nuc_to_AA_dict_sort
end
###########################################################################################################################################################################
###########################################################################################################################################################################
function mixed_muts_to_AA(ref_pango::String, muts::String)
    mut_strings = muts_to_strings(muts)
    mixed_muts = mixed_nucs_filter(mut_strings)             ### New, 2025-1-26
    mixed_muts_regular = mixed_mut_to_regular_mut(mixed_muts)    ### New, 2025-1-26
    ct = 0
    total_mixed_muts = length(mixed_muts)
#    println("Total Mixed Nucs = $(total_mixed_muts)")
#    print("\n"^2)
    for i in 1:length(mixed_muts)
        if ct == 0
#            print(mixed_muts[i])
            ct = 1
        else
#            print(", ", mixed_muts[i])
        end
    end
    ct2 = 0
#    print("\n"^2)
    for i in 1:length(mixed_muts_regular)
        if ct2 == 0
#            print(mixed_muts_regular[i])
            ct2 = 1
#        else
#            print(", ", mixed_muts_regular[i])
        end
    end
#    println()
    syn_nuc_to_AA_dict_sort, nonsyn_nuc_to_AA_dict_sort, all_nuc_to_AA_dict_sort = nuc_to_AA(ref_pango, mixed_muts_regular)
    AA__AAprintlen__vec = []
    nonsynnuc__nonsynprintlen__vec = []
    for nuc___AA in nonsyn_nuc_to_AA_dict_sort
        nucmut = nuc___AA[1]
        nucmutlen = length(nucmut)
        AAmut = nuc___AA[2]
        AAmutlen = length(AAmut)
        push!(AA__AAprintlen__vec, [AAmut, AAmutlen])
        push!(nonsynnuc__nonsynprintlen__vec, [nucmut, nucmutlen])
    end
    aa_pad_vec = String[]
    nuc_pad_vec = String[]
    for i in 1:length(AA__AAprintlen__vec)
        aa = AA__AAprintlen__vec[i][1]
        nuc = nonsynnuc__nonsynprintlen__vec[i][1]
        aapad = AA__AAprintlen__vec[i][2]
        nucpad = nonsynnuc__nonsynprintlen__vec[i][2]
        pads = [nucpad, aapad]
        pad = maximum(pads)
        push!(aa_pad_vec, rpad(aa, pad))
        push!(nuc_pad_vec, rpad(nuc, pad))
    end
    aapad_join = join(aa_pad_vec, ", ")
    nucpad_join = join(nuc_pad_vec, ", ")
#    println("\n"^1)
#    println(aapad_join)
#    println(nucpad_join)
#    println("\n"^1)
    return syn_nuc_to_AA_dict_sort, nonsyn_nuc_to_AA_dict_sort, all_nuc_to_AA_dict_sort
end
######################################################################################################################################
function count_nuc_mut_types(mut_strings::Vector{String})
    mut_types_arr = ["TC", "TA", "TG", "CT", "CA", "CG", "AT", "AC", "AG", "GT", "GC", "GA"]
    mut_type_cts = Dict{String, Int}(mut_nuc_type=>0 for mut_nuc_type in mut_types_arr)
    for nuc_mut in mut_strings
        ref = ref_nuc_comprehensive_dict[nuc_mut]
        qry = ref_nuc_comprehensive_dict[nuc_mut]
        if ref == "T"
            if qry == "C"
                mut_type_cts["TC"] += 1
            elseif qry == "A"
                mut_type_cts["TA"] += 1
            elseif qry == "G"
                mut_type_cts["TG"] += 1
            end
        end
        if ref == "C"
            if qry == "T"
                mut_type_cts["CT"] += 1
            elseif qry == "A"
                mut_type_cts["CA"] += 1
            elseif qry == "G"
                mut_type_cts["CG"] += 1
            end
        end 
        if ref == "A"
            if qry == "T"
                mut_type_cts["AT"] += 1
            elseif qry == "C"
                mut_type_cts["AC"] += 1
            elseif qry == "G"
                mut_type_cts["AG"] += 1
            end
        end   
        if ref == "G"
            if qry == "T"
                mut_type_cts["GT"] += 1
            elseif qry == "C"
                mut_type_cts["GC"] += 1
            elseif qry == "A"
                mut_type_cts["GA"] += 1
            end
        end
    end
    mut_type_cts_sort_by_type = sort(collect(mut_type_cts), by = x -> x[1])
    mut_type_cts_sort_by_count = sort(collect(mut_type_cts), by = x -> x[2], rev=true)
    return mut_type_cts_sort_by_count
end
######################################################################################################################################
function AA_triple(pos::Int, rem0::BitSet, rem1::BitSet, rem2::BitSet, mut::String, ref_pango::String)
    ref_seq, refAA_ORF1a, refAA_ORF1b, refAA_S, refAA_ORF3a, refAA_E, refAA_M, refAA_ORF6, refAA_ORF7a, refAA_ORF7b, refAA_ORF8, refAA_N, refAA_ORF9b = get_ref_pango_nucseq_and_geneseqs(ref_pango)
    ref_triple = ""
    qry_triple = ""
    if pos in rem0 && pos%3 == 0
        ref_triple = ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
        qry_triple = qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
    elseif pos in rem1 && pos%3 == 1
        ref_triple = ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
        qry_triple = qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
    elseif pos in rem2 && pos%3 == 2
        ref_triple = ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
        qry_triple = qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
    elseif pos in rem0 && pos%3 == 1
        ref_triple = string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
        qry_triple = string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
    elseif pos in rem1 && pos%3 == 2
        ref_triple = string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
        qry_triple = string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
    elseif pos in rem2 && pos%3 == 0
        ref_triple = string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
        qry_triple = string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
    elseif pos in rem0 && pos%3 == 2
        ref_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]
        qry_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]
    elseif pos in rem1 && pos%3 == 0
        ref_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]
        qry_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]
    elseif pos in rem2 && pos%3 == 1
        ref_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]
        qry_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]
    end
    return ref_triple, qry_triple
end
######################################################################################################################################
### Fx: SIMPLER_nuc_to_AA (no context)
function SIMPLER_nuc_to_AA(ref_pango::String, muts::Vector{String})
    muts = filter(!isempty, muts)
    if !isempty(muts)
#        all_muts_sort = sort(collect(muts), by = x -> nuc_mut_int_comprehensive_dict[x])
        ref_seq, refAA_ORF1a, refAA_ORF1b, refAA_S, refAA_ORF3a, refAA_E, refAA_M, refAA_ORF6, refAA_ORF7a, refAA_ORF7b, refAA_ORF8, refAA_N, refAA_ORF9b = get_ref_pango_nucseq_and_geneseqs(ref_pango)
    ###############################################################################
        coding_ranges = BitSet([266:13467..., 13469:21555..., 21563:25384..., 25393:26220..., 26245:26472..., 26523:27191..., 27202:27387..., 27394:27755..., 27760:27887..., 27894:28259..., 28274:29533..., 28284:28577...])
        noncoding_ranges = BitSet([1:265..., 21556:21562..., 25385:25392..., 26221:26244..., 26473:26522..., 27192:27201..., 27388:27393..., 27888:27893..., 28260:28273..., 29534:29830...])
        coding_range_9b = BitSet([28284:28577...])
        gene_nuc_starts = Dict{Int, Int}(0=>263, 1=>13465, 2=>21560, 3=>25390, 4=>26242, 5=>26520, 6=>27199, 7=>27391, 8=>27753, 9=>27891, 10=>28271, 11=>28281)
        ref_AA_genes = Dict{Int, String}(0=>refAA_ORF1a, 1=>refAA_ORF1b, 2=>refAA_S, 3=>refAA_ORF3a, 4=>refAA_E, 5=>refAA_M, 6=>refAA_ORF6, 7=>refAA_ORF7a, 8=>refAA_ORF7b, 9=>refAA_ORF8, 10=>refAA_N, 11=>refAA_ORF9b)
        gene_strings = Dict{Int, String}(0=>"ORF1a", 1=>"ORF1b", 2=>"S", 3=>"ORF3a", 4=>"E", 5=>"M", 6=>"ORF6", 7=>"ORF7a", 8=>"ORF7b", 9=>"ORF8", 10=>"N", 11=>"ORF9b")
        nuc_gene_num = Dict{Int, Int}()
        nuc_gene_num_9b = Dict{Int, Int}()
        synonymous_nuc_to_AA_dict = Dict{String, String}()
        nonsynonymous_nuc_to_AA_dict = Dict{String, String}()
        all_nuc_to_AA_dict = Dict{String, String}()
        noncoding_range_dict = Dict{Vector{Int}, String}([1, 265]=>"5' UTR", [21556, 21562]=>"Spike TRS", [25385, 25392]=>"ORF3a TRS", [26221, 26234]=>"ORF3a-E UTR", [26235, 26244]=>"E TRS", [26473, 26522]=>"E-M UTR", [27192, 27201]=>"M-ORF6 UTR", [27388, 27393]=>"ORF7a TRS", [27888, 27893]=>"ORF8 TRS", [28260, 28273]=>"N/ORF9b TRS", [29534, 29830]=>"3' UTR", [29794, 29801]=>"OctanucMotif")
################################################
        noncoding_nuc_vector = Vector{String}()
################################################ 
        gene_nuc_arr = [[266:13467...], [13469:21555...], [21563:25384...], [25393:26220...], [26245:26472...], [26523:27191...], [27202:27387...], [27394:27755...], [27760:27887...], [27894:28259...], [28274:29533...], [28284:28577...]]
        for i in 1:length(gene_nuc_arr)-1
            for nuc_pos in gene_nuc_arr[i]
                nuc_gene_num[nuc_pos] = i-1
            end
        end
        for nuc_pos in gene_nuc_arr[end]
            nuc_gene_num_9b[nuc_pos] = 11
        end
        rem0_gene = [5, 8, 9, 11]
        rem1_gene = [1, 3, 4, 6, 7]
        rem2_gene = [0, 2, 10]
        rem0 = BitSet([26523:27191..., 27760:27887..., 27894:28259..., 28284:28577...])
        rem1 = BitSet([13469:21555..., 25393:26220..., 26245:26472..., 27202:27387..., 27394:27755...])
        rem2 = BitSet([266:13467..., 21563:25384..., 28274:29533...])
        rem9b = BitSet([28284:28577...])
        rem7ab = BitSet([27756:27759...])

        gene_num(nuc_mut) = nuc_gene_num[nuc_mut_int_comprehensive_dict[nuc_mut]] ## Fx ##
        nuc_to_AA_pos(nuc_mut) = string((nuc_mut_int_comprehensive_dict[nuc_mut] - gene_nuc_starts[gene_num(nuc_mut)])÷3) ## Fx ##
        nuc_to_AA_pos_9b(nuc_mut) = string((nuc_mut_int_comprehensive_dict[nuc_mut] - 28281)÷3) ## Fx ##
        nuc2AA_ORF1a(nuc_mut, refAA, qryAA) = gene_strings[gene_num(nuc_mut)]*":"*refAA*nuc_to_AA_pos(nuc_mut)*qryAA ## Fx ##
        nuc2AA_ORF9b(nuc_mut, refAA, qryAA) = "ORF9b:"*refAA*nuc_to_AA_pos_9b(nuc_mut)*qryAA
        nuc_codon_pos_dict = Dict{Int, Int}()
        for nuc_pos in coding_ranges
            gene_number = nuc_gene_num[nuc_pos]
            gene_start = gene_nuc_starts[gene_number]
            codon_num = (nuc_pos-gene_start)%3 + 1
            nuc_codon_pos_dict[nuc_pos] = codon_num
        end
        nuc_codon_pos_dict_9b = Dict{Int, Int}()
        for nuc_pos in coding_range_9b
            gene_number = 11
            gene_start = gene_nuc_starts[gene_number]
            codon_num = (nuc_pos-gene_start)%3 + 1
            nuc_codon_pos_dict_9b[nuc_pos] = codon_num
        end    
        N3_syn = ["TCT", "TCC", "TCA", "TCG", "CTT", "CTC", "CTA", "CTG", "CCT", "CCC", "CCA", "CCG", "CGT", "CGC", "CGA", "CGG", "ACT", "ACC", "ACA", "ACG", "GTT", "GTC", "GTA", "GTG", "GCT", "GCC", "GCA", "GCG", "GGT", "GGC", "GGA", "GGG"]
        N3_tv = ["TTT", "TTC", "TTA", "TTG", "TAT", "TAC", "TAA", "TAG", "AAT", "AAC", "AAA", "AAG", "AGT", "AGC", "AGA", "AGG", "GAT", "GAC", "GAA", "GAG"]
        for nuc_mut in muts
            if !isempty(muts)
                mut = mixed2nuc(nuc_mut)
                if ',' in mut
                    mut1 = string(split(mut, ",")[1])
                    mut2 = string(split(mut, ",")[2])
                    push!(muts, mut1)
                    push!(muts, mut2)
                    filter!(x -> !(length(x)>6), muts)
                end
            end
        end
        coding_ranges = BitSet([266:13467..., 13468:21555..., 21563:25384..., 25393:26220..., 26245:26472..., 26523:27191..., 27202:27387..., 27394:27755..., 27760:27887..., 27894:28259..., 28274:29533...])
        N_9b_synonymous = Set(["C28379A", "C28394A", "T28406C", "C28475A", "C28535A", "A28547C"])
        AA_mut_set = Set{String}()
        AA_mut = ""
        for nuc_mut in muts
            pos = nuc_mut_int_comprehensive_dict[nuc_mut]
            if pos in coding_ranges
                mut = mixed2nuc(nuc_mut)  
                gene_number = gene_num(mut)
                ref_triple = ""
                qry_triple = ""
                if nuc_codon_pos_dict[pos] == 1
                    ref_triple = ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
                    qry_triple = qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
                elseif nuc_codon_pos_dict[pos] == 2
                    ref_triple = string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                    qry_triple = string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                elseif nuc_codon_pos_dict[pos] == 3
                    ref_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]
                    qry_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]
                end
                refAA = AA_triplets[ref_triple]
                qryAA = AA_triplets[qry_triple]
                AA_mut = nuc2AA_ORF1a(mut, refAA, qryAA)
                push!(AA_mut_set, AA_mut)
                all_nuc_to_AA_dict[mut] = AA_mut
                if refAA == qryAA && !(pos in rem9b)
                    synonymous_nuc_to_AA_dict[mut] = AA_mut
                elseif refAA == qryAA && pos in rem9b && mut in N_9b_synonymous
                    synonymous_nuc_to_AA_dict[mut] = AA_mut
                else
                    nonsynonymous_nuc_to_AA_dict[mut] = AA_mut
#                    push!(nonsynonymous_nuc_muts, mut)
                end
###################################
                for nuc_mut2 in muts
                    mut2 = mixed2nuc(nuc_mut2)
                    pos2 = nuc_mut_int_comprehensive_dict[mut2]
                    if pos2 in coding_ranges
                        gene_number2 = gene_num(mut2)
                        if mut ≠ mut2 && gene_number == gene_number2 && nuc_to_AA_pos(mut) == nuc_to_AA_pos(mut2)
                            if nuc_codon_pos_dict[pos] == 1 && nuc_codon_pos_dict[pos2] == 2
                                ref_triple = ref_nuc_comprehensive_dict[mut]*ref_nuc_comprehensive_dict[mut2]*string(ref_seq[pos+2])
                                qry_triple = qry_nuc_comprehensive_dict[mut]*qry_nuc_comprehensive_dict[mut2]*string(ref_seq[pos+2])
                            elseif nuc_codon_pos_dict[pos] == 1 && nuc_codon_pos_dict[pos2] == 3
                                ref_triple = ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*ref_nuc_comprehensive_dict[mut2]
                                qry_triple = qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*qry_nuc_comprehensive_dict[mut2]
                            elseif nuc_codon_pos_dict[pos] == 2 && nuc_codon_pos_dict[pos2] == 3
                                ref_triple = string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]*ref_nuc_comprehensive_dict[mut2]
                                qry_triple = string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]*qry_nuc_comprehensive_dict[mut2]
                            elseif nuc_codon_pos_dict[pos2] == 1 && nuc_codon_pos_dict[pos] == 2
                                ref_triple = ref_nuc_comprehensive_dict[mut2]*ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                                qry_triple = qry_nuc_comprehensive_dict[mut2]*qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                            elseif nuc_codon_pos_dict[pos2] == 1 && nuc_codon_pos_dict[pos] == 3
                                ref_triple = ref_nuc_comprehensive_dict[mut2]*string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]
                                qry_triple = qry_nuc_comprehensive_dict[mut2]*string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]
                            elseif nuc_codon_pos_dict[pos2] == 2 && nuc_codon_pos_dict[pos] == 3
                                ref_triple = string(ref_seq[pos-2])*ref_nuc_comprehensive_dict[mut2]*ref_nuc_comprehensive_dict[mut]
                                qry_triple = string(ref_seq[pos-2])*qry_nuc_comprehensive_dict[mut2]*qry_nuc_comprehensive_dict[mut]
                            end
                            refAA2 = AA_triplets[ref_triple]
                            qryAA2 = AA_triplets[qry_triple]
                            AA_mut2 = nuc2AA_ORF1a(mut2, refAA2, qryAA2)
                            push!(AA_mut_set, AA_mut2)
                            delete!(AA_mut_set, AA_mut)
                            all_nuc_to_AA_dict[mut2] = AA_mut2
                            all_nuc_to_AA_dict[mut] = AA_mut2
                            if refAA2 == qryAA2 && !(pos2 in rem9b)
                                synonymous_nuc_to_AA_dict[mut2] = AA_mut2
                            else
                                nonsynonymous_nuc_to_AA_dict[mut2] = AA_mut2
                                nonsynonymous_nuc_to_AA_dict[mut] = AA_mut2
                            end
                        end
                    end
                end
            elseif !isempty(nuc_mut)
                qry_nuc = qry_nuc_comprehensive_dict[nuc_mut]
                for (start_end, place) in noncoding_range_dict
                    frst = start_end[1]
                    last = start_end[2]
                    if pos ≥ frst && pos ≤ last
                        mut_vec = [nuc_mut, place]
                        push!(noncoding_nuc_vector, nuc_mut)
                    end
                end
            end
        end
#########################################################################################################
        for nuc_mut in muts
            pos_9b = nuc_mut_int_comprehensive_dict[nuc_mut]
            if pos_9b in rem9b
                mut_9b = mixed2nuc(nuc_mut)
                pos_9b = nuc_mut_int_comprehensive_dict[mut_9b]   
                gene_number_9b = 11
                ref_triple_9b = ""
                qry_triple_9b = ""
                if nuc_codon_pos_dict_9b[pos_9b] == 1
                    ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])
                    qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])
                elseif nuc_codon_pos_dict_9b[pos_9b] == 2
                    ref_triple_9b = string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                    qry_triple_9b = string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                elseif nuc_codon_pos_dict_9b[pos_9b] == 3
                    ref_triple_9b = string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]
                    qry_triple_9b = string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]
                end
                refAA_9b = AA_triplets[ref_triple_9b]
                qryAA_9b = AA_triplets[qry_triple_9b]
                AA_mut_9b = nuc2AA_ORF9b(mut_9b, refAA_9b, qryAA_9b)
                push!(AA_mut_set, AA_mut_9b)
                all_nuc_to_AA_dict[mut_9b] = AA_mut_9b
                if refAA_9b == qryAA_9b && nuc_mut in N_9b_synonymous
                    synonymous_nuc_to_AA_dict[mut_9b] = AA_mut_9b
                end
                if refAA_9b ≠ qryAA_9b
                    nonsynonymous_nuc_to_AA_dict[mut_9b] = AA_mut_9b
#                    push!(nonsynonymous_nuc_muts, mut_9b)
                end
###################################
                for nuc_mut2_9b in muts
                    mut2_9b = mixed2nuc(nuc_mut2_9b)
                    pos2_9b = nuc_mut_int_comprehensive_dict[mut2_9b]
                    if pos2_9b in rem9b
                        gene_number2_9b = 11
                        if mut_9b ≠ mut2_9b && nuc_to_AA_pos_9b(mut_9b) == nuc_to_AA_pos_9b(mut2_9b)
                            if nuc_codon_pos_dict_9b[pos_9b] == 1 && nuc_codon_pos_dict_9b[pos2_9b] == 2
                                ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*ref_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b+2])
                                qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*qry_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b+2])
                            elseif nuc_codon_pos_dict_9b[pos_9b] == 1 && nuc_codon_pos_dict_9b[pos2_9b] == 3
                                ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*ref_nuc_comprehensive_dict[mut2_9b]
                                qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*qry_nuc_comprehensive_dict[mut2_9b]
                            elseif nuc_codon_pos_dict_9b[pos_9b] == 2 && nuc_codon_pos_dict_9b[pos2_9b] == 3
                                ref_triple_9b = string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]*ref_nuc_comprehensive_dict[mut2_9b]
                                qry_triple_9b = string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]*qry_nuc_comprehensive_dict[mut2_9b]
                            elseif nuc_codon_pos_dict_9b[pos2_9b] == 1 && nuc_codon_pos_dict_9b[pos_9b] == 2
                                ref_triple_9b = ref_nuc_comprehensive_dict[mut2_9b]*ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                                qry_triple_9b = qry_nuc_comprehensive_dict[mut2_9b]*qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                            elseif nuc_codon_pos_dict_9b[pos2_9b] == 1 && nuc_codon_pos_dict_9b[pos_9b] == 3
                                ref_triple_9b = ref_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]
                                qry_triple_9b = qry_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]
                            elseif nuc_codon_pos_dict_9b[pos2_9b] == 2 && nuc_codon_pos_dict_9b[pos_9b] == 3
                                ref_triple_9b = string(ref_seq[pos_9b-2])*ref_nuc_comprehensive_dict[mut2_9b]*ref_nuc_comprehensive_dict[mut_9b]
                                qry_triple_9b = string(ref_seq[pos_9b-2])*qry_nuc_comprehensive_dict[mut2_9b]*qry_nuc_comprehensive_dict[mut_9b]
                            end
                            refAA2_9b = AA_triplets[ref_triple_9b]
                            qryAA2_9b = AA_triplets[qry_triple_9b]
                            AA_mut2_9b = nuc2AA_ORF9b(mut2_9b, refAA2_9b, qryAA2_9b)
                            push!(AA_mut_set, AA_mut2_9b)
                            delete!(AA_mut_set, AA_mut_9b)
                            if refAA2_9b == qryAA2_9b && nuc_mut2_9b in N_9b_synonymous 
                                synonymous_nuc_to_AA_dict[mut2_9b] = AA_mut2_9b
                                synonymous_nuc_to_AA_dict[mut_9b] = AA_mut2_9b
                            else
                                nonsynonymous_nuc_to_AA_dict[mut2_9b] = AA_mut2_9b
                                nonsynonymous_nuc_to_AA_dict[mut_9b] = AA_mut2_9b
                            end
                        end
                    end
                end
            end
        end
#####################################################################################################################################
        AA_sort = sort(collect(AA_mut_set), by = x -> AA_order_key(x))
        AA_sort2 = Vector{String}()
        for aa in AA_sort
            if !(refAA_comprehensive_dict[aa] == qryAA_comprehensive_dict)
                push!(AA_sort2, aa)
            end
        end
#        print("\n"^1)
#        ct = 0
#        println("############# AA Mutations #############")
#        for i in 1:length(AA_sort2)
#            mut = AA_sort2[i]
#            gene = string(split(mut, ":")[1])
#            non_gene = string(split(mut, ":")[2])
#            if i == 1
#                print("        ", AA_sort2[i])
#            elseif i > 1
#                lastmut = AA_sort2[i-1]
#                last_gene = string(split(lastmut, ":")[1])
#                last_non_gene = string(split(lastmut, ":")[2])
#                if gene == last_gene
#                    print(", $(non_gene)")
#                else
#                    print("        $(mut)")
#                end
#            end
#        end
#####################################################################################################################################
        all_nuc_to_AA_dict_sort = sort(collect(all_nuc_to_AA_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
        nonsynonymous_nuc_to_AA_dict_sort = sort(collect(nonsynonymous_nuc_to_AA_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
        nonsynonymous_nuc_sort = sort(collect(keys(nonsynonymous_nuc_to_AA_dict)), by = x -> nuc_mut_int_comprehensive_dict[x])
        nonsynonymous_AA_sort = sort(collect(values(nonsynonymous_nuc_to_AA_dict)), by = x -> AA_gene_sortKey_2(x))
        synonymous_nuc_to_AA_dict_sort = sort(collect(synonymous_nuc_to_AA_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
        synonymous_nuc_sort = sort(collect(keys(synonymous_nuc_to_AA_dict)), by = x -> nuc_mut_int_comprehensive_dict[x])
        synonymous_AA_sort = sort(collect(values(synonymous_nuc_to_AA_dict)), by = x -> AA_gene_sortKey_2(x))
        noncoding_nuc_vector_sort = sort(collect(noncoding_nuc_vector), by = x -> nuc_mut_int_comprehensive_dict[x])
        synonymous_nuc_total = length(synonymous_nuc_sort)
        for i in 1:length(synonymous_nuc_sort)
            nucpad = rpad("$(synonymous_nuc_to_AA_dict_sort[i][1])", 10)
        end
#        return synonymous_nuc_to_AA_dict, nonsynonymous_nuc_to_AA_dict_sort, all_nuc_to_AA_dict_sort
        return synonymous_nuc_to_AA_dict_sort, nonsynonymous_nuc_to_AA_dict_sort, all_nuc_to_AA_dict_sort
    end
    if isempty(muts)
        aa = Vector{Pair{String, String}}()
        bb = Vector{Pair{String, String}}()
        cc = Vector{Pair{String, String}}()
        return aa, bb, cc
    end 
end
###########################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
function pango_minus_1_fx(pango::String)
    if '.' in pango
        dot_ct = count(".", pango)
        dotsplits = split(pango, ".")
        minus_1 = join(dotsplits(1:dot_ct-1), ".")
        return minus_1
    else
        return pango
    end
end
############################################################################################################################################################################
############################################################################################################################################################################
### Fx: SIMPLER_syn_noncoding_nonsyn_nuc (no context)
############################################################################################################################################################################
        coding_ranges = BitSet([266:13467..., 13469:21555..., 21563:25384..., 25393:26220..., 26245:26472..., 26523:27191..., 27202:27387..., 27394:27755..., 27760:27887..., 27894:28259..., 28274:29533..., 28284:28577...])
        noncoding_ranges = BitSet([1:265..., 21556:21562..., 25385:25392..., 26221:26244..., 26473:26522..., 27192:27201..., 27388:27393..., 27888:27893..., 28260:28273..., 29534:29830...])
        coding_range_9b = BitSet([28284:28577...])
        gene_nuc_starts = Dict{Int, Int}(0=>263, 1=>13465, 2=>21560, 3=>25390, 4=>26242, 5=>26520, 6=>27199, 7=>27391, 8=>27753, 9=>27891, 10=>28271, 11=>28281)
        gene_strings = Dict{Int, String}(0=>"ORF1a", 1=>"ORF1b", 2=>"S", 3=>"ORF3a", 4=>"E", 5=>"M", 6=>"ORF6", 7=>"ORF7a", 8=>"ORF7b", 9=>"ORF8", 10=>"N", 11=>"ORF9b")
        noncoding_range_dict = Dict{Vector{Int}, String}([1, 265]=>"5' UTR", [21556, 21562]=>"Spike TRS", [25385, 25392]=>"ORF3a TRS", [26221, 26234]=>"ORF3a-E UTR", [26235, 26244]=>"E TRS", [26473, 26522]=>"E-M UTR", [27192, 27201]=>"M-ORF6 UTR", [27388, 27393]=>"ORF7a TRS", [27888, 27893]=>"ORF8 TRS", [28260, 28273]=>"N/ORF9b TRS", [29534, 29830]=>"3' UTR", [29794, 29801]=>"OctanucMotif")
################################################
        gene_nuc_arr = [[266:13467...], [13469:21555...], [21563:25384...], [25393:26220...], [26245:26472...], [26523:27191...], [27202:27387...], [27394:27755...], [27760:27887...], [27894:28259...], [28274:29533...], [28284:28577...]]
###################################################################################################################################
        rem0_gene = [5, 8, 9, 11]
        rem1_gene = [1, 3, 4, 6, 7]
        rem2_gene = [0, 2, 10]
        rem0 = BitSet([26523:27191..., 27760:27887..., 27894:28259..., 28284:28577...])
        rem1 = BitSet([13469:21555..., 25393:26220..., 26245:26472..., 27202:27387..., 27394:27755...])
        rem2 = BitSet([266:13467..., 21563:25384..., 28274:29533...])
        rem9b = BitSet([28284:28577...])
        rem7ab = BitSet([27756:27759...])

#        coding_ranges = BitSet([266:13467..., 13468:21555..., 21563:25384..., 25393:26220..., 26245:26472..., 26523:27191..., 27202:27387..., 27394:27755..., 27760:27887..., 27894:28259..., 28274:29533...])
        N_9b_synonymous = Set(["C28379A", "C28394A", "T28406C", "C28475A", "C28535A", "A28547C"])
############################################################################################################################################################################
function SIMPLER_syn_noncoding_nuc(ref_pango::String, muts::Set{String})
    B_1_1_list = ["B.1.1.53", "B.1.1.273"]
    if ref_pango in B_1_1_list
        ref_pango = "B.1.1"
    end
    if ref_pango == "XBB.1.5.82"
         ref_pango = "XBB.1.5"
    end
#########################################################################################################
    if !isempty(muts)
#        all_muts_sort = sort(collect(muts), by = x -> nuc_mut_int_comprehensive_dict[x])
        ref_seq, refAA_ORF1a, refAA_ORF1b, refAA_S, refAA_ORF3a, refAA_E, refAA_M, refAA_ORF6, refAA_ORF7a, refAA_ORF7b, refAA_ORF8, refAA_N, refAA_ORF9b = get_ref_pango_nucseq_and_geneseqs(ref_pango)
#########################################################################################################
        ref_AA_genes = Dict{Int, String}(0=>refAA_ORF1a, 1=>refAA_ORF1b, 2=>refAA_S, 3=>refAA_ORF3a, 4=>refAA_E, 5=>refAA_M, 6=>refAA_ORF6, 7=>refAA_ORF7a, 8=>refAA_ORF7b, 9=>refAA_ORF8, 10=>refAA_N, 11=>refAA_ORF9b)
        noncoding_nuc_vector = Vector{String}()
################################################
        synonymous_nuc_to_AA_dict = Dict{String, String}()
        nuc_codon_pos_dict = Dict{Int, Int}()
        nuc_gene_num = Dict{Int, Int}()
        nuc_gene_num_9b = Dict{Int, Int}()
#########################################################################################################
        for i in 1:length(gene_nuc_arr)-1
            for nuc_pos in gene_nuc_arr[i]
                nuc_gene_num[nuc_pos] = i-1
            end
        end
        for nuc_pos in gene_nuc_arr[end]
            nuc_gene_num_9b[nuc_pos] = 11
        end
#########################################################################################################
        gene_num(nuc_mut) = nuc_gene_num[nuc_mut_int_comprehensive_dict[nuc_mut]] ## Fx ##
        nuc_to_AA_pos(nuc_mut) = string((nuc_mut_int_comprehensive_dict[nuc_mut] - gene_nuc_starts[gene_num(nuc_mut)])÷3) ## Fx ##
        nuc_to_AA_pos_9b(nuc_mut) = string((nuc_mut_int_comprehensive_dict[nuc_mut] - 28281)÷3) ## Fx ##
        nuc2AA_ORF1a(nuc_mut, refAA, qryAA) = gene_strings[gene_num(nuc_mut)]*":"*refAA*nuc_to_AA_pos(nuc_mut)*qryAA ## Fx ##
        nuc2AA_ORF9b(nuc_mut, refAA, qryAA) = "ORF9b:"*refAA*nuc_to_AA_pos_9b(nuc_mut)*qryAA
#########################################################################################################
        for nuc_pos in coding_ranges
            gene_number = nuc_gene_num[nuc_pos]
            gene_start = gene_nuc_starts[gene_number]
            codon_num = (nuc_pos-gene_start)%3 + 1
            nuc_codon_pos_dict[nuc_pos] = codon_num
        end
        nuc_codon_pos_dict_9b = Dict{Int, Int}()
        for nuc_pos in coding_range_9b
            gene_number = 11
            gene_start = gene_nuc_starts[gene_number]
            codon_num = (nuc_pos-gene_start)%3 + 1
            nuc_codon_pos_dict_9b[nuc_pos] = codon_num
        end    
        for nuc_mut in muts
            if !isempty(muts)
                mut = mixed2nuc(nuc_mut)
                if ',' in mut
                    mut1 = string(split(mut, ",")[1])
                    mut2 = string(split(mut, ",")[2])
                    push!(muts, mut1)
                    push!(muts, mut2)
                    filter!(x -> !(length(x)>7), muts)
                end
            end
        end
        AA_mut = ""
        for nuc_mut in muts
            if !isempty(muts) && !isempty(ref_seq) && nuc_mut ≠ ""
                pos = nuc_mut_int_comprehensive_dict[nuc_mut]
                if pos in coding_ranges
                    mut = mixed2nuc(nuc_mut)
                    if isempty(mut)
                        println(nuc_mut)
                        println(mut)
                        println("$(pango)")
                    end
                    try
                        nuc_mut_int_comprehensive_dict[mut]
                    catch e
                        println("Problematic mutation string: ", repr(mut))
                        println("Length: ", length(mut))
                        println("Extracted substring: ", repr(mut[2:end-1]))
                        rethrow(e)
                    end
                    gene_number = nuc_gene_num[pos]
                    ref_triple = ""
                    qry_triple = ""
                    if nuc_codon_pos_dict[pos] == 1
                        ref_triple = ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
                        qry_triple = qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
                    elseif nuc_codon_pos_dict[pos] == 2
                        ref_triple = string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                        qry_triple = string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                    elseif nuc_codon_pos_dict[pos] == 3
                        ref_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]
                        qry_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]
                    end
                    refAA = AA_triplets[ref_triple]
                    qryAA = AA_triplets[qry_triple]
                    AA_mut = nuc2AA_ORF1a(mut, refAA, qryAA)
                    if refAA == qryAA && !(pos in rem9b)
                        synonymous_nuc_to_AA_dict[mut] = AA_mut
                    elseif refAA == qryAA && pos in rem9b && mut in N_9b_synonymous
                        synonymous_nuc_to_AA_dict[mut] = AA_mut
                    end
###################################
                    for nuc_mut2 in muts
                        mut2 = mixed2nuc(nuc_mut2)
                        pos2 = nuc_mut_int_comprehensive_dict[mut2]
                        if pos2 in coding_ranges
                            gene_number2 = gene_num(mut2)
                            if mut ≠ mut2 && gene_number == gene_number2 && nuc_to_AA_pos(mut) == nuc_to_AA_pos(mut2)
                                if nuc_codon_pos_dict[pos] == 1 && nuc_codon_pos_dict[pos2] == 2
                                    ref_triple = ref_nuc_comprehensive_dict[mut]*ref_nuc_comprehensive_dict[mut2]*string(ref_seq[pos+2])
                                    qry_triple = qry_nuc_comprehensive_dict[mut]*qry_nuc_comprehensive_dict[mut2]*string(ref_seq[pos+2])
                                elseif nuc_codon_pos_dict[pos] == 1 && nuc_codon_pos_dict[pos2] == 3
                                    ref_triple = ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*ref_nuc_comprehensive_dict[mut2]
                                    qry_triple = qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*qry_nuc_comprehensive_dict[mut2]
                                elseif nuc_codon_pos_dict[pos] == 2 && nuc_codon_pos_dict[pos2] == 3
                                    ref_triple = string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]*ref_nuc_comprehensive_dict[mut2]
                                    qry_triple = string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]*qry_nuc_comprehensive_dict[mut2]
                                elseif nuc_codon_pos_dict[pos2] == 1 && nuc_codon_pos_dict[pos] == 2
                                    ref_triple = ref_nuc_comprehensive_dict[mut2]*ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                                    qry_triple = qry_nuc_comprehensive_dict[mut2]*qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                                elseif nuc_codon_pos_dict[pos2] == 1 && nuc_codon_pos_dict[pos] == 3
                                    ref_triple = ref_nuc_comprehensive_dict[mut2]*string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]
                                    qry_triple = qry_nuc_comprehensive_dict[mut2]*string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]
                                elseif nuc_codon_pos_dict[pos2] == 2 && nuc_codon_pos_dict[pos] == 3
                                    ref_triple = string(ref_seq[pos-2])*ref_nuc_comprehensive_dict[mut2]*ref_nuc_comprehensive_dict[mut]
                                    qry_triple = string(ref_seq[pos-2])*qry_nuc_comprehensive_dict[mut2]*qry_nuc_comprehensive_dict[mut]
                                end
                                refAA2 = AA_triplets[ref_triple]
                                qryAA2 = AA_triplets[qry_triple]
                                AA_mut2 = nuc2AA_ORF1a(mut2, refAA2, qryAA2)
                                if refAA2 == qryAA2 && !(pos2 in rem9b)
                                    synonymous_nuc_to_AA_dict[mut2] = AA_mut2
                                end
                            end
                        end
                    end
                else
                    qry_nuc = qry_nuc_comprehensive_dict[nuc_mut]
                    for (start_end, place) in noncoding_range_dict
                        frst = start_end[1]
                        last = start_end[2]
                        if pos ≥ frst && pos ≤ last
                            mut_vec = [nuc_mut, place]
                            push!(noncoding_nuc_vector, nuc_mut)
                        end
                    end
                end
            end
        end
#########################################################################################################
        for nuc_mut in muts
            pos_9b = nuc_mut_int_comprehensive_dict[nuc_mut]
            if pos_9b in rem9b && !isempty(ref_seq) && nuc_mut ≠ ""
                mut_9b = mixed2nuc(nuc_mut)
                pos_9b = nuc_mut_int_comprehensive_dict[mut_9b]   
                gene_number_9b = 11
                ref_triple_9b = ""
                qry_triple_9b = ""
                if nuc_codon_pos_dict_9b[pos_9b] == 1
                    ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])
                    qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])
                elseif nuc_codon_pos_dict_9b[pos_9b] == 2
                    ref_triple_9b = string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                    qry_triple_9b = string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                elseif nuc_codon_pos_dict_9b[pos_9b] == 3
                    ref_triple_9b = string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]
                    qry_triple_9b = string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]
                end
                refAA_9b = AA_triplets[ref_triple_9b]
                qryAA_9b = AA_triplets[qry_triple_9b]
                AA_mut_9b = nuc2AA_ORF9b(mut_9b, refAA_9b, qryAA_9b)
                if refAA_9b == qryAA_9b && nuc_mut in N_9b_synonymous
                    synonymous_nuc_to_AA_dict[mut_9b] = AA_mut_9b
                end
###################################
                for nuc_mut2_9b in muts
                    mut2_9b = mixed2nuc(nuc_mut2_9b)
                    pos2_9b = nuc_mut_int_comprehensive_dict[mut2_9b]
                    if pos2_9b in rem9b
                        gene_number2_9b = 11
                        if mut_9b ≠ mut2_9b && nuc_to_AA_pos_9b(mut_9b) == nuc_to_AA_pos_9b(mut2_9b)
                            if nuc_codon_pos_dict_9b[pos_9b] == 1 && nuc_codon_pos_dict_9b[pos2_9b] == 2
                                ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*ref_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b+2])
                                qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*qry_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b+2])
                            elseif nuc_codon_pos_dict_9b[pos_9b] == 1 && nuc_codon_pos_dict_9b[pos2_9b] == 3
                                ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*ref_nuc_comprehensive_dict[mut2_9b]
                                qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*qry_nuc_comprehensive_dict[mut2_9b]
                            elseif nuc_codon_pos_dict_9b[pos_9b] == 2 && nuc_codon_pos_dict_9b[pos2_9b] == 3
                                ref_triple_9b = string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]*ref_nuc_comprehensive_dict[mut2_9b]
                                qry_triple_9b = string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]*qry_nuc_comprehensive_dict[mut2_9b]
                            elseif nuc_codon_pos_dict_9b[pos2_9b] == 1 && nuc_codon_pos_dict_9b[pos_9b] == 2
                                ref_triple_9b = ref_nuc_comprehensive_dict[mut2_9b]*ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                                qry_triple_9b = qry_nuc_comprehensive_dict[mut2_9b]*qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                            elseif nuc_codon_pos_dict_9b[pos2_9b] == 1 && nuc_codon_pos_dict_9b[pos_9b] == 3
                                ref_triple_9b = ref_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]
                                qry_triple_9b = qry_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]
                            elseif nuc_codon_pos_dict_9b[pos2_9b] == 2 && nuc_codon_pos_dict_9b[pos_9b] == 3
                                ref_triple_9b = string(ref_seq[pos_9b-2])*ref_nuc_comprehensive_dict[mut2_9b]*ref_nuc_comprehensive_dict[mut_9b]
                                qry_triple_9b = string(ref_seq[pos_9b-2])*qry_nuc_comprehensive_dict[mut2_9b]*qry_nuc_comprehensive_dict[mut_9b]
                            end
                            refAA2_9b = AA_triplets[ref_triple_9b]
                            qryAA2_9b = AA_triplets[qry_triple_9b]
                            AA_mut2_9b = nuc2AA_ORF9b(mut2_9b, refAA2_9b, qryAA2_9b)
                            if refAA2_9b == qryAA2_9b && nuc_mut2_9b in N_9b_synonymous 
                                synonymous_nuc_to_AA_dict[mut2_9b] = AA_mut2_9b
                                synonymous_nuc_to_AA_dict[mut_9b] = AA_mut2_9b
                            end
                        end
                    end
                end
            end
        end
#####################################################################################################################################
        synonymous_nuc_to_AA_dict_sort = sort(collect(synonymous_nuc_to_AA_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
        synonymous_nuc_sort = sort(collect(keys(synonymous_nuc_to_AA_dict)), by = x -> nuc_mut_int_comprehensive_dict[x])
        synonymous_AA_sort = sort(collect(values(synonymous_nuc_to_AA_dict)), by = x -> AA_gene_sortKey_2(x))
        noncoding_nuc_vector_sort = sort(collect(noncoding_nuc_vector), by = x -> nuc_mut_int_comprehensive_dict[x])
        synonymous_nuc_total = length(synonymous_nuc_sort)
#        for i in 1:length(synonymous_nuc_sort)
#            nucpad = rpad("$(synonymous_nuc_to_AA_dict_sort[i][1])", 10)
#        end
        return synonymous_nuc_sort, noncoding_nuc_vector_sort
    else
        aa = Vector{Pair{String, String}}()
        bb = Vector{Pair{String, String}}()
        cc = Vector{Pair{String, String}}()
        return aa, bb, cc
    end 
end
#################################################################################
function add_leading_zero(int_str::String)
    int_str2 = int_str
    if length(int_str) == 1 && int_str ≠ "0"
        int_str2 = "0"*int_str
    end
    return int_str2
end     
##############################################################################################################################
###################################################################################################################
index_to_date_str = Dict{Int, String}()
date_str_to_index = Dict{String, Int}()
function convert_date_to_date_index(date_str::String)
    date_arr = string.(collect(date_str))
    date_tuple = nothing
### This counts the number of times "-" appears in the date_arr ---> sum(date_arr .== "-")
    if sum(date_arr .== "-") == 0
        year = parse(Int, date_str)
        date_tuple = (year, 0, 0)
    end
    if sum(date_arr .== "-") > 0
        year = parse(Int, split(date_str, "-")[1])
        month = parse(Int, split(date_str, "-")[2])
        if sum(date_arr .== "-") == 1
            date_tuple = (year, month, 0)
        else
            day = parse(Int, split(date_str, "-")[3])
            date_tuple = (year, month, day)
        end
    end
    date_index = tuple_to_index[date_tuple]
    return date_index
end
print("\n"^1)
println("Done Loading Functions (line #1545 as of 2026_04_02)!")
nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime); print("\n"^1)
##############################################################################################################################################
##############################################################################################################################################
for tuple in keys(tuple_to_index)
    one = add_leading_zero(string(tuple[1]))
    two = add_leading_zero(string(tuple[2]))
    three = add_leading_zero(string(tuple[3]))
    date_string = one*"-"*two*"-"*three
    date_str_to_index[date_string] = convert_date_to_date_index(date_string)
end
for (index, tuple) in index_to_tuple
    one = add_leading_zero(string(tuple[1]))
    two = add_leading_zero(string(tuple[2]))
    three = add_leading_zero(string(tuple[3]))
    date_string = one*"-"*two*"-"*three
    index_to_date_str[index] = date_string
end
println("Done Making date_str_to_index & index_to_date_str (line #1563 as of 2026_04_02)!")
nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime); print("\n"^1)
##################################################################
date_to_tuple = Dict{String, Tuple{Int, Int, Int}}()
tuple_to_date = Dict{Tuple{Int, Int, Int}, String}()
function convert_date_to_date_tuple(date_str::String)
    date_arr = string.(collect(date_str))
    date_tuple = nothing
### This counts the number of times "-" appears in the date_arr ---> sum(date_arr .== "-")
    if sum(date_arr .== "-") == 0
        year = parse(Int, date_str)
        date_tuple = (year, 0, 0)
    end
    if sum(date_arr .== "-") > 0
        year = parse(Int, split(date_str, "-")[1])
        month = parse(Int, split(date_str, "-")[2])
        if sum(date_arr .== "-") == 1
            date_tuple = (year, month, 0)
        else
            day = parse(Int, split(date_str, "-")[3])
            date_tuple = (year, month, day)
        end
    end
    return date_tuple
end
for date in keys(date_str_to_index)
    date_to_tuple[date] = convert_date_to_date_tuple(date)
end
for (date, date_tuple) in date_to_tuple
    tuple_to_date[date_tuple] = date
end
println("Done Making date_to_tuple & tuple_to_date (line #1594 as of 2022_02_22)!")
nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime); print("\n"^1)
##################################################################################################################
function find_X_pct_date(clade_pango::String, pct::Float64, cp_date_cumul_dict::Dict{String, Dict{Int, Int}}, cp_total_dict::Dict{String, Int})
    cp_total = cp_total_dict[clade_pango]
    pct_date_index = 0
    pct_date_tuple = nothing
    for date_index in 1:2500
        cumulative_ct = cp_date_cumul_dict[clade_pango][date_index]
        if 100*cumulative_ct/cp_total ≥ pct
            pct_date_index = date_index
            pct_date_tuple = index_to_tuple[date_index]
            break
        end
    end
    return pct_date_index, pct_date_tuple
end
##################################################################################################################
### Truly necessary stuff from old megacell | 2026_02_08
############################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
all_chr_seqs_pangos = Dict{String, Int}()
all_chr_seqs_inherited = Dict{String, Int}()
all_chr_seqs_inherited_pos_only = Dict{String, Int}()
for seq in all_unique_chr_seqs
    pango = seq_pango[seq]
    all_chr_seqs_pangos[pango] = get(all_chr_seqs_pangos, pango, 0) + 1
    if pango == "B.1.1.529"
        pango_AAsub_WT[pango] = union(pango_AAsub_WT["BA.1"], pango_AAsub_WT["BA.2"])
        pango_AAsub_WT_pos_only[pango] = union(pango_AAsub_WT_pos_only["BA.1"], pango_AAsub_WT_pos_only["BA.2"])
        pango_AAdel_WT[pango] = union(pango_AAdel_WT["BA.1"], pango_AAdel_WT["BA.2"])
    end
    if !haskey(pango_AAdel_WT, pango)
        for i in 1:5
            if haskey(pango_predecessor_meta_dict, pango)
                if haskey(pango_predecessor_meta_dict[pango], i)
                    pango_i = pango_predecessor_meta_dict[pango][i]
                    if haskey(pango_AAdel_WT, pango_i)
                        pango_AAdel_WT[pango] = pango_AAdel_WT[pango_i]
                        println("Del: $(pango) => $(pango_i)")
                        break
                    end
                end
            end
        end
    end
    if !haskey(pango_AAsub_WT, pango)
        for i in 1:5
            if haskey(pango_predecessor_meta_dict, pango)
                if haskey(pango_predecessor_meta_dict[pango], i)
                    pango_i = pango_predecessor_meta_dict[pango][i]
                    if haskey(pango_AAsub_WT, pango_i)
                        pango_AAsub_WT[pango] = pango_AAsub_WT[pango_i]
                        println("Sub: $(pango) => $(pango_i)")
                        break
                    end
                end
            end
        end
    end
    if !haskey(pango_AAsub_WT_pos_only, pango)
        for i in 1:5
            if haskey(pango_predecessor_meta_dict, pango)
                if haskey(pango_predecessor_meta_dict[pango], i)
                    pango_i = pango_predecessor_meta_dict[pango][i]
                    if haskey(pango_AAsub_WT_pos_only, pango_i)
                        pango_AAsub_WT_pos_only[pango] = pango_AAsub_WT_pos_only[pango_i]
                        println("Pos_only: $(pango) => $(pango_i)")
                        break
                    end
                end
            end
        end
    end
    if haskey(pango_AAsub_WT, pango)
        for mut in pango_AAsub_WT[pango]
            mutpo = aa_gene_and_pos_comprehensive_dict[mut]
            if !(mutpo in seq_unknown_AA[seq])
                all_chr_seqs_inherited[mut] = get(all_chr_seqs_inherited, mut, 0) + 1
            end
        end
        for del in pango_AAdel_WT[pango]
            delpo = aa_gene_and_pos_comprehensive_dict[del]
            if !(delpo in seq_unknown_AA[seq])
                all_chr_seqs_inherited[del] = get(all_chr_seqs_inherited, del, 0) + 1
            end
        end
    else
        println("Problem: No pango_AAsub_WT key for $(pango)")
    end
end
println("Done Filling all_chr_seqs_pangos, all_chr_seqs_inherited, pango_AAsub_WT (leftovers)! (Line 1686 as of 2026_04_02)")
nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime); print("\n"^1)
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
### Making megaclade_EPCI_ct_dict and megaclade_EPCI_proportion_dict
date_now = Dates.format(now(), "yyyy_mm_dd__HMMp"); println(date_now)
nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime)
x_minor_ct = 0
x_minor_dict_ct = Dict{String, Int}()
unknown_EPCI_ct = 0
pre_VOC_ct = 0
pre_VOC_dict_ct = Dict{String, Int}()
pre_VOC_vec = String[]
total_EPCI_seq_ct = length(EPCI_set)
seq_megaclade = Dict{String, String}()
megaclade_EPCI_ct_dict = Dict{String, Int}()
megaclade_EPCI_proportion_dict = Dict{String, Float64}()
for seq in EPCI_set
    complete = 0
    pango = seq_pango[seq]
    unaliased = get(pango_to_pango_unaliased_v2, pango, pango)
    if length(unaliased) ≥ 3
        if unaliased[1:3] == "XBB" || unaliased[1:3] == "XCH"
            megaclade_EPCI_ct_dict["XBB"] = get(megaclade_EPCI_ct_dict, "XBB", 0) + 1
            seq_megaclade[seq] = "XBB"
            complete = 1
        elseif unaliased[1:3] == "XEC" || unaliased[1:3] == "XEK" || unaliased[1:3] == "XFG" || unaliased[1:3] == "XFC" || unaliased[1:3] == "XFJ"
            megaclade_EPCI_ct_dict["BA.2.86"] = get(megaclade_EPCI_ct_dict, "BA.2.86", 0) + 1
            seq_megaclade[seq] = "BA.2.86"
            complete = 1
        end
        if length(unaliased) ≥ 7 && complete == 0
            if unaliased[1:7] == "B.1.1.7"
                megaclade_EPCI_ct_dict["B.1.1.7"] = get(megaclade_EPCI_ct_dict, "B.1.1.7", 0) + 1
                seq_megaclade[seq] = "B.1.1.7"
                complete = 1
            elseif unaliased[1:7] == "B.1.351"
                megaclade_EPCI_ct_dict["B.1.351"] = get(megaclade_EPCI_ct_dict, "B.1.351", 0) + 1
                seq_megaclade[seq] = "B.1.351"
                complete = 1
            elseif unaliased[1:7] == "B.1.617"
                megaclade_EPCI_ct_dict["B.1.617"] = get(megaclade_EPCI_ct_dict, "B.1.617", 0) + 1
                seq_megaclade[seq] = "B.1.617"
                complete = 1
            end
            if length(unaliased) ≥ 9 && complete == 0
                if unaliased == "B.1.1.529"
                    megaclade_EPCI_ct_dict["OmiUnkn"] = get(megaclade_EPCI_ct_dict, "OmiUnkn", 0) + 1
                    seq_megaclade[seq] = "OmiUnkn"
                    complete = 1
                end
                if length(unaliased) ≥ 10 && complete == 0
                    if unaliased[1:10] == "B.1.1.28.1"
                        megaclade_EPCI_ct_dict["P.1"] = get(megaclade_EPCI_ct_dict, "P.1", 0) + 1
                        seq_megaclade[seq] = "P.1"
                        complete = 1
                    end
                    if length(unaliased) ≥ 11 && complete == 0
                        if unaliased[1:11] == "B.1.1.529.1"
                            megaclade_EPCI_ct_dict["BA.1"] = get(megaclade_EPCI_ct_dict, "BA.1", 0) + 1
                            seq_megaclade[seq] = "BA.1"
                            complete = 1
                        elseif unaliased[1:11] == "B.1.1.529.2"
                            megaclade_EPCI_ct_dict["BA.2"] = get(megaclade_EPCI_ct_dict, "BA.2", 0) + 1
                            seq_megaclade[seq] = "BA.2"
                            complete = 1
                        elseif unaliased[1:11] == "B.1.1.529.4"
                            megaclade_EPCI_ct_dict["BA.5"] = get(megaclade_EPCI_ct_dict, "BA.5", 0) + 1
                            seq_megaclade[seq] = "BA.5"
                            complete = 1
                        elseif unaliased[1:11] == "B.1.1.529.5"
                            megaclade_EPCI_ct_dict["BA.5"] = get(megaclade_EPCI_ct_dict, "BA.5", 0) + 1
                            seq_megaclade[seq] = "BA.5"
                            complete = 1
                        end
                        if length(unaliased) ≥ 14
                            if unaliased[1:14] == "B.1.1.529.2.86"
                                megaclade_EPCI_ct_dict["BA.2.86"] = get(megaclade_EPCI_ct_dict, "BA.2.86", 0) + 1
                                seq_megaclade[seq] = "BA.2.86"
                                megaclade_EPCI_ct_dict["BA.2"] = megaclade_EPCI_ct_dict["BA.2"] - 1
                                complete = 1
                            end
                        end
                    end
                end
            end
        end
    end
    if complete == 0
        if unaliased[1] == 'X'
            x_minor_ct += 1
            x_minor_dict_ct[unaliased] = get(x_minor_dict_ct, unaliased, 0) + 1
            megaclade_EPCI_ct_dict["Recomb"] = get(megaclade_EPCI_ct_dict, "Recomb", 0) + 1
            seq_megaclade[seq] = "Recomb"
            complete = 1
        end
        if unaliased[1] == 'A' || unaliased[1] == 'B'
            pre_VOC_ct += 1
            pre_VOC_dict_ct[unaliased] = get(pre_VOC_dict_ct, unaliased, 0) + 1
            megaclade_EPCI_ct_dict["pre_VOC"] = get(megaclade_EPCI_ct_dict, "pre_VOC", 0) + 1
            seq_megaclade[seq] = "pre_VOC"
            complete = 1
        end
    end
    if complete == 0
        println("Messed Up Pango = $(pango)")
        seq_megaclade[seq] = "Unknown"
        unknown_EPCI_ct += 1
    end
end
######################################################################################################################
for (megaclade, count) in megaclade_EPCI_ct_dict
    megaclade_EPCI_proportion_dict[megaclade] = count/total_EPCI_seq_ct
end
######################################################################################################################
print("\n"^1)
println("unknown_EPCI_ct = $(unknown_EPCI_ct)")
println("x_minor_ct = $(x_minor_ct)")
println("pre_VOC_ct = $(pre_VOC_ct)")
print("\n"^1)
x_minor_dict_ct_sort = sort(collect(x_minor_dict_ct), by = x -> x[1])
for minorX___ct in x_minor_dict_ct_sort
    minorX_pad = rpad(minorX___ct[1], 12)
    count_pad = lpad(minorX___ct[2], 2)
#    println("$(minorX_pad) = $(count_pad)")
end 
pre_VOC_dict_ct_sort = sort(collect(pre_VOC_dict_ct), by = x -> x[1])
for preVOC___ct in pre_VOC_dict_ct_sort
    preVOC_pad = rpad(preVOC___ct[1], 12)
    count_pad = lpad(preVOC___ct[2], 2)
#    println("$(preVOC_pad) = $(count_pad)")
end
######################################################################################################################
megaclade_EPCI_ct_dict_sort = sort(collect(megaclade_EPCI_ct_dict), by = x -> x[2], rev=true)
for megaclade____count in megaclade_EPCI_ct_dict_sort
    megaclade = megaclade____count[1]
    count = megaclade____count[2]
    println("$(megaclade) = $(count)")
end
print("\n"^1)
date_now = Dates.format(now(), "yyyy_mm_dd__HMMp"); println(date_now)
nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime)
print("\n"^1)
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
### Making AA_mut_pango/clade/cladepango/megaclade dicts
start = time()
AA_mut_pango_ct = Dict{String, Dict{String, Int}}()
AA_mut_clade_ct = Dict{String, Dict{String, Int}}()
AA_mut_cladepango_ct = Dict{String, Dict{String, Int}}()
AA_mut_megaclade_ct = Dict{String, Dict{String, Int}}()
for mut in keys(AA_muts_ct)
    AA_mut_pango_ct[mut] = Dict{String, Int}()
    AA_mut_clade_ct[mut] = Dict{String, Int}()
    AA_mut_cladepango_ct[mut] = Dict{String, Int}()
    AA_mut_megaclade_ct[mut] = Dict{String, Int}()
    for seq in all_unique_chr_seqs
        if mut in seq_AA_muts[seq]
            pango = seq_pango[seq]
            clade = seq_clade[seq]
            cladepango = clade_to_pango[clade]
            megaclade = seq_megaclade[seq]
            AA_mut_pango_ct[mut][pango] = get(AA_mut_pango_ct[mut], pango, 0) + 1
            AA_mut_clade_ct[mut][clade] = get(AA_mut_clade_ct[mut], clade, 0) + 1
            AA_mut_cladepango_ct[mut][cladepango] = get(AA_mut_cladepango_ct[mut], cladepango, 0) + 1
            AA_mut_megaclade_ct[mut][megaclade] = get(AA_mut_megaclade_ct[mut], megaclade, 0) + 1
        end
    end
end
println("Done Filling AA_mut_pango_ct, AA_mut_clade_ct, AA_mut_cladepango_ct, AA_mut_megaclade_ct! (Line #1857 as of 2026_04_02")
nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime)
print("\n"^2)
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
#seq_syn_nucs = Dict{String, Vector{String}}()
#seq_noncoding_nucs = Dict{String, Vector{String}}()
#for (seq, nuc_set) in seq_nuc_muts
#    pango = seq_pango[seq]
#    if !haskey(nuc_genome_pango_dict, pango)
#        println("No nuc_genome_pango_dict key, $(pango)")
#    end
#    synonymous_nucmuts, noncoding_nucmuts = SIMPLER_syn_noncoding_nuc(pango, nuc_set)
#    seq_syn_nucs[seq] = synonymous_nucmuts
#    seq_noncoding_nucs[seq] = noncoding_nucmuts
#end
#println("Done Filling seq_syn_nucs, seq_noncoding_nucs! (Line 1874 as of 2026_04_02")
# nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime); print("\n"^1)
############################################################################################################################################################################
########################################################### Create pango_seq_total + pango_date_index_pct ##################################################################
############################################################################################################################################################################
pango_seq_total = Dict{String, Int}()
for pango in keys(pango_date_index_ct)
    pango_seq_sum = sum(values(pango_date_index_ct[pango]))
    pango_seq_total[pango] = pango_seq_sum
end
for pango in pango_set
    if !haskey(pango_date_index_ct, pango)
        for i in 1:5
            if haskey(pango_predecessor_meta_dict, pango)
                if haskey(pango_predecessor_meta_dict[pango], i)
                    pango_i = pango_predecessor_meta_dict[pango][i]
                    if haskey(pango_date_index_ct, pango_i)
                        pango_date_index_ct[pango] = pango_date_index_ct[pango_i]
                        println("$(pango) => $(pango_i)")
                        pango_seq_sum = sum(values(pango_date_index_ct[pango]))
                        pango_seq_total[pango] = pango_seq_sum
                        break
                    end
                end
            end
        end
    end
end
print("\n"^1)
pango_date_index_pct = Dict{String, Dict{Int, Float64}}()
for (pango, pango_total) in pango_seq_total
    pango_date_index_pct[pango] = Dict{Int, Float64}()
    pango_cumuluative = 0
    for i in 1:3000
        pango_cumuluative += get(pango_date_index_ct[pango], i, 0)
        pango_date_index_pct[pango][i] = 100*pango_cumuluative/pango_total
    end
end
#for seq in pango_set
#    if !haskey(pango_seq_total, pango)
#        for i in 1:5
#            if haskey(pango_predecessor_meta_dict, pango)
#                if haskey(pango_predecessor_meta_dict[pango], i)
#                    pango_i = pango_predecessor_meta_dict[pango][i]
#                    pango_total = pango_seq_total[pango_i]
#                    break
#                end
#            end
#        end
#    end
#end 
############################################################################################################################################################################
############################################################################################################################################################################
runtime = time() - start
runtime_rd = round(digits=2, runtime)
runtime1, runtime2 = seconds_to_hrs_min_sec(runtime)
println(); println("EPCI_qc_str = $(EPCI_qc_str) | HQCS_qc_string = $(HQCS_qc_string)"); println()
println("Runtime v0 = $(runtime) seconds")
println("Runtime v2 = $(runtime2)")
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
println("Finished!"); print("\n"^2)


2026_05_05__2357PM
11:57.02_PM


Done Loading Functions (line #1545 as of 2026_04_02)!
11:57.04_PM

Done Making date_str_to_index & index_to_date_str (line #1563 as of 2026_04_02)!
11:57.04_PM

Done Making date_to_tuple & tuple_to_date (line #1594 as of 2022_02_22)!
11:57.04_PM

Done Filling all_chr_seqs_pangos, all_chr_seqs_inherited, pango_AAsub_WT (leftovers)! (Line 1686 as of 2026_04_02)
11:57.04_PM

2026_05_05__2357PM
11:57.04_PM

unknown_EPCI_ct = 0
x_minor_ct = 25
pre_VOC_ct = 269

BA.5 = 532
BA.2 = 493
BA.1 = 441
XBB = 376
pre_VOC = 269
BA.2.86 = 162
B.1.617 = 145
B.1.1.7 = 27
Recomb = 25
OmiUnkn = 9
B.1.351 = 8
P.1 = 3

2026_05_05__2357PM
11:57.05_PM

Done Filling AA_mut_pango_ct, AA_mut_clade_ct, AA_mut_cladepango_ct, AA_mut_megaclade_ct! (Line #1857 as of 2026_04_02
11:57.13_PM




EPCI_qc_str = 15_20_95 | HQCS_qc_string = 5_1_5

Runtime v0 = 13.243319988250732 seconds
Runtime v2 = 0 hr, 0 min, 13.24 sec
Finished!




In [136]:
### There are several different versions of this function, and the parameters below specify which one to run
                                                        sub_0__posonly_1 = 1
                                                        normal_0__spikeonly_1__spikeWithRBD_2 = 0
                                                        noBAL_0__withBAL_1 = 1
                                                        include_RBM = 0
###########################################################################################################################################################################

0

In [137]:
### Load mp_meta_fake_chr_dict_DQ  | This loads the APCI datasets, which were created in the main EPCI notebook | Runtime = ~3 seconds
date_now = Dates.format(now(), "yyyy_mm_dd__HMMp"); println(date_now)
start = time()
########################################################################################
folder_name = ""
save_name = ""
date = "2026_05_03"
if sub_0__posonly_1 == 0
    if normal_0__spikeonly_1__spikeWithRBD_2 == 0
        folder_name = "mp_fake__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)__$(date)"
        save_name = "mp_meta_fake_chr_dict_DQ__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)"
    elseif normal_0__spikeonly_1__spikeWithRBD_2 == 1
        folder_name = "mp_fake_spike_only__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)__$(date)"
        save_name = "mp_meta_fake_chr_dict_DQ__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)"
    end
elseif sub_0__posonly_1 == 1
    if normal_0__spikeonly_1__spikeWithRBD_2 == 0
        folder_name = "mp_fake_pos_only__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)__$(date)"
        save_name = "mp_meta_fake_pos_only_chr_dict_DQ__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)"
    elseif normal_0__spikeonly_1__spikeWithRBD_2 == 1
        folder_name = "mp_fake_pos_only_spike_only__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)__$(date)"
        save_name = "mp_meta_fake_pos_only_spike_only_chr_dict_DQ__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)"
    end
end
println("EPCI_qc_str = $(EPCI_qc_str)")
########################################################################################
#                                Run#     seqIndex     seqAAsubs
# mp_meta_fake_chr_dict_DQ = Dict{Int, Dict{Int, Vector{String}}}()
mp_meta_fake_chr_dict_DQ = load("$(folder_name)/$(save_name)_$(date).jld2", "mp_meta_fake_chr_dict_DQ")
mp_meta_fake_chr_DQ_bad_cts = load("$(folder_name)/$(save_name)_bad_cts_$(date).jld2", "mp_meta_fake_chr_DQ_bad_cts")
runtime = time() - start
runtime1, runtime2 = seconds_to_hrs_min_sec(runtime)
println("Runtime v2 = $(runtime2)")
println()
fake_seq_AA_muts_test = mp_meta_fake_chr_dict_DQ[1]
total_fake_seqs = length(fake_seq_AA_muts_test)
total_runs = length(mp_meta_fake_chr_dict_DQ)
println("Total Fake Seqs = $(total_fake_seqs)")
println("Total Fake Runs = $(total_runs)")
total_AA = 0
for i in 1:total_fake_seqs
    tot_AA = length(fake_seq_AA_muts_test[i])
    total_AA += tot_AA
end
avg_AA = total_AA/total_fake_seqs
println("Avg AA per seq = $(avg_AA)")
######################################################################################################################################
under5_ct = 0
#                          Run#     Seq#  seqAAsub_ct
seq_privAA_len_meta = Dict{Int, Dict{Int,Int}}()
for i in 1:total_runs
    seq_privAA_len_meta[i] = Dict{Int,Int}()
    for j in 1:total_fake_seqs
        seq_total_AA = length(mp_meta_fake_chr_dict_DQ[i][j])
        if seq_total_AA < 5
            under5_ct += 1
        end
        seq_privAA_len_meta[i][j] = seq_total_AA
    end
end
println("under5_ct = $(under5_ct)")
print("\n"^2) 
if sub_0__posonly_1 == 1
    println("Category: pos_only")
end
if sub_0__posonly_1 == 0
    println("Category: subs")
end
print("\n"^2)
println("EPCI_qc_str = $(EPCI_qc_str)")
println("HQCS_qc_string = $(HQCS_qc_string)")
print("\n"^2) 
println(typeof(mp_meta_fake_chr_dict_DQ))
println(typeof(seq_privAA_len_meta[1]))
println("seq_privAA_len_meta[1][1111] = $(seq_privAA_len_meta[1][2222])"); print("\n"^1)
println("Finished!"); print("\n"^1)
######################################################################################################################################
#print("\n"^1)

2026_05_05__2357PM
EPCI_qc_str = 15_20_95
Runtime v2 = 0 hr, 0 min, 02.00 sec

Total Fake Seqs = 2490
Total Fake Runs = 100
Avg AA per seq = 20.019277108433734
under5_ct = 0


Category: pos_only


EPCI_qc_str = 15_20_95
HQCS_qc_string = 5_1_5


Dict{Int64, Dict{Int64, Vector{String}}}
Dict{Int64, Int64}
seq_privAA_len_meta[1][1111] = 17

Finished!



In [138]:
### Various checks on mp_meta_fake_chr_dict_DQ to make sure it's correct
date_now = Dates.format(now(), "yyyy_mm_dd__HMMp"); println(date_now)

total_runs = length(collect(keys(mp_meta_fake_chr_dict_DQ)))
println("total_runs = $(total_runs)")

for i in 1:1
    println(mp_meta_fake_chr_dict_DQ[i][24])
end
####################################################################################################################################
####################################################################################################################################
println("Size of mp_meta_fake_chr_dict_DQ = $(length(keys(mp_meta_fake_chr_dict_DQ)))")
println("Size of mp_meta_fake_chr_dict_DQ[1] = $(length(keys(mp_meta_fake_chr_dict_DQ[1])))")
#println("Size of mp_meta_fake_chr_dict_DQ[1][1] = $(length(keys(mp_meta_fake_chr_dict_DQ[1][1])))")
#for (run, badct) in mp_meta_fake_chr_DQ_bad_cts
#    println("$(run) = $(badct)")
#end
####################################################################################################################################
####################################################################################################################################
date_now = Dates.format(now(), "yyyy_mm_dd__HMMp"); println(date_now)
fake_seq_AA_muts_test = mp_meta_fake_chr_dict_DQ[1]
println(length(fake_seq_AA_muts_test))
total_AA = 0
for i in 1:2222
    tot_AA = length(fake_seq_AA_muts_test[i])
    total_AA += tot_AA
end
avg_AA = total_AA/2222
println("Avg AA per seq = $(avg_AA)"); print("\n"^2)
####################################################################################################################################
####################################################################################################################################

2026_05_05__2357PM
total_runs = 100
["S:499", "S:368", "S:405", "S:243", "ORF7b:24", "ORF1a:3588", "ORF7a:103", "ORF3a:155", "ORF1a:4174", "M:94", "ORF8:32", "ORF1a:2790", "ORF1a:12", "E:68"]
Size of mp_meta_fake_chr_dict_DQ = 100
Size of mp_meta_fake_chr_dict_DQ[1] = 2490
2026_05_05__2357PM
2490
Avg AA per seq = 20.10936093609361




In [139]:
### Get all RBM & RBD muts + ORF9b/N Doubles + artifactual_private_muts_subs + subs vs pos_only | Runtime = 1 sec
start = time(); print("\n"^1)
print("\n"^1); date_now = Dates.format(now(), "yyyy_mm_dd__HMMp"); println(date_now); nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime); print("\n"^1)
date_hour = Dates.format(now(), "yyyy_mm_dd_Hp"); print("\n"^1)
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
#                                                        sub_0__posonly_1 = 1
#                                                        normal_0__spikeonly_1__spikeWithRBD_2 = 0
#                                                        noBAL_0__withBAL_1 = 1
#                                                        include_RBM = 0
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
#--------------------------------------------------------------------------------------------------------------------------------------------------------------------------
########################################################################################################################################################################### 
############################################################# Begin EXCLUDED MUTS SECTION #################################################################################
########################################################################################################################################################################### 
##### The mutations added below are ones that have been found to produce artifactual correlations. For example, 18/19 EPCI sequences 
####      with ORF1a:M2606I are B.1.2 (the other is likely a B.1.2 miscategorized as a B.1), and seven of these have N:D377Y, indicating
####      that these are both inherited (not private) mutations, which can be confirmed by viewing the Usher Tree for these sequences. 
####      Nextclade miscategorizes both of these as private mutations. Similarly, 5/6 sequences with ORF1a:E3764D are B.1.258, and all five also 
####      have ORF3a:A72S. Both are inherited and mischaracterized as private by Nextclade. 
global artifactual_private_muts_subs = Set{String}()
#push!(artifactual_private_muts_subs, "ORF1a:T2501I") ## Artifactual reversion in B.1 sequences (possibly real? Unclear)
push!(artifactual_private_muts_subs, "ORF1a:M2606I") ## Inherited B.1.2 mutation
push!(artifactual_private_muts_subs, "N:D377Y")      ## Inherited B.1.2 mutation
push!(artifactual_private_muts_subs, "ORF3a:A72S")   ## Inherited B.1.258 mutation
push!(artifactual_private_muts_subs, "ORF1a:E3764D") ## Inherited B.1.258 mutation
push!(artifactual_private_muts_subs, "ORF1a:I2283T") ## Artifactual reversion in XEC miscategorized as KP.3 or KP.3.3 by Nextclade
push!(artifactual_private_muts_subs, "ORF1a:A599T")  ## Artifactual "private" mutation in XEC miscategorized as KP.3 or KP.3.3 by Nextclade
push!(artifactual_private_muts_subs, "S:F59S")       ## Artifactual "private" mutation in XEC miscategorized as KP.3 or KP.3.3 by Nextclade
push!(artifactual_private_muts_subs, "ORF8:L60F")    ## Artifactual "private" mutation in AY.44. Over 125000 seqs have both ORF8:L60F & ORF1a:H2125Y
push!(artifactual_private_muts_subs, "ORF1a:H2125Y") ## Artifactual "private" mutation in AY.44. Over 125000 seqs have both ORF8:L60F & ORF1a:H2125Y
push!(artifactual_private_muts_subs, "S:Q146K")      ## Extremely homoplasic XBB mutation and/or artifact. Impossible to tell if inherited or private. 
push!(artifactual_private_muts_subs, "N:M203K")      ## Recombinant Delta/Omicron "mutation"
push!(artifactual_private_muts_subs, "N:G204R")      ## Recombinant Delta/Omicron "mutation"
push!(artifactual_private_muts_subs, "N:R204G")      ## Recombinant Delta/Omicron reversion "mutation"
push!(artifactual_private_muts_subs, "ORF1b:M1156I") ## BQ.1 mutation misattributed as private in 3 sequences
push!(artifactual_private_muts_subs, "ORF6:L61D")    ## Artifactual 3-nuc reversion, often in BA.1/2 or BA.2/5 recombinants
push!(artifactual_private_muts_subs, "ORF1a:S135R")  ## Omicron mut misattributed as private in 3 recombinants
push!(artifactual_private_muts_subs, "M:A30T")       ## Very common artifactual reversion in BA.2.86* lineages
push!(artifactual_private_muts_subs, "ORF1a:S1612L") ## Inherited Beta mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF7b:E39*")   ## Inherited Beta mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "S:F18L")       ## All five on list in Beta, likely artifactual
push!(artifactual_private_muts_subs, "ORF1a:A2554V") ## Inherited AY.44 mutation miscategorized by Nextclade as private 
push!(artifactual_private_muts_subs, "ORF1b:H1087Y") ## Inherited AY.44 mutation miscategorized by Nextclade as private 
push!(artifactual_private_muts_subs, "ORF1a:M2796T") ## Inherited BA.1.17 mutation miscategorized by Nextclade as private 
push!(artifactual_private_muts_subs, "ORF1a:P1803S") ## Inherited BA.1.17 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF3a:P104S")  ## Inherited AY.103 mutation miscategorized by Nextclade as private 
push!(artifactual_private_muts_subs, "N:A208S")      ## Inherited AY.103 mutation miscategorized by Nextclade as private 
push!(artifactual_private_muts_subs, "ORF8:K68*")    ## Inherited B.1.1.7 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:P1975S") ## Inherited B.1.1.7 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:V2178F") ## Inherited BA.1.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:I97V")   ## Inherited BA.1.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:A2589T") ## Inherited BA.5.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF9b:T83I")   ## Inherited BA.5.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:T842I")  ## From 5' End Recombination with BA.2
push!(artifactual_private_muts_subs, "ORF1a:S135R")  ## From 5' End Recombination with BA.2
push!(artifactual_private_muts_subs, "ORF3a:V48F")   ## Inherited BE.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF3a:G49C")   ## Inherited BE.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:A1204T") ## Inherited BE.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "S:A376P")      ## Butchered mutation interpretation by Nextclade for ∆374-375 or ∆374-376
push!(artifactual_private_muts_subs, "S:T376P")      ## Butchered mutation interpretation by Nextclade for ∆374-375 or ∆374-376
push!(artifactual_private_muts_subs, "S:F375A")      ## Butchered mutation interpretation by Nextclade for ∆374-375 or ∆374-376
push!(artifactual_private_muts_subs, "S:S375A")      ## Butchered mutation interpretation by Nextclade for ∆374-375 or ∆374-376
push!(artifactual_private_muts_subs, "S:D339G")      ## An artifactual reversion >90% of the time, likely 100%
push!(artifactual_private_muts_subs, "S:N417K")      ## An artifactual reversion >90% of the time, likely 100%
push!(artifactual_private_muts_subs, "S:K440N")      ## An artifactual reversion >90% of the time, likely 100%
push!(artifactual_private_muts_subs, "ORF1b:T1555I") ## Inherited B.1.2 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:A4285V") ## Inherited B.1.2 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:N2361K") ## Inherited B.1.2 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:V3782I") ## Inherited BA.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "S:K1191N")     ## Inherited B.1.1.7 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "S:R158G")      ## Artifactual Delta reversion
push!(artifactual_private_muts_subs, "ORF1a:I1023-") ## Dumb mistake I made that I have to undo with this.
push!(artifactual_private_muts_subs, "ORF1a:T3646A") ## Inherited Delta mut, miscategorized in recombinants 
push!(artifactual_private_muts_subs, "ORF1b:A1918V") ## Inherited Delta mut, miscategorized in recombinants
push!(artifactual_private_muts_subs, "ORF7b:I2V")    ## Corresponds to ORF7a:*122W
push!(artifactual_private_muts_subs, "ORF1b:P218L")  ## Inherited B.1.1.7 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:V3782I") ## Inherited mutation in Canadian BA.1 branch miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:G2118D") ## B.1 Inherited mutation miscategorized by Nextclade as private (co-occurs with S:P681H)
push!(artifactual_private_muts_subs, "ORF1a:T3646A") ## Inherited Delta mut, miscategorized in recombinants 
push!(artifactual_private_muts_subs, "ORF1b:A1918V") ## Inherited Delta mut, miscategorized in recombinants
push!(artifactual_private_muts_subs, "ORF1a:A1298V") ## Inherited BA.1 mut, miscategorized by Nextclade as private


push!(artifactual_private_muts_subs, "ORF3a:V112F")  ## Inherited B.1.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:N2695I") ## Inherited B.1.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF3a:I35K")   ## Inherited XBB.1.16.11 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:E1871G") ## Inherited B.1.1.7 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:D1903Y") ## Inherited XBB.1.16.31 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:K2219R") ## Inherited XBB.1.16.31 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:K1094R") ## Inherited B.1.2 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:S771I")  ## Inherited XBB.1.5 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "N:S26C")       ## Inherited BA.5 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:V108I")  ## Inherited BA.5.1 mutation in Scandinavian branch miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:G128S")  ## Inherited BA.5.1 mutation in Scandinavian branch miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:N2452S") ## Inherited BA.5.1 mutation in Scandinavian branch miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:S1182T") ## Inherited FL.4 mutation in Australian branch miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:P1570T") ## Inherited BA.1.1.1 mutation in Belgian branch miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:E2355Q") ## Inherited BA.1.1.1 mutation in Belgian branch miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "S:T19R")       ## Delta mutation present in recombinants, falsely labeled as private mutation
push!(artifactual_private_muts_subs, "ORF7a:A66V")   ## Inherited B.1.429 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:T3459M") ## Inherited B.1.429 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF7b:M1T")    ## Corresponds to ORF7a:*122R
push!(artifactual_private_muts_subs, "ORF1b:A869V")  ## Inherited KF.1 (FL.15.1.1.1) mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:E1015G") ## Inherited KF.1 (FL.15.1.1.1) mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF7b:L4F")    ## Inherited BA.1 mutation on Swedish branch miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:V365I")  ## Inherited B.1.2 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:K1094R") ## Inherited B.1.2 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:D1869Y") ## Inherited C.37 and AY.85 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF3a:L65H")   ## Inherited BN.1.3.1 mutation (Italian branch) miscategorized by Nextclade as private
   
push!(artifactual_private_muts_subs, "ORF3a:S220I")  ## Inherited B.1.466 (Indonesia) mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:A176T")  ## Inherited EG.5.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:E352D")  ## Inherited B.1.1.176 (Canada) mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1b:A434S")  ## Inherited B.1.1.176 (Canada) mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:G3676C") ## Inherited B.1.1.176 (Canada) mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "ORF1a:Q991L")  ## Inherited BA.5.2 (North America) mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_subs, "S:V143D")      ## This is actually ∆143 combined with an inherited S:G142D
push!(artifactual_private_muts_subs, "S:Y144D")      ## This is actually ∆143-144 combined with an inherited S:G142D
######################################################################################################################################
######################################################################################################################################
#### Double_N_ORF9b_muts are muts that result in both N and ORF9b mutations, which of course correlate perfectly but trivially.
####    Default is to block N muts, but it's also possible to change that and instead block ORF9b muts to view their N equivalents
####    by commenting out the top list and uncommenting out the bottom list.
#### This list was created in one of the above cells (the one with the ORF9b_N_nucmuts_to_AAres function) & then copied & pasted here.
global Double_N_ORF9b_muts = Set(["N:G63D", "N:F33S, N:L13-", "N:L13I", "N:L13H", "N:L13S", "N:N4I", "N:N4T", "N:N4S", "N:N4K", "N:N4K", "N:G5*", "N:G5R", "N:G5R", "N:G5V", "N:G5A", "N:G5E", "N:P6T", "N:P6A", "N:P6L", "N:P6H", "N:P6R", "N:Q7L", "N:Q7P", "N:Q7R", "N:Q7H", "N:Q7H", "N:N8Y", "N:N8H", "N:N8I", "N:N8T", "N:N8S", "N:N8K", "N:N8K", "N:Q9E", "N:Q9L", "N:Q9P", "N:Q9R", "N:Q9H", "N:Q9H", "N:R10G", "N:R10L", "N:R10P", "N:R10Q", "N:N11Y", "N:N11H", "N:N11I", "N:N11T", "N:N11S", "N:N11K", "N:N11K", "N:A12S", "N:A12P", "N:A12T", "N:A12V", "N:A12E", "N:A12G", "N:P13T", "N:P13A", "N:P13L", "N:L13P", "N:P13H", "N:P13R", "N:R14L", "N:R14P", "N:R14H", "N:I15N", "N:I15S", "N:I15M", "N:T16S", "N:T16P", "N:T16M", "N:T16K", "N:T16R", "N:F17Y", "N:F17C", "N:F17L", "N:F17L", "N:G18C", "N:G18R", "N:G18V", "N:G18A", "N:G18D", "N:G19V", "N:G19A", "N:G19E", "N:P20T", "N:P20A", "N:P20L", "N:P20H", "N:P20R", "N:S21L", "N:S21*", "N:S21*", "N:D22Y", "N:D22H", "N:D22V", "N:D22A", "N:D22G", "N:D22E", "N:D22E", "N:S23A", "N:S23L", "N:S23*", "N:S23*", "N:T24S", "N:T24P", "N:T24N", "N:T24S", "N:G25V", "N:G25A", "N:G25D", "N:S26I", "N:S26T", "N:S26N", "N:S26R", "N:S26R", "N:N27I", "N:N27T", "N:N27S", "N:N27K", "N:N27K", "N:Q28L", "N:Q28R", "N:Q28H", "N:Q28H", "N:N29Y", "N:N29H", "N:N29I", "N:N29T", "N:N29S", "N:N29K", "N:N29K", "N:G30*", "N:G30R", "N:G30R", "N:G30V", "N:G30A", "N:G30E", "N:E31*", "N:E31Q", "N:E31V", "N:E31A", "N:E31G", "N:E31D", "N:E31D", "N:R32S", "N:R32G", "N:R32L", "N:R32P", "N:R32H", "N:S33I", "N:S33T", "N:S33N", "N:S33R", "N:S33R", "N:G34V", "N:G34A", "N:G34E", "N:A35V", "N:A35E", "N:A35G", "N:R36L", "N:R36P", "N:R36Q", "N:S37T", "N:S37A", "N:S37L", "N:S37*", "N:S37*", "N:K38*", "N:K38Q", "N:K38I", "N:K38T", "N:K38R", "N:K38N", "N:K38N", "N:Q39K", "N:Q39E", "N:Q39L", "N:Q39P", "N:Q39R", "N:Q39H", "N:Q39H", "N:R40S", "N:R40G", "N:R40L", "N:R40P", "N:R40H", "N:R41L", "N:R41P", "N:R41Q", "N:P42L", "N:P42H", "N:P42R", "N:Q43L", "N:Q43P", "N:Q43R", "N:Q43H", "N:Q43H", "N:G44C", "N:G44R", "N:G44V", "N:G44A", "N:G44D", "N:L45S", "N:L45*", "N:L45*", "N:L45F", "N:L45F", "N:P46T", "N:P46A", "N:P46L", "N:P46H", "N:P46R", "N:N47I", "N:N47T", "N:N47S", "N:N47K", "N:N47K", "N:N48D", "N:N48I", "N:N48T", "N:N48S", "N:N48K", "N:N48K", "N:T49A", "N:T49N", "N:T49S", "N:A50V", "N:A50E", "N:A50G", "N:S51F", "N:S51Y", "N:S51C", "N:W52L", "N:W52S", "N:W52*", "N:W52C", "N:W52C", "N:W52*", "N:F53S", "N:F53Y", "N:F53C", "N:F53L", "N:F53L", "N:T54I", "N:T54N", "N:T54S", "N:A55V", "N:A55D", "N:A55G", "N:L56P", "N:L56H", "N:L56R", "N:T57I", "N:T57N", "N:T57S", "N:Q58L", "N:Q58P", "N:Q58R", "N:Q58H", "N:Q58H", "N:H59N", "N:H59D", "N:H59L", "N:H59P", "N:H59R", "N:H59Q", "N:H59Q", "N:G60C", "N:G60R", "N:G60S", "N:G60V", "N:G60A", "N:G60D", "N:K61M", "N:K61R", "N:K61N", "N:K61N", "N:E62*", "N:E62Q", "N:E62V", "N:E62A", "N:E62G", "N:E62D", "N:E62D", "N:D63Y", "N:D63H", "N:D63V", "N:D63A", "N:D63G", "N:D63E", "N:D63E", "N:G63D", "N:L64H", "N:L64R", "N:K65*", "N:K65Q", "N:K65I", "N:K65T", "N:K65R", "N:K65N", "N:K65N", "N:F66I", "N:F66V", "N:F66S", "N:F66Y", "N:F66C", "N:F66L", "N:F66L", "N:P67L", "N:P67H", "N:P67R", "N:R68L", "N:R68P", "N:R68Q", "N:G69*", "N:G69R", "N:G69V", "N:G69A", "N:G69E", "N:Q70K", "N:Q70E", "N:Q70L", "N:Q70P", "N:Q70R", "N:Q70H", "N:Q70H", "N:G71C", "N:G71R", "N:G71V", "N:G71A", "N:G71D", "N:V72A", "N:V72D", "N:V72G", "N:P73T", "N:P73A", "N:P73L", "N:P73Q", "N:P73R", "N:I74F", "N:I74L", "N:I74N", "N:I74S", "N:I74M", "N:N75Y", "N:N75H", "N:N75I", "N:N75T", "N:N75S", "N:N75K", "N:N75K", "N:T76I", "N:T76N", "N:T76S", "N:N77I", "N:N77T", "N:N77S", "N:N77K", "N:N77K", "N:S78G", "N:S78I", "N:S78T", "N:S78N", "N:S78R", "N:S78R", "N:S79I", "N:S79T", "N:S79N", "N:S79R", "N:S79R", "N:P80L", "N:P80Q", "N:P80R", "N:D81Y", "N:D81H", "N:D81V", "N:D81A", "N:D81G", "N:D81E", "N:D81E", "N:D82Y", "N:D82H", "N:D82N", "N:D82V", "N:D82A", "N:D82G", "N:D82E", "N:D82E", "N:Q83L", "N:Q83P", "N:Q83R", "N:Q83H", "N:Q83H", "N:I84F", "N:I84L", "N:I84N", "N:I84S", "N:I84M", "N:G85C", "N:G85R", "N:G85V", "N:G85A", "N:G85D", "N:Y86F", "N:Y86S", "N:Y86C", "N:Y86*", "N:Y86*", "N:Y87F", "N:Y87S", "N:Y87C", "N:Y87*", "N:Y87*", "N:R88L", "N:R88P", "N:R88Q", "N:R89*", "N:R89I", "N:R89T", "N:R89K", "N:R89S", "N:R89S", "N:A90S", "N:A90P", "N:A90D", "N:A90G", "N:T91I", "N:T91N", "N:T91S", "N:R92I", "N:R92T", "N:R92K", "N:R92S", "N:R92S", "N:R93G", "N:R93L", "N:R93P", "N:R93Q", "N:I94F", "N:I94L", "N:I94T", "N:I94N", "N:I94S", "N:I94M", "N:R95S", "N:R95G", "N:R95L", "N:R95P", "N:R95H", "N:G96V", "N:G96A", "N:G96D", "N:G97V", "N:G97A", "N:G97D", "N:D98V", "N:D98A", "N:D98G", "N:D98E", "N:D98E", "N:G99V", "N:G99A", "N:G99D", "N:K100I", "N:K100T", "N:K100R", "N:K100N", "N:K100N", "N:M101L", "N:M101L", "N:M101T", "N:M101K", "N:M101R", "N:M101I", "N:M101I", "N:K102*", "N:K102Q", "N:K102E"])            
# global  Double_N_ORF9b_muts = Set(["ORF9b:M1L", "ORF9b:M1L", "ORF9b:M1V", "ORF9b:M1K", "ORF9b:M1R", "ORF9b:M1I", "ORF9b:M1I", "ORF9b:M1I", "ORF9b:D2Y", "ORF9b:D2H", "ORF9b:D2N", "ORF9b:D2E", "ORF9b:D2E", "ORF9b:P3S", "ORF9b:P3T", "ORF9b:P3A", "ORF9b:K4*", "ORF9b:K4Q", "ORF9b:K4E", "ORF9b:K4I", "ORF9b:K4T", "ORF9b:K4N", "ORF9b:K4N", "ORF9b:I5F", "ORF9b:I5L", "ORF9b:I5V", "ORF9b:I5N", "ORF9b:I5S", "ORF9b:I5M", "ORF9b:S6C", "ORF9b:S6R", "ORF9b:S6G", "ORF9b:S6I", "ORF9b:S6T", "ORF9b:S6R", "ORF9b:E7*", "ORF9b:E7Q", "ORF9b:E7K", "ORF9b:E7D", "ORF9b:E7D", "ORF9b:M8L", "ORF9b:M8L", "ORF9b:M8V", "ORF9b:M8K", "ORF9b:M8R", "ORF9b:M8I", "ORF9b:M8I", "ORF9b:M8I", "ORF9b:H9Y", "ORF9b:H9N", "ORF9b:H9D", "ORF9b:H9Q", "ORF9b:P10S", "ORF9b:P10T", "ORF9b:P10A", "ORF9b:A11S", "ORF9b:A11P", "ORF9b:A11T", "ORF9b:L12I", "ORF9b:L12V", "ORF9b:L12*", "ORF9b:L12F", "ORF9b:L12F", "ORF9b:R13C", "ORF9b:R13S", "ORF9b:R13G", "ORF9b:L14M", "ORF9b:L14V", "ORF9b:L14*", "ORF9b:L14W", "ORF9b:L14F", "ORF9b:L14F", "ORF9b:V15L", "ORF9b:V15L", "ORF9b:V15M", "ORF9b:D16Y", "ORF9b:D16H", "ORF9b:D16N", "ORF9b:D16E", "ORF9b:D16E", "ORF9b:P17S", "ORF9b:P17T", "ORF9b:P17A", "ORF9b:Q18*", "ORF9b:Q18K", "ORF9b:Q18E", "ORF9b:Q18H", "ORF9b:Q18H", "ORF9b:I19F", "ORF9b:I19L", "ORF9b:I19V", "ORF9b:I19N", "ORF9b:I19S", "ORF9b:I19M", "ORF9b:Q20*", "ORF9b:Q20K", "ORF9b:Q20E", "ORF9b:Q20H", "ORF9b:Q20H", "ORF9b:L21M", "ORF9b:L21V", "ORF9b:A22S", "ORF9b:A22P", "ORF9b:A22T", "ORF9b:V23L", "ORF9b:V23L", "ORF9b:V23I", "ORF9b:V23E", "ORF9b:V23G", "ORF9b:T24S", "ORF9b:T24P", "ORF9b:T24A", "ORF9b:T24N", "ORF9b:T24S", "ORF9b:R25*", "ORF9b:R25G", "ORF9b:R25I", "ORF9b:R25T", "ORF9b:R25S", "ORF9b:R25S", "ORF9b:M26L", "ORF9b:M26L", "ORF9b:M26V", "ORF9b:M26K", "ORF9b:M26R", "ORF9b:M26I", "ORF9b:M26I", "ORF9b:M26I", "ORF9b:E27*", "ORF9b:E27Q", "ORF9b:E27K", "ORF9b:E27D", "ORF9b:E27D", "ORF9b:N28Y", "ORF9b:N28H", "ORF9b:N28D", "ORF9b:N28I", "ORF9b:N28T", "ORF9b:N28K", "ORF9b:N28K", "ORF9b:A29S", "ORF9b:A29P", "ORF9b:A29T", "ORF9b:V30L", "ORF9b:V30L", "ORF9b:V30M", "ORF9b:V30E", "ORF9b:V30G", "ORF9b:G31W", "ORF9b:G31R", "ORF9b:G31R", "ORF9b:R32C", "ORF9b:R32S", "ORF9b:R32G", "ORF9b:D33Y", "ORF9b:D33H", "ORF9b:D33N", "ORF9b:D33E", "ORF9b:D33E", "ORF9b:Q34*", "ORF9b:Q34K", "ORF9b:Q34E", "ORF9b:Q34H", "ORF9b:Q34H", "ORF9b:N35Y", "ORF9b:N35H", "ORF9b:N35D", "ORF9b:N35I", "ORF9b:N35T", "ORF9b:N35K", "ORF9b:N35K", "ORF9b:N36Y", "ORF9b:N36H", "ORF9b:N36D", "ORF9b:N36I", "ORF9b:N36T", "ORF9b:N36K", "ORF9b:N36K", "ORF9b:V37F", "ORF9b:V37L", "ORF9b:V37I", "ORF9b:G38C", "ORF9b:G38R", "ORF9b:G38S", "ORF9b:P39S", "ORF9b:P39T", "ORF9b:P39A", "ORF9b:K40*", "ORF9b:K40Q", "ORF9b:K40E", "ORF9b:K40M", "ORF9b:K40T", "ORF9b:K40N", "ORF9b:K40N", "ORF9b:V41F", "ORF9b:V41L", "ORF9b:V41I", "ORF9b:Y42H", "ORF9b:Y42N", "ORF9b:Y42D", "ORF9b:Y42F", "ORF9b:Y42S", "ORF9b:Y42*", "ORF9b:Y42*", "ORF9b:P43S", "ORF9b:P43T", "ORF9b:P43A", "ORF9b:I44L", "ORF9b:I44L", "ORF9b:I44V", "ORF9b:I44K", "ORF9b:I44R", "ORF9b:I44M", "ORF9b:I45L", "ORF9b:I45L", "ORF9b:I45V", "ORF9b:I45K", "ORF9b:I45R", "ORF9b:I45M", "ORF9b:L46M", "ORF9b:L46V", "ORF9b:R47C", "ORF9b:R47S", "ORF9b:R47G", "ORF9b:L48F", "ORF9b:L48I", "ORF9b:L48V", "ORF9b:G49C", "ORF9b:G49R", "ORF9b:G49S", "ORF9b:G49V", "ORF9b:G49A", "ORF9b:G49D", "ORF9b:S50P", "ORF9b:S50T", "ORF9b:S50A", "ORF9b:S50*", "ORF9b:S50*", "ORF9b:P51S", "ORF9b:P51T", "ORF9b:P51A", "ORF9b:L52F", "ORF9b:L52I", "ORF9b:L52V", "ORF9b:S53P", "ORF9b:S53T", "ORF9b:S53A", "ORF9b:L54F", "ORF9b:L54I", "ORF9b:L54V", "ORF9b:N55Y", "ORF9b:N55H", "ORF9b:N55D", "ORF9b:N55I", "ORF9b:N55T", "ORF9b:N55K", "ORF9b:N55K", "ORF9b:M56L", "ORF9b:M56L", "ORF9b:M56V", "ORF9b:M56K", "ORF9b:M56R", "ORF9b:M56I", "ORF9b:M56I", "ORF9b:M56I", "ORF9b:A57S", "ORF9b:A57P", "ORF9b:A57T", "ORF9b:R58W", "ORF9b:R58G", "ORF9b:R58M", "ORF9b:R58T", "ORF9b:R58S", "ORF9b:R58S", "ORF9b:K59*", "ORF9b:K59Q", "ORF9b:K59E", "ORF9b:K59M", "ORF9b:K59T", "ORF9b:K59N", "ORF9b:K59N", "ORF9b:T60S", "ORF9b:T60P", "ORF9b:T60A", "ORF9b:T60N", "ORF9b:T60S", "ORF9b:L61I", "ORF9b:L61V", "ORF9b:L61F", "ORF9b:L61F", "ORF9b:N62Y", "ORF9b:N62H", "ORF9b:N62D", "ORF9b:N62I", "ORF9b:N62T", "ORF9b:N62K", "ORF9b:N62K", "ORF9b:S63P", "ORF9b:S63T", "ORF9b:S63A", "ORF9b:S63Y", "ORF9b:S63C", "ORF9b:L64F", "ORF9b:L64I", "ORF9b:L64V", "ORF9b:E65*", "ORF9b:E65Q", "ORF9b:E65K", "ORF9b:E65D", "ORF9b:E65D", "ORF9b:D66Y", "ORF9b:D66H", "ORF9b:D66N", "ORF9b:D66E", "ORF9b:D66E", "ORF9b:K67*", "ORF9b:K67Q", "ORF9b:K67E", "ORF9b:K67M", "ORF9b:K67T", "ORF9b:K67N", "ORF9b:K67N", "ORF9b:A68S", "ORF9b:A68P", "ORF9b:A68T", "ORF9b:F69L", "ORF9b:F69I", "ORF9b:F69V", "ORF9b:F69L", "ORF9b:F69L", "ORF9b:Q70*", "ORF9b:Q70K", "ORF9b:Q70E", "ORF9b:Q70H", "ORF9b:Q70H", "ORF9b:L71I", "ORF9b:L71V", "ORF9b:L71*", "ORF9b:L71F", "ORF9b:L71F", "ORF9b:T72S", "ORF9b:T72P", "ORF9b:T72A", "ORF9b:T72K", "ORF9b:T72R", "ORF9b:P73S", "ORF9b:P73T", "ORF9b:P73A", "ORF9b:I74L", "ORF9b:I74L", "ORF9b:I74V", "ORF9b:I74K", "ORF9b:I74R", "ORF9b:I74M", "ORF9b:A75S", "ORF9b:A75P", "ORF9b:A75T", "ORF9b:A75E", "ORF9b:A75G", "ORF9b:V76F", "ORF9b:V76L", "ORF9b:V76I", "ORF9b:V76D", "ORF9b:V76G", "ORF9b:Q77*", "ORF9b:Q77K", "ORF9b:Q77E", "ORF9b:Q77H", "ORF9b:Q77H", "ORF9b:M78L", "ORF9b:M78L", "ORF9b:M78V", "ORF9b:M78K", "ORF9b:M78R", "ORF9b:M78I", "ORF9b:M78I", "ORF9b:M78I", "ORF9b:T79S", "ORF9b:T79P", "ORF9b:T79A", "ORF9b:T79N", "ORF9b:T79S", "ORF9b:K80*", "ORF9b:K80Q", "ORF9b:K80E", "ORF9b:K80I", "ORF9b:K80T", "ORF9b:K80N", "ORF9b:K80N", "ORF9b:L81M", "ORF9b:L81V", "ORF9b:L81W", "ORF9b:L81F", "ORF9b:L81F", "ORF9b:A82S", "ORF9b:A82P", "ORF9b:A82T", "ORF9b:T83S", "ORF9b:T83P", "ORF9b:T83A", "ORF9b:T83N", "ORF9b:T83S", "ORF9b:T84S", "ORF9b:T84P", "ORF9b:T84A", "ORF9b:T84N", "ORF9b:T84S", "ORF9b:E85*", "ORF9b:E85Q", "ORF9b:E85K", "ORF9b:E85D", "ORF9b:E86*", "ORF9b:E86Q", "ORF9b:E86K", "ORF9b:E86V", "ORF9b:E86A", "ORF9b:E86D", "ORF9b:E86D", "ORF9b:L87I", "ORF9b:L87V", "ORF9b:P88S", "ORF9b:P88T", "ORF9b:P88A", "ORF9b:D89Y", "ORF9b:D89H", "ORF9b:D89N", "ORF9b:D89V", "ORF9b:D89A", "ORF9b:D89E", "ORF9b:E90*", "ORF9b:E90Q", "ORF9b:E90K", "ORF9b:E90D", "ORF9b:E90D", "ORF9b:F91L", "ORF9b:F91I", "ORF9b:F91V", "ORF9b:F91C", "ORF9b:F91L", "ORF9b:F91L", "ORF9b:V92L", "ORF9b:V92L", "ORF9b:V92M", "ORF9b:V93L", "ORF9b:V93L", "ORF9b:V93M", "ORF9b:V94L", "ORF9b:V94L", "ORF9b:V94M", "ORF9b:T95S", "ORF9b:T95P", "ORF9b:T95A", "ORF9b:T95K", "ORF9b:T95R", "ORF9b:V96L", "ORF9b:V96L", "ORF9b:V96I", "ORF9b:K97*", "ORF9b:K97Q", "ORF9b:K97E", "ORF9b:K97I", "ORF9b:K97T", "ORF9b:K97N", "ORF9b:K97N", "ORF9b:*98R", "ORF9b:*98R", "ORF9b:*98G", "ORF9b:*98L", "ORF9b:*98S", "ORF9b:*98C", "ORF9b:*98C", "ORF9b:*98W"])
######################################################################################################################################
#### BAL_muts are all the mutations found that are part of the bronchoalveolar lavage mutation pattern (determined by a separate
####    function described in the paper and which is available for viewing on Github).
#### Like RBM_muts and artifactual_private_muts, the original mutation patterns are entirely unaffected by these mutations. However,
####    it is necessary to exclude them from serving **as seed mutations** in the control function, which tests all other mutations 
####    that occur more than 5 times in the EPCI dataset. Otherwise, a large number of BAL-associated mut patterns are reproduced. 
####    A few will sneak by anyway.
global BAL_muts_OG = list_to_set("ORF1a:T2183I, ORF1a:S2972F, ORF1a:S2972P, ORF1a:A3049V, ORF1a:T3058I, ORF1a:A3070V, ORF1a:G3072C, ORF1a:H3076Y, ORF1a:L3123F, ORF1a:S3195G, ORF1a:P3359L, ORF1a:A3454V, ORF1a:A3456V, ORF1a:Q4110R, ORF1a:T4175I, ORF1a:P4197S, ORF1a:I4205V, ORF1a:T4207A, ORF1a:R4387S, ORF1b:L314P, ORF1b:I1181T, ORF1b:Y1247C, ORF1b:T1424I, ORF1b:S2339F, ORF1b:K2340N, ORF1b:T2537I, S:S50L, S:P330S, S:N354D, S:V367F, S:F374L, S:F375-, S:A376V, S:N405D, S:Y508H, S:V551I, S:T573I, S:A647S, S:A653V, S:N657K, S:S659P, S:A668V, S:T859I, S:A944T, S:A958D, S:N978D, S:I1169T, S:I1179T, S:L1186F, E:V5A, E:V5F, E:T9I, E:G10S, E:G10C, E:I13-, E:V14-, E:S16N, E:F23S, E:T30I, E:A36V, E:Y42C, M:A2V, M:S4P, M:V10I, M:F28S, M:T77N, M:S94R, M:S94N, M:H125Y, M:H148R, M:A188T, N:P80L, N:S416L, N:T417I, ORF7a:T115I, ORF9b:M1T, ORF9b:Q77*") 
artifactual_private_muts_pos_only = Set{String}()
#push!(artifactual_private_muts_pos_only, "ORF1a:2501) ## Artifactual reversion in B.1 sequences (possibly real? Unclear)
push!(artifactual_private_muts_pos_only, "ORF1a:2606") ## Inherited B.1.2 mutation
push!(artifactual_private_muts_pos_only, "N:377")      ## Inherited B.1.2 mutation
push!(artifactual_private_muts_pos_only, "ORF3a:72")   ## Inherited B.1.25 8 mutation
push!(artifactual_private_muts_pos_only, "ORF1a:3764") ## Inherited B.1.258 mutation
push!(artifactual_private_muts_pos_only, "ORF1a:2283") ## Artifactual reversion in XEC miscategorized as KP.3 or KP.3.3 by Nextclade
push!(artifactual_private_muts_pos_only, "ORF1a:599")  ## Artifactual "private" mutation in XEC miscategorized as KP.3 or KP.3.3 by Nextclade
push!(artifactual_private_muts_pos_only, "S:59")       ## Artifactual "private" mutation in XEC miscategorized as KP.3 or KP.3.3 by Nextclade
push!(artifactual_private_muts_pos_only, "ORF8:L60")   ## Artifactual "private" mutation in AY.44. Over 125000 seqs have both ORF8:L60F & ORF1a:2125Y
push!(artifactual_private_muts_pos_only, "ORF1a:2125") ## Artifactual "private" mutation in AY.44. Over 125000 seqs have both ORF8:L60F & ORF1a:2125Y
push!(artifactual_private_muts_pos_only, "S:146")      ## Extremely homoplasic XBB mutation and/or artifact. Impossible to tell if inherited or private. 
#push!(artifactual_private_muts_pos_only, "N:203")      ## Recombinant Delta/Omicron "mutation"
#push!(artifactual_private_muts_pos_only, "N:204")      ## Recombinant Delta/Omicron "mutation"
#push!(artifactual_private_muts_pos_only, "N:204")      ## Recombinant Delta/Omicron reversion "mutation"
push!(artifactual_private_muts_pos_only, "ORF1b:1156") ## BQ.1 mutation misattributed as private in 3 sequences
push!(artifactual_private_muts_pos_only, "ORF:61")     ## Artifactual 3-nuc reversion
push!(artifactual_private_muts_pos_only, "ORF1a:135")  ## Omicron mut misattributed as private in 3 recombinants
#push!(artifactual_private_muts_pos_only, "M:30")       ## Very common artifactual reversion in BA.2.86* lineages
push!(artifactual_private_muts_pos_only, "ORF1a:1612") ## Inherited Beta mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF7b:39")   ## Inherited Beta mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "S:18")       ## All five on list in Beta, likely artifactual
push!(artifactual_private_muts_pos_only, "ORF1a:2554") ## Inherited AY.44 mutation miscategorized by Nextclade as private 
push!(artifactual_private_muts_pos_only, "ORF1b:1087") ## Inherited AY.44 mutation miscategorized by Nextclade as private 
push!(artifactual_private_muts_pos_only, "ORF1a:2796") ## Inherited BA.1.17 mutation miscategorized by Nextclade as private 
push!(artifactual_private_muts_pos_only, "ORF1a:1803") ## Inherited BA.1.17 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF3a:104")  ## Inherited AY.103 mutation miscategorized by Nextclade as private 
push!(artifactual_private_muts_pos_only, "N:208")      ## Inherited AY.103 mutation miscategorized by Nextclade as private 
push!(artifactual_private_muts_pos_only, "ORF8:68")    ## Inherited B.1.1.7 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1b:1975") ## Inherited B.1.1.7 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1b:2178") ## Inherited BA.1.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1b:97")   ## Inherited BA.1.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1b:2589") ## Inherited BA.5.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF9b:83")   ## Inherited BA.5.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1a:842")  ## From 5' End Recombination with BA.2
push!(artifactual_private_muts_pos_only, "ORF1a:135")  ## From 5' End Recombination with BA.2
push!(artifactual_private_muts_pos_only, "ORF3a:48")   ## Inherited BE.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF3a:49")   ## Inherited BE.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1a:1204") ## Inherited BE.1 mutation miscategorized by Nextclade as private
#push!(artifactual_private_muts_pos_only, "S:376")      ## Butchered mutation interpretation by Nextclade for ∆374-375 or ∆374-376
#push!(artifactual_private_muts_pos_only, "S:376")      ## Butchered mutation interpretation by Nextclade for ∆374-375 or ∆374-376
#push!(artifactual_private_muts_pos_only, "S:375")      ## Butchered mutation interpretation by Nextclade for ∆374-375 or ∆374-376
#push!(artifactual_private_muts_pos_only, "S:375")      ## Butchered mutation interpretation by Nextclade for ∆374-375 or ∆374-376
#push!(artifactual_private_muts_pos_only, "S:339")      ## An artifactual reversion >90% of the time, likely 100%
#push!(artifactual_private_muts_pos_only, "S:417")      ## An artifactual reversion >90% of the time, likely 100%
#push!(artifactual_private_muts_pos_only, "S:440")      ## An artifactual reversion >90% of the time, likely 100%
push!(artifactual_private_muts_pos_only, "ORF1b:1555") ## Inherited B.1.2 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1a:4285") ## Inherited B.1.2 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1a:2361") ## Inherited B.1.2 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1a:3782") ## Inherited BA.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "S:1191")     ## Inherited B.1.1.7 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF7b:44")   ## Deletions here that maintain the stop codon get misinterpreted as substitutions
push!(artifactual_private_muts_pos_only, "ORF1a:3646") ## Inherited Delta mut, miscategorized in recombinants 
push!(artifactual_private_muts_pos_only, "ORF1b:1918") ## Inherited Delta mut, miscategorized in recombinants 
push!(artifactual_private_muts_pos_only, "ORF7b:1")    ## Necessarily overlaps with ORF7a:122 mutations/deletions
push!(artifactual_private_muts_pos_only, "ORF7b:2")    ## Overlaps with ORF7a:122 mutations/deletions
push!(artifactual_private_muts_pos_only, "ORF1b:218")  ## Inherited B.1.1.7 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1a:3782") ## Inherited mutation in Canadian BA.1 branch miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1a:2118") ## B.1 Inherited mutation miscategorized by Nextclade as private (co-occurs with S:P681H)
push!(artifactual_private_muts_pos_only, "ORF1a:3646") ## Inherited Delta mut, miscategorized in recombinants 
push!(artifactual_private_muts_pos_only, "ORF1b:1918") ## Inherited Delta mut, miscategorized in recombinants
push!(artifactual_private_muts_pos_only, "ORF1a:1298") ## Inherited BA.1 mut, miscategorized by Nextclade as private

push!(artifactual_private_muts_pos_only, "ORF3a:112")  ## Inherited B.1.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1b:2695") ## Inherited B.1.1 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF3a:35")   ## Inherited XBB.1.16.11 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1b:1903") ## Inherited XBB.1.16.31 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1b:1094") ## Inherited B.1.2 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1a:771")  ## Inherited XBB.1.5 mutation miscategorized by Nextclade as private 

push!(artifactual_private_muts_pos_only, "ORF1a:108")  ## Inherited BA.5.1 mutation in Scandinavian branch miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1a:365")  ## Inherited B.1.2 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1a:397")  ## Inherited B.1.466 (Indonesia) mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1a:925")  ## Inherited BN.1.3.1 mutation (Italian branch) miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1a:1015") ## Inherited KF.1 (FL.15.1.1.1) mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1a:3459") ## Inherited B.1.429 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1b:128")  ## Inherited BA.5.1 mutation in Scandinavian branch miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1b:869")  ## Inherited KF.1 (FL.15.1.1.1) mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1b:1094") ## Inherited B.1.2 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1b:1871") ## Inherited B.1.1.7 mutation miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF1b:2452") ## Inherited BA.5.1 mutation in Scandinavian branch miscategorized by Nextclade as private
push!(artifactual_private_muts_pos_only, "ORF7b:4")    ## Inherited BA.1 mutation on Swedish branch miscategorized by Nextclade as private
######################################################################################################################################
#### Double_N_ORF9b_muts are muts that result in both N and ORF9b mutations, which of course correlate perfectly but trivially.
####    Default is to block N muts, but it's also possible to change that and instead block ORF9b muts to view their N equivalents
####    by commenting out the top list and uncommenting out the bottom list.
global Double_N_ORF9b_muts_pos_only = Set(["N:4", "N:5", "N:6", "N:7", "N:8", "N:9", "N:10", "N:11", "N:12", "N:13", "N:14", "N:15", "N:16", "N:17", "N:18", "N:19", "N:20", "N:21", "N:22", "N:23", "N:24", "N:25", "N:26", "N:27", "N:28", "N:29", "N:30", "N:31", "N:32", "N:33", "N:34", "N:35", "N:36", "N:37", "N:38", "N:39", "N:40", "N:41", "N:42", "N:43", "N:44", "N:45", "N:46", "N:47", "N:48", "N:49", "N:50", "N:51", "N:52", "N:53", "N:54", "N:55", "N:56", "N:57", "N:58", "N:59", "N:60", "N:61", "N:62", "N:63", "N:64", "N:65", "N:66", "N:67", "N:68", "N:69", "N:70", "N:71", "N:72", "N:73", "N:74", "N:75", "N:76", "N:77", "N:78", "N:79", "N:80", "N:81", "N:82", "N:83", "N:84", "N:85", "N:86", "N:87", "N:88", "N:89", "N:90", "N:91", "N:92", "N:93", "N:94", "N:95", "N:96", "N:97", "N:98", "N:99", "N:100", "N:101"])
# global  Double_N_ORF9b_muts = Set(["ORF9b:1", "ORF9b:2", "ORF9b:3", "ORF9b:4", "ORF9b:5", "ORF9b:6", "ORF9b:7", "ORF9b:8", "ORF9b:9", "ORF9b:10", "ORF9b:11", "ORF9b:12", "ORF9b:13", "ORF9b:14", "ORF9b:15", "ORF9b:16", "ORF9b:17", "ORF9b:18", "ORF9b:19", "ORF9b:20", "ORF9b:21", "ORF9b:22", "ORF9b:23", "ORF9b:24", "ORF9b:25", "ORF9b:26", "ORF9b:27", "ORF9b:28", "ORF9b:29", "ORF9b:30", "ORF9b:31", "ORF9b:32", "ORF9b:33", "ORF9b:34", "ORF9b:35", "ORF9b:36", "ORF9b:37", "ORF9b:38", "ORF9b:39", "ORF9b:40", "ORF9b:41", "ORF9b:42", "ORF9b:43", "ORF9b:44", "ORF9b:45", "ORF9b:46", "ORF9b:47", "ORF9b:48", "ORF9b:49", "ORF9b:50", "ORF9b:51", "ORF9b:52", "ORF9b:53", "ORF9b:54", "ORF9b:55", "ORF9b:56", "ORF9b:57", "ORF9b:58", "ORF9b:59", "ORF9b:60", "ORF9b:61", "ORF9b:62", "ORF9b:63", "ORF9b:64", "ORF9b:65", "ORF9b:66", "ORF9b:67", "ORF9b:68", "ORF9b:69", "ORF9b:70", "ORF9b:71", "ORF9b:72", "ORF9b:73", "ORF9b:74", "ORF9b:75", "ORF9b:76", "ORF9b:77", "ORF9b:78", "ORF9b:79", "ORF9b:80", "ORF9b:81", "ORF9b:82", "ORF9b:83", "ORF9b:84", "ORF9b:85", "ORF9b:86", "ORF9b:87", "ORF9b:88", "ORF9b:89", "ORF9b:90", "ORF9b:91", "ORF9b:92", "ORF9b:93", "ORF9b:94", "ORF9b:95", "ORF9b:96", "ORF9b:97", "ORF9b:98"])
######################################################################################################################################
#### BAL_muts are all the mutations found that are part of the bronchoalveolar lavage mutation pattern (determined by a separate
####    function described in the paper and which is available for viewing on Github).
#### Like RBM_muts and artifactual_private_muts, the original mutation patterns are entirely unaffected by these mutations. However,
####    it is necessary to exclude them from serving **as seed mutations** in the control function, which tests all other mutations 
####    that occur more than 5 times in the EPCI dataset. Otherwise, a large number of BAL-associated mut patterns are reproduced. 
####    A few will sneak by anyway.
global BAL_muts_pos_only = list_to_set("ORF1a:2183, ORF1a:2972, ORF1a:2972, ORF1a:3049, ORF1a:3058, ORF1a:3070, ORF1a:3072, ORF1a:3076, ORF1a:3123, ORF1a:3195, ORF1a:3359, ORF1a:3454, ORF1a:3456, ORF1a:4110, ORF1a:4175, ORF1a:4197, ORF1a:4205, ORF1a:4207, ORF1a:4387, ORF1a:314, ORF1b:1181, ORF1b:1247, ORF1b:1424, ORF1b:2339, ORF1b:2340, ORF1b:2537, S:50, S:330, S:354, S:367, S:374, S:375-, S:376, S:405, S:508, S:551, S:573, S:647, S:653, S:657, S:659, S:668, S:859, S:944, S:958, S:978, S:1169, S:1179, S:1186, E:5, E:5, E:9, E:10, E:10, E:13-, E:14-, E:16, E:23, E:30, E:36, E:42, M:2, M:4, M:10, M:28, M:77, M:94, M:94, M:125, M:148, M:188, N:80, N:416, N:417, ORF7a:115, ORF9b:1, ORF9b:77")
###########################################################################################################################################################################
#####################################################   BEGIN Sub/pos_only Section   ######################################################################################
###########################################################################################################################################################################
artifactual_private_muts = Set{String}()
pango_AAsub_WT_universal = Dict{String, Set{String}}
global mp_folder_universal = ""
mp_chr_all_ratio = Dict()
if sub_0__posonly_1 == 0
    pango_AAsub_WT_universal = pango_AAsub_WT
    artifactual_private_muts = artifactual_private_muts_subs
    Double_N_ORF9b_muts = Double_N_ORF9b_muts
    BAL_muts = BAL_muts_OG
    mp_chr_all_ratio = AA_muts_ct_chr_all_ratio
elseif sub_0__posonly_1 == 1
    pango_AAsub_WT_universal = pango_AAsub_WT_pos_only
    artifactual_private_muts = artifactual_private_muts_pos_only
    Double_N_ORF9b_muts = Double_N_ORF9b_muts_pos_only
    BAL_muts = BAL_muts_pos_only
    mp_chr_all_ratio = AA_muts_ct_pos_only_no_dels_chr_all_ratio
end
for pango in pango_set
    mutset = pango_AAsub_WT_universal[pango]
    if "" in mutset
        println(pango)
    end
end
Bset = 
delete!(pango_AAsub_WT_universal["B"], "")
#################################################################################################
function sel_muts_pt1_pos_only_sort_key(n)
    if n == "" || isempty(n)
        return (0, 0)  # Return a default sort key for empty strings
    else
        return mp_AA_gene_pos_only_sortKey_2(string(split(n, ", ")[1]))
    end
end
############################################################################################
function sel_muts_pt1_sort_key_universal(n::String, sub_0__posonly_1::Int)
    if sub_0__posonly_1 == 0
        return sel_muts_pt1_sort_key(n)
    elseif sub_0__posonly_1 == 1
        return sel_muts_pt1_pos_only_sort_key(n)
    end
end
############################################################
function mp_AA_gene_sortKey_2_universal(n::String, sub_0__posonly_1::Int)
    if sub_0__posonly_1 == 0
        return mp_AA_gene_sortKey_2(n)
    elseif sub_0__posonly_1 == 1
        return mp_AA_gene_pos_only_sortKey_2(n)
    end
end    
############################################################
function AA_muts_ct_no_dels__sub__or__pos_only(sub_0__posonly_1::Int)
    if sub_0__posonly_1 == 0
        return AA_muts_ct_no_dels
    end
    if sub_0__posonly_1 == 1
        return AA_muts_ct_pos_only_no_dels
    end
end
AA_muts_ct_no_dels_universal = AA_muts_ct_no_dels__sub__or__pos_only(sub_0__posonly_1)
if AA_muts_ct_no_dels_universal == AA_muts_ct_no_dels
    println("AA_muts_ct_no_dels_universal == AA_muts_ct_no_dels")
else
    println("AA_muts_ct_no_dels_universal ≠ AA_muts_ct_no_dels")
end
if AA_muts_ct_no_dels_universal == AA_muts_ct_pos_only_no_dels
    println("AA_muts_ct_no_dels_universal == AA_muts_ct_pos_only_no_dels")
else
    println("AA_muts_ct_no_dels_universal ≠ AA_muts_ct_pos_only_no_dels")
end
############################################################
function seq_AA_muts_no_dels__sub__or__pos_only(sub_0__posonly_1::Int)
    if sub_0__posonly_1 == 0
        return seq_AA_muts_no_dels
    elseif sub_0__posonly_1 == 1
        return seq_AA_muts_pos_only_no_dels
    end
end
seq_AA_muts_no_dels_universal = seq_AA_muts_no_dels__sub__or__pos_only(sub_0__posonly_1)
####################################################################################
artifacts_ORF9bdoubles = union(artifactual_private_muts, Double_N_ORF9b_muts)
###########################################################################################################################################################################
###########################################################################################################################################################################
global RBM_muts = Set{String}()
global RBD_muts = Set{String}()
global RBD_not_RBM_muts = Set{String}()
global spike_not_RBD_muts = Set{String}()
global spike_muts = Set{String}()
global nonspike_muts = Set{String}()
RBD_sites = BitSet(335:528)
RBD_no_RBM_1 = BitSet(335:437)
RBD_no_RBM_2 = BitSet(507:528)
RBD_not_RBM_sites = union(RBD_no_RBM_1, RBD_no_RBM_2)
RBM_sites = BitSet(438:506)
spike_not_RBD1 = BitSet(1:334)
spike_not_RBD2 = BitSet(529:1273)
spike_not_RBD_sites = union(spike_not_RBD1, spike_not_RBD2)
for mut in keys(AA_muts_ct_no_dels_universal)
    if aa_gene_comprehensive_dict[mut] == "S"
        push!(spike_muts, mut)
    else
        push!(nonspike_muts, mut)
    end
    if aa_gene_comprehensive_dict[mut] == "S" && aa_pos_comprehensive_dict[mut] in RBD_not_RBM_sites
        push!(RBD_not_RBM_muts, mut)
    end
    if aa_gene_comprehensive_dict[mut] == "S" && aa_pos_comprehensive_dict[mut] in spike_not_RBD_sites
        push!(spike_not_RBD_muts, mut)
    end
    if aa_gene_comprehensive_dict[mut] == "S" && aa_pos_comprehensive_dict[mut] in RBM_sites
        push!(RBM_muts, mut)
    end
    if aa_gene_comprehensive_dict[mut] == "S" && aa_pos_comprehensive_dict[mut] in RBD_sites
        push!(RBD_muts, mut)
    end
end
###########################################################################################################################################################################
###########################################################################################################################################################################
global purespikebanned_0__purespikeallowed_1 = 0
global nonspike_seqfactor__0off__1RBM__2RBD__3SpikeNonRBD__4AllSpike = 0
global all_excluded_muts = Set{String}()
if normal_0__spikeonly_1__spikeWithRBD_2 == 0
    purespikebanned_0__purespikeallowed_1 = 0
    nonspike_seqfactor__0off__1RBM__2RBD__3SpikeNonRBD__4AllSpike = 0
    if noBAL_0__withBAL_1 == 0
        all_excluded_muts = union(artifactual_private_muts, Double_N_ORF9b_muts, RBD_muts, BAL_muts)
    elseif noBAL_0__withBAL_1 == 1
        all_excluded_muts = union(artifactual_private_muts, Double_N_ORF9b_muts, RBD_muts)
    end
elseif normal_0__spikeonly_1__spikeWithRBD_2 == 1
    purespikebanned_0__purespikeallowed_1 = 1
    nonspike_seqfactor__0off__1RBM__2RBD__3SpikeNonRBD__4AllSpike = 3
    if noBAL_0__withBAL_1 == 0
        all_excluded_muts = union(artifactual_private_muts, Double_N_ORF9b_muts, RBD_muts, BAL_muts, nonspike_muts)
    elseif noBAL_0__withBAL_1 == 1
        all_excluded_muts = union(artifactual_private_muts, Double_N_ORF9b_muts, RBD_muts, nonspike_muts)
    end
elseif normal_0__spikeonly_1__spikeWithRBD_2 == 2
    purespikebanned_0__purespikeallowed_1 = 1
    nonspike_seqfactor__0off__1RBM__2RBD__3SpikeNonRBD__4AllSpike = 0
    if noBAL_0__withBAL_1 == 0
        all_excluded_muts = union(artifactual_private_muts, Double_N_ORF9b_muts, BAL_muts, nonspike_muts)
    elseif noBAL_0__withBAL_1 == 1
        all_excluded_muts = union(artifactual_private_muts, Double_N_ORF9b_muts, nonspike_muts)
    end    
end
if include_RBM == 1
    all_excluded_muts = union(artifactual_private_muts, Double_N_ORF9b_muts)
end
###########################################################################################################################################################################
###########################################################################################################################################################################
RBD_not_RBM_muts_sort = sort(collect(RBD_not_RBM_muts), by = x -> aa_pos_comprehensive_dict[x])
RBD_not_RBM_tot = length(RBD_not_RBM_muts_sort)
println("Total Different RBD-not-RBM Muts = $(RBD_not_RBM_tot)")
RBD_not_RBM_muts_sort_join = join(RBD_not_RBM_muts_sort, ", ")
########################
RBM_muts_sort = sort(collect(RBM_muts), by = x -> aa_pos_comprehensive_dict[x])
RBM_tot = length(RBM_muts_sort)
println("Total Different RBM Muts = $(RBM_tot)")
RBM_muts_sort_join = join(RBM_muts_sort, ", ")
########################
RBD_muts_sort = sort(collect(RBD_muts), by = x -> aa_pos_comprehensive_dict[x])
RBD_tot = length(RBD_muts_sort)
println("Total Different RBD Muts = $(RBD_tot)")
RBD_muts_sort_join = join(RBD_muts_sort, ", ")
########################
spike_not_RBD_muts_sort = sort(collect(spike_not_RBD_muts), by = x -> aa_pos_comprehensive_dict[x])
spike_not_RBD_tot = length(spike_not_RBD_muts_sort)
println("Total Different spike_not_RBD Muts = $(spike_not_RBD_tot)")
spike_not_RBD_muts_sort_join = join(spike_not_RBD_muts_sort, ", ")
#####################################################################################################################################
ORF9bNdoubles_artifacts = union(Double_N_ORF9b_muts, artifactual_private_muts)
tot_grp_ct = length(rep_seq_grps_maxmut_seqs)
tot_single_ct = length(non_rep_seqs)
tot_chr_seq_ct = tot_grp_ct + tot_single_ct
total_chronic_AA_ct = 0
total_chronic_AA_ct_nonRBM = 0
total_chronic_AA_ct_nonRBD = 0
total_chronic_AA_ct_spike_nonRBD = 0
total_chronic_AA_ct_nonspike = 0
total_chronic_AA_ct_spike = 0
for (mut, ct) in AA_muts_ct_no_dels_universal
    if !(mut in artifacts_ORF9bdoubles)
        total_chronic_AA_ct += ct
        if aa_gene_comprehensive_dict[mut] ≠ "S"
            total_chronic_AA_ct_nonspike += ct
            total_chronic_AA_ct_nonRBM += ct
            total_chronic_AA_ct_nonRBD += ct
        end
        if aa_gene_comprehensive_dict[mut] == "S" && !(aa_pos_comprehensive_dict[mut] in RBM_sites)
            total_chronic_AA_ct_nonRBM += ct
        end
        if aa_gene_comprehensive_dict[mut] == "S" && !(aa_pos_comprehensive_dict[mut] in RBD_sites)
            total_chronic_AA_ct_nonRBD += ct
            total_chronic_AA_ct_spike_nonRBD += ct
        end
    end
end
for (mut, ct) in AA_muts_ct_no_dels_universal
    if aa_gene_comprehensive_dict[mut] == "S"
        total_chronic_AA_ct_spike += ct
    end
end

for (mut, ct) in AA_muts_ct_no_dels_universal
    if aa_gene_comprehensive_dict[mut] == "S"
        total_chronic_AA_ct_spike += ct
    end
end
println("Spike_Mut_Ct_v1 (total_chronic_AA_ct_spike) = $(total_chronic_AA_ct_spike)")
print("\n"^2)
#####################################################################################################################
avg_AA_sub_ct_per_chronic_seq = round(digits=2, total_chronic_AA_ct/tot_chr_seq_ct)
println("avg_AA_sub_ct_per_chronic_seq = $(avg_AA_sub_ct_per_chronic_seq)")
avg_AA_sub_ct_per_chronic_seq_nonspike = round(digits=2, total_chronic_AA_ct_nonspike/tot_chr_seq_ct)
println("avg_AA_sub_ct_per_chronic_seq_nonspike = $(avg_AA_sub_ct_per_chronic_seq_nonspike)")
avg_AA_sub_ct_per_chronic_seq_nonRBM = round(digits=2, total_chronic_AA_ct_nonRBM/tot_chr_seq_ct)
println("avg_AA_sub_ct_per_chronic_seq_nonRBM = $(avg_AA_sub_ct_per_chronic_seq_nonRBM)")
avg_AA_sub_ct_per_chronic_seq_nonRBD = round(digits=2, total_chronic_AA_ct_nonRBD/tot_chr_seq_ct)
println("avg_AA_sub_ct_per_chronic_seq_nonRBD = $(avg_AA_sub_ct_per_chronic_seq_nonRBD)")
avg_AA_sub_ct_per_chronic_seq_spike_nonRBD = round(digits=2, total_chronic_AA_ct_spike_nonRBD/tot_chr_seq_ct)
println("avg_AA_sub_ct_per_chronic_seq_spike_nonRBD = $(avg_AA_sub_ct_per_chronic_seq_spike_nonRBD)")
avg_AA_sub_ct_per_chronic_seq_spike = round(digits=2, total_chronic_AA_ct_spike/tot_chr_seq_ct)
println("avg_AA_sub_ct_per_chronic_seq_spike = $(avg_AA_sub_ct_per_chronic_seq_spike)")
######################################################################################################################################
#seq_privAA_nonRBD_len = Dict{String, Int}()
#for (seq, AAsubvec) in seq_AA_muts_no_dels_universal
#    seq_nonRBD_ct = 0
#    for sub in AAsubvec
#        if !(sub in RBD_muts)
#            seq_nonRBD_ct += 1
#        end
#    end
#    seq_privAA_nonRBD_len[seq] = seq_nonRBD_ct
#end
######################################################################################################################################
#seq_privAA_nonRBD_len_relative = Dict{String, Float64}()
#for (seq, ct) in seq_privAA_nonRBD_len
#    relative_nonRBD = round(digits=2, ct/avg_AA_sub_ct_per_chronic_seq_nonRBD)
#    seq_privAA_nonRBD_len_relative[seq] = relative_nonRBD
#end
#####################################################################################################################
global avg_AA_sub_ct_per_chronic_seq_for_main_fx = 0.0
#####################################################################################################################
if normal_0__spikeonly_1__spikeWithRBD_2 == 0
    avg_AA_sub_ct_per_chronic_seq_for_main_fx = avg_AA_sub_ct_per_chronic_seq_nonRBD
elseif normal_0__spikeonly_1__spikeWithRBD_2 == 1
    avg_AA_sub_ct_per_chronic_seq_for_main_fx = avg_AA_sub_ct_per_chronic_seq_spike_nonRBD
end
#####################################################################################################################
# all_excluded_muts
seq_privAA_len = Dict{String, Int}()
for (seq, AAsubvec) in seq_AA_muts_no_dels_universal
    seq_mut_ct = 0
    for sub in AAsubvec
        if !(sub in all_excluded_muts)
            seq_mut_ct += 1
        end
    end
    seq_privAA_len[seq] = seq_mut_ct
end
#####################################################################################################################
seq_privAA_len_relative = Dict{String, Float64}()
for (seq, ct) in seq_privAA_len
    relative_muts = round(digits=2, ct/avg_AA_sub_ct_per_chronic_seq_for_main_fx)
    seq_privAA_len_relative[seq] = relative_muts
end
#####################################################################################################################
total_chronic_AA_ct_v2 = 0
total_chronic_AA_ct_v2_nonRBM = 0
total_chronic_AA_ct_v2_nonRBD = 0
total_chronic_AA_ct_v2_spike_nonRBD = 0
total_chronic_AA_ct_v2_nonspike = 0
total_chronic_AA_ct_v2_spike = 0
for seq in EPCI_set
    for mut in seq_AA_muts_no_dels_universal[seq]
        if !(mut in artifacts_ORF9bdoubles)
            total_chronic_AA_ct_v2 += 1
            if aa_gene_comprehensive_dict[mut] ≠ "S"
                total_chronic_AA_ct_v2_nonspike += 1
                total_chronic_AA_ct_v2_nonRBM += 1
                total_chronic_AA_ct_v2_nonRBD += 1
            else
                total_chronic_AA_ct_v2_spike += 1
            end
            if aa_gene_comprehensive_dict[mut] == "S" && !(aa_pos_comprehensive_dict[mut] in RBM_sites)
                total_chronic_AA_ct_v2_nonRBM += 1
            end
            if aa_gene_comprehensive_dict[mut] == "S" && !(aa_pos_comprehensive_dict[mut] in RBD_sites)
                total_chronic_AA_ct_v2_nonRBD += 1
                total_chronic_AA_ct_v2_spike_nonRBD += 1
            end
        end
    end
end
####################################################################################################################################
avg_AA_sub_ct_per_chronic_seq_v2 = round(digits=2, total_chronic_AA_ct_v2/all_unique_chr_seqs_ct)  #### Double checking the first count
println("avg_AA_sub_ct_per_chronic_seq_v2 = $(avg_AA_sub_ct_per_chronic_seq_v2)")
println("total_chronic_AA_ct = $(total_chronic_AA_ct)")
println("total_chronic_AA_ct_v2 = $(total_chronic_AA_ct_v2)")
avg_AA_sub_ct_per_chronic_seq_v2_nonspike = round(digits=2, total_chronic_AA_ct_v2_nonspike/all_unique_chr_seqs_ct)
println("avg_AA_sub_ct_per_chronic_seq_v2_nonspike = $(avg_AA_sub_ct_per_chronic_seq_v2_nonspike)")
println("total_chronic_AA_ct_nonspike = $(total_chronic_AA_ct_nonspike)")
println("total_chronic_AA_ct_v2_nonspike = $(total_chronic_AA_ct_v2_nonspike)")
avg_AA_sub_ct_per_chronic_seq_v2_nonRBM = round(digits=2, total_chronic_AA_ct_v2_nonRBM/all_unique_chr_seqs_ct)
println("avg_AA_sub_ct_per_chronic_seq_v2_nonRBM = $(avg_AA_sub_ct_per_chronic_seq_v2_nonRBM)")
println("total_chronic_AA_ct_nonRBM = $(total_chronic_AA_ct_nonRBM)")
println("total_chronic_AA_ct_v2_nonRBM = $(total_chronic_AA_ct_v2_nonRBM)")
avg_AA_sub_ct_per_chronic_seq_v2_nonRBD = round(digits=2, total_chronic_AA_ct_v2_nonRBD/all_unique_chr_seqs_ct)
println("avg_AA_sub_ct_per_chronic_seq_v2_nonRBD = $(avg_AA_sub_ct_per_chronic_seq_v2_nonRBD)")
println("total_chronic_AA_ct_nonRBD = $(total_chronic_AA_ct_nonRBD)")
println("total_chronic_AA_ct_v2_nonRBD = $(total_chronic_AA_ct_v2_nonRBD)")
avg_AA_sub_ct_per_chronic_seq_v2_spike_nonRBD = round(digits=2, total_chronic_AA_ct_v2_spike_nonRBD/all_unique_chr_seqs_ct)
println("avg_AA_sub_ct_per_chronic_seq_v2_spike_nonRBD = $(avg_AA_sub_ct_per_chronic_seq_v2_spike_nonRBD)")
println("total_chronic_AA_ct_spike_nonRBD = $(total_chronic_AA_ct_spike_nonRBD)")
println("total_chronic_AA_ct_v2_spike_nonRBD = $(total_chronic_AA_ct_v2_spike_nonRBD)")
avg_AA_sub_ct_per_chronic_seq_v2_spike = round(digits=2, total_chronic_AA_ct_v2_spike/all_unique_chr_seqs_ct)
println("avg_AA_sub_ct_per_chronic_seq_v2_spike = $(avg_AA_sub_ct_per_chronic_seq_v2_spike)")
println("total_chronic_AA_ct_spike = $(total_chronic_AA_ct_spike)")
println("total_chronic_AA_ct_v2_spike = $(total_chronic_AA_ct_v2_spike)"); print("\n"^1)
println("Finished!"); print("\n"^1)
println("all_unique_chr_seqs_ct = $(all_unique_chr_seqs_ct)")
println("tot_chr_seq_ct = $(tot_chr_seq_ct)")
##############################################################################################
runtime = time() - start
runtime_rd = round(digits=2, runtime)
runtime1, runtime2 = seconds_to_hrs_min_sec(runtime)
println("Runtime v0 = $(runtime) seconds")
println("Runtime v2 = $(runtime2)")
date_now = Dates.format(now(), "yyyy_mm_dd__HMMp"); println(date_now); print("\n"^1)
date_hour = Dates.format(now(), "yyyy_mm_dd_Hp"); print("\n"^2)
####################################################################################################################################
####################################################################################################################################



2026_05_05__2357PM
11:57.34_PM


AA_muts_ct_no_dels_universal ≠ AA_muts_ct_no_dels
AA_muts_ct_no_dels_universal == AA_muts_ct_pos_only_no_dels
Total Different RBD-not-RBM Muts = 80
Total Different RBM Muts = 53
Total Different RBD Muts = 133
Total Different spike_not_RBD Muts = 779
Spike_Mut_Ct_v1 (total_chronic_AA_ct_spike) = 33368


avg_AA_sub_ct_per_chronic_seq = 20.31
avg_AA_sub_ct_per_chronic_seq_nonspike = 13.69
avg_AA_sub_ct_per_chronic_seq_nonRBM = 18.83
avg_AA_sub_ct_per_chronic_seq_nonRBD = 17.82
avg_AA_sub_ct_per_chronic_seq_spike_nonRBD = 4.14
avg_AA_sub_ct_per_chronic_seq_spike = 13.4
avg_AA_sub_ct_per_chronic_seq_v2 = 20.02
total_chronic_AA_ct = 50575
total_chronic_AA_ct_v2 = 49848
avg_AA_sub_ct_per_chronic_seq_v2_nonspike = 13.58
total_chronic_AA_ct_nonspike = 34080
total_chronic_AA_ct_v2_nonspike = 33813
avg_AA_sub_ct_per_chronic_seq_v2_nonRBM = 18.55
total_chronic_AA_ct_nonRBM = 46883
total_chronic_AA_ct_v2_nonRBM = 46183
avg_AA_sub_ct_per_chronic_seq_v2_nonRBD = 17.

In [140]:
###   MAIN APCI FUNCTION Part 1  | APCI  ### The empty lines here are so that the lines match up exactly with the main EPCI function 
###########################################################################################################################################
################################### Abbreviations used in this function (including in comments) ###########################################
#                       mut = mutation, which in this function means amino acid substitution (not insertions/deletions)
#                       EPCI = manually curated list of confirmed & likely chronic-infection sequences
#                       chronic = chronic-infection sequence or lineage, i.e. EPCI sequence
#                       mp = mutation pattern
#                       AA = amino acid
#                       del = deletion
#                       nd = no deletions (also no_dels)
#                       po = position only (also pos_only); indicates all AA substitutions at a given AA position
#                       ct = count
#                       seq = sequence
#                       grp = group
#                       RBD = spike receptor-binding domain (S:335-519)
#                       RBM = spike receptor-binding motif (S:438-506)
#                       log10pvFISH = negative log10 p-value for Fisher's exact test
#                       chi2 = chi-squared test (also Chi2 or Chi)
#                       dict = Dictionary (type of Julia coding object)
#                       qual = qualifying (i.e. meeting the necessary requirements)
#                       coAAmut = co-occuring amino acid substitutions (i.e. in the same sequence)
#                       priv = private (as in private mutations, i.e. those not inherited from documented sequences, according to the best phylogenetic reconstructions)
#                       min = minimum (as in min_mut_ct, i.e. minimum mutation count requirement)
#                       max = maximum
#                       avg = average, i.e. mean (for all you stats snobs, who had to invent a new word for average that already had a dozen other meanings)
#                       tot = total
#                       chr = chronic (see above)
#                       vec = Vector type (array or tuple)
#                       str = string type
#                       df = DataFrame type
#                       fake = from the artificially created list of EPCI-like sequences (called APCI in methods section)
#                       BAL = bronchoalveoloar lavage, as in mutations associated with bronchoalveoloar lavage-sample sequences
#################################################################################################################################################
print("\n"^1)
date_now = Dates.format(now(), "yyyy_mm_dd__HMMp"); println(date_now)
date_hour = Dates.format(now(), "yyyy_mm_dd_Hp")
date = Dates.format(today(), "yyyy_mm_dd")
nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime); print("\n"^1)
#### When testing the original mut patterns, zero RBD or RBM mutations appeared in any of the final or intermediary results. 
####    Their exclusion, therefore, has no effect. 
#### However, when testing all other mutations that occur in the EPCI dataset more than five times (for control purposes), 
####    RBM mutation groups frequently occur in enormous "mutation patterns", which is simply a result of the extremely dense
####    mutation rate in the RBM in a large number of EPCI sequences. These "patterns" are therefore not very meaningful and only
####    serve to identify the mutations present in sequences with a high density of mutations in the RBM.
#### RBM mutations are therefore entirely prohibited as seed mutations AND excluded from inclusion in any mut pattern.
###########################################################################################################################################################################
###########################################################################################################################################################################
####### Below dicts were an attempt to use a more sophisticated & less blunt exclusion of N/9b double muts. Not currently used.
                        # ORF9b_N_doubles = Dict{String, Vector{String}}("ORF9b:D2E"=>["N:P6T"], "ORF9b:P10S"=>["N:P13L"], "ORF9b:S10P"=>["N:L13P"], "ORF9b:T60A"=>["N:D63G"], "ORF9b:Q77E"=>["N:P80R"], "ORF9b:E86D"=>["N:A90S", "N:A90P"], "ORF9b:V92M"=>["N:R95H"], "ORF9b:V93L"=>["N:G96V", "N:G96A"],            "ORF9b:V93M"=>["N:G96D"], "ORF9b:V94M"=>["N:G97D"],      "ORF9b:V94L"=>["N:G97V", "N:G97A"],    "ORF9b:T95A"=>["N:D98G"], "ORF9b:K97E"=>["N:K100R"], "ORF9b:K97N"=>["N:M101L"])  # "ORF9b:"=>["N:"], "ORF9b:"=>["N:"]
                        # N_ORF9b_doubles = Dict{String, Vector{String}}("N:P6T"=>["ORF9b:D2E"], "N:P13L"=>["ORF9b:P10S"], "N:L13P"=>["ORF9b:S10P"], "N:D63G"=>["ORF9b:T60A"], "N:P80R"=>["ORF9b:Q77E"], "N:A90S"=>["ORF9b:E86D"], "N:R95H"=>["ORF9b:V92M"], "N:G96V"=>["ORF9b:V93L"], "N:G96A"=>["ORF9b:V93L"], "N:G96D"=>["ORF9b:V93M"], "N:G97D"=>["ORF9b:V94M"], "N:G97V"=>["ORF9b:V94L"], "N:G97A"=>["ORF9b:V94L"], "N:D98G"=>["ORF9b:T95A"], "N:K100R"=>["ORF9b:K97E"], "N:M101L"=>["ORF9b:K97N"])
######################################################################################################################################
#open("$(mp_folder_universal)/mp_mp_round_by_round_debug/round_by_round_debug_seed$(seedmut)_$(mut_pattern_name)_minGrpFish$(min_grp_fish_int)_minFish$(min_log_pv_fish_int)_seqfac$(seqfac)_$(date).txt", "w") do g; println(g); end
######################################################################################################################################
####       This fx begins with a minimal number of mutations that have a clear & obvious connection (e.g. ORF1a:T1322I & ORF1a:T1638I)
####   A minimum number of mutations is required for a sequence to qualify. The exact minimum depends on the number of 
####   genomic regions contained in the searched mutations (i.e. the parameter sel_muts_pt1). If there are 1-2 regions represented,
####   a sequence must have ≥1 mutation in the 2 regions. For 3-4 regions, a sequence must have ≥2 muts. If ≥5 regions are represented,
####   the minimum is 3 muts. 
####       Once the qualifying sequences have been determined, all private mutations in those sequences are tallied. Each mutation 
####   in this dataset is then assessed to find whether it is statistically overrepresented in the sequence set. A Chi^2 value is
####   calculated based on the number of times the mutation appears in the "qualifying sequences" dataset, the number of times
####   the mutation appears in the entire chronic-sequences list (EPCI), and the number of qualifying sequences. 
####       If a mutation is in the same region as one of the existing sel_muts_pt1 regions (defined as being within 4 AA of one of the
####   muts in a sel_muts_pt1 region), it must have a Chi^2 value ≥ 12 to qualify for inclusion in that region. If it is not near any 
####   of the sel_muts_pt1 regions, then it must meet the stricter criteria of having a -log10 p-value of for the Fisher's exact test
####   in order to be included as a new mutation group in sel_muts_pt1. 
####       When the new mutations (sel_muts2_pt1) have been determined, they become the mutations to begin the next round. The entire 
####   process is repeated until there is no change from the previous round, at which point this part of the 2-part function stops and
####   its data fed into part 2, which is described below. 
######################################################################################################################################
##### NOTE: Mutations in mp_masked_muts are excluded from inclusion in the mut pattern and from the sequence-qualifying process; 
##        i.e. if E:T9I is in mp_masked_muts, E:T9I is BLOCKED from inclusion in the mut pattern.
##    The justification for mp_masked_muts (which is rarely used) is that sometimes one or more extremely common chronic mutations
##        might have a strong statistical association with a mutation pattern, even though its fold increase might be relatively small
##        compared to other mutations——maybe 2-fold higher compared to >10-fold for others. But because it is such a common mutation,
##        the statistical association appears strong. The problem is that the sequence-selection process becomes dominated by these
##        common chronic mutations. For example, ORF1a:K1795Q and ORF7a:T39I are both very common in chronics (288 and 200 sequences, 
##        respectively) and highly correlated with each other. They also both have somewhat less intense correlations with the
##        3-7a pattern (which involves 5 other regions). ORF1a:K1795Q and ORF7a:T39I, because of how common they are and how often
##        they appear together, dominate the sequence-selection process, drowning out the other regions/mutations, which are the most
##        distinctive parts of the 3-7a mutational pattern. mp_masked_muts allows us to test the 3-7a mutational pattern both with 
##        and without ORF1a:K1795Q and ORF7a:T39I.
######################################################################################################################################
#### ind_or_grp is either "ind" or "grp". The former ("ind") means all muts counted individually; 
####     The latter ("grp") means each group/region is only counted once, no matter how many mutations are in that group.
####     Right now, "grp" is the ony option that is ever used.
###########################################################################################################################################################################
###########################################################################################################################################################################
#####################################################   Part 2 Description   ##############################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
######   Part 2 looks at each mut region individually by first removing that region from the list & then reassessing the statistical
######   significance of each mut in that region (for how it correlates w/sequences w/muts in the remaining region(s)). The same
######   Chi^2 value of ≥ 12 is used to determine if a mutation qualifies, though this value can be adjusted up or down if one wishes.
######   It does this for each region in turn. A round is finished once all regions have been assessed this way. Mutations can
######   be added or removed during this process.
######       Once the final region is finished, it checks to see if the round just finished caused any changes in the full mutation 
######   list relative to the beginning of the round.  If there were changes, it goes through another full round. 
######   If there were no changes in the round, it finishes and prints a bunch of shit to record the final results.
######       Part 1 took care of locating the regions to be analyzed, so no new regions are added in Part 2, though individual 
######   mutations are lost or gained.
######   Originally, Part 1 and Part 2 were separate functions, but due to the inexplicable inability of Part 1 to correctly
######       deliver its results to Part 2, the two functions were combined into a single function that uses global variables
######       instead of parameters. 
######################################################################################################################################
##  • mp_masked_muts are excluded mutations. They are not eligible for inclusion in the mutational pattern. (See part 1 for justification
##        and explanation for mp_masked_muts.)
##  • ind_or_grp is either "ind" or "grp". The former ("ind") means all muts counted individually; *only* grp is currently used
##    The latter ("grp") means each group/region is only counted once, not matter how many mutations are in that group
##  • all_muts_round_dict_pt2 (line below) keeps track of all the muts after each full round. Used to determine when to stop iterating.
##  • pt2_iteration_ct keeps track of the total number of iterations. Each round has as many iterations as the mut pattern has regions.
##    E.g., if there are 5 regions in the mut pattern, there will be 5 iterations in each round, one for each region
##  • pt2_region_analyzed tells which region is being analyzed; e.g. if there are 4 regions
##    pt2_region_analyzed should *always* be set to *one* when starting the function.
###########################################################################################################################################################################
       fake_dict_num_ct = 1
######### pt1_round_ct = round # we are on in part 1.
       pt1_round_ct = 0
       pt1_finished = 0
######################################################################################################################################
####### all_muts_round_dict_pt2 records all qualifying muts at the end of each round (i.e. after all regions have been independently checked)
#######    The stats for each mutation MUST ONLY BE RECORDED IN THE ITERATION DURING WHICH ITS GROUP IS EXCLUDED
       all_muts_round_dict_pt2 = Dict{Int, Set{String}}()   # all_muts_round_dict_pt2 stores the qualifying mutations at the end of each round
#####################################################################################################################################
####### all_muts_round_mutgroup_dict_pt2 records all qualifying *groups* of mutations at the end of each round
####### As with the original sel_muts_pt2, each group is a string of comma-separated mutations
       all_muts_round_mutgroup_dict_pt2 = Dict{Int, Vector{String}}()   
#####################################################################################################################################
#### df_props_dict_pt2 is needed to record all the specific stats for each mutation in each round (for printing results & csv/tsv purposes)
#                             Round RegionAnalyzed  Mutation      EPCI_pct, MP_pct, totChrMutCt, MP_Tot_Mut_ct, Adjusted_MP_seq_ct, Chi^2,  pvFish, log10pvFISH, fold_incr
global df_props_dict_pt2 = Dict{Int, Dict{Int, Dict{String, Tuple{Float64,Float64,Int,Int,Int,Float64,Float64,Float64,Union{Float64,String},Float64}}}}()
global pt2_round_ct = 1
global pt2_region_analyzed = 1
global sel_muts_pt2 = Vector{String}()
global sel_muts_pt2_ind_dict = Dict{Int, Vector{String}}()
global region_ct2
global sel_muts_pt1_round_dict = Dict{Int, Vector{String}}()
global sel_muts_pt1_global = Vector{String}()
global sel_muts_pt1_sort = Vector{String}()
global sel_muts_pt1_ind = Vector{Vector{String}}()
global pt1_seed_mut = ""
global coAAmut_ct_pt1 = Dict{String,Int}()
global coAAmut_ct_pt2 = Dict{String,Int}()
global coAAmut_ct_pt1_unknown = Dict{String,Int}()
global coAAmut_ct_pt2_unknown = Dict{String,Int}()
############################################################################################################################################################################
#                         Mutation        EPCI_pct, MP_pct, totChrMutCt, MP_Tot_Mut_ct, Adjusted_MP_seq_ct, Chi^2,  pvFish, log10pvFISH, fold_incr
global prop_dict_pt1 = Dict{String, Tuple{Float64,Float64,Int,Int,Int,Float64,Float64,Float64,Union{Float64,String},Float64} }()
global prop_dict_pt2 = Dict{String, Tuple{Float64,Float64,Int,Int,Int,Float64,Float64,Float64,Union{Float64,String},Float64} }()
############################################################################################################################################################################




############################################################################################################################################################################
global cumulative_seq_round_set_pt2 = Set{Int}() ### Set of all seqs that qualify during any iteration of a round (erased upon the start of each round so that only the final round's results are left at end)
global cumulative_seq_round_set_pt2_sort = Vector{String}()
global cumulative_seq_round_set_pt2_sort_join = ""
############################################################################################################################################################################





global df_mp = DataFrame("Mutation" => String[], "MP Tot Mut ct" => Float64[], "EPCI Mut ct" => Int[], "Adjusted MP seq ct" => Int[], "EPCI pct" => Float64[], "MP pct" => Float64[], "non MP pct" => Float64[], "log10pvFISH" => Float64[], "Fold Incr" => Union{Float64,String}[], "Chi2" => Float64[], "EPCI HQCS Ratio" => Float64[], "avg NonRBD AAct" => Float64[], "avg nonRBD rel" => Float64[])
global df_mp_META
global df_mp_META_sort
global df_mp_META_sort_unique
global df_mp_META_set_set = Set{Set{String}}()
global df_mp_META_region_set_set = Set{Set{Set{String}}}() 
global df_mp_META_set_set_dict = Dict{Set{String}, String}() ## Values = seedmuts
global df_mp_META_region_set_set_dict = Dict{Set{Set{String}}, String}() ## Values = seedmuts
global df_mp_META_groups_dict = Dict{String,Vector{Tuple{String,Int,Int,Int,Int,String,Int,Int,Int,Float64,Float64,Float64,Float64,Float64,Union{Float64,String},Float64,Float64,Float64,Float64,Int,String,Int,String,Int,Int,Int,Int,Int,Int,Int,Int,String,String,String,String,String,String,String,Int,Int,Int,Int,Int,Int,Int,String,String,String,String,String,String,String,String}}}()
############################################################################################################################################################################
global total_EPCI_seq_ct = length(EPCI_set)
global mp_index










###########################################################################################################################################################################
###########################################################################################################################################################################
min_log_pv_fish = 5.0
min_grp_fish = 2.0
plus_minus = 5
min_log_pv_fish_int = Int(min_log_pv_fish)
min_grp_fish_int = Int(min_grp_fish)
global mp_folder_universal = ""
if sub_0__posonly_1 == 0
    if normal_0__spikeonly_1__spikeWithRBD_2 == 0
        mp_folder_universal = "mp_subs_FAKE_$(date_hour)__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)__minFsh$(min_log_pv_fish_int)_grpFsh_$(min_grp_fish_int)"
    elseif normal_0__spikeonly_1__spikeWithRBD_2 == 1
        mp_folder_universal = "mp_subs_FAKE_spike_only_$(date_hour)__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)__minFsh$(min_log_pv_fish_int)_grpFsh_$(min_grp_fish_int)"
    end
elseif sub_0__posonly_1 == 1
    if normal_0__spikeonly_1__spikeWithRBD_2 == 0
        mp_folder_universal = "mp_pos_only_FAKE_$(date_hour)__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)__minFsh$(min_log_pv_fish_int)_grpFsh_$(min_grp_fish_int)"
    elseif normal_0__spikeonly_1__spikeWithRBD_2 == 1
        mp_folder_universal = "mp_pos_only_FAKE_spike_only_$(date_hour)__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)__minFsh$(min_log_pv_fish_int)_grpFsh_$(min_grp_fish_int)"
    end
end

###########################################################################################################################################################################
###########################################################################################################################################################################
global pt1_seed_mut_set = Set{String}()
global pt1_seed_mut_grp_total_dict = Dict{String, Int}()
###########################################################################################################################################################################






















###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
function AA_2plus__2026_05_02_subs_only_FAKE(min_match_ct::Int, PMR1::Float64, PMR2::Float64, purespikebanned_0__purespikeallowed_1::Int, nonmulti0__multi1::Int, notrandom0__random1::Int, nonspike_seqfactor__0off__1RBM__2RBD__3SpikeNonRBD__4AllSpike::Int, plus_minus::Int, mut_pattern_name::String, ind_or_grp::String, min_grp_fish::Float64, min_log_pv_fish::Float64, mp_masked_muts::Set{String}, sel_muts_pt1::Vector{String}, seq_factor_OFF_HALF_ON__0_1_2::Int)
#    mkpath(mp_folder_universal)
    start_time = time()
    global sel_muts_pt1_round_dict # = Dict{Int, Vector{String}}()
    global sel_muts_pt1_global # = Vector{String}()
    global sel_muts_pt1_sort # = Vector{String}()
    global sel_muts_pt1_ind # = Vector{Vector{String}}()
    global sel_muts_pt2 # = Vector{String}()
    global sel_muts_pt2_ind_dict # = Dict{Int, Vector{String}}()
    global pt1_round_ct # = 0
    global pt1_finished # = 0
    global pt2_round_ct # = 0
    global pt2_region_analyzed # = 1
    global all_muts_round_dict_pt2 # = Dict{Int, Set{String}}()
    global all_muts_round_mutgroup_dict_pt2 # = Dict{Int, Vector{String}}()
    global df_props_dict_pt2 # = Dict{Int, Dict{Int, Dict{String,Tuple{Float64,Float64,Int,Int,Int,Float64,Float64,Float64,Union{Float64,String},Float64}}}}()
    global coAAmut_ct_pt1 # = Dict{String,Int}()
    global coAAmut_ct_pt2 # = Dict{String,Int}()
    global coAAmut_ct_pt1_unknown # = Dict{String,Int}()
    global coAAmut_ct_pt2_unknown # = Dict{String,Int}()
    global prop_dict_pt1 # = Dict{String,Tuple{Float64,Float64,Int,Int,Int,Float64,Float64,Float64,Float64}}()
    global prop_dict_pt2 # = Dict{String,Tuple{Float64,Float64,Int,Int,Int,Float64,Float64,Float64,Float64}}()
    global cumulative_seq_round_set_pt2 # = Set{String}()
    global cumulative_seq_round_set_pt2_sort # = Vector{String}()
    global cumulative_seq_round_set_pt2_sort_join # = ""




    

    global df_mp
    
    global all_excluded_muts
    global purespikebanned_0__purespikeallowed_1
    global nonspike_seqfactor__0off__1RBM__2RBD__3SpikeNonRBD__4AllSpike
    global mp_folder_universal
    global df_mp_META_seedmut_set_dict
    global pt1_seed_mut_set    
    global df_mp_META_set_set
    global df_mp_META_region_set_set
    global pt1_seed_mut_grp_total_dict
    global pt1_seed_mut
#    global region_ct2
    global fake_dict_num_ct    
    fake_seq_AA_muts = mp_meta_fake_chr_dict_DQ[fake_dict_num_ct]
    global mp_index
    global df_mp_META
    global df_mp_META_META
    global seq_privAA_len
    global seq_privAA_len_meta






    
    pt1_seed_mut = ""
    date_and_AAlen_seqs = Set{Int}()
    all_muts_set_pt2 = Set{Int}()
    chr_all_ratio = 0.0
    min_abs_mut_ct = 1
    max_pt1_rounds = 16
    seqfac_dict = Dict(0=>"OFF", 1=>"ON", 2=>"FULLBLAST")
    seqfac_dict_spike = Dict(0=>"OFF", 1=>"RBM", 2=>"RBD", 3=>"SpikeNonRBD")
    seqfac = seqfac_dict[seq_factor_OFF_HALF_ON__0_1_2]
    SpikeFac = seqfac_dict_spike[nonspike_seqfactor__0off__1RBM__2RBD__3SpikeNonRBD__4AllSpike]
    error_ct = 0
#   avg_mp_pango_date_index_int_50 = 0
#   avg_mp_pango_date_string_50 = "00-00-00"
#   pango_ct_dict = Dict{String,Int}()
#   pango_ct_dict_sort = Vector{Pair{String,Int64}}()
#   clade_ct_dict = Dict{String,Int}()
#   clade_ct_dict_sort = Vector{Pair{String,Int64}}()
#   cladepango_ct_dict = Dict{String,Int}()
#   cladepango_ct_dict_sort = Vector{Pair{String,Int64}}()    
    seedmut = sel_muts_pt1[1]
    min_date_raw_ct = 2
##########################################################################################################################################
    all_excluded_muts_plus_mp_masked_muts = union(mp_masked_muts, all_excluded_muts)
    fake_seq_AA_muts_not_excluded2 = Dict{Int, Set{String}}()
    for (seq, mutset) in fake_seq_AA_muts
        fake_seq_AA_muts_not_excluded2[seq] = Set{String}()
        for mut in mutset
            if !(mut in all_excluded_muts_plus_mp_masked_muts)
                push!(fake_seq_AA_muts_not_excluded2[seq], mut)
            end
        end
    end
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
### These dictionaries determine the number of MP mutations required for a sequence to qualify to be part of the set of sequences included in the MP
### As the number of regions in an MP increases the additional mutations required foreach additional MP (which starts at 0.4) is decreased to 0.3, 
### then 0.2 for each additional MP region beyond a certain number (different for parts 1 and 2.
### PMR1 = private mutation requirement factor for part 1
### PMR2 = private mutation requirement factor for part 1
######################################################################################################################################
    min_mut_ct_dict_pt1 = Dict{Int,Float64}(i=>round(digits=1, (max(i, 1)*PMR1)) for i in 0:4)
    for i in 5:8
        min_mut_ct_dict_pt1[i] = 4*PMR1 + (i-4)*(PMR1 - 0.1)
    end
    for i in 9:30
        min_mut_ct_dict_pt1[i] = 4*PMR1 + 4*(PMR1 - 0.1) + (i-8)*(PMR1 - 0.2)
    end
    min_abs_mut_ct_dict_pt1 = Dict{Int,Int}(-1=>1, 0=>1, 1=>1, 2=>1, 3=>1, 4=>2, 5=>2, 6=>2, 7=>2, 8=>3, 9=>3, 10=>3, 11=>3, 12=>3, 13=>3, 14=>4, 15=>4, 16=>4, 17=>4, 18=>5, 19=>5, 20=>5, 21=>6, 22=>6, 23=>6, 24=>6, 25=>6, 26=>7, 27=>7, 28=>8, 29=>9, 30=>10, 31=>20, 32=>20, 33=>20, 34=>20, 35=>20, 36=>20, 37=>20, 38=>20, 39=>30)
    min_date_abs_ct_dict_pt1 = Dict{Int,Int}(-1=>1, 0=>1, 1=>1, 2=>2, 3=>2, 4=>2, 5=>3, 6=>3, 7=>3, 8=>3, 9=>4, 10=>4, 11=>4, 12=>4, 13=>5, 14=>5, 15=>5, 16=>5, 17=>5, 18=>5, 19=>5, 20=>5, 21=>6, 22=>6, 23=>6, 24=>6, 25=>6, 26=>7, 27=>7, 28=>8, 29=>9, 30=>10, 31=>20, 32=>20, 33=>20, 34=>20, 35=>20, 36=>20, 37=>20, 38=>20, 39=>30)
######################################################################################################################################
    min_mut_ct_dict_pt2 = Dict{Int,Float64}(i=>round(digits=1, (max(i, 1)*PMR2)) for i in -1:1)
    for i in 2:8
        min_mut_ct_dict_pt2[i] = PMR2 + (i-1)*(PMR2 - 0.1) - 0.1
    end
    for i in 9:11
        min_mut_ct_dict_pt2[i] = PMR2 + 7*(PMR2 - 0.1) + (i-8)*(PMR2 - 0.2)
    end
    for i in 12:33
        min_mut_ct_dict_pt2[i] = PMR2 + 7*(PMR2 - 0.1) + 3*(PMR2 - 0.2) + (i-11)*(PMR2 - 0.25)
    end
    min_abs_mut_ct_dict_pt2 = Dict{Int,Int}(-1=>1, 0=>1, 1=>1, 2=>1, 3=>1, 4=>2, 5=>2, 6=>2, 7=>2, 8=>3, 9=>3, 10=>3, 11=>3, 12=>3, 13=>3, 14=>4, 15=>4, 16=>4, 17=>4, 18=>5, 19=>5, 20=>5, 21=>6, 22=>6, 23=>6, 24=>6, 25=>6, 26=>7, 27=>7, 28=>8, 29=>9, 30=>10, 31=>20, 32=>20, 33=>20, 34=>20, 35=>20, 36=>20, 37=>20, 38=>20, 39=>30)
    min_date_abs_ct_dict_pt2 = Dict{Int,Int}(-1=>1, 0=>1, 1=>1, 2=>2, 3=>2, 4=>2, 5=>3, 6=>3, 7=>3, 8=>3, 9=>4, 10=>4, 11=>4, 12=>4, 13=>4, 14=>4, 15=>5, 16=>5, 17=>5, 18=>5, 19=>5, 20=>5, 21=>6, 22=>6, 23=>6, 24=>6, 25=>6, 26=>7, 27=>7, 28=>8, 29=>9, 30=>10, 31=>20, 32=>20, 33=>20, 34=>20, 35=>20, 36=>20, 37=>20, 38=>20, 39=>30)    
############################################################################################################################################################################
############################################################################################################################################################################
##############################################################  Below: OG version  #########################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
#    min_mut_ct_dict_pt1 = Dict{Int,Float64}(i=>round(digits=1, (max(i, 1)*PMR1)) for i in 0:4)
#    for i in 5:10
#        min_mut_ct_dict_pt1[i] = 4*PMR1 + (i-4)*(PMR1 - 0.1)
#    end
#    for i in 11:30
#        min_mut_ct_dict_pt1[i] = 4*PMR1 + 6*(PMR1 - 0.1) + (i-10)*(PMR1 - 0.2)
#    end
#    min_abs_mut_ct_dict_pt1 = Dict{Int,Int}(-1=>1, 0=>1, 1=>1, 2=>1, 3=>1, 4=>2, 5=>2, 6=>2, 7=>2, 8=>3, 9=>3, 10=>3, 11=>3, 12=>3, 13=>3, 14=>4, 15=>4, 16=>4, 17=>4, 18=>5, 19=>5, 20=>5)
#    min_date_abs_ct_dict_pt1 = Dict{Int,Int}(-1=>1, 0=>1, 1=>1, 2=>2, 3=>2, 4=>2, 5=>3, 6=>3, 7=>3, 8=>3, 9=>4, 10=>4, 11=>4, 12=>4, 13=>5, 14=>5, 15=>5, 16=>5, 17=>5, 18=>5, 19=>5, 20=>5)
############################################################################################################################################################################
#    min_mut_ct_dict_pt2 = Dict{Int,Float64}(i=>round(digits=1, (max(i, 1)*PMR2)) for i in -1:3)
#    for i in 4:8
#        min_mut_ct_dict_pt2[i] = 3*PMR2 + (i-3)*(PMR2 - 0.1)
#    end
#    for i in 9:11
#        min_mut_ct_dict_pt2[i] = PMR2 + 7*(PMR2 - 0.1) + (i-8)*(PMR2 - 0.2)
#    end
#    for i in 12:30
#        min_mut_ct_dict_pt2[i] = PMR2 + 6*(PMR2 - 0.1) + 4*(PMR2 - 0.2) + (i-11)*(PMR2 - 0.25)
#    end
#    min_abs_mut_ct_dict_pt2 = Dict{Int,Int}(-1=>1, 0=>1, 1=>1, 2=>1, 3=>1, 4=>2, 5=>2, 6=>2, 7=>2, 8=>3, 9=>3, 10=>3, 11=>3, 12=>3, 13=>3, 14=>4, 15=>4, 16=>4, 17=>4, 18=>5, 19=>5, 20=>5)
#    min_date_abs_ct_dict_pt2 = Dict{Int,Int}(-1=>1, 0=>1, 1=>1, 2=>2, 3=>2, 4=>2, 5=>3, 6=>3, 7=>3, 8=>3, 9=>4, 10=>4, 11=>4, 12=>4, 13=>4, 14=>4, 15=>5, 16=>5, 17=>5, 18=>5, 19=>5, 20=>5)   
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
    while true
        if pt2_region_analyzed == 1 || pt1_finished == 0
            date_and_AAlen_seqs = Set{Int}()
            all_muts_round_dict_pt2[pt2_round_ct] = Set{String}()
            cumulative_seq_round_set_pt2 = Set{Int}()
#           mp_pango_50_dates_pt2 = Vector{Int}()
#           pango_vec_pt2 = Vector{String}()
#           clade_vec_pt2 = Vector{String}()
#           cladepango_vec_pt2 = Vector{String}()
#           collection_date_index_vec_pt2 = Vector{Int}()
        end 
        coAAmut_ct_pt1 = Dict{String,Int}()               
        coAAmut_ct_pt2 = Dict{String,Int}()
        coAAmut_ct_pt1_unknown = Dict{String,Int}()
        coAAmut_ct_pt2_unknown = Dict{String,Int}()
        prop_dict_pt1 = Dict{String, Tuple{Float64,Float64,Int,Int,Int,Float64,Float64,Float64,Union{Float64,String},Float64} }()
        prop_dict_pt2 = Dict{String, Tuple{Float64,Float64,Int,Int,Int,Float64,Float64,Float64,Union{Float64,String},Float64} }()
        df_mp = DataFrame("SeedMut" => String[], "MP Tot Mut ct" => Int[], "MP Tot Reg ct" => Int[], "Reg Tot Mut ct" => Int[], "MP Region Num" => Int[], "Mutation" => String[], "MP Mut ct" => Float64[], "EPCI Mut ct" => Int[], "Adjusted MP seq ct" => Int[], "EPCI pct" => Float64[], "MP pct" => Float64[], "non MP pct" => Float64[], "Fishers Exact Test pval" => Float64[], "log10pvFISH" => Float64[], "Fold Incr" => Union{Float64,String}[], "Chi2" => Float64[], "EPCI HQCS Ratio" => Float64[], "avg NonRBD AAct" => Float64[], "avg nonRBD rel" => Float64[])
    ########### df_mp columns key #################
    #### Mutation = mutation being analyzed
    #### MP_Tot_Mut_ct = total qualifying sequences with given mutation
    #### EPCI_Mut_ct = total chronic seqs w/given mutation
    #### Adjusted_MP_seq_ct = number of qualifying seqs (with ≥ min_mut_ct_pt2 in mut pattern)
    #### EPCI_pct = % of all chronic seqs w/given AA mut that appear in qualifying seqs (e.g. if 10 seqs have E:T30I & 4 of those are in seqs meeting the criteria, EPCI_prop = 0.40, EPCI_pct = 40%)
    #### MP_pct = % of all qualifying chronic seqs that have a given mut (e.g. if 100 seqs meet criteria & 4 of those have E:T30I, MP_prop = 0.04, MP_pct = 4%)
    #### log10pvFISH = -log10 Fisher's exact test p-value
    #### Fold_Incr = fold-increase of mutation in MP compared to non-MP EPCI sequences
    #### Chi2 = chi-squared
    #### EPCI_HQCS_Ratio = adjusted ratio of mutation frequency in EPCI vs HQCS

        
        masked_muts = union(mp_masked_muts, all_excluded_muts)









        
        if pt1_round_ct == 0
            sel_muts_pt1_global = sel_muts_pt1
            pt1_seed_mut = join(sel_muts_pt1_global, ", ")
        end
        if pt1_finished == 0  
#####################################################################################################################
#### Debug required to prevent empty "groups" from appearing in sel_muts_pt1_global
            blank_indices = Int[]
            for i in 1:length(sel_muts_pt1_global)
                mutstring = sel_muts_pt1_global[i]
                if mutstring == ""
                    push!(blank_indices, i)
                end
            end
            temp_sel_muts_pt1_global_standin = String[]
            for i in 1:length(sel_muts_pt1_global)
                mutstring = sel_muts_pt1_global[i]
                if !(i in blank_indices)
                    push!(temp_sel_muts_pt1_global_standin, mutstring)
                end
            end
            sel_muts_pt1_global = temp_sel_muts_pt1_global_standin
#####################################################################################################################
#### Rules for minimum required mutation count (min_mut_ct) for mut patterns w/a given number of regions (region_ct)
            region_ct = length(sel_muts_pt1_global)
            min_mut_ct_dict_region_ct = region_ct
            for mutstring in sel_muts_pt1_global
                if isempty(mutstring) || mutstring == ""
                    
                    min_mut_ct_dict_region_ct -= 1
                end
            end
            min_mut_ct = min_mut_ct_dict_pt1[min_mut_ct_dict_region_ct]
            min_abs_mut_ct = min_abs_mut_ct_dict_pt1[min_mut_ct_dict_region_ct]
            min_date_raw_ct = min_date_abs_ct_dict_pt1[min_mut_ct_dict_region_ct]
#####################################################################################################################################
#### Splitting all mutation groups into individual mutations in order to determine the range for each region.
#### This is necessary because the qualifying criteria for mutations in a groups's range are different (less strict) than for mutations outside it. 
            sel_muts_pt1_ind = Vector{Vector{String}}()
            dict_ct = 1
            sel_muts_pt1_sort = sort(sel_muts_pt1_global, by = x -> sel_muts_pt1_sort_key_universal(x, sub_0__posonly_1))
            for mut_grp_str in sel_muts_pt1_sort
                if mut_grp_str ≠ ""
                    grp_muts = string.(split(mut_grp_str, ", "))
                    tmp_mut_vec = Vector{String}()
                    for mut in grp_muts
                        push!(tmp_mut_vec, mut)
                    end
                    tmp_mut_vec_sort = sort(tmp_mut_vec, by = x -> aa_pos_comprehensive_dict[x])
                    push!(sel_muts_pt1_ind, tmp_mut_vec_sort)
                end
            end
#####################################################################################################################################
#### This determines the start & end of a given mutation region's range (which extends plus_minus AA up & downstream its current limits)
#                                            mut_grp  gene   min  max
            sel_muts_pt1_min_max = Vector{Tuple{Int,String,Int,Int}}()








            for grp_num in 1:length(sel_muts_pt1_ind)
                
                
                mut_vec = sel_muts_pt1_ind[grp_num]
                AA_num_vec = Int[]
                gene = ""
                for mut in mut_vec
                    if mut ≠ ""
                        pos = aa_pos_comprehensive_dict[mut]
                        push!(AA_num_vec, pos)
                        gene = aa_gene_comprehensive_dict[mut]


                        


                        
                    end
                end
                if !isempty(AA_num_vec)





                    


                    
                    min_site = minimum(AA_num_vec) - plus_minus
                    max_site = maximum(AA_num_vec) + plus_minus
                    minmaxtup = (grp_num, gene, min_site, max_site)
                    push!(sel_muts_pt1_min_max, minmaxtup)
                    
                else
                    minmaxtup = (grp_num, "", 0, 0)
                    push!(sel_muts_pt1_min_max, minmaxtup)
                end
            end
#####################################################################################################################################
            total_mut_ct = 0
#            total_mut_ct_nonspike = 0
#            total_mut_ct_nonRBD = 0
#            total_mut_ct_spike_nonRBD = 0
#            total_mut_ct_nonRBM = 0
######################################################################################################################################
## coAAmut_ct_pt1 = number of sequences meeting specified criteria (i.e. that have ≥X AA muts from a given mutational pattern) that possess a given mut. 
#                                AA_mut   Count  
            coAAmut_ct_pt1 = Dict{String,Int}()                  
###################################
            mut_groups_ln = length(sel_muts_pt1_global)   ## Total # of mut regions searched for (= # of muts if all indicated individually, e.g. "E:16, E:30, E:31"=1 region, i.e. length=1; "E:16", "E:30", "E:31"=3 mutations, i.e. length=3)
###################################
#### Each mut_group is assigned a number, and the value for each group number is a vector containing all the mutations in that group.
#### If every mutation is listed as a separate string in the `sel_muts_pt1_global` parameter, then each mutation is its own mut_group.
            mut_groups = Dict{Int, Vector{String}}(i => Vector{String}() for i in 1:mut_groups_ln)  ## 
######################################################################################################################################
### sel_muts_pt1_global = selected mutations/mut groups. If muts are listed individually (e.g. "E:F4L", "E:V5I", "E:I9T", etc) then each mutation
#      is a "group." If all the mutations in a given region are grouped together (e.g. "E:F4L, E:V5I, E:I9T") each group is a group.
#      only the "grp" optio is currently used. May just dump the "ind" version at some point. 
            mut_grp_ct = 0
            for mut_grp in sel_muts_pt1_global
                if mut_grp ≠ ""
                    mut_grp_ct += 1
                    muts = split(mut_grp, ", ")
                    for m in muts
                        str = string(m)
                        push!(mut_groups[mut_grp_ct], str)
                    end
                end
            end
######################################################################################################################################
            MP_seqs = Set{Int}()     ## MP_seqs = all seqs that meet criteria
#           MP_seqs_pangos = Dict{String,Int}()  ## MP_seqs_pangos = Count for Pango lineages for all seqs that meet criteria
#           MP_seqs_inherited = Dict{String,Int}()  ## MP_seqs_inherited = Counts for inherited mutations among sequences on list. 
#            MP_seqs_priv_AA_adjustment_factors = Set{Float64}()  ## factors used to adjust for # of priv AA muts in a sequence
######################################################################################################################################
#### masked_muts are masked mutations. They are excluded from being included in the mutational pattern (see above for more info).
#### MP_seqs is a set containing all seqs that meet qualifying criteria (e.g. that have ≥ min_mut_ct muts in sel_muts_pt1_global)
            if ind_or_grp == "grp"
                for seq in 1:length(fake_seq_AA_muts)
                    seq_factor = 0
#####################################################
                    seq_priv_AA_mut_total = length(fake_seq_AA_muts_not_excluded2[seq])
                    seq_factor = clamp(0.6, (avg_AA_sub_ct_per_chronic_seq_for_main_fx/seq_priv_AA_mut_total), 1.5)
                    half_seq_factor = (seq_factor+1)/2
                    if seq_factor_OFF_HALF_ON__0_1_2 == 0
                        seq_factor = 1
                    end
                    if seq_factor_OFF_HALF_ON__0_1_2 == 1
                        seq_factor = half_seq_factor
                    end
#####################################################
                    seq_grp_mut_ct = 0
                    abs_grp_mut_ct = 0
                    for (group, muts) in mut_groups
                        group_ct = 0
                        for mut in muts
                            if mut in fake_seq_AA_muts_not_excluded2[seq] # && !(mut in masked_muts)
                                group_ct += 1
                            end
                        end
                        if group_ct > 0
                            seq_grp_mut_ct += 1*seq_factor
                            abs_grp_mut_ct += 1
                        end
                    end
#                    if pt1_seed_mut in seedmuts2chk
#                        muts2chk_intersect = intersect(muts2chk, seq_AA_muts[seq])
#                        if length(muts2chk_intersect) ≥ 2
#                            for mut_vec in sel_muts_pt1_ind
#                                mut_vec_string = join(mut_vec, "|")
#                                println("     $(mut_vec_string)")
#                            end
#                            for mut in muts2chk_intersect
#                                seqpad = rpad(seq, 16)
#                                mutpad = rpad(mut, 12)
#                                seedmutpad = rpad(pt1_seed_mut, 12)
#                                seq_grp_mut_ct_rd = @sprintf("%.2f", round(digits=2,seq_grp_mut_ct))
#                                seq_grp_mut_ct_rd_pad = rpad(seq_grp_mut_ct_rd, 2)
#                                min_mut_ct_rd = round(digits=2, min_mut_ct)
#                                min_mut_ct_rd_pad = lpad(min_mut_ct_rd, 4)
#                                println("Pt1,Rd$(pt1_round_ct)|seed=$(seedmutpad)|$(seqpad)|$(mutpad)|Grps=$(region_ct)|seqGrpCt=$(seq_grp_mut_ct_rd_pad)|MinMutCt=$(min_mut_ct_rd_pad)|AbsMutCt=$(abs_grp_mut_ct)|min_abs_mut_ct=$(min_abs_mut_ct)")
#                            end
#                        end
#                    end
            ##### the '-0.00001' part here is because, for reasons I don't understand, min_mut_ct_pt2 is often 0.000000000000001 larger than what it should be.
                    if seq_grp_mut_ct ≥ (min_mut_ct - 0.00001) && abs_grp_mut_ct ≥ min_abs_mut_ct
                        push!(MP_seqs, seq)
                    end
                end
######################################################################################################
            elseif ind_or_grp == "ind"
                for seq in 1:length(fake_seq_AA_muts)
                    seq_factor = 0
#####################################################
                    seq_priv_AA_mut_total = length(fake_seq_AA_muts_not_excluded2[seq])
                    seq_factor = clamp(0.6, (avg_AA_sub_ct_per_chronic_seq/seq_priv_AA_mut_total), 1.5)
                    half_seq_factor = (seq_factor+1)/2
                    if seq_factor_OFF_HALF_ON__0_1_2 == 0
                        seq_factor = 1
                    end
                    if seq_factor_OFF_HALF_ON__0_1_2 == 1
                        seq_factor = half_seq_factor
                    end
#####################################################
                    seq_mut_ct = 0
                    abs_mut_ct = 0
                    for (group, muts) in mut_groups
                        for mut in muts
                            if mut in fake_seq_AA_muts_not_excluded2[seq] # && !(mut in masked_muts)
                                seq_mut_ct +=1*seq_factor
                                abs_mut_ct += 1
                            end
                        end
                    end
            ##### the '-0.00001' part here is because, for reasons I don't understand, min_mut_ct is often 0.000000000000001 larger than what it should be.
                    if seq_mut_ct ≥ (min_mut_ct - 0.00001) && abs_mut_ct ≥ min_abs_mut_ct
                        push!(MP_seqs, seq)
                    end
                end
            end
            pt1_qual_seq_number = length(MP_seqs)
#           if notrandom0__random1 == 0
#               print("MP_seqs pt1 = $(pt1_qual_seq_number) sequences | ")
#           end
###############################################################################################################################
            for seq in MP_seqs
                total_mut_ct += length(fake_seq_AA_muts_not_excluded2[seq])
#### For each qualifying sequence, all of its private mutations are tallied in the coAAmut_ct_pt1 dictionary (keys = mutations, values = counts)
                for mut in fake_seq_AA_muts_not_excluded2[seq]
                    coAAmut_ct_pt1[mut] = get(coAAmut_ct_pt1, mut, 0) + 1
                end
            end
##########################################################################################################################################
#### This counts the pango lineages for each sequence on the list, then counts all the mutations inherited by that pango lineage.
#### Similar to the "unknown" dicts below, the purpose here is to remove from the denominator sequences that could not possibly have had a given 
####    private mutation (because you can't acquire a mutation you already have). For example, let's say we're testing whether the private mutation 
####    S:H655Y correlates with other mutations in a list of 100 sequences. If 70 of thoser sequences are Omicron, all of which inherited S:H655Y,
####    then only the 30 non-Omicron sequences will be used in calculating the Fisher's exact test (as well as all other calculations).
#           for seq in MP_seqs
#               pango = seq_pango[seq]
#               MP_seqs_pangos[pango] = get(MP_seqs_pangos, pango, 0) + 1
#               if !haskey(pango_AAsub_WT_universal, pango)
#                   pango = pango_predecessor_meta_dict[pango][1]
#                   if !haskey(pango_AAsub_WT_universal, pango)
#                       pango = pango_predecessor_meta_dict[pango][1]
#                   end
#               end
#               for mut in pango_AAsub_WT_universal[pango]
#                   mutpo = aa_gene_and_pos_comprehensive_dict[mut]
#                   if !(mutpo in seq_unknown_AA[seq])
#                       MP_seqs_inherited[mut] = get(MP_seqs_inherited, mut, 0) + 1
#                   end
#               end
#               for del in pango_AAdel_WT[pango]
#                   delpo = aa_gene_and_pos_comprehensive_dict[del]
#                   if !(delpo in seq_unknown_AA[seq])
#                       MP_seqs_inherited[del] = get(MP_seqs_inherited, del, 0) + 1
#                   end
#               end     
#           end
######################################################################################################################################
#### coAAmut_ct_pt1_unknown counts the # of times a given AA position is "unknown," either due to dropout or a mixed nucleotide for the sequences that qualify
#                                 unknown_AA_pos | count
#           coAAmut_ct_pt1_unknown = Dict{String,Int}()
##############################################################################################################
### MP_seqs_unknown_region_ct tallies the number of mutational regions with at least one unknown AA
#                                                    EPI_ISL | total_regions_with_unknown_AA_count_in_seq
#            MP_seqs_unknown_region_ct = Dict{String,Int}()
#            total_unknown_region_ct = 0
##########################################################
#           for seq in MP_seqs
#                MP_seqs_unknown_region_ct[seq] = 0
#               for region_num in 1:length(sel_muts_pt1_ind)
#                   region_unkn_ct = 0
#                   mut_vec = sel_muts_pt1_ind[region_num]
#                   for mut in mut_vec
#                       if mut ≠ ""
#                           mutpo = aa_gene_and_pos_comprehensive_dict[mut]
#                           if mutpo in seq_unknown_AA[seq]
#                                region_unkn_ct += 1
#                               coAAmut_ct_pt1_unknown[mutpo] = get(coAAmut_ct_pt1_unknown, mutpo, 0) + 1
#                           end
#                       end
#                   end
#                    if region_unkn_ct > 0
#                        MP_seqs_unknown_region_ct[seq] += 1
#                        total_unknown_region_ct += 1
#                    end
#               end
#           end
###################################################################################################################################### 
            MP_seq_ct = length(MP_seqs)
            avg_mutations_per_seq_no_dels = round(digits=2, total_mut_ct/MP_seq_ct)
            ratio_of_AAmuts_in_Grp_vs_AAmuts_in_all_chronics = avg_mutations_per_seq_no_dels/avg_AA_sub_ct_per_chronic_seq
            ratio_of_AAmuts_in_Grp_vs_AAmuts_in_all_chronics_rd = round(digits = 2, ratio_of_AAmuts_in_Grp_vs_AAmuts_in_all_chronics)
            MP_seqs_sort = sort(collect(MP_seqs), by = x -> (length(x), x) )
######################################################################################################################################
#### for prop_dict_pt1, keys = mutations; values = tuple of (EPCI_pct_rd, MP_pct_rd, ct, chi_squared, pv_fish, log_pv_fish)
#### See below for meanings of each part of this tuple
            prop_dict_pt1 = Dict{String, Tuple{Float64,Float64,Int,Int,Int,Float64,Float64,Float64,Union{Float64,String},Float64} }()
#### prop_dict_pt1_unknown: keys = mutations, values = percentage of qualifying sequences that have an unknown AA at that position
#            prop_dict_pt1_unknown = Dict{String,Float64}()
###### EPCI_prop = proportion of all chronic seqs with a given AA mut that appear in seqs meeting the criteria (e.g. if 10 seqs have E:T30I & 4 of those are in seqs meeting the criteria, EPCI_prop = 0.40)
###### MP_prop = proportion of all chronic seqs that meet the criteria that have a given AA mut (e.g. if 100 seqs meet the criteria & 4 of those have E:T30I, MP_prop = 0.04)
###### EPCI_pct_rd = same as EPCI prop but in % form and rounded to 1 decimal place
###### MP_pct_rd = same as MP prop but in % form and rounded to 1 decimal place
###### mut_prop_of_EPCI = proportion of all chronic seqs that possess given AA mut
###### MP_prop_of_EPCI = proportion of all chronic seqs that meet criteria
###### fold_inc = fold-change where 1.0 = expected proportion, i.e. MP_prop_of_EPCI = mut_prop_of_EPCI, or (equivalently), EPCI_pct = MP_pct
######################################################################################################################################
######################################################################################################################################
#           for (mut, ct) in coAAmut_ct_pt1_unknown
#               MP_pct_unk = ct/MP_seq_ct
#               prop_dict_pt1_unknown[mut] = MP_pct_unk
#           end
            total_EPCI_seq_ct = length(EPCI_set)
######################################################################################################################################
            for (mut, co_mut_count) in coAAmut_ct_pt1
                mut_po = aa_gene_and_pos_comprehensive_dict[mut]
#               unknown = get(coAAmut_ct_pt1_unknown, mut_po, 0)                                           ### unknown = total number of qualifying sequences w/an unknown AA at the mut_po position
#               total_unknown = get(total_AApos_ct_unknown, mut_po, 0)                                     ### total_unknown = total number of sequences in entire EPCI list w/an unknown AA at the mut_po position
                inherited = 0
#               all_chr_inherited = 0
#               if sub_0__posonly_1 == 0
#                   inherited = get(MP_seqs_inherited, mut, 0)
#                   all_chr_inherited = get(all_chr_seqs_inherited, mut, 0)
#               end
#### This section accounts for inherited muts that make the mut being analyzed impossible. E.g., all BA.2 & XBB have ORF1a:L3201F.
#### This means it is impossible for any BA.2/XBB sequence to have ORF1a:L3201P, so they must be removed from the denominator.
#### The whole pango part is to prevent double-counting muts that are both inherited and either lack coverage or have mixed nuc muts there (i.e. are "unknown")
#               if sub_0__posonly_1 == 0
#                   for seq in MP_seqs
#                       pango = seq_pango[seq]
#                       if mut_po in pango_AAsub_WT_pos_only[pango] && !(mut_po in seq_unknown_AA[seq])
#                           if !haskey(pango_AAsub_WT_universal, pango)
#                               pango = pango_predecessor_meta_dict[pango][1]
#                               if !haskey(pango_AAsub_WT_universal, pango)
#                                   pango = pango_predecessor_meta_dict[pango][1]
#                               end
#                           end
#                           for inherited_mut in pango_AAsub_WT_universal[pango]
#                               if aa_gene_and_pos_comprehensive_dict[inherited_mut] == mut_po && inherited_mut ≠ mut && qryAA_comprehensive_dict[inherited_mut] ≠ refAA_comprehensive_dict[mut]
#                                   inherited += 1
#                               end
#                           end
#                       end
#                   end
#               end
                adjusted_MP_seq_ct = MP_seq_ct #- unknown                                              ### MP_seq_ct = All qualifying seqs (which have ≥ min_mut_ct muts from current mut pattern list)
                adjusted_EPCI_seq_ct = total_EPCI_seq_ct #- total_unknown                              ### total_EPCI_seq_ct = total # of independent chronic seqs in entire list
                EPCI_mut_ct = AA_muts_ct_no_dels_universal[mut]                                        ### AA_muts_ct_no_dels_universal[mut] = total AA mut ct in all independent chronic seqs/lineages
                EPCI_prop = co_mut_count/AA_muts_ct_no_dels_universal[mut]                             ### EPCI_prop = proportion of the total EPCI count of a mut that appears in the mp sequence list
                MP_prop = co_mut_count/adjusted_MP_seq_ct                                              ### MP_prop = proportion of all qualifying MP sequences that have the mut as a private mutation
                non_MP_prop = (EPCI_mut_ct - co_mut_count)/(adjusted_EPCI_seq_ct - adjusted_MP_seq_ct) ### non_MP_prop = proportion of all non-MP EPCI sequences that have the mut as a private mutation 
                EPCI_pct = 100*EPCI_prop                                         
                MP_pct = 100*MP_prop
                non_MP_pct = 100*non_MP_prop                                       
                mut_prop_of_EPCI = AA_muts_ct_no_dels_universal[mut]/adjusted_EPCI_seq_ct                  
                MP_prop_of_EPCI = adjusted_MP_seq_ct/adjusted_EPCI_seq_ct
                fold_incr::Union{Float64,String} = 0.0
                if non_MP_prop ≠ 0
                    fold_incr = MP_prop/non_MP_prop
                elseif non_MP_prop == 0
                    fake_non_MP_prop = 1/(adjusted_EPCI_seq_ct - adjusted_MP_seq_ct)
                    fake_fold_incr = MP_prop/fake_non_MP_prop
                    fake_fold_incr_rd = round(digits=2, fake_fold_incr)
                    fold_incr = ">$(fake_fold_incr_rd)"
                end
#                fold_incr_rd = round(digits=2, fold_incr)
                if fold_incr == ">Inf"
                    println("For $(mut), fold_incr = $(fold_incr) [should show >Inf]")
                end
                observed = co_mut_count
                expected = mut_prop_of_EPCI*adjusted_MP_seq_ct
                if observed ≥ expected
#            diff = observed-expected; chi_squared = diff^2/expected; chi_squared_rd = round(digits=1, chi_squared)
                    fishexact_up_left = co_mut_count                                                                                           ### seq with mut + meet search criteria (i.e. ≥ min_mut_ct muts from current mut pattern list)
                    fishexact_up_rt = max(adjusted_MP_seq_ct - co_mut_count, 0)                                                                ### seq without mut + meet search criteria (i.e. ≥ min_mut_ct muts from current mut pattern list)
                    fishexact_down_left = max(AA_muts_ct_no_dels_universal[mut] - co_mut_count, 0)                                             ### seq with mut + does not meet search criteria (i.e. < min_mut_ct muts from current mut pattern list)
                    fishexact_down_rt = max(adjusted_EPCI_seq_ct - AA_muts_ct_no_dels_universal[mut] - adjusted_MP_seq_ct + co_mut_count, 0)   ### seq w/o mut + does not meet search criteria (i.e. < min_mut_ct muts from current mut pattern list)
                    fish = FisherExactTest(fishexact_up_left, fishexact_up_rt, fishexact_down_left, fishexact_down_rt)
                    pv_fish = pvalue(fish)
                    log_pv_fish = -log10(pv_fish)
################# Chi-squared formula —— I got this straight from Claude.ai which says I was doing it all wrong before. No perceivable difference in the results though.
                    a = fishexact_up_left; b = fishexact_up_rt; c = fishexact_down_left; d = fishexact_down_rt
                    n = a + b + c + d    # total sample size
                    chi_2_top = n*(a*d - b*c)^2
                    chi_2_bottom = (a+b)*(c+d)*(a+c)*(b+d)
                    chi_squared = 0.0
                    chi_squared_rd = 0.0
                    if chi_2_bottom > 0
                        chi_squared = chi_2_top/chi_2_bottom
                        expected_a = (a + b) * (a + c) / n
                        expected_b = (a + b) * (b + d) / n
                        expected_c = (c + d) * (a + c) / n
                        expected_d = (c + d) * (b + d) / n
                        if min(expected_a, expected_b, expected_c, expected_d) < 5   # Yates' continuity correction (another Claude command)
                            corrected_numerator = n * (abs(a * d - b * c) - n/2)^2
                            chi_squared_yates = corrected_numerator/chi_2_bottom
                            chi_squared = chi_squared_yates
                            chi_squared_rd = round(digits=1, chi_squared)
                        end
#                        chi_squared_pvalue = 1 - cdf(Chisq(1), chi_squared)
                    end
################# End Claude's Chi-squared formula calculations ################
#                                1       2          3            4                5                6          7          8           9          10
                    props = (EPCI_pct, MP_pct, EPCI_mut_ct, co_mut_count, adjusted_MP_seq_ct, chi_squared, pv_fish, log_pv_fish, fold_incr, non_MP_prop)
                    prop_dict_pt1[mut] = props
#                    if pt1_seed_mut in seedmuts2chk && mut in muts2chk
#                        log_pv_fish_rd = round(digits=2, log_pv_fish)
#                        fold_incr_rd::Union{String, Float64} = ""
#                        if (fold_incr isa String)
#                            fold_incr_rd = fold_incr
#                        else
#                            fold_incr_rd = round(digits=2, fold_incr)
#                        end
#                        seedmutpad = rpad(pt1_seed_mut, 12)
#                        mutpad = rpad(mut, 12)
#                        co_mut_count_pad = lpad(co_mut_count, 3)
#                        log_pv_fish_rd_pad = lpad(log_pv_fish_rd, 5)
#                        fold_incr_rd_pad = lpad(fold_incr_rd, 6)
#                        println("Pt1,Rd$(pt1_round_ct)|Grps:$(region_ct)|$(mutpad)|seed=$(seedmutpad)|co_mut_ct=$(co_mut_count_pad)|logpvfish=$(log_pv_fish_rd_pad)|fold_incr=$(fold_incr_rd_pad)")
#                    end
                end
            end














            
#####################################################################################################################################
##### sel_muts2_pt1_sort is the updated, sorted set of mutation regions to be included in the next round. 
##### sel_muts2_pt1 is the unsorted list, which is later sorted to make sel_muts2_pt1_sort
            sel_muts2_pt1 = Vector{String}() 
            sel_muts2_pt1_sort = Vector{String}()                                  
#### sel_muts2_pt1_pre_dict = Each key represents a mutation group; values are mut grp vectors (later joined to become strings in sel_muts2_pt1)                      
            sel_muts2_pt1_pre_dict = Dict{Int, Vector{String}}()  
#### Two sets below created to distinguish between mutations within one of the mut_grp ranges and those not within an existing group
####    Mutations within an existing group are subjected to a less stringent qualification requirement.
            prop_dict_pt1_muts_set = Set{String}()
            MP_prop_dict_pt1_muts_set = Set{String}()
            for mut in keys(prop_dict_pt1)
                push!(prop_dict_pt1_muts_set, mut)
            end
            for (mut, props) in prop_dict_pt1
                mut_count = props[4]
                log_pv_fish = props[8]
                mut_gene = aa_gene_comprehensive_dict[mut]
                mut_poz = aa_pos_comprehensive_dict[mut]
#               grp_join_ct = 0
                for grp_num in 1:length(sel_muts_pt1_min_max)
#                   if grp_join_ct == 0
                        gene = sel_muts_pt1_min_max[grp_num][2]
                        min_site = sel_muts_pt1_min_max[grp_num][3]
                        max_site = sel_muts_pt1_min_max[grp_num][4]
                        if mut_gene == gene && mut_poz ≥ min_site && mut_poz ≤ max_site
                            push!(MP_prop_dict_pt1_muts_set, mut)
                            if log_pv_fish ≥ min_grp_fish && mut_count ≥ min_match_ct  
                                if haskey(sel_muts2_pt1_pre_dict, grp_num)
                                    push!(sel_muts2_pt1_pre_dict[grp_num], mut)
                                else
                                    sel_muts2_pt1_pre_dict[grp_num] = Vector{String}()
                                    push!(sel_muts2_pt1_pre_dict[grp_num], mut)
                                end
                            end
#                           grp_join_ct += 1
#                       end
                    end
                end
            end
#### non_MP_prop_dict_pt1_muts_set are the muts not within or near one of the existing mut regions (which must meet stricter stat criteria)
            non_MP_prop_dict_pt1_muts_set = setdiff(prop_dict_pt1_muts_set, MP_prop_dict_pt1_muts_set)
            new_region_ct = 0
            maxFish = 0
            winner = ""
            for (mut, props) in prop_dict_pt1
                if mut in non_MP_prop_dict_pt1_muts_set
                    log_pv_fish = props[8]
                    mut_count   = props[4]
                    if log_pv_fish ≥ min_log_pv_fish && mut_count ≥ min_match_ct 
                        if log_pv_fish > maxFish
                            winner = mut
                            maxFish = log_pv_fish
                        end
                    end
                end
            end
            if winner ≠ ""                   















                
                grp_num = length(keys(sel_muts2_pt1_pre_dict)) + 1
                sel_muts2_pt1_pre_dict[grp_num] = Vector{String}()
                push!(sel_muts2_pt1_pre_dict[grp_num], winner)
                new_region_ct += 1
#                        if mut == "ORF7a:E22D" && "ORF7a:T39I" in sel_muts_pt1_global || mut == "ORF7a:T39I" && "ORF7a:E22D" in sel_muts_pt1_global
#                            mut_gene = aa_gene_comprehensive_dict[mut]
#                            mut_pos = aa_pos_comprehensive_dict[mut]
#                            grp_num = length(keys(sel_muts2_pt1_pre_dict))
#                            sel_muts2_pt1_pre_dict[grp_num] = Vector{String}()
#                            push!(sel_muts2_pt1_pre_dict[grp_num], mut)
#                            new_region_ct += 1
#                        end
#                        if mut == "ORF7a:E22D" && "ORF7a:F59I" in sel_muts_pt1_global || mut == "ORF7a:F59I" && "ORF7a:E22D" in sel_muts_pt1_global
#                            mut_gene = aa_gene_comprehensive_dict[mut]
#                            mut_pos = aa_pos_comprehensive_dict[mut]
#                            grp_num = length(keys(sel_muts2_pt1_pre_dict))
#                            sel_muts2_pt1_pre_dict[grp_num] = Vector{String}()
#                            push!(sel_muts2_pt1_pre_dict[grp_num], mut)
#                            new_region_ct += 1
#                        end
#                        if mut == "ORF7a:T39I" && "ORF7a:F59I" in sel_muts_pt1_global || mut == "ORF7a:F59I" && "ORF7a:T39I" in sel_muts_pt1_global
#                            mut_gene = aa_gene_comprehensive_dict[mut]
#                            mut_pos = aa_pos_comprehensive_dict[mut]
#                            grp_num = length(keys(sel_muts2_pt1_pre_dict))
#                            sel_muts2_pt1_pre_dict[grp_num] = Vector{String}()
#                            push!(sel_muts2_pt1_pre_dict[grp_num], mut)
#                            new_region_ct += 1
#                        end
#                        if mut == "ORF7a:T39I" && "ORF7a:N43K" in sel_muts_pt1_global || mut == "ORF7a:N43K" && "ORF7a:T39I" in sel_muts_pt1_global
#                            mut_gene = aa_gene_comprehensive_dict[mut]
#                            mut_pos = aa_pos_comprehensive_dict[mut]
#                            grp_num = length(keys(sel_muts2_pt1_pre_dict))
#                            sel_muts2_pt1_pre_dict[grp_num] = Vector{String}()
#                            push!(sel_muts2_pt1_pre_dict[grp_num], mut)
#                            new_region_ct += 1
#                        end
#                    end
#                end
            end
#### Sorting muts within each mut group so they're easier to make sense of
            for (grp_num, mut_vec) in sel_muts2_pt1_pre_dict
                mut_vec_sort = sort(mut_vec, by = x -> (aa_gene_comprehensive_dict[x], aa_pos_comprehensive_dict[x]))
                sel_muts2_pt1_pre_dict[grp_num] = mut_vec_sort
            end
#### Turning mut groups into strings so they can be used in the next round of the function
#### Also excluding all groups that consist only of spike muts, if that parameter (purespikebanned_0__purespikeallowed_1) is enabled
            grp_1stmut_set = Set{String}()
            for (grp_num, mut_vec) in sel_muts2_pt1_pre_dict
                push!(grp_1stmut_set, mut_vec[1])
            end
            for (grp_num, mut_vec) in sel_muts2_pt1_pre_dict
                if purespikebanned_0__purespikeallowed_1 == 1
                    sel_mut_str = join(mut_vec, ", ")
                    push!(sel_muts2_pt1, sel_mut_str)
                elseif purespikebanned_0__purespikeallowed_1 == 0
                    if !all(aa_gene_comprehensive_dict[mut] == "S" for mut in grp_1stmut_set)
                        sel_mut_str = join(mut_vec, ", ")
                        push!(sel_muts2_pt1, sel_mut_str)
                    end
                end     
            end
#####################################################################################################################################
            sel_muts_pt1_sort = sort(sel_muts_pt1_global, by = x -> sel_muts_pt1_sort_key_universal(x, sub_0__posonly_1))
            sel_muts2_pt1_sort = sort(sel_muts2_pt1, by = x -> sel_muts_pt1_sort_key_universal(x, sub_0__posonly_1))
            pt1_round_ct += 1
            sel_muts_pt1_round_dict[pt1_round_ct] = sel_muts2_pt1_sort
#####################################################################################################################################
#### If there are no changes from previous round, we're finished & everything is sorted & printed. Otherwise, the whole thing runs again with the new mutation set. 
            sel_muts_pt1_set = Set{String}()
            sel_muts2_pt1_set = Set{String}()
            for m in sel_muts_pt1_global
                push!(sel_muts_pt1_set, m)
            end
            for m in sel_muts2_pt1
                push!(sel_muts2_pt1_set, m)
            end
            sel_muts_pt1_global = sel_muts2_pt1_sort
            sel_muts_pt2 = sel_muts_pt1_round_dict[pt1_round_ct] ### already sorted!!
            if sel_muts2_pt1_set ≠ sel_muts_pt1_set && pt1_round_ct < max_pt1_rounds
                continue
            elseif pt1_round_ct < 3
                continue
            else
##############################################################################################################
                pt1_finished += 1
##############################################################################################################

























                ### Checking to make sure there are actually two regions before doing part 2 (Claude.ai suggestion)
                blank_group_ct = 0
                for i in 1:length(sel_muts_pt2)
                    mutstring = sel_muts_pt2[i]
                    blank_mutstrings = Set(["", " ", "  ", "   ", "    ", "     ", "      ", "       ", "        ", "         ", "          ", "           ", "            "])
                    if mutstring in blank_mutstrings || isempty(mutstring)
                        blank_group_ct += 1
                    end
                end
                real_region_ct = length(sel_muts_pt2) - blank_group_ct
                if real_region_ct < 2
                    return (Dict{Int, Dict{String, Tuple{Float64,Float64,Int,Int,Int,Float64,Float64,Float64,Union{Float64,String},Float64}}}(), "", 0, "00-00-00", Pair{String,Int64}[])
                end
##############################################################################################################
                all_muts_set_pt2 = Set{String}()
                sel_muts_pt2_ind_dict = Dict{Int, Vector{String}}()
                for i in 1:length(sel_muts_pt2)
                    grp_str = sel_muts_pt2[i]
                    grp_muts = string.(split(grp_str, ", "))
                    tmp_mvec = String[]
                    for mut in grp_muts
                        push!(tmp_mvec, mut)
                        push!(all_muts_set_pt2, mut)
                    end
                    tmp_mvec_sort = sort(tmp_mvec, by = x -> aa_pos_comprehensive_dict[x])
                    sel_muts_pt2_ind_dict[i] = tmp_mvec_sort
                end
                if !(pt1_seed_mut in all_muts_set_pt2)
#                    break
                    return (Dict{Int, Dict{String, Tuple{Float64,Float64,Int,Int,Int,Float64,Float64,Float64,Union{Float64,String},Float64}}}(), "", 0, "00-00-00", Pair{String,Int64}[])
                end
                continue
            end
        end
        if pt1_finished > 0 && !isempty(sel_muts_pt2) && sel_muts_pt2 ≠ [""]
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
####################################################################   PART 2   ###########################################################################################
####################################################################   PART 2   ###########################################################################################
####################################################################   PART 2   ###########################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
            blank_group_ct = 0
############## Once again getting rid of any blank or empty groups.
            blank_mutstrings = Set(["", " ", "  ", "   ", "    ", "     ", "      ", "       ", "        ", "         ", "          ", "           ", "            "])
            for i in 1:length(sel_muts_pt2)
                mutstring = sel_muts_pt2[i]
                if mutstring in blank_mutstrings || isempty(mutstring)

                    
                    blank_group_ct += 1
                end
            end
            
            
            
            
            
            
            
            
            
            
            
            
            
            
            
            
            
            
            
            
            
            
            
            
            
            
            
            
            
            
            
            
            
            
            
            
            
            
            
            region_ct2 = length(sel_muts_pt2) - blank_group_ct
            min_mut_ct_pt2 = 0.1
            min_mut_ct_pt2 = min_mut_ct_dict_pt2[region_ct2-1]
            min_abs_mut_ct = min_abs_mut_ct_dict_pt2[region_ct2-1]
            min_date_raw_ct = min_date_abs_ct_dict_pt2[region_ct2-1]
####################################################################################################################################
#### During each iteration, we are only testing the region that is excluded when selecting the sequences used to test correlations.
####################################################################################################################################
#### Splitting all mutation groups into individual mutations in order to determine the range for each region.
#### Also making all_muts_set_pt2 to update all_muts_round_dict_pt2 with (if we've just finished a full round).
####   This is necessary because the qualifying criteria for mutations in a groups's range are different than for mutations outside it.
####   Has to be done before removing the region being analyzed so that the favorable criteria apply to that region.                                       
####################################################################################################################################
### This is to determine the start & end of a given mutation region's range (which extends 6 AA up & downstream its current limits)
####################################################################################################################################
#                                     mut_grp  gene   min  max
            sel_muts_pt2_min_max = Tuple{Int,String,Int,Int}
            mut_vec = sel_muts_pt2_ind_dict[pt2_region_analyzed]
            AA_num_vec = Int[]
            gene = ""
            for mut in mut_vec
                if mut ≠ ""
                    pos = aa_pos_comprehensive_dict[mut]
                    push!(AA_num_vec, pos)
                    gene = aa_gene_comprehensive_dict[mut]
                end
            end
            min_site = 1
            max_site = 1
            if !isempty(AA_num_vec)
                min_site = minimum(AA_num_vec) - plus_minus
                max_site = maximum(AA_num_vec) + plus_minus
                sel_muts_pt2_min_max = (pt2_region_analyzed, gene, min_site, max_site)
            else
                sel_muts_pt2_min_max = (pt2_region_analyzed, gene, min_site, max_site)
#                push!(sel_muts_pt2_min_max, (pt2_region_analyzed, gene, min_site, max_site))
            end
############################################ End New Version (2025-11-1) #####################################################
#####################################################################################################################################
            total_mut_ct_pt2 = 0             ## Total mutations in all *counted* (i.e. qualifying) seqs
## coAAmut_ct_pt2 = number of seqs meeting specified criteria (i.e. that have ≥min_mut_ct AA muts from a given mutational pattern) that possess a given mut. 
#                                 AA_mut   Count  
            coAAmut_ct_pt2 = Dict{String,Int}()                  
######################################################################################################################################
            MP_seqs_pt2 = Set{Int}()     ## MP_seqs_pt2 = all seqs that meet criteria
#           MP_seqs_pt2_pangos = Dict{String,Int}()  ## MP_seqs_pangos = Count for Pango lineages for all seqs that meet criteria
#           MP_seqs_pt2_inherited = Dict{String,Int}()  ## MP_seqs_inherited = Counts for inherited mutations among sequences on list.
#            MP_seqs_pt2_priv_AA_adjustment_factors = Set{Float64}()  ## factors used to adjust for # of priv AA muts in a sequence
######################################################################################################################################
#### all_unique_chr_seqs combines all rep_seq_grps_maxmut_seqs and non_rep_seqs (both explained below). 
#### rep_seq_grps_maxmut_seqs is a dictionary containing one sequence that represents each group of related chronic sequences. 
#### Each representative sequence is the one with the most private AA mutations in the group. Its keys are the group's numbers 
####   (which are arbitrary) and its values are the EPI_ISL number for the sequence.
#### non_rep_seqs are chronic singlets (i.e. seqs with no closely related sequences from the same patient)
#### MP_seqs_pt2 is a set containing all sequences that meet the qualifying criteria (e.g. that have ≥2 mutations in sel_muts_pt2)
#### masked_muts are excluded from inclusion in the mutational pattern. (See above for explanation & justification.)
            if ind_or_grp == "grp"
                for seq in 1:length(fake_seq_AA_muts)
                    seq_factor = 0
#####################################################
#                    if nonspike_seqfactor__0off__1RBM__2RBD__3SpikeNonRBD__4AllSpike == 0
                    seq_priv_AA_mut_total = length(fake_seq_AA_muts_not_excluded2[seq])
                    seq_factor = clamp(0.6, (avg_AA_sub_ct_per_chronic_seq_for_main_fx/seq_priv_AA_mut_total), 1.5)
                    half_seq_factor = (seq_factor+1)/2
                    if seq_factor_OFF_HALF_ON__0_1_2 == 0
                        seq_factor = 1
                    end
                    if seq_factor_OFF_HALF_ON__0_1_2 == 1
                        seq_factor = half_seq_factor
                    end
#####################################################
                    seq_grp_mut_ct = 0
                    abs_grp_mut_ct = 0
                    for grp_num in 1:length(sel_muts_pt2_ind_dict)
                        if !(grp_num == pt2_region_analyzed)
                            mut_vec = sel_muts_pt2_ind_dict[grp_num]
                            group_ct = 0
                            for mut in mut_vec
                                if mut in fake_seq_AA_muts_not_excluded2[seq] # && !(mut in masked_muts)
                                    group_ct += 1
                                end
                            end
                            if group_ct > 0
                                seq_grp_mut_ct += 1*seq_factor
                                abs_grp_mut_ct += 1
                            end
                        end
                    end
#                    if pt1_seed_mut in seedmuts2chk
#                        muts2chk_intersect = intersect(muts2chk, seq_AA_muts[seq])
#                        if length(muts2chk_intersect) ≥ 2
#                            for grp_num in 1:length(sel_muts_pt2_ind_dict)
#                                mut_vec = sel_muts_pt2_ind_dict[grp_num]
#                                mut_vec_string = join(mut_vec, "|")
#                                println("     Grp$(grp_num):$(mut_vec_string)")
#                            end
#                            for mut in muts2chk_intersect
#                                seqpad = rpad(seq, 16)
#                                mutpad = rpad(mut, 12)
#                                seedmutpad = rpad(pt1_seed_mut, 12)
#                                seq_grp_mut_ct_rd = @sprintf("%.2f", round(digits=2,seq_grp_mut_ct))
#                                seq_grp_mut_ct_rd_pad = rpad(seq_grp_mut_ct_rd, 2)                                
#                               println("Pt2,Rd$(pt2_round_ct)|seed=$(seedmutpad)|$(seqpad)|$(mutpad)|Grps=$(region_ct2)|seqGrpCt=$(seq_grp_mut_ct_rd_pad)|MinMutCtPt2=$(min_mut_ct_pt2)|AbsMutCt=$(abs_grp_mut_ct)|min_abs_mut_ct=$(min_abs_mut_ct)")
#                            end
#                        end
#                    end
                ##### the '-0.00001' part here is because, for reasons I don't understand, min_mut_ct is often 0.000000000000001 larger than what it should be.
                    if seq_grp_mut_ct ≥ (min_mut_ct_pt2 - 0.00001) && abs_grp_mut_ct ≥ min_abs_mut_ct
                        push!(MP_seqs_pt2, seq)
                        push!(cumulative_seq_round_set_pt2, seq)
                    end
                    if abs_grp_mut_ct ≥ min_date_raw_ct
                        push!(date_and_AAlen_seqs, seq)
                    end
                end
############ Part below only utilized if we want to count multiple mutations in each region (not used in normal calculations)
            elseif ind_or_grp == "ind"
                for seq in 1:length(fake_seq_AA_muts)
##################################################
                    seq_priv_AA_mut_total = length(fake_seq_AA_muts_not_excluded2[seq])
                    seq_factor = clamp(0.6, (avg_AA_sub_ct_per_chronic_seq_for_main_fx/seq_priv_AA_mut_total), 1.5)
                    half_seq_factor = (seq_factor+1)/2
                    if seq_factor_OFF_HALF_ON__0_1_2 == 0
                        seq_factor = 1
                    end
                    if seq_factor_OFF_HALF_ON__0_1_2 == 1
                        seq_factor = half_seq_factor
                    end
###################################################
                    seq_mut_ct = 0
                    abs_mut_ct = 0
                    for grp_num in 1:length(sel_muts_pt2_ind_dict)
                        if !(grp_num == pt2_region_analyzed)
                            mut_vec = sel_muts_pt2_ind_dict[grp_num]
                            for mut in mut_vec
                                if mut in fake_seq_AA_muts_not_excluded2[seq] # && !(mut in masked_muts)
                                    seq_mut_ct +=1*seq_factor
                                    abs_mut_ct += 1
                                end
                            end
                        end
                    end
        ##### the '-0.00001' part here is because, for reasons I don't understand, min_mut_ct is often 0.000000000000001 larger than what it should be.
                    if seq_mut_ct ≥ (min_mut_ct_pt2 - 0.00001) && abs_mut_ct ≥ min_abs_mut_ct
                        push!(MP_seqs_pt2, seq)
                        push!(cumulative_seq_round_set_pt2, seq) 
                    end
                    if abs_grp_mut_ct ≥ min_date_raw_ct
                        push!(date_and_AAlen_seqs, seq)
                    end
                end
            end
#           if notrandom0__random1 == 0
#               println("MP_seqs_pt2 length = $(length(MP_seqs_pt2)) | region_ct2 = $(region_ct2) | ")
#           end
#####################################################################################
            for seq in MP_seqs_pt2
                total_mut_ct_pt2 += length(fake_seq_AA_muts_not_excluded2[seq])
#### For each qualifying sequence, all of its private mutations are tallied in the coAAmut_ct_pt2 dictionary (keys = mutations, values = counts)
                for mut in fake_seq_AA_muts_not_excluded2[seq]
                    #if !(mut in masked_muts)
                    coAAmut_ct_pt2[mut] = get(coAAmut_ct_pt2, mut, 0) + 1
                    #end
                end
            end
######################################################################################################################################
#           for seq in MP_seqs_pt2
#               WT_universal = Set{String}()
#               WT_del_universal = Set{String}()
#               pango = seq_pango[seq]
#               MP_seqs_pt2_pangos[pango] = get(MP_seqs_pt2_pangos, pango, 0) + 1
#               if !haskey(pango_AAsub_WT_universal, pango)
#                   for i in 1:5
#                       if haskey(pango_predecessor_meta_dict, pango)
#                           if haskey(pango_predecessor_meta_dict[pango], i)
#                               pango_i = pango_predecessor_meta_dict[pango][i]
#                               if haskey(pango_AAsub_WT_universal, pango_i)
#                                   WT_universal = pango_AAsub_WT_universal[pango_i]
#                                   WT_del_universal = pango_AAdel_WT[pango_i]
#                                   break
#                               end
#                           end
#                       end
#                   end
#               else
#                   WT_universal = pango_AAsub_WT_universal[pango]
#                   WT_del_universal = pango_AAdel_WT[pango]
#               end
#               for mut in WT_universal
#                   mutpo = aa_gene_and_pos_comprehensive_dict[mut]
#                   if !(mutpo in seq_unknown_AA[seq])
#                       MP_seqs_pt2_inherited[mut] = get(MP_seqs_pt2_inherited, mut, 0) + 1
#                   end
#               end
#               for del in WT_del_universal
#                   delpo = aa_gene_and_pos_comprehensive_dict[del]
#                   if !(delpo in seq_unknown_AA[seq])
#                       MP_seqs_pt2_inherited[del] = get(MP_seqs_pt2_inherited, del, 0) + 1
#                   end
#               end
#           end
######################################################################################################################################
#### coAAmut_ct_pt2_unknown counts the number of timesa given AA position is "unknown," either due to dropout or a mixed nucleotide for
#       the sequences that qualify as matching a given mutational pattern 
#                                  unknown_AA_pos|count
#           coAAmut_ct_pt2_unknown = Dict{String,Int}()
### MP_seqs_pt2_unknown_region_ct tallies the number of mutational regions with at least one unknown AA (not strictly necessary, tbh)
#                                                        EPI_ISL|total_regions_with_unknown_AA_count_in_seq
#            MP_seqs_pt2_unknown_region_ct = Dict{String,Int}()
#            total_unknown_region_ct_pt2 = 0
###################################################################
#           for seq in MP_seqs_pt2
#                MP_seqs_pt2_unknown_region_ct[seq] = 0
#               for region_num in 1:length(sel_muts_pt2_ind_dict)
#                    region_mut_ct = 0
#                   mut_vec = sel_muts_pt2_ind_dict[region_num]
#                   for mut in mut_vec
#                       if mut ≠ ""
#                           mutpo = aa_gene_and_pos_comprehensive_dict[mut]
#                           if mutpo in seq_unknown_AA[seq]
#                                region_mut_ct += 1
#                               coAAmut_ct_pt2_unknown[mutpo] = get(coAAmut_ct_pt2_unknown, mutpo, 0) + 1
#                           end
#                       end
#                   end
#                    if region_mut_ct > 0
#                        MP_seqs_pt2_unknown_region_ct[seq] += 1
#                        total_unknown_region_ct_pt2 += 1
#                    end
#               end
#           end
###################################################################################################################################### 
            MP_seq_ct_pt2 = length(MP_seqs_pt2)
#           println("MP_seq_ct_pt2 = $(MP_seq_ct_pt2) | Seed Mut = $(pt1_seed_mut) ")
###################
            avg_mutations_per_seq_pt2 = round(digits=2, total_mut_ct_pt2/MP_seq_ct_pt2)
            ratio_of_AAmuts_in_Grp_vs_AAmuts_in_all_chronics_pt2 = avg_mutations_per_seq_pt2/avg_AA_sub_ct_per_chronic_seq
            ratio_of_AAmuts_in_Grp_vs_AAmuts_in_all_chronics_pt2_rd = round(digits = 2, ratio_of_AAmuts_in_Grp_vs_AAmuts_in_all_chronics_pt2)
###################
            MP_seqs_pt2_sort = sort(collect(MP_seqs_pt2), by = x -> (length(x), x) )
######################################################################################################################################
#### for prop_dict_pt2, keys = mutations; values = tuple (EPCI_pct_rd, MP_pct_rd, ct, chi_squared, pv_fish, log_pv_fish))
#### See below for meanings of each part of this tuple
            prop_dict_pt2 = Dict{String, Tuple{Float64,Float64,Int,Int,Int,Float64,Float64,Float64,Union{Float64,String},Float64} }()
#### prop_dict_pt2_unknown: keys = mutations, values = percentage of qualifying sequences that have an unknown AA at that position
            prop_dict_pt2_unknown = Dict{String,Float64}()
##### EPCI_prop = proportion of all chronic seqs w/given AA mut that appear in qualifying seqs (e.g. if 10 seqs have E:T30I & 4 of those are in seqs meeting the criteria, EPCI_prop = 0.40)
##### MP_prop = proportion of all qualifying chronic seqs w/given mut (e.g. if 100 seqs meet criteria & 4 of those have E:T30I, MP_prop = 0.04)
##### EPCI_pct_rd = same as EPCI prop but in % form and rounded to 1 decimal place
##### MP_pct_rd = same as MP prop but in % form and rounded to 1 decimal place
##### mut_prop_of_EPCI = proportion of all chronic seqs that possess given AA mut
##### MP_prop_of_EPCI = proportion of all chronic seqs that meet criteria
#           for (mut, ct) in coAAmut_ct_pt2_unknown
#               MP_pct_unk = ct/MP_seq_ct_pt2
#               prop_dict_pt2_unknown[mut] = MP_pct_unk
#           end
            total_EPCI_seq_ct = length(EPCI_set)
###########################################################################################################################################################################
###########################################################################################################################################################################
            for (mut, co_mut_count) in coAAmut_ct_pt2
                prop_dict_pt2[mut] = (0.0, 0.0, 0, 0, 0, 0.0, 0.0, 0.0, 0.0, 0.0)
                unknown = 0                                                                               ### unknown = total number of qualifying sequences w/an unknown AA at the mut_po position
                mut_po = aa_gene_and_pos_comprehensive_dict[mut]
#               if haskey(coAAmut_ct_pt2_unknown, mut_po)
#                   unknown = coAAmut_ct_pt2_unknown[mut_po]
#               else
#                   coAAmut_ct_pt2_unknown[mut_po] = 0
#               end
                total_unknown = 0                                                                         ### total_unknown = total number of sequences in EPCI w/an unknown AA at the mut_po position
#               if haskey(total_AApos_ct_unknown_pt2, mut_po)
#                   total_unknown = total_AApos_ct_unknown_pt2[mut_po]
#               end
#               inherited = 0
#               all_chr_inherited = 0
#               if sub_0__posonly_1 == 0
#                   inherited = get(MP_seqs_pt2_inherited, mut, 0)
#                   all_chr_inherited = get(all_chr_seqs_inherited, mut, 0)
#               end
#### This section accounts for inherited muts that make the mut being analyzed impossible. E.g., all BA.2 & XBB have ORF1a:L3201F.
#### This means it is impossible for any BA.2/XBB sequence to have ORF1a:L3201P, so they must be removed from the denominator.
#               if sub_0__posonly_1 == 0
#                   for seq in MP_seqs_pt2
#                       pango = seq_pango[seq]
#                       if mut_po in pango_AAsub_WT_pos_only[pango] && !(mut_po in seq_unknown_AA[seq])
#                           if !haskey(pango_AAsub_WT_universal, pango)
#                               pango = pango_predecessor_meta_dict[pango][1]
#                               if !haskey(pango_AAsub_WT_universal, pango)
#                                   pango = pango_predecessor_meta_dict[pango][1]
#                               end
#                           end
#                           for inherited_mut in pango_AAsub_WT_universal[pango]
#                               if aa_gene_and_pos_comprehensive_dict[inherited_mut] == mut_po && inherited_mut ≠ mut && qryAA_comprehensive_dict[inherited_mut] ≠ refAA_comprehensive_dict[mut]
#                                   inherited += 1
#                               end
#                           end
#                       end
#                   end
#               end
                adjusted_MP_seq_ct = MP_seq_ct_pt2 - unknown
                adjusted_EPCI_seq_ct = total_EPCI_seq_ct - total_unknown
###################################################################
                EPCI_mut_ct = AA_muts_ct_no_dels_universal[mut]               
                EPCI_prop = co_mut_count/AA_muts_ct_no_dels_universal[mut]                                 ### AA_muts_ct_no_dels_universal[mut] = total AA mut count in all independent chronic seqs/lineages
                MP_prop = co_mut_count/adjusted_MP_seq_ct                                                  ### MP_prop = proportion of all qualifying MP sequences that have the mut as a private mutation
                non_MP_prop = (EPCI_mut_ct - co_mut_count)/(adjusted_EPCI_seq_ct - adjusted_MP_seq_ct)     ### non_MP_prop = proportion of all non-MP EPCI sequences that have the mut as a private mutation
                EPCI_pct = 100*EPCI_prop
                MP_pct = 100*MP_prop
                non_MP_pct = 100*non_MP_prop
                mut_prop_of_EPCI = AA_muts_ct_no_dels_universal[mut]/adjusted_EPCI_seq_ct                  ### total_EPCI_seq_ct = total # of independent chronic seqs in entire list
                MP_prop_of_EPCI = adjusted_MP_seq_ct/adjusted_EPCI_seq_ct         
                fold_incr::Union{Float64,String} = 0.0
                if non_MP_prop ≠ 0
                    fold_incr = MP_prop/non_MP_prop
                elseif non_MP_prop == 0
                    fake_non_MP_prop = 1/(adjusted_EPCI_seq_ct - adjusted_MP_seq_ct)
                    fake_fold_incr = MP_prop/fake_non_MP_prop
                    fake_fold_incr_rd = round(digits=2, fake_fold_incr)
                    fold_incr = ">$(fake_fold_incr_rd)"
                end
#                fold_incr_rd = round(digits=2, fold_incr)
                if fold_incr == ">Inf"
                    println("For $(mut), fold_incr = $(fold_incr) [should show >Inf]")
                end
                observed = co_mut_count
                expected = mut_prop_of_EPCI*adjusted_MP_seq_ct
                if observed ≥ expected
                    fishexact_up_left = co_mut_count                                                                                            ### seq with mut + meet search criteria (≥ min_mut_ct_pt2 muts in current mut pattern list)
                    fishexact_up_rt = max(adjusted_MP_seq_ct - co_mut_count, 0)                                                                 ### seq without mut + meet search criteria (≥ min_mut_ct_pt2 muts in current mut pattern list)
                    fishexact_down_left = max(AA_muts_ct_no_dels_universal[mut] - co_mut_count, 0)                                              ### seq with mut + does not meet search criteria (i.e. < min_mut_ct_pt2 muts in current mut pattern list)
                    fishexact_down_rt = max(adjusted_EPCI_seq_ct - AA_muts_ct_no_dels_universal[mut] - adjusted_MP_seq_ct + co_mut_count , 0)   ### seq w/o mut + does not meet search criteria (i.e. < min_mut_ct_pt2 muts in current mut pattern list)
                    fish = FisherExactTest(fishexact_up_left, fishexact_up_rt, fishexact_down_left, fishexact_down_rt)
                    pv_fish = pvalue(fish)
                    log_pv_fish = -log10(pv_fish)
#### Chi2 formula —— Straight from Claude.ai which says I was doing it all wrong before. Seems to make ~zero difference though ########
                    a = fishexact_up_left;  b = fishexact_up_rt;  c = fishexact_down_left;  d = fishexact_down_rt
                    n = a + b + c + d  # total sample size
                    chi_2_top = n*(a*d - b*c)^2
                    chi_2_bottom = (a+b)*(c+d)*(a+c)*(b+d)
                    chi_squared = 0.0
                    chi_squared_rd = 0.0
                    if chi_2_bottom > 0
                        chi_squared = chi_2_top/chi_2_bottom
                        expected_a = (a + b) * (a + c) / n
                        expected_b = (a + b) * (b + d) / n
                        expected_c = (c + d) * (a + c) / n
                        expected_d = (c + d) * (b + d) / n
                        if min(expected_a, expected_b, expected_c, expected_d) < 5   # Yates' continuity correction (another Claude command)
                            corrected_numerator = n * (abs(a * d - b * c) - n/2)^2
                            chi_squared_yates = corrected_numerator/chi_2_bottom
                            chi_squared = chi_squared_yates
                            chi_squared_rd = round(digits=1, chi_squared)
                        end
#                        chi_squared_pvalue = 1 - cdf(Chisq(1), chi_squared)
                    end
################# End Claude's Chi-squared formula calculations ################
                    props_pt2 = (EPCI_pct, MP_pct, EPCI_mut_ct, co_mut_count, adjusted_MP_seq_ct, chi_squared, pv_fish, log_pv_fish, fold_incr, non_MP_pct)
                    prop_dict_pt2[mut] = props_pt2
#                    if pt1_seed_mut in seedmuts2chk && mut in muts2chk
#                        log_pv_fish_rd = round(digits=2, log_pv_fish)
#                        fold_incr_rd::Union{String, Float64} = ""
#                        if (fold_incr isa String)
#                            fold_incr_rd = fold_incr
#                        else
#                            fold_incr_rd = round(digits=2, fold_incr)
#                        end
#                        mutpad = rpad(mut, 12)
#                        seedmutpad = rpad(pt1_seed_mut, 12)
#                        log_pv_fish_rd_pad = lpad(log_pv_fish_rd, 5)
#                        fold_incr_rd_pad = lpad(log_pv_fish_rd, 6)
#                        co_mut_count_pad = lpad(co_mut_count, 3)
#                        println("Pt2,Rd$(pt2_round_ct)|Grps:$(region_ct2)|seed=$(seedmutpad)|$(mutpad)|co_mut_ct=$(co_mut_count_pad)|logpvfish=$(log_pv_fish_rd_pad)|fold_incr=$(fold_incr_rd_pad)")
 #                   end
                end
            end
###################################################################################################################################
#### sel_muts_pt2 is the updated set of mutation regions to be included in the next round of the function.
#### sel_muts_pt2_pre_dict = Each key represents a mutation group; values are mut grp vectors (later joined to become strings in sel_muts_pt2)                      
#### Two sets below created in order to distinguish between mutations within one of the mut_grp ranges and those not
#####################################################################################################################################
#### This mini-section is to determine if any mutations in the region being analyzed qualify. If none do, we move on.
            sel_muts2_pt2 = Vector{String}()
            reg_analyzed_qualified = 0
            for (mut, mut_props_pt2) in prop_dict_pt2
                co_mut_count = mut_props_pt2[4]
                log_pv_fish = mut_props_pt2[8]
                mut_gene = aa_gene_comprehensive_dict[mut]
                mut_poz = aa_pos_comprehensive_dict[mut]
################
                gene_reg_analyzed = sel_muts_pt2_min_max[2]
                min_reg_analyzed = sel_muts_pt2_min_max[3]
                max_reg_analyzed = sel_muts_pt2_min_max[4]
                if co_mut_count ≥ min_match_ct && mut_gene == gene_reg_analyzed && mut_poz ≥ min_reg_analyzed && mut_poz ≤ max_reg_analyzed && log_pv_fish ≥ min_log_pv_fish
                    reg_analyzed_qualified = 1
                    break
                end
            end
###############################
#### If the region qualifies, we examine all mutations in that region to see if they meet the minimum requirements to be included in the MP (namely, log_pv_fish ≥ min_grp_fish)
            if reg_analyzed_qualified == 1
                for (mut, mut_props_pt2) in prop_dict_pt2
                    co_mut_count = mut_props_pt2[4]
                    log_pv_fish = mut_props_pt2[8]
                    mut_gene = aa_gene_comprehensive_dict[mut]
                    mut_poz = aa_pos_comprehensive_dict[mut]
##################
                    gene_reg_analyzed = sel_muts_pt2_min_max[2]
                    min_reg_analyzed = sel_muts_pt2_min_max[3]
                    max_reg_analyzed = sel_muts_pt2_min_max[4]
                    if co_mut_count ≥ min_match_ct && mut_gene == gene_reg_analyzed && mut_poz ≥ min_reg_analyzed && mut_poz ≤ max_reg_analyzed && log_pv_fish ≥ min_grp_fish
                        push!(sel_muts2_pt2, mut)
                        push!(all_muts_round_dict_pt2[pt2_round_ct], mut)
                    end
                end
            end
#########################################################################################################################################
############ Sorting these so that the results of each round an iteration are easy to compare——also the final results
            sel_muts2_pt2_sort = sort(sel_muts2_pt2, by = x -> sel_muts_pt1_sort_key_universal(x, sub_0__posonly_1))
            sel_muts_pt2_ind_dict[pt2_region_analyzed] = sel_muts2_pt2_sort
#########################################################################################################################################
#################################################### Below: Debug Code ############################################
#           if notrandom0__random1 == 0
#               if length(sel_muts2_pt2) ≥ 1
#                   println("length of sel_muts_pt2 = $(length(sel_muts_pt2))  |  number of regions = $(region_ct2)")
#                   println("## sel_muts2_pt2, Seed Mut = $(pt1_seed_mut)  |  Region #$(pt2_region_analyzed) ")
#                   sel_muts2_pt2_string = join(sel_muts2_pt2_sort, ", ")
#                   println(sel_muts2_pt2_string); println()
#               end
#           end
#################################################### Above: Debug Code ##################################################################
#########################################################################################################################################
            if length(keys(prop_dict_pt2)) > 1
                MP_Tot_Mut_ct = 0
                for mut in sel_muts2_pt2
                    props2 = prop_dict_pt2[mut]                      
                    EPCI_pct = props2[1]
                    MP_pct = props2[2]
                    EPCI_mut_ct = props2[3]
                    MP_Tot_Mut_ct = props2[4]
                    qual_seq_ct = props2[5]
                    Chi2 = props2[6]
                    pvFish = props2[7]
                    log10pvFISH = props2[8]
                    fold_incr = props2[9]
                    non_MP_pct = props2[10]
                    chr_all_ratio = mp_chr_all_ratio[mut]
                    if !haskey(df_props_dict_pt2, pt2_round_ct)
                        df_props_dict_pt2[pt2_round_ct] = Dict{Int, Dict{String, Tuple{Float64,Float64,Int,Int,Int,Float64,Float64,Float64,Union{Float64,String},Float64}}}()
                    end
                    if !haskey(df_props_dict_pt2[pt2_round_ct], pt2_region_analyzed)
                        df_props_dict_pt2[pt2_round_ct][pt2_region_analyzed] = Dict{String, Tuple{Float64,Float64,Int,Int,Int,Float64,Float64,Float64,Union{Float64,String},Float64}}()
                    end
                    df_props_dict_pt2[pt2_round_ct][pt2_region_analyzed][mut] = (EPCI_pct, MP_pct, EPCI_mut_ct, MP_Tot_Mut_ct, qual_seq_ct, Chi2, pvFish, log10pvFISH, fold_incr, non_MP_pct)
                end
            end
#####################################################################################################################################
#### Move on to the next region (as long as we didn't just finish the final region)
#### Sel_muts_pt2 has to be turned into a vector of strings, with each string containing all a region's mutations, separated by commas
#            prev_round_pt2 = pt2_round_ct-1
            all_muts_set_pt2 = Set{String}()
            sel_muts_pt2 = Vector{String}()
            for grp in 1:length(sel_muts_pt2_ind_dict)
                mut_vec = sel_muts_pt2_ind_dict[grp]
                for mut in mut_vec
                    push!(all_muts_set_pt2, mut)
                end
                mut_join = join(mut_vec, ", ")
                push!(sel_muts_pt2, mut_join)
            end
            if pt2_region_analyzed < region_ct2 + blank_group_ct
                pt2_region_analyzed += 1
                continue
#### If on last region, start over on 1st region w/updated list——unless there were 0 changes this round, in which case we're done.
            elseif pt2_region_analyzed > region_ct2 + blank_group_ct #DEBUG
                print("\n"^8); println("ERROR ALERT: pt2_region_analyzed is greater than region_ct2; SHOULD NOT HAPPEN | Seed Mut = $(pt1_seed_mut) "); print("\n"^8)  #DEBUG 
            elseif pt2_round_ct < 3 && pt2_region_analyzed == region_ct2 + blank_group_ct
                pt2_round_ct += 1
                pt2_region_analyzed = 1
                continue
            elseif pt2_region_analyzed == region_ct2 + blank_group_ct && all_muts_round_dict_pt2[pt2_round_ct] ≠ all_muts_round_dict_pt2[pt2_round_ct-1] && pt2_round_ct ≤ 12 # && all_muts_round_dict_pt2[pt2_round_ct] ≠ all_muts_round_dict_pt2[pt2_round_ct-2]
                pt2_round_ct += 1
                pt2_region_analyzed = 1
                continue
            elseif (pt2_region_analyzed == region_ct2 + blank_group_ct && all_muts_round_dict_pt2[pt2_round_ct] == all_muts_round_dict_pt2[pt2_round_ct-1]) || (pt2_region_analyzed == region_ct2 + blank_group_ct && pt2_round_ct ≥ 12)
############################################################################################################################################################################
# Shutting everything down if seedmut is not in final MP
#                if !(pt1_seed_mut in all_muts_set_pt2)
#                    return (Dict{Int, Dict{String, Tuple{Float64,Float64,Int,Int,Int,Float64,Float64,Float64,Float64}}}(), "", 0, "00-00-00", Pair{String,Int64}[], Pair{String,Int64}[], Pair{String,Int64}[])
#                end
                
                
                
                
                
                
                
                




















                
                
                
                
                                
############################################################################################################################################################################
###################################################### Begin: Pango lineage & collection date section ######################################################################
############################################################################################################################################################################

                cumulative_seq_round_set_pt2_sort = sort(collect(cumulative_seq_round_set_pt2), by = x -> (length(x), x))
                cumulative_seq_round_set_pt2_sort_join = join(cumulative_seq_round_set_pt2_sort, ", ") 





















############################################################################################



############################################################################################



















####################################







####################################









####################################



















####################################



############################################################################################################################################################################
###################################################### End: Pango lineage & collection date section ########################################################################
############################################################################################################################################################################






                
############################################################################################################################################################################
###################################### Begin: Rigorous Avg_AA, Avg_AA_nonRBD len, & collection date section ################################################################
############################################################################################################################################################################
                
                mp_AA_len_vec = Int[]
                
                
                
                
                avg_hard_mp_AA_len = 0
                avg_hard_mp_AA_len_raw = 0
                avg_hard_mp_AA_len_relative = 0
                
                
                
                if !isempty(date_and_AAlen_seqs)
                    for seq in date_and_AAlen_seqs
                        
                        
                        
                        
                        
                        
                        seqaa_len = seq_privAA_len[seq]
                        push!(mp_AA_len_vec, seqaa_len)
#                            seqaa_len_rel = seq_privAA_len_relative[seq]
#                            push!(mp_AA_len_vec, seqaa_len)
                    end
















                    
                    total_AA_len = 0
                    AA_len_seq_count = 0
                    if !isempty(mp_AA_len_vec)
                        total_AA_len = sum(mp_AA_len_vec)
                        AA_len_seq_count = length(mp_AA_len_vec)
                    end
                    if total_AA_len ≠ 0 && AA_len_seq_count ≠ 0
                        avg_hard_mp_AA_len_raw = total_AA_len/AA_len_seq_count
                        avg_hard_mp_AA_len = round(digits=2, avg_hard_mp_AA_len_raw)
                    end
                    avg_hard_mp_AA_len_relative = round(digits=2, avg_hard_mp_AA_len_raw/avg_AA_sub_ct_per_chronic_seq_for_main_fx)
                end
############################################################################################################################################################################
################################################# End: Rigorous Avg AA len & collection date section #######################################################################
############################################################################################################################################################################
                if !isempty(keys(df_props_dict_pt2)) && region_ct2 > 1 && haskey(df_props_dict_pt2, pt2_round_ct)
                    regions_to_delete_pt1 = Int[]
                    for region in keys(df_props_dict_pt2[pt2_round_ct])
                        if length(df_props_dict_pt2[pt2_round_ct][region]) == 0
                            push!(regions_to_delete_pt1, region)
                        end
                    end
                    for reg in regions_to_delete_pt1
                        delete!(df_props_dict_pt2[pt2_round_ct], reg)
                    end
                    region_mut_array_dict = Dict{Int, Union{Set{String},Vector{String}}}()
                    region_key_sort = sort(collect(keys(df_props_dict_pt2[pt2_round_ct])))
                    region_duplicate_test_set = Set{Set{String}}()
                    regions_to_delete_pt2 = Int[]
                    for region in region_key_sort
                        region_mut_set = Set{String}
                        region_mut_set = Set(collect(keys(df_props_dict_pt2[pt2_round_ct][region])))
                        region_mut_array_dict[region] = Set{String}()
                        if region_mut_set in region_duplicate_test_set
                            continue
                        end
                        push!(region_duplicate_test_set, region_mut_set)
                        region_mut_array_dict[region] = Set{String}()
                        for (mut, props_arr) in df_props_dict_pt2[pt2_round_ct][region]
                            EPCI_pct = props_arr[1]
                            MP_pct = props_arr[2]
                            log10pvFISH = props_arr[8]
                            if length(df_props_dict_pt2[pt2_round_ct][region]) < 2  
                                if EPCI_pct > 18 || log10pvFISH > 6 || MP_pct > 50
                                    push!(region_mut_array_dict[region], mut)
                                else
                                    EPCI_pct_rd = round(digits=2, EPCI_pct); MP_pct_rd = round(digits=2, MP_pct); log10pvFISH_rd = round(digits=2, log10pvFISH)
                                    pt1_seed_mutpad = rpad(pt1_seed_mut, 12); mutpad = rpad(mut, 12); EPCI_pct_pad = lpad(EPCI_pct_rd, 5); MP_pct_pad = lpad(MP_pct_rd, 5); log10pvFISH_pad = lpad(log10pvFISH_rd, 5)
                                    println("Booted, pt1: $(mutpad) |seed=$(pt1_seed_mutpad)|EPCI_pct = $(EPCI_pct_pad)|MP_pct = $(MP_pct_pad)|log10pvFISH = $(log10pvFISH_pad)|")
                                end
                            else
                                push!(region_mut_array_dict[region], mut)
                            end
                        end
                        if !isempty(region_mut_array_dict[region])
                            region_mut_array_sort = sort(collect(region_mut_array_dict[region]), by = x -> (aa_gene_comprehensive_dict[x], aa_pos_comprehensive_dict[x]))
                            region_mut_array_dict[region] = region_mut_array_sort
                        else
                            push!(regions_to_delete_pt2, region)
                        end
                    end
                    for reg in regions_to_delete_pt2
                        delete!(df_props_dict_pt2[pt2_round_ct], reg)
                    end
########################### Begin: Section to eliminate duplicate mut patterns #############################
                    df_mp_META_set = Set{String}()
#######                    df_mp_META_region_max_pvals = 
                    for region in region_key_sort
                        region_muts_set = Set(region_mut_array_dict[region])
                        for mut in region_muts_set
                            EPCI_pct__MP_pct__totChrMutCt__MP_Tot_Mut_ct__Adjusted_MP_seq_ct__Chi2__pvFish__log10pvFISH = df_props_dict_pt2[pt2_round_ct][region][mut]
                            EPCI_pct = EPCI_pct__MP_pct__totChrMutCt__MP_Tot_Mut_ct__Adjusted_MP_seq_ct__Chi2__pvFish__log10pvFISH[1]
                            MP_pct = EPCI_pct__MP_pct__totChrMutCt__MP_Tot_Mut_ct__Adjusted_MP_seq_ct__Chi2__pvFish__log10pvFISH[2]
                            log10pvFISH = EPCI_pct__MP_pct__totChrMutCt__MP_Tot_Mut_ct__Adjusted_MP_seq_ct__Chi2__pvFish__log10pvFISH[8]
                            if EPCI_pct ≤ 18 && log10pvFISH < 6 && MP_pct < 50
                                region_mut_array_dict[region] = setdiff(region_mut_array_dict[region], [mut])
                                EPCI_pct_rd = round(digits=2, EPCI_pct); MP_pct_rd = round(digits=2, MP_pct); log10pvFISH_rd = round(digits=2, log10pvFISH)
                                pt1_seed_mutpad = rpad(pt1_seed_mut, 12); mutpad = rpad(mut, 12); EPCI_pct_pad = lpad(EPCI_pct_rd, 5); MP_pct_pad = lpad(MP_pct_rd, 5); log10pvFISH_pad = lpad(log10pvFISH_rd, 5)
                                print("\n"); println("Booted, pt2: $(mutpad) |seed=$(pt1_seed_mutpad)|EPCI_pct = $(EPCI_pct_pad)|MP_pct = $(MP_pct_pad)|log10pvFISH = $(log10pvFISH_pad)|")
##                               delete!(region_mut_array_dict[region], mut)
#                                delete!(df_mp_META_set, mut)
                            else
                                push!(df_mp_META_set, mut)
                            end
                        end
                    end
#                   if nonmulti0__multi1 == 1                    
                        if df_mp_META_set in df_mp_META_set_set
# DEBUG                          println("DUPLICATE EARLY RETURN | seed=$(pt1_seed_mut) | length=$(length(df_mp_META_set))") # | $(all_muts_at_end_of_pt2_join[1:80])", "..."))
                            return (Dict{Int, Dict{String, Tuple{Float64,Float64,Int,Int,Int,Float64,Float64,Float64,Union{Float64,String},Float64}}}(), "", 0, "", Pair{String,Int64}[])
                        end
#                   end
                    df_mp_META_region_set = Set{Set{String}}()
                    for region_mut_array in values(region_mut_array_dict)
                        region_mut_array_set = Set(region_mut_array)
                        if length(region_mut_array_set) > 0
                            push!(df_mp_META_region_set, region_mut_array_set)
                        end
                    end
######################### End: Section to eliminate duplicate mut patterns ###########################
#                    region_keys_to_delete = Set{Int}()
#                    if isempty(region_mut_array_dict[region])
########################### Begin: Section to eliminate low % mutations ##############################
#                    for region in region_key_sort
#                        for mut in region_mut_array_dict[region]
#                            EPCI_pct__MP_pct__totChrMutCt__MP_Tot_Mut_ct__Adjusted_MP_seq_ct__Chi2__pvFish__log10pvFISH = df_props_dict_pt2[pt2_round_ct][region][mut]
#                            EPCI_pct = EPCI_pct__MP_pct__totChrMutCt__MP_Tot_Mut_ct__Adjusted_MP_seq_ct__Chi2__pvFish__log10pvFISH[1]
#                            MP_pct = EPCI_pct__MP_pct__totChrMutCt__MP_Tot_Mut_ct__Adjusted_MP_seq_ct__Chi2__pvFish__log10pvFISH[2]
#                            log10pvFISH = EPCI_pct__MP_pct__totChrMutCt__MP_Tot_Mut_ct__Adjusted_MP_seq_ct__Chi2__pvFish__log10pvFISH[8]
#                            if EPCI_pct ≤ 20 && MP_pct ≤ 20 && log10pvFISH ≤ 7
#                                region_mut_array_dict[region] = setdiff(region_mut_array_dict[region], [mut])
##                                delete!(region_mut_array_dict[region], mut)
#                                delete!(df_mp_META_set, mut)
#                            end
#                        end
#                    end
############################ End: Section to eliminate low % mutations ###############################











                    
                    if length(df_mp_META_set) ≥ 2
                        df_mp_META_groups_dict[pt1_seed_mut] = Vector{Tuple{String,Int,Int,Int,Int,String,Float64,Int,Int,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Int,String,Int,String,Int,Int,Int,Int,Int,Int,Int,Int,String,String,String,String,String,String,String,Int,Int,Int,Int,Int,Int,Int,String,String,String,String,String,String,String,String}}()
                        mp_tot_region_ct = length(region_key_sort)
                        pt1_seed_mut_grp_total_dict[pt1_seed_mut] = 0
                        mp_total_mut_ct = length(df_mp_META_set)
                        region_number_count = 0
                        for region in region_key_sort
                            if !isempty(region_mut_array_dict[region])
                                region_number_count += 1
                                pt1_seed_mut_grp_total_dict[pt1_seed_mut] += 1
                                region_total_mut_ct = length(region_mut_array_dict[region])
                                for mut in region_mut_array_dict[region]
                                    EPCI_pct__MP_pct__totChrMutCt__MP_Tot_Mut_ct__Adjusted_MP_seq_ct__Chi2__pvFish__log10pvFISH = df_props_dict_pt2[pt2_round_ct][region][mut]
                                    EPCI_pct = EPCI_pct__MP_pct__totChrMutCt__MP_Tot_Mut_ct__Adjusted_MP_seq_ct__Chi2__pvFish__log10pvFISH[1]
                                    MP_pct = EPCI_pct__MP_pct__totChrMutCt__MP_Tot_Mut_ct__Adjusted_MP_seq_ct__Chi2__pvFish__log10pvFISH[2]
                                    EPCI_mut_ct = EPCI_pct__MP_pct__totChrMutCt__MP_Tot_Mut_ct__Adjusted_MP_seq_ct__Chi2__pvFish__log10pvFISH[3]
                                    MP_Tot_Mut_ct = EPCI_pct__MP_pct__totChrMutCt__MP_Tot_Mut_ct__Adjusted_MP_seq_ct__Chi2__pvFish__log10pvFISH[4]
                                    qual_seq_ct = EPCI_pct__MP_pct__totChrMutCt__MP_Tot_Mut_ct__Adjusted_MP_seq_ct__Chi2__pvFish__log10pvFISH[5]
                                    Chi2 = EPCI_pct__MP_pct__totChrMutCt__MP_Tot_Mut_ct__Adjusted_MP_seq_ct__Chi2__pvFish__log10pvFISH[6]
                                    pvFish = EPCI_pct__MP_pct__totChrMutCt__MP_Tot_Mut_ct__Adjusted_MP_seq_ct__Chi2__pvFish__log10pvFISH[7]
                                    log10pvFISH = EPCI_pct__MP_pct__totChrMutCt__MP_Tot_Mut_ct__Adjusted_MP_seq_ct__Chi2__pvFish__log10pvFISH[8]
                                    fold_incr = EPCI_pct__MP_pct__totChrMutCt__MP_Tot_Mut_ct__Adjusted_MP_seq_ct__Chi2__pvFish__log10pvFISH[9]
                                    non_MP_pct = EPCI_pct__MP_pct__totChrMutCt__MP_Tot_Mut_ct__Adjusted_MP_seq_ct__Chi2__pvFish__log10pvFISH[10]
                                    chr_all_ratio = mp_chr_all_ratio[mut]
                                    if mp_total_mut_ct == 2 && MP_Tot_Mut_ct < 5
                                        return (Dict{Int, Dict{String, Tuple{Float64,Float64,Int,Int,Int,Float64,Float64,Float64,Union{Float64,String},Float64}}}(), "", 0, "", Pair{String,Int64}[], Pair{String,Int64}[], Pair{String,Int64}[])
                                    
                                    
                                    
                                    
                                    end
#                                    if normal_0__spikeonly_1__spikeWithRBD_2 == 0
#                                        if EPCI_pct ≤ 20 && MP_pct ≤ 20 && log10pvFISH ≤ 7                                            
#                                            mp_total_mut_ct -= 1
#                                            if length(region_mut_array_dict[region]) == 1
#                                                region_number_count -= 1
#                                                mp_tot_region_ct -= 1
#                                            end
#                                            continue
#                                        end
#                                    end
                                    if mp_total_mut_ct > 1
                                        push!(df_mp, (pt1_seed_mut, mp_total_mut_ct, region_number_count, region_total_mut_ct, region, mut, MP_Tot_Mut_ct, EPCI_mut_ct, qual_seq_ct, EPCI_pct, MP_pct, non_MP_pct, pvFish, log10pvFISH, fold_incr, Chi2, chr_all_ratio, avg_hard_mp_AA_len, avg_hard_mp_AA_len_relative))
                                        if nonmulti0__multi1 == 1
                                            push!(pt1_seed_mut_set, pt1_seed_mut)                                    
                                            push!(df_mp_META, (fake_dict_num_ct, mp_index, mp_total_mut_ct, mp_tot_region_ct, region_total_mut_ct, region, mut, MP_Tot_Mut_ct, EPCI_mut_ct, qual_seq_ct, EPCI_pct, MP_pct, non_MP_pct, pvFish, log10pvFISH, fold_incr, Chi2, chr_all_ratio, avg_hard_mp_AA_len, avg_hard_mp_AA_len_relative))
                                            push!(df_mp_META_META, (fake_dict_num_ct, mp_index, mp_total_mut_ct, mp_tot_region_ct, region_total_mut_ct, region, mut, MP_Tot_Mut_ct, EPCI_mut_ct, qual_seq_ct, EPCI_pct, MP_pct, non_MP_pct, pvFish, log10pvFISH, fold_incr, Chi2, chr_all_ratio, avg_hard_mp_AA_len, avg_hard_mp_AA_len_relative))
                                        end
                                    end
                                end
                            end
                        end
# DEBUG                          println("REGISTERING | seed=$(pt1_seed_mut) | length=$(length(df_mp_META_set))")
                        push!(df_mp_META_set_set, df_mp_META_set)
                        push!(df_mp_META_region_set_set, df_mp_META_region_set)
                        df_mp_META_set_set_dict[df_mp_META_set] = pt1_seed_mut
                        df_mp_META_region_set_set_dict[df_mp_META_region_set] = pt1_seed_mut
                    end
                end
######################################################################################################################################### 
############ All the sections directly below just sort the dictionaries according to their various components ########################
###################################################################################################################################### 




















############################################################################################################# 












############################################################################################################# 







############################################################################################################# 






#####################################################################################################################################



#####################################################################################################################################                
            else
                println("No Conditions Satisified: Something is seriously wrong with the recursion conditionals"^8)
            end
        end
        if !haskey(df_props_dict_pt2, pt2_round_ct)
            df_props_dict_pt2[pt2_round_ct] = Dict{Int, Dict{String, Tuple{Float64,Float64,Int,Int,Int,Float64,Float64,Float64,Union{Float64,String},Float64}}}()
            df_props_dict_pt2[pt2_round_ct][0] = Dict{String, Tuple{Float64,Float64,Int,Int,Int,Float64,Float64,Float64,Union{Float64,String},Float64}}()
            df_props_dict_pt2[pt2_round_ct][0][""] = (0.0, 0.0, 0, 0, 0, 0.0, 0.0, 0.0, 0.0, 0.0)
        end
        return df_props_dict_pt2[pt2_round_ct], cumulative_seq_round_set_pt2_sort_join
    end
    return (Dict{Int, Dict{String, Tuple{Float64,Float64,Int,Int,Int,Float64,Float64,Float64,Union{Float64,String},Float64}}}(), "")
end
date_now_end = Dates.format(now(), "yyyy_mm_dd__HMMp"); println(date_now_end); println("Finished."); print("\n"^1)
nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime); print("\n"^1)
#############################################################################################################################
#############################################################################################################################


2026_05_05__2357PM
11:57.40_PM

2026_05_05__2357PM
Finished.

11:57.40_PM



In [141]:
### NEW (2025-11-18) v2 for candidate muts: only Double_N_ORF9b_muts & artifactual_private_muts excluded
## Reminder: all_excluded_muts is a global variable defined in RBD cell & takes on its value depending on the values of: 
#                                                        sub_0__posonly_1 = 0
#                                                        normal_0__spikeonly_1__spikeWithRBD_2 = 0
#                                                        noBAL_0__withBAL_1 = 1
print("\n"^1)
date_now = Dates.format(now(), "yyyy_mm_dd__HMMp"); println(date_now)
nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime)
min_ct = 3
max_ct = 999
mut_cand_ct = 0
spike_muts_to_exclude = list_to_set("S:D145Y, S:I95T, S:I212L, S:V67A, S:A27S")
candidate_test_muts = Set{String}()
for (mut, ct) in AA_muts_ct_no_dels_universal
    if !(mut in all_excluded_muts) && !(mut in spike_muts_to_exclude) && ct ≤ max_ct && ct ≥ min_ct
        push!(candidate_test_muts, mut)
    end
end
print("\n"^2)
cand_length = length(candidate_test_muts)
println("Total Number of Test Mut Candidates = $cand_length")
print("\n"^3)
for mut in candidate_test_muts
    print("$(mut), ")
end
print("\n"^2)
nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime); print("\n")
println("Finished")
print("\n"^2)


2026_05_05__2357PM
11:57.44_PM


Total Number of Test Mut Candidates = 3099



N:230, ORF1b:1274, ORF3a:131, ORF1a:676, ORF1b:391, ORF3a:31, S:22, ORF7b:26, ORF1a:2865, ORF8:21, ORF1b:2019, S:245, ORF1a:2994, ORF1a:819, ORF1b:2575, N:185, ORF1a:335, ORF1b:2197, M:162, ORF1a:433, ORF1a:730, ORF8:115, S:198, ORF3a:12, ORF3a:123, ORF1b:1178, ORF1a:1577, ORF1a:1716, ORF1a:206, S:1260, M:8, ORF1a:3627, ORF1a:2181, ORF3a:9, S:810, ORF1a:2460, ORF1b:2425, S:1027, ORF1a:283, ORF7b:37, ORF1a:4311, S:780, ORF1a:904, ORF1a:2712, S:867, ORF1b:3, ORF1b:1899, ORF1a:1246, ORF1a:2299, ORF1a:2966, ORF1a:1089, ORF1a:1314, ORF1a:991, ORF1b:1076, ORF1a:1639, ORF1b:15, S:831, ORF9b:34, ORF7a:45, ORF1a:2219, ORF1b:731, ORF1b:2269, ORF9b:8, ORF1a:1500, ORF1a:10, S:256, S:625, ORF1a:1504, ORF7a:72, S:54, ORF1a:3577, ORF1a:1812, ORF1a:438, ORF1a:2114, N:392, ORF1b:329, ORF1a:3149, ORF1a:192, ORF1a:3148, ORF7b:15, S:561, ORF3a:10, ORF1a:272, S:576, S:15, ORF7a:106, ORF3a:63, ORF1a:3922, ORF1a:324, N:243, ORF1a

In [142]:
### Testing all EPCI/APCI mutations for mutational patterns | ## 2 hr, 38 min, 52.28 sec, 2026_05_03 (100 Runs, subs, 8-10-95)
date_now = Dates.format(now(), "yyyy_mm_dd__HMMp"); print("\n"^1); println(date_now)
print("\n"^1); nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime)
print("\n"^1); println("EPCI_qc_str = $(EPCI_qc_str)"); println("HQCS_qc_string = $(HQCS_qc_string)"); print("\n"^1) 
date = Dates.format(today(), "yyyy_mm_dd")
date_hour = Dates.format(now(), "yyyy_mm_dd_Hp")
start = time()
############################################################################################################################################################################
#                              Run#
global df_final_mp_META = Dict{Int, DataFrame}()
for fake in 1:length(mp_meta_fake_chr_dict_DQ)
    df_final_mp_META[fake] = DataFrame(Run = Int[], mpNum = Int[], Region = Int[], MutNum = Int[], Mutation = String[])
end
#                               Run#       mp#    region#     mpmut#    mut
global df_final_dict_META = Dict{Int, Dict{Int, Dict{Int, Dict{Int,String}}}}()
for fake in 1:length(mp_meta_fake_chr_dict_DQ)
    df_final_dict_META[fake] = Dict{Int, Dict{Int, Dict{Int,String}}}()
end
global fake_dict_num_ct = 0
global total_mp_ct = 0
global mp_index
## Int,Int,Int,Int,String,Float64,Int,Int,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
## run|seedmutindex|mpregion|regionMutTot|Mut|MP_Tot_Mut_ct|EPCI_Mut_ct = |Adjusted_MP_seq_ct|EPCI_pct|MP_pct|log10pvFISH|FoldIncr|Chi2|ChrAllRatio|avgNonRBD_AAct|avgNonRBDrel|
global df_mp_META_META = DataFrame(Run = Int[], Seedmut_Index = Int[], MP_Tot_Mut_ct = Int[], MP_Tot_Reg_ct = Int[], Reg_Tot_Mut_ct = Int[], MP_Region_Num = Int[], Mutation = String[], MP_Mut_ct = Float64[], EPCI_Mut_ct = Int[], Adjusted_MP_seq_ct = Int[], EPCI_pct = Float64[], MP_pct = Float64[], non_MP_pct = Float64[], Fishers_Exact_Test_pval = Float64[], log10pvFISH = Float64[], Fold_Incr = Union{Float64,String}[], Chi2 = Float64[], EPCI_HQCS_Ratio = Float64[], avg_NonRBD_AAct = Float64[], avg_nonRBD_rel = Float64[])
############################################################################################################################################################################
global rand_folder = "$(mp_folder_universal)_RANDOM"
mkpath(rand_folder)
global tsv_folder = "$(rand_folder)/TSV"
mkpath(tsv_folder)
global rand_folder_ind = "$(rand_folder)/FAKE_specific_mut_stats"
mkpath(rand_folder_ind)













############################################################################################################################################################################
mut_gene_Dict = Dict{String,Int}("ORF1a"=>1, "ORF1b"=>2, "S"=>3, "E"=>4, "M"=>5, "N"=>6, "ORF3a"=>7, "ORF6"=>8, "ORF7a"=>9, "ORF7b"=>10, "ORF8"=>11, "ORF9b"=>12)
#################################################################
pure_spike_ct = 0
cand_length = length(candidate_test_muts)
println("Total Number of Candidate Test Mutations = $(cand_length)"); print("\n"^1)
random_sample_size = cand_length
min_mut_ct = 5
max_mut_ct = 999
tot_tested_muts = random_sample_size
total_EPCI_seq_ct = length(EPCI_set)
############################################################################################################################################################################
    min_match_ct = 2
    PMR1 = 0.4
    PMR2 = 0.4
    nonmulti0__multi1 = 1        
    notrandom0__random1 = 1
    plus_minus = 5
    if normal_0__spikeonly_1__spikeWithRBD_2 == 1
        plus_minus = 2
    elseif normal_0__spikeonly_1__spikeWithRBD_2 == 2   
        plus_minus = 0
    end
    ind_or_grp = "grp"
    min_grp_fish = 2.0
    min_log_pv_fish = 5.0
    mp_masked_muts = Set{String}()
### see `starter` below for sel_muts
    seq_factor_OFF_HALF_ON__0_1_2 = 1
    if normal_0__spikeonly_1__spikeWithRBD_2 == 1
        seq_factor_OFF_HALF_ON__0_1_2 = 1
    end
    group_overlap_thresh = 0.5
#################################################################################
    seqfac_dict = Dict(0=>"OFF", 1=>"ON", 2=>"FULLBLAST")
    seqfac_dict_spike = Dict(0=>"OFF", 1=>"RBM", 2=>"RBD", 3=>"SpikeNonRBD")
    seqfac = seqfac_dict[seq_factor_OFF_HALF_ON__0_1_2]
    SpikeFac = seqfac_dict_spike[nonspike_seqfactor__0off__1RBM__2RBD__3SpikeNonRBD__4AllSpike]
############################################################################################################################################################################
nowtime1 = Dates.format(now(), "I:MM.SS_p") #; println(nowtime1)
time1 = time()
###############################
for fake_dict_num in 1:length(mp_meta_fake_chr_dict_DQ)
    nowtime1 = Dates.format(now(), "I:MM.SS_p"); println("Run #$(fake_dict_num) Start Time = $(nowtime1)")
    time1 = time()
    fake_dict_num_ct += 1
    global fake_dict_num_ct
    global seq_privAA_len = seq_privAA_len_meta[fake_dict_num_ct]
####################################################################################################################
    global df_mp_META_set_set = Set{Set{String}}()
## (fake_dict_num_ct, mp_index, mp_total_mut_ct, mp_tot_region_ct, region_total_mut_ct, mut, MP_Tot_Mut_ct, EPCI_mut_ct, qual_seq_ct, EPCI_pct, MP_pct, log10pvFISH, fold_incr, Chi2, chr_all_ratio, avg_hard_mp_AA_len, avg_hard_mp_AA_len_relative))
#            df_mp_META, (fake_dict_num_ct,       mp_index,           mp_total_mut_ct,     mp_tot_region_ct,      region_total_mut_ct,      region, mut, MP_Tot_Mut_ct, EPCI_mut_ct, qual_seq_ct, EPCI_pct, MP_pct, non_MP_pct, pvFish, log10pvFISH, fold_incr, Chi2, chr_all_ratio, avg_hard_mp_AA_len, avg_hard_mp_AA_len_relative))
# push!(df_mp_META_META, (fake_dict_num_ct, mp_index, mp_total_mut_ct, mp_tot_region_ct, region_total_mut_ct, region, mut, MP_Tot_Mut_ct, EPCI_mut_ct, qual_seq_ct, EPCI_pct, MP_pct, non_MP_pct, pvFish, log10pvFISH, fold_incr, Chi2, chr_all_ratio, avg_hard_mp_AA_len, avg_hard_mp_AA_len_relative))
       
    global df_mp_META = DataFrame(Run = Int[], Seedmut_Index = Int[], MP_Tot_Mut_ct = Int[], MP_Tot_Reg_ct = Int[], Reg_Tot_Mut_ct = Int[], MP_Region_Num = Int[], Mutation = String[], MP_Mut_ct = Float64[], EPCI_Mut_ct = Int[], Adjusted_MP_seq_ct = Int[], EPCI_pct = Float64[], MP_pct = Float64[], non_MP_pct = Float64[], Fishers_Exact_Test_pval = Float64[], log10pvFISH = Float64[], Fold_Incr = Union{Float64,String}[], Chi2 = Float64[], EPCI_HQCS_Ratio = Float64[], avg_NonRBD_AAct = Float64[], avg_nonRBD_rel = Float64[])
    global df_mp_META_META
    global df_mp_META_set
    global df_mp_META_set_set
    global df_mp_META_seedmut_set_dict = Dict{String, Set{String}}()
#################################### Put in later: avgMutPerSeq1 = Float64[] —— requires changing main fx so that only the qualifying seqs with this mut are included. Likely need it in in props/props_dict2.
############################################################################################################################################################################
    chr_all_ratio = 0.0
    all_correlated_rand_muts_set = Set{String}()
    mp_index = 0
    seedmut_mut_set_set = Set{Set{String}}()
    blank_mutstrings = Set(["", " ", "  ", "   ", "    ", "     ", "      ", "       ", "        ", "         ", "          ", "           ", "            "])
    blank_mut_ct = 0
    repeat_ct = 0
    for seed_mut in candidate_test_muts
        mp_index +=1



        
###############################################################
    global pt1_round_ct = 0
    global pt1_finished = 0
    global sel_muts_pt1_round_dict = Dict{Int, Vector{String}}()
    global all_muts_round_dict_pt2 = Dict{Int, Set{String}}()   # all_muts_round_dict_pt2 stores the qualifying mutations at the end of each round
    global all_muts_round_mutgroup_dict_pt2 = Dict{Int, Vector{String}}()
#                                     Round RegionAnalyzed  Mutation      EPCI_pct, MP_pct, totChrMutCt, MP_Tot_Mut_ct, Adjusted_MP_seq_ct, Chi^2,  pvFish, log10pvFISH, fold_incr, non_MP_pct
        global df_props_dict_pt2 = Dict{Int, Dict{Int, Dict{String, Tuple{Float64,Float64,Int,Int,Int,Float64,Float64,Float64,Union{Float64,String},Float64}}}}()
        global pt2_round_ct = 0
        global pt2_region_analyzed = 1
        global sel_muts_pt2 = Vector{String}()
        global sel_muts_pt1_global = Vector{String}()
        global sel_muts_pt1_sort = Vector{String}()
        global sel_muts_pt1_ind = Vector{Vector{String}}()
        global region_ct2
        global pt1_seed_mut = ""
        global coAAmut_ct_pt1 = Dict{String,Int}()
        global coAAmut_ct_pt2 = Dict{String,Int}()
        global coAAmut_ct_pt1_unknown = Dict{String,Int}()
        global coAAmut_ct_pt2_unknown = Dict{String,Int}()
        global prop_dict_pt1 = Dict{String, Tuple{Float64,Float64,Int,Int,Int,Float64,Float64,Float64,Union{Float64,String},Float64} }()
        global prop_dict_pt2 = Dict{String, Tuple{Float64,Float64,Int,Int,Int,Float64,Float64,Float64,Union{Float64,String},Float64} }()  
        global cumulative_seq_round_set_pt2 = Set{String}() ### Set of all seqs that qualify during any iteration of a round (erased upon the start of each round so that only the final round's results are left at end)
        global df_mp = DataFrame(Mutation = String[], MP_Tot_Mut_ct = Float64[], EPCI_Mut_ct = Int[], Adjusted_MP_seq_ct = Int[], EPCI_pct = Float64[], MP_pct = Float64[], non_MP_pct = Float64[], log10pvFISH = Float64[], Fold_Incr = Union{Float64,String}[], Chi2 = Float64[], EPCI_HQCS_Ratio = Float64[], avg_NonRBD_AAct = Float64[], avg_nonRBD_rel = Float64[])
        global df_mp_META
        
        global df_mp_META_seedmut_set_dict

        
        starter = [seed_mut]
        mut_pattern_name = seed_mut
############################################################################################################################################################################
############################################################################################################################################################################ 
############################################################################################################################################################################ 
        df_props_dict_pt2__keyRoundCt, cumulative_seq_round_set_pt2_sort_join = AA_2plus__2026_05_02_subs_only_FAKE(min_match_ct,PMR1, PMR2, purespikebanned_0__purespikeallowed_1, nonmulti0__multi1, notrandom0__random1, nonspike_seqfactor__0off__1RBM__2RBD__3SpikeNonRBD__4AllSpike, plus_minus, mut_pattern_name, ind_or_grp, min_grp_fish, min_log_pv_fish, mp_masked_muts, starter, seq_factor_OFF_HALF_ON__0_1_2)
############################################################################################################################################################################
############################################################################################################################################################################ 
############################################################################################################################################################################ 



        
        all_mp_seqs_sorted = split(cumulative_seq_round_set_pt2_sort_join, ", ")
        cumulative_seq_round_set_pt2 = Set(all_mp_seqs_sorted)
        df_mp_META_seedmut_set_dict[seed_mut] = cumulative_seq_round_set_pt2
        if "" in cumulative_seq_round_set_pt2
            delete!(cumulative_seq_round_set_pt2, "")
        end
        total_MP_sequence_ct = length(all_mp_seqs_sorted)
        all_mp_muts_collected_set = Set{String}()
        for region_num in keys(df_props_dict_pt2__keyRoundCt)
            df_props_dict_pt2__keyRoundCt__keyRegion__keyMutsValueProps = df_props_dict_pt2__keyRoundCt[region_num]
            all_mp_muts_collected_set = Set(collect(keys(df_props_dict_pt2__keyRoundCt__keyRegion__keyMutsValueProps)))
        end
    end
###########################################################################################################################################################################
## Julia's DataFrame sorting syntax: sort(dataframe_to_be_sorted, column_to_sort, by = sort_function)
############################################################################################################################################################################
min_log_pv_fish_int = Int(min_log_pv_fish)
min_grp_fish_int = Int(min_grp_fish)
    CSV.write("$(rand_folder)/RUN$(fake_dict_num_ct)_df_FAKE_MP_META__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)__minGrpFish$(min_grp_fish_int)_minFish$(min_log_pv_fish_int)_seqfac$(seqfac)_$(date_hour).csv", df_mp_META)
    CSV.write("$(tsv_folder)/RUN$(fake_dict_num_ct)_df_FAKE_MP_META__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)__minGrpFish$(min_grp_fish_int)_minFish$(min_log_pv_fish_int)_seqfac$(seqfac)_$(date_hour).tsv", df_mp_META, delim='\t')
min_log_pv_fish = 5.0
min_grp_fish = 2.0


    
############################################################################################################################################################################    
    nowtime2 = Dates.format(now(), "I:MM.SS_p")
    time2 = time()

    
    rd_finish_time = round(digits=1, time2 - time1)
    print("Run #$(fake_dict_num_ct) Finish Time = $(nowtime2)  |  ")
    println("Time to Finish Round = $(rd_finish_time) seconds")
end
#############################################################################################################################################################################

############ Insert merge groups code here when finished.

################### Merging Related mp groups #################################
date_now = Dates.format(now(), "yyyy_mm_dd__HMMp"); println(date_now)
date_hour = Dates.format(now(), "yyyy_mm_dd_Ip")
date = Dates.format(today(), "yyyy_mm_dd")
####################################################################################################################
df_final_mp = DataFrame(mpNum = Int[], Region = Int[], MutNum = Int[], Mutation = String[])
#                    mp#    region#     mpmut#    mut
df_final_dict = Dict{Int, Dict{Int, Dict{Int,String}}}()
####################################################################################################################
df_mp_META_set_set_sort = sort(collect(df_mp_META_set_set), by = x -> length(x), rev=true)
df_mp_META_set_grp_dict = Dict{Int, Set{String}}()
mp_set_ct = 0
for mp_set in df_mp_META_set_set_sort
    mp_set_ct += 1
    df_mp_META_set_grp_dict[mp_set_ct] = mp_set
end
#                       #shared_muts     grp1,grp2   proportion of smaller in larger
shared_total_pairs = Dict{Int, Set{Tuple{Int,Int,Float64}}}()
#               proportion_small_in_big  grp1,grp2,#shared_muts  
shared_prop_pairs = Dict{Float64, Set{Tuple{Int,Int,Int}}}()
#                         grp1     grp2  #sharedmuts  prop_shared
mp_set_shared_muts = Dict{Int, Dict{Int, Tuple{Int,Float64}}}()
############################################################################################################################################################################
# Part below documents the number of shared mutations and the proportion of mutations shared, which is (# shared muts)/(muts in smaller of 2 mut grps)
############################################################################################################################################################################
for i in 1:length(df_mp_META_set_grp_dict)
    mp_set_shared_muts[i] = Dict{Int,Int}()
    grp1 = df_mp_META_set_grp_dict[i]
    for j in 1:length(df_mp_META_set_grp_dict)
        grp2 = df_mp_META_set_grp_dict[j]
        len1 = length(grp1)
        len2 = length(grp2)
        big = Set{String}()
        small = Set{String}()
        if len1 ≥ len2
            big = grp1
            small = grp2
        else
            big = grp2
            small = grp1
        end
        small_len = length(small)
        shared_muts = intersect(grp1, grp2)
        overlap = length(shared_muts)
        if overlap ≠ 0
            prop_shared = round(digits=3, overlap/small_len)
            mp_set_shared_muts[i][j] = (overlap, prop_shared)
            get!(shared_total_pairs, overlap, Set{Tuple{Int,Int,Float64}}() )
            push!(shared_total_pairs[overlap], (i, j, prop_shared) )
            get!(shared_prop_pairs, prop_shared, Set{Tuple{Int,Int,Int}}() )
            push!(shared_prop_pairs[prop_shared], (i, j, overlap))
        end
    end
end
############################################################################################################################################################################
############################################################################################################################################################################
##  After the initial formation of a meta_group, all other grps are checked to see if they belong to the group (this prevents duplication)
function check_against_meta_grps(grp::Set{String}, meta_grp::Set{String}, min_prop::Float64)
    grp_len = length(grp)
    meta_len = length(meta_grp)
    big = Set{String}()
    small = Set{String}()
    if meta_len ≥ grp_len
        big = meta_grp
        small = grp
    else
        big = grp
        small = meta_grp
    end
    big_len = length(big)
    small_len = length(small)
    shared_muts = intersect(grp, meta_grp)
    tot_shared = length(shared_muts)
    prop = tot_shared/small_len
    if prop ≥ min_prop && tot_shared ≥ 2
        return shared_muts, tot_shared, prop
    else
        return nothing, nothing, nothing
    end
end       
############################################################################################################################################################################
############################################################################################################################################################################
min_prop = 0.5
df_mp_META_groups = Dict{Int, Set{String}}()
df_mp_META_group_mut_cts = Dict{Int, Dict{String,Int}}()
used_grp_numbers = Set{Int}()
for (grp1, grpdict) in mp_set_shared_muts
    if !(grp1 in used_grp_numbers)
        tot_grps = length(df_mp_META_groups)
        grp1_mutset = df_mp_META_set_grp_dict[grp1]
        for (grp2, overlap__propshare) in grpdict
            if !(grp2 in used_grp_numbers)
                grp2_mutset = df_mp_META_set_grp_dict[grp2]
                if overlap__propshare[1] ≥ 2 && overlap__propshare[2] ≥ min_prop
                    df_mp_META_groups[tot_grps + 1] = Set{String}()
                    df_mp_META_group_mut_cts[tot_grps + 1] = Dict{String,Int}()
#                    df_mp_META_group_mut_cts[tot_grps + 1] = get(df_mp_META_group_mut_cts, tot_grps + 1, Dict{String,Int}() )
#                    df_mp_META_groups[tot_grps + 1] = get(df_mp_META_groups, tot_grps+1, Set{String}() )
#                    get!(df_mp_META_groups, tot_grps + 1, Set{String}() )
                    for mut in grp1_mutset
                        push!(df_mp_META_groups[tot_grps + 1], mut)
                        df_mp_META_group_mut_cts[tot_grps + 1][mut] = get(df_mp_META_group_mut_cts[tot_grps + 1], mut, 0) + 1
                    end
                    for mut in grp2_mutset
                        push!(df_mp_META_groups[tot_grps + 1], mut)
                        df_mp_META_group_mut_cts[tot_grps + 1][mut] = get(df_mp_META_group_mut_cts[tot_grps + 1], mut, 0) + 1
                    end
                    push!(used_grp_numbers, grp1)
                    push!(used_grp_numbers, grp2)
                    for grp3 in 1:length(df_mp_META_set_grp_dict)
                        grp3_mutset = df_mp_META_set_grp_dict[grp3]
                        if grp3 ≠ grp1 && grp3 ≠ grp2 && !(grp3 in used_grp_numbers)
                            shared_muts, tot_shared, prop = check_against_meta_grps(grp3_mutset, df_mp_META_groups[tot_grps + 1], min_prop)
                            if shared_muts ≠ nothing
                                if tot_shared ≥ 2 && prop ≥ min_prop
                                    for mut in grp3_mutset
                                        push!(df_mp_META_groups[tot_grps + 1], mut)
                                        df_mp_META_group_mut_cts[tot_grps + 1][mut] = get(df_mp_META_group_mut_cts[tot_grps + 1], mut, 0) + 1
                                    end
                                    push!(used_grp_numbers, grp3)
                                end
                            end
                        end
                    end
                end
            end
        end
    end
end
print("\n"^4)
#############################################################################################################################################################################
println("################### All non-merged groups after part 1 ###################")
for i in 1:length(df_mp_META_set_grp_dict)
    if !(i in used_grp_numbers)
        println("nonused grp_number = $(i)")
        nonmerge_grp_mutset = df_mp_META_set_grp_dict[i]
        nonmerge_grp_mutset_sort = sort(collect(nonmerge_grp_mutset), by = x -> mp_AA_gene_sortKey_2_universal(x, sub_0__posonly_1))
        nonmerge_grp_mutset_join = join(nonmerge_grp_mutset_sort, ", ")
        println("Group #$(i) = $(nonmerge_grp_mutset_join)")
    end
end
print("\n"^1)
#############################################################################################################################################################################
println("#################################### Groups After One Run ######################################")
println("Total Groups = $(length(df_mp_META_groups))")
for i in 1:length(df_mp_META_groups)
    mut_set = df_mp_META_groups[i]
    mut_set_sort = sort(collect(mut_set), by = x -> mp_AA_gene_sortKey_2_universal(x, sub_0__posonly_1))
    mut_set_sort_join = join(mut_set_sort, ", ")
    println("Group #$(i) = $(mut_set_sort_join)")
end
print("\n"^4)
#############################################################################################################################################################################
min_prop = 0.5
df_mp_META_groups_2 = Dict{Int, Set{String}}()
used_grps_2 = Set{Int}()
confounders = Set(["ORF1a:K1795Q", "ORF1a:E102K", "ORF1a:1795", "ORF1a:102"])
for i in 1:length(df_mp_META_groups)
    if !(i in used_grps_2) && !((length(intersect(confounders, df_mp_META_groups[i]))) ≥ 2)
        num_of_grps = length(df_mp_META_groups_2)
        grp1_mutset = df_mp_META_groups[i]
        for j in 1:length(df_mp_META_groups)
            if !(j in used_grps_2) && !(length(intersect(confounders, df_mp_META_groups[j])) ≥ 2)
                grp2_mutset = df_mp_META_groups[j]
                if i ≠ j
                    overlap_muts, overlap, prop = check_against_meta_grps(grp1_mutset, grp2_mutset, min_prop)
                    if overlap_muts ≠ nothing
                        if prop ≥ min_prop && overlap ≥ 2
                            df_mp_META_groups_2[num_of_grps+1] = Set{String}()
                            for mut in grp1_mutset
                                push!(df_mp_META_groups_2[num_of_grps+1], mut)
                            end
                            for mut in grp2_mutset
                                push!(df_mp_META_groups_2[num_of_grps+1], mut)
                            end
                            push!(used_grps_2, i)
                            push!(used_grps_2, j)
                        end
                    end
                end
            end
        end
    end
end
df_mp_META_set_vec_new = Set{Vector{String}}()
for i in 1:length(df_mp_META_groups_2)
    new_grp = df_mp_META_groups_2[i]
    new_grp_sort = sort(collect(new_grp), by = x -> mp_AA_gene_sortKey_2_universal(x, sub_0__posonly_1))
    push!(df_mp_META_set_vec_new, new_grp_sort)
end
for i in 1:length(df_mp_META_groups)
    if !(i in used_grps_2)
        new_grp = df_mp_META_groups[i]
        new_grp_sort = sort(collect(new_grp), by = x -> mp_AA_gene_sortKey_2_universal(x, sub_0__posonly_1))
        push!(df_mp_META_set_vec_new, new_grp_sort)
    end
end
for i in 1:length(df_mp_META_set_grp_dict)
    if !(i in used_grp_numbers) && !(i in used_grps_2)
        new_grp = df_mp_META_groups[i]
        new_grp_sort = sort(collect(new_grp), by = x -> mp_AA_gene_sortKey_2_universal(x, sub_0__posonly_1))
        push!(df_mp_META_set_vec_new, new_grp_sort)
    end
end
df_mp_META_set_vec_new_sort = sort(collect(df_mp_META_set_vec_new), by = x -> length(x), rev = true)
mp_ct = 0
gene_dict = Dict{Int, Dict{Int,String}}()
pos_dict = Dict{Int, Dict{Int,Int}}()
for i in 1:length(df_mp_META_set_vec_new_sort)
    df_final_dict[i] = Dict{Int, Dict{Int,String}}()
    gene_dict[i] = Dict{Int,String}()
    pos_dict[i] = Dict{Int,Int}()
    mp_ct += 1
    mut_vec = df_mp_META_set_vec_new_sort[i]
    region_ct = 1
#    for j in 1:length(mut_vec)
#        df_final_dict[i][j] = Dict{Int,String}()
#    end
    for j in 1:length(mut_vec)
        mp_mut_ct = j
        mut = mut_vec[j]
        mut_gene = aa_gene_comprehensive_dict[mut]
        gene_dict[i][j] = mut_gene
        mootpos = aa_pos_comprehensive_dict[mut]
        pos_dict[i][j] = mootpos
        if j == 1
            df_final_dict[i][region_ct] = get(df_final_dict[i], region_ct, Dict{Int,String}())
            df_final_dict[i][region_ct][mp_mut_ct] = mut
            push!(df_final_mp, (i, region_ct, mp_mut_ct, mut))
        else
            if mut_gene == gene_dict[i][j-1] && abs(mootpos - pos_dict[i][j-1]) ≤ 5
                df_final_dict[i][region_ct] = get(df_final_dict[i], region_ct, Dict{Int,String}())
                df_final_dict[i][region_ct][mp_mut_ct] = mut
                push!(df_final_mp, (i, region_ct, mp_mut_ct, mut))
            else
                region_ct += 1
                df_final_dict[i][region_ct] = Dict{Int,String}()
                df_final_dict[i][region_ct][mp_mut_ct] = mut
                push!(df_final_mp, (i, region_ct, mp_mut_ct, mut))
            end
        end 
    end
end
min_log_pv_fish_int = Int(min_log_pv_fish)
min_grp_fish_int = Int(min_grp_fish)
CSV.write("$(rand_folder)_final_merged_patterns_$(date_hour)__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)__1fish$(min_log_pv_fish_int)_2fish$(min_grp_fish_int).csv", df_final_mp)
println("############################# Final post-merge #2 groups #############################")
fake_merged_mp_grp_ct = 0
for fake_merged_mp_grp_vec in df_mp_META_set_vec_new
    fake_merged_mp_grp_ct += 1
    fake_merged_mp_grp_vec_join = join(fake_merged_mp_grp_vec, ", ")
    println("#$(fake_merged_mp_grp_ct): $(fake_merged_mp_grp_vec_join)")
end; print("\n"^1)
############################################################################################################################################################################
#    CSV.write("$(rand_folder)/final_merged_patterns_$(date_hour)__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)__1fish$(min_log_pv_fish_int)_2fish$(min_grp_fish_int).csv", df_final_mp)
#    CSV.write("$(tsv_folder)/final_merged_patterns_$(date_hour)__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)__1fish$(min_log_pv_fish_int)_2fish$(min_grp_fish_int).tsv", df_final_mp, delim='\t')
############################################################################################################################################################################
############################################################################################################################################################################
nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime)
RUNS = length(mp_meta_fake_chr_dict_DQ)
############################################################################################################################################################################
CSV.write("$(rand_folder)/FINAL_df_fake_MP_META_META_$(RUNS)RUNS__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)___minGrpFish$(min_grp_fish_int)_minFish$(min_log_pv_fish_int)_seqfac$(seqfac)_$(date_hour).csv", df_mp_META_META)
#CSV.write("$(tsv_folder)/FINAL_df_fake_MP_META_META_$(RUNS)RUNS__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)___minGrpFish$(min_grp_fish_int)_minFish$(min_log_pv_fish_int)_seqfac$(seqfac)_$(date_hour).tsv", df_mp_META_META, delim='\t')
############################################################################################################################################################################
CSV.write("$(rand_folder)/META_final_merged_patterns_$(date_hour)__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)___1fish$(min_log_pv_fish_int)_2fish$(min_grp_fish_int).csv", df_final_mp_META)
#CSV.write("$(tsv_folder)/META_final_merged_patterns_$(date_hour)__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_string)___1fish$(min_log_pv_fish_int)_2fish$(min_grp_fish_int).tsv", df_final_mp_META, delim='\t')
############################################################################################################################################################################
runtime = time() - start
runtime1, runtime2 = seconds_to_hrs_min_sec(runtime)
println("Runtime v2 = $(runtime2)")
open("fake_rand_correlated_muts_runtime_printout_$(date_hour).txt", "w") do g
    println(g, "Runtime v2 = $(runtime2)")
end
print("\n"^1); println("EPCI_qc_str = $(EPCI_qc_str)"); println("HQCS_qc_string = $(HQCS_qc_string)"); print("\n"^1) 
print("\n"^1); nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime); print("\n"^1)
#### Runtime = 
#### Runtime = 
#### Runtime = 2 hr, 29 min, 10.38 sec, 2026_02_21 (100 runs), muts
#### Runtime = 1 hr, 34 min, 11.52 sec, 2026_01_31 (100 runs), muts
#### Runtime = 1 hr, 41 min, 47.05 sec, 2026_01_31 (100 Runs), pos_only
#### Runtime = 3 hr, 11 min, 12.70 sec, 2025-11-04 (100 Runs)
#####################################################################################################################################
#####################################################################################################################################


2026_05_05__2358PM

11:58.09_PM

EPCI_qc_str = 15_20_95
HQCS_qc_string = 5_1_5

Total Number of Candidate Test Mutations = 3099

Run #1 Start Time = 11:58.11_PM
Booted, pt1: ORF1a:1323   |seed=ORF1a:1323  |EPCI_pct = 17.71|MP_pct = 17.44|log10pvFISH =  5.36|
Booted, pt1: ORF1a:4175   |seed=ORF1a:1323  |EPCI_pct = 17.09|MP_pct = 18.09|log10pvFISH =  5.37|
Booted, pt1: ORF1a:110    |seed=ORF6:61     |EPCI_pct = 15.38|MP_pct = 20.69|log10pvFISH =  5.14|
Booted, pt1: ORF1a:1323   |seed=ORF1a:4175  |EPCI_pct = 17.71|MP_pct = 17.44|log10pvFISH =  5.36|
Booted, pt1: ORF1a:4175   |seed=ORF1a:4175  |EPCI_pct = 17.09|MP_pct = 18.09|log10pvFISH =  5.37|
Booted, pt1: ORF1a:110    |seed=ORF1a:110   |EPCI_pct = 15.38|MP_pct = 20.69|log10pvFISH =  5.14|
Booted, pt1: N:134        |seed=N:134       |EPCI_pct =   9.8|MP_pct = 38.46|log10pvFISH =  5.17|
Booted, pt1: N:134        |seed=ORF7b:43    |EPCI_pct =   9.8|MP_pct = 38.46|log10pvFISH =  5.17|
Run #1 Finish Time = 11:59.41_PM  |  Time to Finish Ro

In [143]:
### APCI Version | Getting mp stats from df_mp to compare to fake runs, i.e. df_mp_fake
date_now = Dates.format(now(), "yyyy_mm_dd__HMMp"); println(date_now)
nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime)
################################################################################################
### Converting Excel .xlsx file back into CSV file
#Pkg.add("XLSX"); using XLSX
#df_mp_subs_CSV_from_Excel_xlsx = DataFrame(XLSX.readtable("mp_subs_2026_02_28_12PM__EPCI_8_10_95__HQCS_5_1_5__minFsh5_grpFsh_2_RANDOM/df_rand_MP_META_sorted__EPCI_8_10_95__HQCS_5_1_5___plusminus5_minGrpFish2_minFish5_seqfacON_2026_02_28_12PM___repeats_removed.xlsx", "df_rand_MP_META_sorted__EPCI_8_"))
#CSV.write("df_rand_MP_META_sorted__EPCI_8_10_95__HQCS_5_1_5___plusminus5_minGrpFish2_minFish5_seqfacON_2026_02_28_12PM___repeats_removed.csv", df_mp_subs_CSV_from_Excel_xlsx)
#############################################################################################################################################################################
#############################################################################################################################################################################
#############################################################################################################################################################################
folder = "mp_pos_only_FAKE_2026_05_05_23PM__EPCI_15_20_95__HQCS_5_1_5__minFsh5_grpFsh_2_RANDOM"
filename = "FINAL_df_fake_MP_META_META_100RUNS__EPCI_15_20_95__HQCS_5_1_5___minGrpFish2_minFish5_seqfacON_2026_05_06_2AM"
#############################################################################################################################################################################
#############################################################################################################################################################################
#############################################################################################################################################################################
df_mp_subs_csv = CSV.read("$(folder)/$(filename).csv", DataFrame)
headers = names(df_mp_subs_csv)
ncols = ncol(df_mp_subs_csv)
nrows = nrow(df_mp_subs_csv)
################################################################################################
rows = collect(eachrow(df_mp_subs_csv))

mp_mut_total_dict = Dict{Int, Int}()

total_mp_ct = 0
total_MP_Mut_ct = 0 
total_EPCI_Mut_ct = 0
total_Adjusted_MP_seq_ct = 0
Fishers_Exact_Test_pval_sum = 0
avg_NonRBD_AAct_sum = 0
avg_nonRBD_rel_sum = 0
log10pvFISH_sum = 0
#Fold_Incr_sum = 0

MP_Tot_Different_Mut_ct_sum = 0
MP_Tot_Different_Reg_ct_sum = 0
Reg_Tot_Different_Mut_ct_sum = 0

Region_total_mut_ct = 0
Region_total_mut_ct_vec = Int[]
MP_total_mut_ct = 0
MP_total_mut_ct_vec = Int[]
#Tot_Different_Mut_ct_per_Region_vec = Float64[]

SeedMut_vec = Float64[]
MP_Tot_Different_Mut_ct_vec = Int[]
MP_Tot_Different_Reg_ct_vec = Int[]
Reg_Tot_Different_Mut_ct_vec = Int[]
MP_Region_Num_vec = Int[]
Mutation_vec = String[]
MP_Mut_ct_vec = Int[]
EPCI_Mut_ct_vec = Int[]
Adjusted_MP_seq_ct_vec = Int[]
EPCI_pct_vec = Float64[]
MP_pct_vec = Float64[]
non_MP_pct_vec = Float64[]
Fishers_Exact_Test_pval_vec = Float64[]
log10pvFISH_vec = Float64[]
#Fold_Incr_vec = Union{Float64,String}[]
#Chi2_vec = Float64[]
EPCI_HQCS_Ratio_vec = Float64[]
avg_NonRBD_AAct_vec = Float64[]
avg_nonRBD_rel_vec = Float64[]
#PangoDateIndex50_Avg_vec = Int[]
#PangoDateStr50_Avg_vec = String[]
#CollDateIndex_Avg_vec = Int[]
#CollDateString_Avg_vec = String[]
for i in 1:nrows
    row = rows[i]
#################################
    Run = row[1]
    SeedMut = row[2]
    MP_Tot_Different_Mut_ct = row[3]
    MP_Tot_Different_Reg_ct = row[4]
    Reg_Tot_Different_Mut_ct = row[5]    
    MP_Region_Num = row[6]
    Mutation = row[7]
    MP_Mut_ct = row[8]
    EPCI_Mut_ct = row[9]
    Adjusted_MP_seq_ct = row[10]
    EPCI_pct = row[11]
    MP_pct = row[12]
    non_MP_pct = row[13]
    Fishers_Exact_Test_pval = row[14]
    log10pvFISH = row[15]
#    Fold_Incr = row[16]
#    if !(Fold_Incr isa Float64)
#        Fold_Incr = parse(Float64, Fold_Incr[2:end])/2
#    end
    Chi2 = row[17]
    EPCI_HQCS_Ratio = row[18]
    avg_NonRBD_AAct = row[19]
    avg_nonRBD_rel = row[20]
#   PangoDateIndex50_Avg = row[21]
#   PangoDateStr50_Avg = row[22]
#   CollDateIndex_Avg = row[23]
#   CollDateString_Avg = row[24]
#################################
    if !(MP_Mut_ct isa Int) && !(MP_Mut_ct isa Float64)
        println(MP_Mut_ct)
    end
    total_MP_Mut_ct += MP_Mut_ct
    Region_total_mut_ct += MP_Mut_ct
    MP_total_mut_ct += MP_Mut_ct
    total_EPCI_Mut_ct += EPCI_Mut_ct
    total_Adjusted_MP_seq_ct += Adjusted_MP_seq_ct
    Fishers_Exact_Test_pval_sum += Fishers_Exact_Test_pval
    log10pvFISH_sum += log10pvFISH
#    Fold_Incr_sum += Fold_Incr
    push!(MP_Mut_ct_vec, MP_Mut_ct)
    push!(EPCI_Mut_ct_vec, EPCI_Mut_ct)
    push!(Adjusted_MP_seq_ct_vec, Adjusted_MP_seq_ct)
    push!(Fishers_Exact_Test_pval_vec, Fishers_Exact_Test_pval)
    push!(log10pvFISH_vec, log10pvFISH)
#    push!(Fold_Incr_vec, Fold_Incr)
    push!(EPCI_HQCS_Ratio_vec, EPCI_HQCS_Ratio)
    push!(avg_NonRBD_AAct_vec, avg_NonRBD_AAct)
    push!(avg_nonRBD_rel_vec, avg_nonRBD_rel)
    if i == nrows
        total_mp_ct += 1
        MP_Tot_Different_Reg_ct_sum += MP_Tot_Different_Reg_ct
        MP_Tot_Different_Mut_ct_sum += MP_Tot_Different_Mut_ct
        Reg_Tot_Different_Mut_ct_sum += Reg_Tot_Different_Mut_ct
        
        avg_NonRBD_AAct_sum += avg_NonRBD_AAct
        avg_nonRBD_rel_sum += avg_nonRBD_rel
        push!(MP_Tot_Different_Mut_ct_vec, MP_Tot_Different_Mut_ct)
        push!(MP_Tot_Different_Reg_ct_vec, MP_Tot_Different_Reg_ct)
        push!(Reg_Tot_Different_Mut_ct_vec, Reg_Tot_Different_Mut_ct)
#        push!(Tot_Different_Mut_ct_per_Region_vec, /MP_Tot_Different_Reg_ct)      
    elseif i ≠ 1
        prev_row = rows[i-1]
        next_row = rows[i+1]
#################################
        prev_Run = prev_row[1]
        prev_SeedMut = prev_row[2]
        prev_MP_Tot_Different_Mut_ct = prev_row[3]
        prev_MP_Tot_Different_Reg_ct = prev_row[4]
        prev_Reg_Tot_Different_Mut_ct = prev_row[5]
        prev_MP_Region_Num = prev_row[6]
        if prev_MP_Region_Num ≠ MP_Region_Num
            push!(Reg_Tot_Different_Mut_ct_vec, prev_Reg_Tot_Different_Mut_ct)
            Reg_Tot_Different_Mut_ct_sum += prev_Reg_Tot_Different_Mut_ct
        end
#        prev_Mutation = prev_row[7]
#        prev_MP_Mut_ct = prev_row[8]
#        prev_EPCI_Mut_ct = prev_row[9]
#        prev_Adjusted_MP_seq_ct = prev_row[10]
#        prev_EPCI_pct = prev_row[11]
#        prev_MP_pct = prev_row[12]
#        prev_non_MP_pct = prev_row[13]
#        prev_Fishers_Exact_Test_pval = prev_row[14]
#        prev_log10pvFISH = prev_row[15]
#        prev_Fold_Incr = prev_row[16]
#        if prev_Fold_Incr isa String
#            prev_Fold_Incr = parse(Float64, prev_Fold_Incr[2:end])/2
#        end
#        prev_Chi2 = prev_row[17]
#        prev_EPCI_HQCS_Ratio = prev_row[18]
        prev_avg_NonRBD_AAct = prev_row[19]
        prev_avg_nonRBD_rel = prev_row[20]
#       prev_PangoDateIndex50_Avg = prev_row[21]
#       prev_PangoDateStr50_Avg = prev_row[22]
#       prev_CollDateIndex_Avg = prev_row[23]
#       prev_CollDateString_Avg = prev_row[24]  
        if prev_SeedMut ≠ SeedMut
            total_mp_ct += 1
            MP_Tot_Different_Reg_ct_sum += prev_MP_Tot_Different_Reg_ct
            MP_Tot_Different_Mut_ct_sum += prev_MP_Tot_Different_Mut_ct
            avg_NonRBD_AAct_sum += prev_avg_NonRBD_AAct
            avg_nonRBD_rel_sum += prev_avg_nonRBD_rel
            push!(MP_Tot_Different_Mut_ct_vec, prev_MP_Tot_Different_Mut_ct)
            push!(MP_Tot_Different_Reg_ct_vec, prev_MP_Tot_Different_Reg_ct)
            push!(MP_total_mut_ct_vec, MP_total_mut_ct)
            MP_total_mut_ct = 0
        end
        next_Run = next_row[1]
        next_SeedMut = next_row[2]
        next_MP_Tot_Different_Mut_ct = next_row[3]
        next_MP_Tot_Different_Reg_ct = next_row[4]
        next_Reg_Tot_Different_Mut_ct = next_row[5]
        next_MP_Region_Num = next_row[6]
        if MP_Region_Num ≠ next_MP_Region_Num
            push!(Region_total_mut_ct_vec, Region_total_mut_ct)
            Region_total_mut_ct = 0
        end
    end
end
########################################################################################################################################################
########################################################################################################################################################
average_different_muts_per_mp = MP_Tot_Different_Mut_ct_sum/total_mp_ct
average_different_regions_per_mp = MP_Tot_Different_Reg_ct_sum/total_mp_ct
average_different_muts_per_region = MP_Tot_Different_Mut_ct_sum/MP_Tot_Different_Reg_ct_sum
average_total_MP_mut_count_per_region = total_MP_Mut_ct/MP_Tot_Different_Reg_ct_sum
average_total_MP_mut_count_per_mp = total_MP_Mut_ct/total_mp_ct
########################################################
average_different_muts_per_mp_rd = round(digits=3, average_different_muts_per_mp)
average_different_regions_per_mp_rd = round(digits=3, average_different_regions_per_mp)
average_different_muts_per_region_rd = round(digits=3, average_different_muts_per_region)
average_total_MP_mut_count_per_region_rd = round(digits=3, average_total_MP_mut_count_per_region)
average_total_MP_mut_count_per_mp_rd = round(digits=3, average_total_MP_mut_count_per_mp)
########################################################################################################################################################
average_MP_mut_ct_per_MPmut = total_MP_Mut_ct/nrows
average_EPCI_mut_ct_per_MPmut = total_EPCI_Mut_ct/nrows
average_Adjusted_MP_seq_ct = total_Adjusted_MP_seq_ct/nrows
average_avg_NonRBD_AAct = avg_NonRBD_AAct_sum/total_mp_ct
average_avg_nonRBD_rel = avg_nonRBD_rel_sum/total_mp_ct
average_Fishers_Exact_Test_pval = Fishers_Exact_Test_pval_sum/nrows  ## This is a true "normal" average
average_log10pvFISH = log10pvFISH_sum/nrows                           ## This is a geometric average
#average_Fold_Incr = Fold_Incr_sum/nrows
########################################################
average_MP_mut_ct_per_MPmut_rd = round(digits=2, average_MP_mut_ct_per_MPmut)
average_EPCI_mut_ct_per_MPmut_rd = round(digits=2, average_EPCI_mut_ct_per_MPmut)
average_Adjusted_MP_seq_ct_rd = round(digits=2, average_Adjusted_MP_seq_ct)
average_avg_NonRBD_AAct_rd = round(digits=2, average_avg_NonRBD_AAct)
average_avg_nonRBD_rel_rd = round(digits=3, average_avg_nonRBD_rel)
average_Fishers_Exact_Test_pval_rd = round(digits=12, average_Fishers_Exact_Test_pval)
average_log10pvFISH_rd = round(digits=2, average_log10pvFISH)
#average_Fold_Incr_rd = round(digits=2, average_Fold_Incr)
########################################################################################################################################################
########################################################################################################################################################
median_different_muts_per_mp = median(MP_Tot_Different_Mut_ct_vec)
median_different_regions_per_mp = median(MP_Tot_Different_Reg_ct_vec)
median_different_muts_per_region = median(Reg_Tot_Different_Mut_ct_vec)
median_total_MP_mut_count_per_region = median(Region_total_mut_ct_vec)
median_total_MP_mut_count_per_mp = median(MP_total_mut_ct_vec)
########################################################
median_different_muts_per_mp_rd = round(digits=2, median_different_muts_per_mp)
median_different_regions_per_mp_rd = round(digits=2, median_different_regions_per_mp)
median_different_muts_per_region_rd = round(digits=2, median_different_muts_per_region)
median_total_MP_mut_count_per_region_rd = round(digits=2, median_total_MP_mut_count_per_region)
median_total_MP_mut_count_per_mp_rd = round(digits=2, median_total_MP_mut_count_per_mp)
########################################################################################################################################################
median_MP_mut_ct_per_MPmut = median(MP_Mut_ct_vec)
median_EPCI_mut_ct_per_MPmut = median(EPCI_Mut_ct_vec)
median_Adjusted_MP_seq_ct = median(Adjusted_MP_seq_ct_vec)
median_avg_NonRBD_AAct = median(avg_NonRBD_AAct_vec)
median_avg_nonRBD_rel = median(avg_nonRBD_rel_vec)
median_Fishers_Exact_Test_pval = median(Fishers_Exact_Test_pval_vec)
median_log10pvFISH = median(log10pvFISH_vec)
median_EPCI_HQCS_Ratio = median(EPCI_HQCS_Ratio_vec)
#median_Fold_Incr = median(Fold_Incr_vec)
########################################################
median_MP_mut_ct_per_MPmut_rd = round(digits=2, median_MP_mut_ct_per_MPmut)
median_EPCI_mut_ct_per_MPmut_rd = round(digits=2, median_EPCI_mut_ct_per_MPmut)
median_Adjusted_MP_seq_ct_rd = round(digits=2, median_Adjusted_MP_seq_ct)
median_avg_NonRBD_AAct_rd = round(digits=2, median_avg_NonRBD_AAct)
median_avg_nonRBD_rel_rd = round(digits=3, median_avg_nonRBD_rel)
median_Fishers_Exact_Test_pval_rd = round(digits=3, median_Fishers_Exact_Test_pval)
median_log10pvFISH_rd = round(digits=2, median_log10pvFISH)
median_EPCI_HQCS_Ratio_rd = round(digits=2, median_EPCI_HQCS_Ratio)
#median_Fold_Incr_rd = round(digits=2, median_Fold_Incr)
########################################################################################################################################################
########################################################################################################################################################
print("\n"^2)
println("median_different_muts_per_mp_rd       = $(median_different_muts_per_mp_rd)")
println("median_different_regions_per_mp_rd    = $(median_different_regions_per_mp_rd)")
println("median_different_muts_per_region      = $(median_different_muts_per_region)")
println("median_total_MP_mut_count_per_region  = $(median_total_MP_mut_count_per_region)")
println("median_total_MP_mut_count_per_mp      = $(median_total_MP_mut_count_per_mp)")
###########################
println("median_MP_mut_ct_per_MPmut_rd         = $(median_MP_mut_ct_per_MPmut_rd)")
println("median_EPCI_mut_ct_per_MPmut_rd       = $(median_EPCI_mut_ct_per_MPmut_rd)")
println("median_Adjusted_MP_seq_ct_rd          = $(median_Adjusted_MP_seq_ct_rd)")
println("median_avg_NonRBD_AAct_rd             = $(median_avg_NonRBD_AAct_rd)")
println("median_avg_nonRBD_rel_rd              = $(median_avg_nonRBD_rel_rd)")
println("median_Fishers_Exact_Test_pval_rd     = $(median_Fishers_Exact_Test_pval_rd)")
println("median_log10pvFISH_rd                 = $(median_log10pvFISH_rd)")
println("median_EPCI_HQCS_Ratio_rd             = $(median_EPCI_HQCS_Ratio_rd)")
#println("median_Fold_Incr_rd                  = $(median_Fold_Incr_rd)")
########################################################
print("\n"^4)
println("average_different_muts_per_mp_rd      = $(average_different_muts_per_mp_rd)")
println("average_different_regions_per_mp_rd   = $(average_different_regions_per_mp_rd)")
println("average_different_muts_per_region_rd  = $(average_different_muts_per_region_rd)")
println("average_total_MP_mut_count_per_region = $(average_total_MP_mut_count_per_region_rd)")
println("average_total_MP_mut_count_per_mp     = $(average_total_MP_mut_count_per_mp_rd)")
###########################
println("average_MP_mut_ct_per_MPmut_rd        = $(average_MP_mut_ct_per_MPmut_rd)")
println("average_EPCI_mut_ct_per_MPmut_rd      = $(average_EPCI_mut_ct_per_MPmut_rd)")
println("average_Adjusted_MP_seq_ct_rd         = $(average_Adjusted_MP_seq_ct_rd)")
println("average_avg_NonRBD_AAct_rd            = $(average_avg_NonRBD_AAct_rd)")
println("average_avg_nonRBD_rel_rd             = $(average_avg_nonRBD_rel_rd)")
println("average_Fishers_Exact_Test_pval_rd    = $(average_Fishers_Exact_Test_pval_rd)")
println("average_log10pvFISH_rd                = $(average_log10pvFISH_rd)")
#println("average_Fold_Incr_rd                 = $(average_Fold_Incr_rd)")
print("\n"^4)
########################################################################################################################################################
########################################################################################################################################################
df_MP_stats_csv = DataFrame(
    "Median or Mean" => String[],
    "Different Muts per MP" => Float64[],
    "Different Regions per MP" => Float64[],
    "Different Muts per Region" => Float64[],
    "MP Mut ct per MP Mut" => Float64[],
    "Total MP Mut Count per Region" => Float64[],
    "Total MP Mut Count per MP" => Float64[],
    "Avg non-RBD AAct" => Float64[],
    "Avg non-RBD Rel" => Float64[],
    "EPCI/HQCS Ratio" => Float64[])

    
########################################################
median_different_muts_per_mp = median(MP_Tot_Different_Mut_ct_vec)
median_different_regions_per_mp = median(MP_Tot_Different_Reg_ct_vec)
median_different_muts_per_region = median(Reg_Tot_Different_Mut_ct_vec)
median_total_MP_mut_count_per_region = median(Region_total_mut_ct_vec)
median_total_MP_mut_count_per_mp = median(MP_total_mut_ct_vec)
########################################################
median_MP_mut_ct_per_MPmut = median(MP_Mut_ct_vec)
median_EPCI_mut_ct_per_MPmut = median(EPCI_Mut_ct_vec)
median_Adjusted_MP_seq_ct = median(Adjusted_MP_seq_ct_vec)
median_avg_NonRBD_AAct = median(avg_NonRBD_AAct_vec)
median_avg_nonRBD_rel = median(avg_nonRBD_rel_vec)
median_Fishers_Exact_Test_pval = median(Fishers_Exact_Test_pval_vec)
median_log10pvFISH = median(log10pvFISH_vec)
median_EPCI_HQCS_Ratio = median(EPCI_HQCS_Ratio_vec)
########################################################
    
 push!(df_MP_stats_csv, ("Median",  median_different_muts_per_mp,  median_different_regions_per_mp,  median_different_muts_per_region, median_MP_mut_ct_per_MPmut,   median_total_MP_mut_count_per_region,  median_total_MP_mut_count_per_mp,  median_avg_NonRBD_AAct,  median_avg_nonRBD_rel,  median_EPCI_HQCS_Ratio))         
push!(df_MP_stats_csv, ("Average", average_different_muts_per_mp, average_different_regions_per_mp, average_different_muts_per_region, average_MP_mut_ct_per_MPmut, average_total_MP_mut_count_per_region, average_total_MP_mut_count_per_mp, average_avg_NonRBD_AAct, average_avg_nonRBD_rel, 0.0))
########################################################################################################################################################
########################################################################################################################################################
df_MP_stats_csv_flip = DataFrame(
    "Statistic" => String[],
    "Median" => Float64[],
    "Average" => Float64[])
push!(df_MP_stats_csv_flip, ("Different Muts per MP", median_different_muts_per_mp, average_different_muts_per_mp))
push!(df_MP_stats_csv_flip, ("Different Regions per MP", median_different_regions_per_mp, average_different_regions_per_mp))
push!(df_MP_stats_csv_flip, ("Different Muts per_Region", median_different_muts_per_region, average_different_muts_per_region))
push!(df_MP_stats_csv_flip, ("MP Mut ct per MP Mut", median_MP_mut_ct_per_MPmut, average_MP_mut_ct_per_MPmut))
push!(df_MP_stats_csv_flip, ("Total MP Mut Count per Region", median_total_MP_mut_count_per_region, average_total_MP_mut_count_per_region))
push!(df_MP_stats_csv_flip, ("Total MP Mut Count per MP", median_total_MP_mut_count_per_mp, average_total_MP_mut_count_per_mp))
push!(df_MP_stats_csv_flip, ("Avg non-RBD AAct", median_avg_NonRBD_AAct, average_avg_NonRBD_AAct))
push!(df_MP_stats_csv_flip, ("Avg non-RBD Rel", median_avg_nonRBD_rel, average_avg_nonRBD_rel))
push!(df_MP_stats_csv_flip, ("EPCI/HQCS Ratio", median_EPCI_HQCS_Ratio, 0.0))
#push!(df_MP_stats_csv_flip, ("Fold_Incr", median_Fold_Incr, average_Fold_Incr))
########################################################################################################################
########################################################################################################################
CSV.write("$(folder)/$(filename)__STATS.csv", df_MP_stats_csv)
CSV.write("$(folder)/$(filename)__STATS_flip.csv", df_MP_stats_csv_flip)
############################################################################################################################################################################
############################################################################################################################################################################

2026_05_06__648AM
6:48.20_AM


median_different_muts_per_mp_rd       = 2.0
median_different_regions_per_mp_rd    = 2.0
median_different_muts_per_region      = 1.0
median_total_MP_mut_count_per_region  = 5.0
median_total_MP_mut_count_per_mp      = 12.0
median_MP_mut_ct_per_MPmut_rd         = 5.0
median_EPCI_mut_ct_per_MPmut_rd       = 20.0
median_Adjusted_MP_seq_ct_rd          = 25.0
median_avg_NonRBD_AAct_rd             = 23.67
median_avg_nonRBD_rel_rd              = 1.325
median_Fishers_Exact_Test_pval_rd     = 0.0
median_log10pvFISH_rd                 = 5.76
median_EPCI_HQCS_Ratio_rd             = 2.97




average_different_muts_per_mp_rd      = 2.229
average_different_regions_per_mp_rd   = 2.114
average_different_muts_per_region_rd  = 1.054
average_total_MP_mut_count_per_region = 5.946
average_total_MP_mut_count_per_mp     = 12.571
average_MP_mut_ct_per_MPmut_rd        = 5.64
average_EPCI_mut_ct_per_MPmut_rd      = 45.03
average_Adjusted_MP_seq_ct_rd         = 48.04
average_avg_NonR

"mp_pos_only_FAKE_2026_05_05_23PM__EPCI_15_20_95__HQCS_5_1_5__minFsh5_grpFsh_2_RANDOM/FINAL_df_fake_MP_META_META_100RUNS__EPCI_15_20_95__HQCS_5_1_5___minGrpFish2_minFish5_seqfacON_2026_05_06_2AM__STATS_flip.csv"